In [1]:
# ==============================
# LOAD SAVED DATASET
# ==============================
import numpy as np
import torch

data = np.load("data/fertnet_data.npz")

X_train = torch.tensor(data["X_train"], dtype=torch.float32)
y_train = torch.tensor(data["y_train"], dtype=torch.float32)
X_val   = torch.tensor(data["X_val"], dtype=torch.float32)
y_val   = torch.tensor(data["y_val"], dtype=torch.float32)
X_test  = torch.tensor(data["X_test"], dtype=torch.float32)
y_test  = torch.tensor(data["y_test"], dtype=torch.float32)

print("✅ Loaded dataset: ", X_train.shape, y_train.shape)


✅ Loaded dataset:  torch.Size([292903, 23]) torch.Size([292903, 4])


In [ ]:
# ppo_fertnet_rl_weighted_hyperparam_sweep_rmse.py

import os
import random
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.distributions import Normal
from itertools import product

# ==========================================================
# REPRO & DEVICE
# ==========================================================
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# ==========================================================
# CONFIG
# ==========================================================
state_dim = 23
action_dim = 4
steps_per_update = 2048
micro_batch = 256
mini_batch = 64
max_grad_norm = 0.5
EPS = 1e-8

# ==========================================================
# HYPERPARAMETER GRID
# ==========================================================
learning_rates = [0.0001]
clip_eps_list = [0.2, 0.3]
ppo_epochs_list = [8, 12]
num_updates_list =[2000]# adjust as needed

# ==========================================================
# LOAD DATA
# ==========================================================
data = np.load("data/fertnet_data.npz")
X = torch.tensor(data[data.files[0]], dtype=torch.float32, device=device)
y = torch.tensor(data[data.files[1]], dtype=torch.float32, device=device)

perm = torch.randperm(len(X))
split = int(0.8 * len(X))
X_train, y_train = X[perm[:split]], y[perm[:split]]
X_test, y_test = X[perm[split:]], y[perm[split:]]

# ==========================================================
# MODEL
# ==========================================================
class Backbone(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(state_dim, 256),
            nn.ReLU(),
            nn.Linear(256, 256),
            nn.ReLU()
        )

    def forward(self, x):
        return self.net(x)

class Actor(nn.Module):
    def __init__(self, backbone):
        super().__init__()
        self.backbone = backbone
        self.mu = nn.Linear(256, action_dim)
        self.log_std = nn.Parameter(torch.ones(action_dim) * -1.6)

    def forward(self, x):
        h = self.backbone(x)
        mu = torch.tanh(self.mu(h)) * 3.0
        std = torch.exp(self.log_std)
        return mu, std

class Critic(nn.Module):
    def __init__(self, backbone):
        super().__init__()
        self.backbone = backbone
        self.v = nn.Linear(256, 1)

    def forward(self, x):
        return self.v(self.backbone(x))

# ==========================================================
# REWARD & EXPLAINED VARIANCE
# ==========================================================
def weighted_reward(pred, truth):
    rmse_actions = torch.sqrt((pred - truth) ** 2 + 1e-8)
    reward = -(
        1.0 * rmse_actions[:, 0] +
        1.0 * rmse_actions[:, 1] +
        1.0 * rmse_actions[:, 2] +
    )
    reward = reward / 10.0
    return reward.unsqueeze(1)

def explained_variance(y, v):
    var_y = torch.var(y)
    if var_y < 1e-8:
        return 0.0
    return (1 - torch.var(y - v) / (var_y + 1e-8)).item()

# ==========================================================
# EXPERIMENT LOOP
# ==========================================================
results = []

for lr_actor_val, lr_critic_val, clip_eps_val, ppo_epochs_val, num_updates_val in product(
    learning_rates, learning_rates, clip_eps_list, ppo_epochs_list, num_updates_list
):

    print(f"\n=== EXPERIMENT: lr_actor={lr_actor_val}, lr_critic={lr_critic_val}, "
          f"clip_eps={clip_eps_val}, ppo_epochs={ppo_epochs_val}, num_updates={num_updates_val} ===")

    # Reinitialize models
    backbone = Backbone().to(device)
    actor = Actor(backbone).to(device)
    critic = Critic(backbone).to(device)

    opt_actor = optim.Adam(actor.parameters(), lr=lr_actor_val)
    opt_critic = optim.Adam(critic.parameters(), lr=lr_critic_val)

    # Best checkpoint tracking
    best_test_reward = -np.inf
    best_actor_path = f"best_actor_lr{lr_actor_val}_ce{clip_eps_val}_ep{ppo_epochs_val}.pt"
    best_critic_path = f"best_critic_lr{lr_critic_val}_ce{clip_eps_val}_ep{ppo_epochs_val}.pt"

    # ---------------- TRAINING LOOP ----------------
    for update in range(1, num_updates_val + 1):
        S, A, R, V, LP = [], [], [], [], []

        # ROLLOUT
        while len(S) * micro_batch < steps_per_update:
            idx = torch.randint(0, len(X_train), (micro_batch,))
            s = X_train[idx]
            t = y_train[idx]

            with torch.no_grad():
                mu, std = actor(s)
                dist = Normal(mu, std)
                a = dist.sample()
                lp = dist.log_prob(a).sum(-1, keepdim=True)
                v = critic(s)
                r = weighted_reward(a, t)

            S.append(s); A.append(a)
            R.append(r); V.append(v); LP.append(lp)

        S = torch.cat(S)
        A = torch.cat(A)
        R = torch.cat(R)
        V = torch.cat(V)
        LP = torch.cat(LP)

        adv = (R - V)
        adv = (adv - adv.mean()) / (adv.std() + EPS)

        # PPO UPDATE
        actor_loss_v = critic_loss_v = kl_v = clip_v = entropy_v = 0.0
        mb_count = 0

        for _ in range(ppo_epochs_val):
            idx = torch.randperm(len(S))
            for i in range(0, len(S), mini_batch):
                mb = idx[i:i + mini_batch]

                mu, std = actor(S[mb])
                dist = Normal(mu, std)
                new_lp = dist.log_prob(A[mb]).sum(-1, keepdim=True)

                ratio = torch.exp(new_lp - LP[mb])
                surr1 = ratio * adv[mb]
                surr2 = torch.clamp(ratio, 1 - clip_eps_val, 1 + clip_eps_val) * adv[mb]

                actor_loss = -torch.min(surr1, surr2).mean()
                value = critic(S[mb])
                critic_loss = nn.MSELoss()(value, R[mb])

                opt_actor.zero_grad()
                opt_critic.zero_grad()
                actor_loss.backward(retain_graph=True)
                critic_loss.backward()
                torch.nn.utils.clip_grad_norm_(actor.parameters(), max_grad_norm)
                torch.nn.utils.clip_grad_norm_(critic.parameters(), max_grad_norm)
                opt_actor.step()
                opt_critic.step()

                with torch.no_grad():
                    kl = (LP[mb] - new_lp).mean().item()
                    kl_v += kl
                    clip_v += (torch.abs(ratio - 1.0) > clip_eps_val).float().mean().item()
                    entropy_v += dist.entropy().mean().item()
                    actor_loss_v += actor_loss.item()
                    critic_loss_v += critic_loss.item()
                    mb_count += 1

                if kl > 1.5:
                    break

        # TRAIN METRICS
        with torch.no_grad():
            mu_train, _ = actor(X_train)
            train_rmse = torch.sqrt(((mu_train - y_train) ** 2).mean(dim=0))
            train_reward = R.mean().item()
            train_exp = explained_variance(R, V)

        print(
            f"[TRAIN {update}] AvgReward {train_reward:.6f} | "
            f"ActorLoss {actor_loss_v/mb_count:.6f} | "
            f"CriticLoss {critic_loss_v/mb_count:.6f} | "
            f"KL {kl_v/mb_count:.6f} | "
            f"ClipFrac {clip_v/mb_count:.3f} | "
            f"Entropy {entropy_v/mb_count:.4f} | "
            f"ExplVar {train_exp:.4f}"
        )
        print("TRAIN Per-action RMSE:", train_rmse.cpu().numpy())

        # TEST
        with torch.no_grad():
            mu_test, _ = actor(X_test)
            v_test = critic(X_test)
            test_reward = weighted_reward(mu_test, y_test)
            test_avg_reward = test_reward.mean().item()
            test_rmse = torch.sqrt(((mu_test - y_test) ** 2).mean(dim=0))
            test_exp = explained_variance(test_reward, v_test)
            test_critic_loss = nn.MSELoss()(v_test, test_reward).item()

        # SAVE BEST
        if test_avg_reward > best_test_reward:
            best_test_reward = test_avg_reward
            torch.save(actor.state_dict(), best_actor_path)
            torch.save(critic.state_dict(), best_critic_path)

    # ---------------- RECORD EXPERIMENT RESULTS ----------------
    results.append({
        "lr_actor": lr_actor_val,
        "lr_critic": lr_critic_val,
        "clip_eps": clip_eps_val,
        "ppo_epochs": ppo_epochs_val,
        "num_updates": num_updates_val,
        "best_test_reward": best_test_reward,
        "train_rmse": train_rmse.cpu().numpy(),
        "test_rmse": test_rmse.cpu().numpy(),
        "expl_var": test_exp
    })

# ---------------- PRINT ALL RESULTS ----------------
results = sorted(results, key=lambda x: x["best_test_reward"], reverse=True)
print("\n===== HYPERPARAMETER SWEEP RESULTS =====")
for r in results:
    print(r)


Device: cuda

=== EXPERIMENT: lr_actor=0.0003, lr_critic=0.0003, clip_eps=0.2, ppo_epochs=8, num_updates=2000 ===
[TRAIN 1] AvgReward -0.288505 | ActorLoss 0.180189 | CriticLoss 0.028259 | KL 2.400157 | ClipFrac 0.831 | Entropy -0.1810 | ExplVar 0.0874
TRAIN Per-action RMSE: [0.9561073  0.95368105 1.066124   0.32342294]
[TRAIN 2] AvgReward -0.267880 | ActorLoss 0.132288 | CriticLoss 0.011207 | KL 1.233760 | ClipFrac 0.767 | Entropy -0.1798 | ExplVar -0.1087
TRAIN Per-action RMSE: [1.0019927 1.0026817 0.9262506 0.2981989]
[TRAIN 3] AvgReward -0.249204 | ActorLoss 0.130056 | CriticLoss 0.007712 | KL 1.090499 | ClipFrac 0.814 | Entropy -0.1783 | ExplVar 0.2353
TRAIN Per-action RMSE: [0.89748454 0.93413204 0.8540272  0.32576758]
[TRAIN 4] AvgReward -0.226602 | ActorLoss 0.071343 | CriticLoss 0.007102 | KL 0.981512 | ClipFrac 0.766 | Entropy -0.1779 | ExplVar 0.4028
TRAIN Per-action RMSE: [0.7589915 0.8832949 0.7934841 0.3416911]
[TRAIN 5] AvgReward -0.204556 | ActorLoss 0.066649 | CriticLo

In [1]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from torch.distributions import Normal

# -----------------------
# Load backbone & PPO components (reuse your previous classes)
# -----------------------
# Make sure FertNetV2, PPOActor, PPOCritic are defined/imported

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Load your data
# X_all, y_all tensors from training
# Make sure these are on 'device'

# Load last checkpoint
ckpt_path = "ppo_checkpoints/ppo_update_2000.pth"  # change if different
ckpt = torch.load(ckpt_path, map_location=device)

backbone = FertNetV2(input_dim=12, output_dim=4).to(device)
actor = PPOActor(backbone, action_dim=5).to(device)
critic = PPOCritic(backbone).to(device)

actor.load_state_dict(ckpt["actor_state_dict"])
critic.load_state_dict(ckpt["critic_state_dict"])
actor.eval()
critic.eval()

# -----------------------
# Evaluation
# -----------------------
with torch.no_grad():
    mu, std = actor(X_all)
    pred_actions = mu  # take mean as deterministic action
    mse_per_action = ((pred_actions - y_all)**2).mean(dim=0).cpu().numpy()
    total_mse = ((pred_actions - y_all)**2).mean().item()
    
    print("Per-action MSE:", mse_per_action)
    print("Total mean MSE:", total_mse)

# Optional: plot predicted vs true for first few actions
plt.figure(figsize=(12,5))
for i in range(5):  # 5 actions
    plt.subplot(1,5,i+1)
    
    y_cpu = y_all[:, i].cpu().numpy()
    pred_cpu = pred_actions[:, i].cpu().numpy()
    
    plt.scatter(y_cpu, pred_cpu, alpha=0.5)
    plt.plot([y_cpu.min(), y_cpu.max()],
             [y_cpu.min(), y_cpu.max()], 'r--')  # y=x line
    
    plt.xlabel("True")
    plt.ylabel("Pred")
    plt.title(f"Action {i+1}")
plt.tight_layout()
plt.show()

# Optional: reward curve over updates if you saved rewards per update
# If you have saved rewards in a list `reward_history`, you can plot:
# plt.plot(reward_history)
# plt.xlabel("Update")
# plt.ylabel("Avg Reward")
# plt.title("PPO Training Reward Curve")
# plt.show()


NameError: name 'FertNetV2' is not defined

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import os

# -----------------------
# Paths & dataset
# -----------------------
checkpoint_dir = "ppo_checkpoints"
data_path = "data/fertnet_data.npz"

# Load dataset
data = np.load(data_path)
X_all = torch.tensor(data["X_train"].astype(np.float32))
y_all = torch.tensor(data["y_train"].astype(np.float32))

# Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
X_all = X_all.to(device)
y_all = y_all.to(device)

# -----------------------
# Load all checkpoints
# -----------------------
ckpts = sorted([f for f in os.listdir(checkpoint_dir) if f.endswith(".pth")])
updates = []
avg_rewards = []
adv_stds = []
for ckpt_file in ckpts:
    path = os.path.join(checkpoint_dir, ckpt_file)
    ckpt = torch.load(path, map_location=device)
    update = ckpt.get("update", None)
    updates.append(update)

    # Extract stored rewards/adv if you saved them during training
    # If not saved, you can only plot update numbers
    # Here, just plotting placeholder metrics (you can replace with real ones)
    # For example, if you saved avg_reward in checkpoint, use:
    # avg_rewards.append(ckpt["avg_reward"])
    # adv_stds.append(ckpt["adv_std"])
    
    # Temporary: placeholder (just to visualize checkpoint spacing)
    avg_rewards.append(update * 0.001)  # remove when real metrics are available
    adv_stds.append(0.5)                # remove when real metrics are available

# -----------------------
# Plot
# -----------------------
plt.figure(figsize=(12,5))

plt.subplot(1,2,1)
plt.plot(updates, avg_rewards, marker='o')
plt.xlabel("Update")
plt.ylabel("Average Reward")
plt.title("PPO Average Reward per Update")

plt.subplot(1,2,2)
plt.plot(updates, adv_stds, marker='o', color='orange')
plt.xlabel("Update")
plt.ylabel("Advantage Std")
plt.title("Advantage Std per Update")

plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import re

# --- Paste your training log as a multi-line string ---
log_text = """Device: cuda
Loaded X,y shapes: (17499, 12) (17499, 5)
Loaded FertNetV2 weights into backbone.
Starting PPO training (stable adv handling)...
[Update 1] Samples 2048 | Reward mean/std -2.885718/1.583833 | Value mean/std 0.373498/0.834252 | Adv std 1.918126e+00 | ZeroFrac 0.000
Post-update: AvgReward -2.885718 | AvgAdv -0.000000 | AdvAfterStd 7.101409e-01
[Update 2] Samples 2048 | Reward mean/std -2.291085/1.346289 | Value mean/std -2.785841/1.406987 | Adv std 9.235218e-01 | ZeroFrac 0.000
Post-update: AvgReward -2.291085 | AvgAdv 0.000000 | AdvAfterStd 6.363477e-01
[Update 3] Samples 2048 | Reward mean/std -2.005504/1.265993 | Value mean/std -2.398201/1.134327 | Adv std 7.286765e-01 | ZeroFrac 0.000
Post-update: AvgReward -2.005504 | AvgAdv 0.000000 | AdvAfterStd 5.599885e-01
[Update 4] Samples 2048 | Reward mean/std -1.562427/1.030587 | Value mean/std -1.954552/1.053904 | Adv std 6.261586e-01 | ZeroFrac 0.000
Post-update: AvgReward -1.562427 | AvgAdv 0.000000 | AdvAfterStd 4.846630e-01
[Update 5] Samples 2048 | Reward mean/std -1.260466/0.858692 | Value mean/std -1.606319/0.837052 | Adv std 5.219274e-01 | ZeroFrac 0.000
Post-update: AvgReward -1.260466 | AvgAdv 0.000000 | AdvAfterStd 4.058256e-01
[Update 6] Samples 2048 | Reward mean/std -0.993835/0.733507 | Value mean/std -1.231148/0.724842 | Adv std 4.572410e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.993835 | AvgAdv -0.000000 | AdvAfterStd 3.563271e-01
[Update 7] Samples 2048 | Reward mean/std -0.805699/0.659329 | Value mean/std -1.002874/0.638676 | Adv std 4.493046e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.805699 | AvgAdv -0.000000 | AdvAfterStd 3.495797e-01
[Update 8] Samples 2048 | Reward mean/std -0.612833/0.527820 | Value mean/std -0.755134/0.528859 | Adv std 3.683425e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.612833 | AvgAdv 0.000000 | AdvAfterStd 2.844989e-01
[Update 9] Samples 2048 | Reward mean/std -0.533716/0.469487 | Value mean/std -0.603617/0.454865 | Adv std 3.224729e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.533716 | AvgAdv 0.000000 | AdvAfterStd 2.573203e-01
[Update 10] Samples 2048 | Reward mean/std -0.449655/0.401531 | Value mean/std -0.561614/0.400881 | Adv std 2.968059e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.449655 | AvgAdv -0.000000 | AdvAfterStd 2.355556e-01
[Update 11] Samples 2048 | Reward mean/std -0.368504/0.343267 | Value mean/std -0.446139/0.307508 | Adv std 2.763711e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.368504 | AvgAdv 0.000000 | AdvAfterStd 2.169915e-01
[Update 12] Samples 2048 | Reward mean/std -0.322557/0.297792 | Value mean/std -0.339325/0.258148 | Adv std 2.684345e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.322557 | AvgAdv -0.000000 | AdvAfterStd 1.952446e-01
[Update 13] Samples 2048 | Reward mean/std -0.270821/0.246663 | Value mean/std -0.329298/0.215616 | Adv std 2.191780e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.270821 | AvgAdv -0.000000 | AdvAfterStd 1.693030e-01
[Update 14] Samples 2048 | Reward mean/std -0.241717/0.237569 | Value mean/std -0.310645/0.181196 | Adv std 2.120516e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.241717 | AvgAdv -0.000000 | AdvAfterStd 1.635204e-01
[Update 15] Samples 2048 | Reward mean/std -0.223877/0.267424 | Value mean/std -0.241506/0.147504 | Adv std 2.397361e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.223877 | AvgAdv -0.000000 | AdvAfterStd 2.044052e-01
[Update 16] Samples 2048 | Reward mean/std -0.208999/0.268927 | Value mean/std -0.281194/0.164134 | Adv std 2.337348e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.208999 | AvgAdv 0.000000 | AdvAfterStd 1.898941e-01
[Update 17] Samples 2048 | Reward mean/std -0.194289/0.203063 | Value mean/std -0.200688/0.148342 | Adv std 1.998945e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.194289 | AvgAdv 0.000000 | AdvAfterStd 1.517833e-01
[Update 18] Samples 2048 | Reward mean/std -0.182801/0.204486 | Value mean/std -0.198765/0.122485 | Adv std 1.924523e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.182801 | AvgAdv -0.000000 | AdvAfterStd 1.551865e-01
[Update 19] Samples 2048 | Reward mean/std -0.169341/0.213312 | Value mean/std -0.172007/0.135135 | Adv std 1.862490e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.169341 | AvgAdv -0.000000 | AdvAfterStd 1.586796e-01
[Update 20] Samples 2048 | Reward mean/std -0.168992/0.222149 | Value mean/std -0.140196/0.126174 | Adv std 2.059559e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.168992 | AvgAdv 0.000000 | AdvAfterStd 1.718308e-01
[Update 21] Samples 2048 | Reward mean/std -0.158365/0.203592 | Value mean/std -0.182576/0.127671 | Adv std 1.825127e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.158365 | AvgAdv -0.000000 | AdvAfterStd 1.430760e-01
[Update 22] Samples 2048 | Reward mean/std -0.156879/0.238651 | Value mean/std -0.142471/0.123582 | Adv std 2.075374e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.156879 | AvgAdv 0.000000 | AdvAfterStd 1.765596e-01
[Update 23] Samples 2048 | Reward mean/std -0.157865/0.267704 | Value mean/std -0.173170/0.139616 | Adv std 2.242402e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.157865 | AvgAdv 0.000000 | AdvAfterStd 1.862304e-01
[Update 24] Samples 2048 | Reward mean/std -0.149119/0.233861 | Value mean/std -0.140899/0.146660 | Adv std 1.933055e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.149119 | AvgAdv 0.000000 | AdvAfterStd 1.642843e-01
[Update 25] Samples 2048 | Reward mean/std -0.143318/0.249555 | Value mean/std -0.167504/0.138972 | Adv std 2.225236e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.143318 | AvgAdv -0.000000 | AdvAfterStd 1.880979e-01
[Update 26] Samples 2048 | Reward mean/std -0.135486/0.189777 | Value mean/std -0.127193/0.133597 | Adv std 1.658098e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.135486 | AvgAdv -0.000000 | AdvAfterStd 1.402714e-01
[Update 27] Samples 2048 | Reward mean/std -0.143127/0.228273 | Value mean/std -0.128450/0.123297 | Adv std 1.960305e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.143127 | AvgAdv -0.000000 | AdvAfterStd 1.506512e-01
[Update 28] Samples 2048 | Reward mean/std -0.143388/0.253247 | Value mean/std -0.125624/0.122595 | Adv std 2.371973e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.143388 | AvgAdv -0.000000 | AdvAfterStd 1.787113e-01
[Update 29] Samples 2048 | Reward mean/std -0.135857/0.195277 | Value mean/std -0.152237/0.111717 | Adv std 1.753489e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.135857 | AvgAdv 0.000000 | AdvAfterStd 1.321208e-01
[Update 30] Samples 2048 | Reward mean/std -0.137876/0.242556 | Value mean/std -0.124812/0.151449 | Adv std 2.043384e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.137876 | AvgAdv 0.000000 | AdvAfterStd 1.636115e-01
[Update 31] Samples 2048 | Reward mean/std -0.125686/0.187759 | Value mean/std -0.127551/0.143621 | Adv std 1.580367e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.125686 | AvgAdv 0.000000 | AdvAfterStd 1.253016e-01
[Update 32] Samples 2048 | Reward mean/std -0.119801/0.168097 | Value mean/std -0.118280/0.087831 | Adv std 1.585559e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.119801 | AvgAdv -0.000000 | AdvAfterStd 1.273813e-01
[Update 33] Samples 2048 | Reward mean/std -0.133465/0.260780 | Value mean/std -0.147890/0.134196 | Adv std 2.112795e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.133465 | AvgAdv 0.000000 | AdvAfterStd 1.777890e-01
[Update 34] Samples 2048 | Reward mean/std -0.130207/0.234211 | Value mean/std -0.131699/0.131884 | Adv std 1.878408e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.130207 | AvgAdv 0.000000 | AdvAfterStd 1.518571e-01
[Update 35] Samples 2048 | Reward mean/std -0.118101/0.189392 | Value mean/std -0.091577/0.144047 | Adv std 1.464644e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.118101 | AvgAdv 0.000000 | AdvAfterStd 1.181386e-01
[Update 36] Samples 2048 | Reward mean/std -0.123479/0.255224 | Value mean/std -0.136452/0.158877 | Adv std 2.028097e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.123479 | AvgAdv -0.000000 | AdvAfterStd 1.544787e-01
[Update 37] Samples 2048 | Reward mean/std -0.122290/0.247347 | Value mean/std -0.127836/0.172781 | Adv std 2.161307e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.122290 | AvgAdv -0.000000 | AdvAfterStd 1.557287e-01
[Update 38] Samples 2048 | Reward mean/std -0.117248/0.209031 | Value mean/std -0.115216/0.154667 | Adv std 1.835227e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.117248 | AvgAdv 0.000000 | AdvAfterStd 1.451041e-01
[Update 39] Samples 2048 | Reward mean/std -0.114266/0.199709 | Value mean/std -0.094422/0.118970 | Adv std 1.819771e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.114266 | AvgAdv 0.000000 | AdvAfterStd 1.500864e-01
[Update 40] Samples 2048 | Reward mean/std -0.116646/0.219506 | Value mean/std -0.116040/0.132693 | Adv std 1.786419e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.116646 | AvgAdv -0.000000 | AdvAfterStd 1.543820e-01
[Update 41] Samples 2048 | Reward mean/std -0.116224/0.226102 | Value mean/std -0.112934/0.113588 | Adv std 1.866197e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.116224 | AvgAdv 0.000000 | AdvAfterStd 1.397639e-01
[Update 42] Samples 2048 | Reward mean/std -0.112947/0.192593 | Value mean/std -0.102427/0.140653 | Adv std 1.731258e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.112947 | AvgAdv -0.000000 | AdvAfterStd 1.363634e-01
[Update 43] Samples 2048 | Reward mean/std -0.113975/0.251771 | Value mean/std -0.099952/0.107429 | Adv std 2.061698e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.113975 | AvgAdv 0.000000 | AdvAfterStd 1.508515e-01
[Update 44] Samples 2048 | Reward mean/std -0.101521/0.150643 | Value mean/std -0.095900/0.177119 | Adv std 1.621936e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.101521 | AvgAdv 0.000000 | AdvAfterStd 1.074660e-01
[Update 45] Samples 2048 | Reward mean/std -0.112668/0.243366 | Value mean/std -0.116307/0.111725 | Adv std 1.943292e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.112668 | AvgAdv -0.000000 | AdvAfterStd 1.297275e-01
[Update 46] Samples 2048 | Reward mean/std -0.109546/0.214749 | Value mean/std -0.108402/0.156231 | Adv std 1.727975e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.109546 | AvgAdv 0.000000 | AdvAfterStd 1.226737e-01
[Update 47] Samples 2048 | Reward mean/std -0.111992/0.191682 | Value mean/std -0.129190/0.152359 | Adv std 1.734938e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.111992 | AvgAdv -0.000000 | AdvAfterStd 1.304549e-01
[Update 48] Samples 2048 | Reward mean/std -0.114144/0.227232 | Value mean/std -0.119086/0.108973 | Adv std 1.941864e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.114144 | AvgAdv -0.000000 | AdvAfterStd 1.523338e-01
[Update 49] Samples 2048 | Reward mean/std -0.108077/0.198114 | Value mean/std -0.119010/0.153614 | Adv std 1.857527e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.108077 | AvgAdv -0.000000 | AdvAfterStd 1.270765e-01
[Update 50] Samples 2048 | Reward mean/std -0.104768/0.175595 | Value mean/std -0.101662/0.109507 | Adv std 1.641429e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.104768 | AvgAdv -0.000000 | AdvAfterStd 1.408791e-01
Saved checkpoint at update 50
[Update 51] Samples 2048 | Reward mean/std -0.110399/0.266191 | Value mean/std -0.094359/0.116921 | Adv std 2.306967e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.110399 | AvgAdv 0.000000 | AdvAfterStd 1.657523e-01
[Update 52] Samples 2048 | Reward mean/std -0.110883/0.210797 | Value mean/std -0.115128/0.147545 | Adv std 1.593158e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.110883 | AvgAdv 0.000000 | AdvAfterStd 1.255461e-01
[Update 53] Samples 2048 | Reward mean/std -0.107425/0.215399 | Value mean/std -0.116109/0.133291 | Adv std 1.580882e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.107425 | AvgAdv 0.000000 | AdvAfterStd 1.318376e-01
[Update 54] Samples 2048 | Reward mean/std -0.096975/0.166839 | Value mean/std -0.102322/0.163000 | Adv std 1.294949e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.096975 | AvgAdv 0.000000 | AdvAfterStd 9.905273e-02
[Update 55] Samples 2048 | Reward mean/std -0.115227/0.346876 | Value mean/std -0.133168/0.186809 | Adv std 2.395470e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.115227 | AvgAdv 0.000000 | AdvAfterStd 1.749461e-01
[Update 56] Samples 2048 | Reward mean/std -0.109035/0.227435 | Value mean/std -0.116800/0.178309 | Adv std 1.689738e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.109035 | AvgAdv -0.000000 | AdvAfterStd 1.269078e-01
[Update 57] Samples 2048 | Reward mean/std -0.099908/0.198086 | Value mean/std -0.102530/0.137344 | Adv std 1.482124e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.099908 | AvgAdv -0.000000 | AdvAfterStd 1.098174e-01
[Update 58] Samples 2048 | Reward mean/std -0.100577/0.204995 | Value mean/std -0.099925/0.183004 | Adv std 1.734479e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.100577 | AvgAdv -0.000000 | AdvAfterStd 1.139663e-01
[Update 59] Samples 2048 | Reward mean/std -0.097918/0.191249 | Value mean/std -0.104011/0.130221 | Adv std 1.542228e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.097918 | AvgAdv -0.000000 | AdvAfterStd 1.205797e-01
[Update 60] Samples 2048 | Reward mean/std -0.107549/0.205782 | Value mean/std -0.106538/0.162510 | Adv std 1.680001e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.107549 | AvgAdv -0.000000 | AdvAfterStd 1.304397e-01
[Update 61] Samples 2048 | Reward mean/std -0.097102/0.189604 | Value mean/std -0.094832/0.146282 | Adv std 1.503469e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.097102 | AvgAdv 0.000000 | AdvAfterStd 1.031309e-01
[Update 62] Samples 2048 | Reward mean/std -0.103385/0.249748 | Value mean/std -0.097041/0.150866 | Adv std 1.944977e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.103385 | AvgAdv 0.000000 | AdvAfterStd 1.641873e-01
[Update 63] Samples 2048 | Reward mean/std -0.099494/0.159783 | Value mean/std -0.094444/0.119618 | Adv std 1.476316e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.099494 | AvgAdv -0.000000 | AdvAfterStd 1.197634e-01
[Update 64] Samples 2048 | Reward mean/std -0.103715/0.257773 | Value mean/std -0.094497/0.130390 | Adv std 1.913977e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.103715 | AvgAdv 0.000000 | AdvAfterStd 1.425420e-01
[Update 65] Samples 2048 | Reward mean/std -0.105375/0.222310 | Value mean/std -0.102052/0.170487 | Adv std 1.537932e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.105375 | AvgAdv 0.000000 | AdvAfterStd 1.303726e-01
[Update 66] Samples 2048 | Reward mean/std -0.107988/0.259710 | Value mean/std -0.094418/0.132690 | Adv std 1.968681e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.107988 | AvgAdv -0.000000 | AdvAfterStd 1.492643e-01
[Update 67] Samples 2048 | Reward mean/std -0.098010/0.214659 | Value mean/std -0.093509/0.174032 | Adv std 1.841461e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.098010 | AvgAdv 0.000000 | AdvAfterStd 1.189499e-01
[Update 68] Samples 2048 | Reward mean/std -0.095975/0.181171 | Value mean/std -0.080920/0.151631 | Adv std 1.575411e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.095975 | AvgAdv 0.000000 | AdvAfterStd 1.251949e-01
[Update 69] Samples 2048 | Reward mean/std -0.093005/0.236917 | Value mean/std -0.105678/0.126968 | Adv std 1.878704e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.093005 | AvgAdv -0.000000 | AdvAfterStd 1.629244e-01
[Update 70] Samples 2048 | Reward mean/std -0.092070/0.216430 | Value mean/std -0.095993/0.155061 | Adv std 1.463943e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.092070 | AvgAdv 0.000000 | AdvAfterStd 1.068673e-01
[Update 71] Samples 2048 | Reward mean/std -0.102889/0.240465 | Value mean/std -0.093339/0.153908 | Adv std 1.595654e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.102889 | AvgAdv 0.000000 | AdvAfterStd 1.242257e-01
[Update 72] Samples 2048 | Reward mean/std -0.099831/0.224877 | Value mean/std -0.102111/0.162651 | Adv std 1.744699e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.099831 | AvgAdv 0.000000 | AdvAfterStd 1.293613e-01
[Update 73] Samples 2048 | Reward mean/std -0.096672/0.208573 | Value mean/std -0.090667/0.156848 | Adv std 1.412635e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.096672 | AvgAdv 0.000000 | AdvAfterStd 1.240391e-01
[Update 74] Samples 2048 | Reward mean/std -0.092899/0.215399 | Value mean/std -0.080044/0.138581 | Adv std 1.636705e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.092899 | AvgAdv 0.000000 | AdvAfterStd 1.210181e-01
[Update 75] Samples 2048 | Reward mean/std -0.091356/0.160571 | Value mean/std -0.088144/0.139373 | Adv std 1.365052e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.091356 | AvgAdv 0.000000 | AdvAfterStd 1.004373e-01
[Update 76] Samples 2048 | Reward mean/std -0.093077/0.241880 | Value mean/std -0.091122/0.140672 | Adv std 1.973906e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.093077 | AvgAdv -0.000000 | AdvAfterStd 1.611878e-01
[Update 77] Samples 2048 | Reward mean/std -0.094407/0.196704 | Value mean/std -0.082729/0.149698 | Adv std 1.667837e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.094407 | AvgAdv 0.000000 | AdvAfterStd 1.324972e-01
[Update 78] Samples 2048 | Reward mean/std -0.093155/0.166364 | Value mean/std -0.101495/0.114858 | Adv std 1.365459e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.093155 | AvgAdv -0.000000 | AdvAfterStd 9.950984e-02
[Update 79] Samples 2048 | Reward mean/std -0.097913/0.216096 | Value mean/std -0.087305/0.116485 | Adv std 1.720919e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.097913 | AvgAdv -0.000000 | AdvAfterStd 1.195751e-01
[Update 80] Samples 2048 | Reward mean/std -0.102727/0.302946 | Value mean/std -0.097823/0.180202 | Adv std 2.066033e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.102727 | AvgAdv 0.000000 | AdvAfterStd 1.356861e-01
[Update 81] Samples 2048 | Reward mean/std -0.090815/0.199695 | Value mean/std -0.097381/0.172215 | Adv std 1.381323e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.090815 | AvgAdv 0.000000 | AdvAfterStd 1.167150e-01
[Update 82] Samples 2048 | Reward mean/std -0.087003/0.159365 | Value mean/std -0.098879/0.099031 | Adv std 1.524118e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.087003 | AvgAdv -0.000000 | AdvAfterStd 1.294159e-01
[Update 83] Samples 2048 | Reward mean/std -0.094670/0.188185 | Value mean/std -0.116780/0.154761 | Adv std 1.442531e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.094670 | AvgAdv -0.000000 | AdvAfterStd 1.088221e-01
[Update 84] Samples 2048 | Reward mean/std -0.090138/0.192477 | Value mean/std -0.076473/0.110369 | Adv std 1.525584e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.090138 | AvgAdv -0.000000 | AdvAfterStd 1.129141e-01
[Update 85] Samples 2048 | Reward mean/std -0.080738/0.139924 | Value mean/std -0.097318/0.112987 | Adv std 1.172857e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.080738 | AvgAdv 0.000000 | AdvAfterStd 9.631805e-02
[Update 86] Samples 2048 | Reward mean/std -0.093208/0.239918 | Value mean/std -0.096778/0.110922 | Adv std 1.862130e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.093208 | AvgAdv 0.000000 | AdvAfterStd 1.352590e-01
[Update 87] Samples 2048 | Reward mean/std -0.095276/0.297054 | Value mean/std -0.093789/0.186107 | Adv std 1.800643e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.095276 | AvgAdv 0.000000 | AdvAfterStd 1.256353e-01
[Update 88] Samples 2048 | Reward mean/std -0.093296/0.216381 | Value mean/std -0.101931/0.170109 | Adv std 1.803865e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.093296 | AvgAdv -0.000000 | AdvAfterStd 1.364274e-01
[Update 89] Samples 2048 | Reward mean/std -0.087541/0.202420 | Value mean/std -0.068949/0.181980 | Adv std 1.499059e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.087541 | AvgAdv -0.000000 | AdvAfterStd 1.108218e-01
[Update 90] Samples 2048 | Reward mean/std -0.083480/0.147228 | Value mean/std -0.079364/0.124446 | Adv std 1.438152e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.083480 | AvgAdv -0.000000 | AdvAfterStd 1.086442e-01
[Update 91] Samples 2048 | Reward mean/std -0.085897/0.167259 | Value mean/std -0.088052/0.111201 | Adv std 1.375108e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.085897 | AvgAdv 0.000000 | AdvAfterStd 1.057792e-01
[Update 92] Samples 2048 | Reward mean/std -0.094734/0.259295 | Value mean/std -0.084709/0.165874 | Adv std 1.710252e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.094734 | AvgAdv 0.000000 | AdvAfterStd 1.362404e-01
[Update 93] Samples 2048 | Reward mean/std -0.082279/0.159393 | Value mean/std -0.098632/0.101750 | Adv std 1.252900e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.082279 | AvgAdv 0.000000 | AdvAfterStd 1.023260e-01
[Update 94] Samples 2048 | Reward mean/std -0.092994/0.255281 | Value mean/std -0.081583/0.134744 | Adv std 1.975359e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.092994 | AvgAdv -0.000000 | AdvAfterStd 1.374790e-01
[Update 95] Samples 2048 | Reward mean/std -0.083591/0.158484 | Value mean/std -0.081302/0.174311 | Adv std 1.940314e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.083591 | AvgAdv 0.000000 | AdvAfterStd 1.291070e-01
[Update 96] Samples 2048 | Reward mean/std -0.088376/0.205917 | Value mean/std -0.083062/0.106430 | Adv std 1.537753e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.088376 | AvgAdv -0.000000 | AdvAfterStd 1.170005e-01
[Update 97] Samples 2048 | Reward mean/std -0.091562/0.174123 | Value mean/std -0.084465/0.127441 | Adv std 1.457498e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.091562 | AvgAdv 0.000000 | AdvAfterStd 1.217160e-01
[Update 98] Samples 2048 | Reward mean/std -0.096990/0.235156 | Value mean/std -0.088203/0.150284 | Adv std 1.678398e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.096990 | AvgAdv 0.000000 | AdvAfterStd 1.310875e-01
[Update 99] Samples 2048 | Reward mean/std -0.088626/0.204957 | Value mean/std -0.092599/0.189513 | Adv std 1.799551e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.088626 | AvgAdv 0.000000 | AdvAfterStd 1.204543e-01
[Update 100] Samples 2048 | Reward mean/std -0.088964/0.172349 | Value mean/std -0.097427/0.097188 | Adv std 1.528110e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.088964 | AvgAdv 0.000000 | AdvAfterStd 1.034906e-01
Saved checkpoint at update 100
[Update 101] Samples 2048 | Reward mean/std -0.092892/0.239897 | Value mean/std -0.098811/0.171694 | Adv std 1.781948e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.092892 | AvgAdv 0.000000 | AdvAfterStd 1.225714e-01
[Update 102] Samples 2048 | Reward mean/std -0.090882/0.226324 | Value mean/std -0.087703/0.138348 | Adv std 1.679298e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.090882 | AvgAdv -0.000000 | AdvAfterStd 1.287477e-01
[Update 103] Samples 2048 | Reward mean/std -0.090543/0.211394 | Value mean/std -0.076650/0.184106 | Adv std 1.686112e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.090543 | AvgAdv 0.000000 | AdvAfterStd 1.121820e-01
[Update 104] Samples 2048 | Reward mean/std -0.081588/0.152969 | Value mean/std -0.079037/0.114310 | Adv std 1.158623e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.081588 | AvgAdv 0.000000 | AdvAfterStd 8.436856e-02
[Update 105] Samples 2048 | Reward mean/std -0.087181/0.176295 | Value mean/std -0.074577/0.138815 | Adv std 1.389981e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.087181 | AvgAdv 0.000000 | AdvAfterStd 1.154594e-01
[Update 106] Samples 2048 | Reward mean/std -0.087777/0.172554 | Value mean/std -0.069904/0.120803 | Adv std 1.469956e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.087777 | AvgAdv -0.000000 | AdvAfterStd 1.276485e-01
[Update 107] Samples 2048 | Reward mean/std -0.091269/0.322408 | Value mean/std -0.087834/0.142093 | Adv std 2.216161e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.091269 | AvgAdv 0.000000 | AdvAfterStd 1.331252e-01
[Update 108] Samples 2048 | Reward mean/std -0.082277/0.193001 | Value mean/std -0.081126/0.140244 | Adv std 1.317323e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.082277 | AvgAdv -0.000000 | AdvAfterStd 1.127884e-01
[Update 109] Samples 2048 | Reward mean/std -0.102632/0.386996 | Value mean/std -0.088589/0.310938 | Adv std 1.888395e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.102632 | AvgAdv -0.000000 | AdvAfterStd 1.229930e-01
[Update 110] Samples 2048 | Reward mean/std -0.094058/0.392307 | Value mean/std -0.086543/0.299224 | Adv std 1.949899e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.094058 | AvgAdv -0.000000 | AdvAfterStd 1.191148e-01
[Update 111] Samples 2048 | Reward mean/std -0.094526/0.305228 | Value mean/std -0.087983/0.162424 | Adv std 1.964084e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.094526 | AvgAdv -0.000000 | AdvAfterStd 1.771263e-01
[Update 112] Samples 2048 | Reward mean/std -0.083076/0.179334 | Value mean/std -0.088238/0.112289 | Adv std 1.400248e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.083076 | AvgAdv -0.000000 | AdvAfterStd 1.123376e-01
[Update 113] Samples 2048 | Reward mean/std -0.090764/0.206585 | Value mean/std -0.098527/0.166801 | Adv std 1.617104e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.090764 | AvgAdv -0.000000 | AdvAfterStd 1.287362e-01
[Update 114] Samples 2048 | Reward mean/std -0.087016/0.196716 | Value mean/std -0.106427/0.173701 | Adv std 1.569172e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.087016 | AvgAdv 0.000000 | AdvAfterStd 1.133413e-01
[Update 115] Samples 2048 | Reward mean/std -0.092310/0.206411 | Value mean/std -0.079551/0.159850 | Adv std 1.384393e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.092310 | AvgAdv -0.000000 | AdvAfterStd 1.044194e-01
[Update 116] Samples 2048 | Reward mean/std -0.085018/0.182779 | Value mean/std -0.091186/0.140714 | Adv std 1.554690e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.085018 | AvgAdv -0.000000 | AdvAfterStd 1.088435e-01
[Update 117] Samples 2048 | Reward mean/std -0.091952/0.202572 | Value mean/std -0.065717/0.131872 | Adv std 1.774982e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.091952 | AvgAdv -0.000000 | AdvAfterStd 1.281976e-01
[Update 118] Samples 2048 | Reward mean/std -0.079753/0.131520 | Value mean/std -0.086079/0.121742 | Adv std 1.392196e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.079753 | AvgAdv 0.000000 | AdvAfterStd 9.728482e-02
[Update 119] Samples 2048 | Reward mean/std -0.084360/0.181872 | Value mean/std -0.077978/0.115893 | Adv std 1.362370e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.084360 | AvgAdv 0.000000 | AdvAfterStd 1.103638e-01
[Update 120] Samples 2048 | Reward mean/std -0.087616/0.214453 | Value mean/std -0.101918/0.118945 | Adv std 1.596197e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.087616 | AvgAdv 0.000000 | AdvAfterStd 1.297288e-01
[Update 121] Samples 2048 | Reward mean/std -0.078714/0.153893 | Value mean/std -0.063749/0.116196 | Adv std 1.408229e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.078714 | AvgAdv -0.000000 | AdvAfterStd 1.003315e-01
[Update 122] Samples 2048 | Reward mean/std -0.083297/0.149385 | Value mean/std -0.075099/0.121272 | Adv std 1.443356e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.083297 | AvgAdv 0.000000 | AdvAfterStd 1.071858e-01
[Update 123] Samples 2048 | Reward mean/std -0.101110/0.387038 | Value mean/std -0.083044/0.122128 | Adv std 3.139040e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.101110 | AvgAdv 0.000000 | AdvAfterStd 1.309773e-01
[Update 124] Samples 2048 | Reward mean/std -0.091377/0.291838 | Value mean/std -0.100608/0.253276 | Adv std 1.638757e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.091377 | AvgAdv -0.000000 | AdvAfterStd 1.138250e-01
[Update 125] Samples 2048 | Reward mean/std -0.093017/0.220553 | Value mean/std -0.088042/0.245466 | Adv std 1.994695e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.093017 | AvgAdv -0.000000 | AdvAfterStd 1.095621e-01
[Update 126] Samples 2048 | Reward mean/std -0.079391/0.137534 | Value mean/std -0.083814/0.114622 | Adv std 1.195307e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.079391 | AvgAdv 0.000000 | AdvAfterStd 9.282898e-02
[Update 127] Samples 2048 | Reward mean/std -0.092619/0.229827 | Value mean/std -0.086643/0.163221 | Adv std 1.349339e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.092619 | AvgAdv -0.000000 | AdvAfterStd 9.936272e-02
[Update 128] Samples 2048 | Reward mean/std -0.091406/0.205293 | Value mean/std -0.092714/0.159638 | Adv std 1.532531e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.091406 | AvgAdv 0.000000 | AdvAfterStd 1.137751e-01
[Update 129] Samples 2048 | Reward mean/std -0.088987/0.223940 | Value mean/std -0.103116/0.135296 | Adv std 1.654157e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.088987 | AvgAdv -0.000000 | AdvAfterStd 1.350093e-01
[Update 130] Samples 2048 | Reward mean/std -0.080473/0.134998 | Value mean/std -0.066925/0.191588 | Adv std 1.649717e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.080473 | AvgAdv 0.000000 | AdvAfterStd 9.351987e-02
[Update 131] Samples 2048 | Reward mean/std -0.091558/0.219126 | Value mean/std -0.093773/0.109096 | Adv std 1.598815e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.091558 | AvgAdv -0.000000 | AdvAfterStd 1.061612e-01
[Update 132] Samples 2048 | Reward mean/std -0.087634/0.184398 | Value mean/std -0.100560/0.125706 | Adv std 1.473346e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.087634 | AvgAdv 0.000000 | AdvAfterStd 1.052362e-01
[Update 133] Samples 2048 | Reward mean/std -0.088481/0.243654 | Value mean/std -0.085457/0.191226 | Adv std 1.799171e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.088481 | AvgAdv 0.000000 | AdvAfterStd 1.239937e-01
[Update 134] Samples 2048 | Reward mean/std -0.092596/0.294927 | Value mean/std -0.099566/0.230187 | Adv std 1.677235e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.092596 | AvgAdv -0.000000 | AdvAfterStd 1.063718e-01
[Update 135] Samples 2048 | Reward mean/std -0.082698/0.141527 | Value mean/std -0.100977/0.153847 | Adv std 1.375653e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.082698 | AvgAdv -0.000000 | AdvAfterStd 1.024497e-01
[Update 136] Samples 2048 | Reward mean/std -0.094383/0.315618 | Value mean/std -0.087839/0.106384 | Adv std 2.921649e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.094383 | AvgAdv 0.000000 | AdvAfterStd 1.410030e-01
[Update 137] Samples 2048 | Reward mean/std -0.088862/0.207526 | Value mean/std -0.097691/0.123820 | Adv std 1.673530e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.088862 | AvgAdv 0.000000 | AdvAfterStd 1.261482e-01
[Update 138] Samples 2048 | Reward mean/std -0.084334/0.159560 | Value mean/std -0.101258/0.234371 | Adv std 2.204762e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.084334 | AvgAdv 0.000000 | AdvAfterStd 1.058214e-01
[Update 139] Samples 2048 | Reward mean/std -0.087450/0.187863 | Value mean/std -0.104920/0.155333 | Adv std 1.444682e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.087450 | AvgAdv 0.000000 | AdvAfterStd 1.115469e-01
[Update 140] Samples 2048 | Reward mean/std -0.082898/0.187492 | Value mean/std -0.089845/0.099899 | Adv std 1.506703e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.082898 | AvgAdv 0.000000 | AdvAfterStd 1.104426e-01
[Update 141] Samples 2048 | Reward mean/std -0.089789/0.214571 | Value mean/std -0.077935/0.178329 | Adv std 1.887034e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.089789 | AvgAdv 0.000000 | AdvAfterStd 1.512517e-01
[Update 142] Samples 2048 | Reward mean/std -0.099376/0.195547 | Value mean/std -0.094940/0.132512 | Adv std 1.600500e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.099376 | AvgAdv 0.000000 | AdvAfterStd 1.096130e-01
[Update 143] Samples 2048 | Reward mean/std -0.085951/0.174852 | Value mean/std -0.103728/0.132294 | Adv std 1.522882e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.085951 | AvgAdv 0.000000 | AdvAfterStd 1.138619e-01
[Update 144] Samples 2048 | Reward mean/std -0.093907/0.282974 | Value mean/std -0.073916/0.165049 | Adv std 1.934914e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.093907 | AvgAdv -0.000000 | AdvAfterStd 1.165472e-01
[Update 145] Samples 2048 | Reward mean/std -0.093242/0.203991 | Value mean/std -0.085417/0.155858 | Adv std 1.484964e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.093242 | AvgAdv -0.000000 | AdvAfterStd 1.224317e-01
[Update 146] Samples 2048 | Reward mean/std -0.086915/0.193057 | Value mean/std -0.089062/0.146227 | Adv std 1.646099e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.086915 | AvgAdv -0.000000 | AdvAfterStd 1.163484e-01
[Update 147] Samples 2048 | Reward mean/std -0.081247/0.175878 | Value mean/std -0.074526/0.143174 | Adv std 1.170902e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.081247 | AvgAdv -0.000000 | AdvAfterStd 1.030455e-01
[Update 148] Samples 2048 | Reward mean/std -0.094498/0.302529 | Value mean/std -0.094987/0.147589 | Adv std 2.266984e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.094498 | AvgAdv -0.000000 | AdvAfterStd 1.059875e-01
[Update 149] Samples 2048 | Reward mean/std -0.078434/0.188834 | Value mean/std -0.077587/0.140024 | Adv std 1.047960e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.078434 | AvgAdv 0.000000 | AdvAfterStd 8.201713e-02
[Update 150] Samples 2048 | Reward mean/std -0.081756/0.152831 | Value mean/std -0.077050/0.133014 | Adv std 1.295829e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.081756 | AvgAdv 0.000000 | AdvAfterStd 9.995548e-02
Saved checkpoint at update 150
[Update 151] Samples 2048 | Reward mean/std -0.088298/0.238665 | Value mean/std -0.082641/0.124886 | Adv std 1.831063e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.088298 | AvgAdv -0.000000 | AdvAfterStd 1.180440e-01
[Update 152] Samples 2048 | Reward mean/std -0.084296/0.230099 | Value mean/std -0.088164/0.249795 | Adv std 1.901583e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.084296 | AvgAdv -0.000000 | AdvAfterStd 1.261908e-01
[Update 153] Samples 2048 | Reward mean/std -0.086158/0.224174 | Value mean/std -0.072308/0.128530 | Adv std 1.797281e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.086158 | AvgAdv 0.000000 | AdvAfterStd 1.133835e-01
[Update 154] Samples 2048 | Reward mean/std -0.089286/0.241859 | Value mean/std -0.081389/0.220612 | Adv std 1.337905e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.089286 | AvgAdv 0.000000 | AdvAfterStd 1.074194e-01
[Update 155] Samples 2048 | Reward mean/std -0.084264/0.214764 | Value mean/std -0.065996/0.148444 | Adv std 1.713893e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.084264 | AvgAdv 0.000000 | AdvAfterStd 1.226214e-01
[Update 156] Samples 2048 | Reward mean/std -0.086559/0.222625 | Value mean/std -0.081389/0.148210 | Adv std 1.502939e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.086559 | AvgAdv 0.000000 | AdvAfterStd 1.290519e-01
[Update 157] Samples 2048 | Reward mean/std -0.079507/0.165827 | Value mean/std -0.096531/0.170253 | Adv std 1.375982e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.079507 | AvgAdv -0.000000 | AdvAfterStd 1.035903e-01
[Update 158] Samples 2048 | Reward mean/std -0.073998/0.151460 | Value mean/std -0.085547/0.100752 | Adv std 1.111365e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.073998 | AvgAdv -0.000000 | AdvAfterStd 8.964379e-02
[Update 159] Samples 2048 | Reward mean/std -0.079085/0.205693 | Value mean/std -0.089456/0.156256 | Adv std 1.279709e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.079085 | AvgAdv -0.000000 | AdvAfterStd 1.051255e-01
[Update 160] Samples 2048 | Reward mean/std -0.093812/0.303805 | Value mean/std -0.090705/0.240052 | Adv std 1.698545e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.093812 | AvgAdv -0.000000 | AdvAfterStd 1.278219e-01
[Update 161] Samples 2048 | Reward mean/std -0.084458/0.180037 | Value mean/std -0.088352/0.140977 | Adv std 1.454544e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.084458 | AvgAdv 0.000000 | AdvAfterStd 1.162861e-01
[Update 162] Samples 2048 | Reward mean/std -0.082233/0.173455 | Value mean/std -0.083217/0.125217 | Adv std 1.424585e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.082233 | AvgAdv -0.000000 | AdvAfterStd 1.046454e-01
[Update 163] Samples 2048 | Reward mean/std -0.087027/0.199016 | Value mean/std -0.100749/0.145647 | Adv std 1.849615e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.087027 | AvgAdv 0.000000 | AdvAfterStd 1.181974e-01
[Update 164] Samples 2048 | Reward mean/std -0.078652/0.145154 | Value mean/std -0.079368/0.149448 | Adv std 1.401232e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.078652 | AvgAdv 0.000000 | AdvAfterStd 9.733118e-02
[Update 165] Samples 2048 | Reward mean/std -0.085335/0.192979 | Value mean/std -0.085773/0.122746 | Adv std 1.521699e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.085335 | AvgAdv 0.000000 | AdvAfterStd 1.031476e-01
[Update 166] Samples 2048 | Reward mean/std -0.079088/0.161355 | Value mean/std -0.074158/0.098351 | Adv std 1.405037e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.079088 | AvgAdv 0.000000 | AdvAfterStd 1.029805e-01
[Update 167] Samples 2048 | Reward mean/std -0.082527/0.210385 | Value mean/std -0.085345/0.124526 | Adv std 1.662151e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.082527 | AvgAdv 0.000000 | AdvAfterStd 1.249892e-01
[Update 168] Samples 2048 | Reward mean/std -0.077523/0.177046 | Value mean/std -0.081330/0.181193 | Adv std 1.279356e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.077523 | AvgAdv 0.000000 | AdvAfterStd 8.780397e-02
[Update 169] Samples 2048 | Reward mean/std -0.089696/0.215633 | Value mean/std -0.086183/0.167986 | Adv std 1.661557e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.089696 | AvgAdv -0.000000 | AdvAfterStd 1.133587e-01
[Update 170] Samples 2048 | Reward mean/std -0.080765/0.193139 | Value mean/std -0.081765/0.183998 | Adv std 1.369248e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.080765 | AvgAdv 0.000000 | AdvAfterStd 1.032161e-01
[Update 171] Samples 2048 | Reward mean/std -0.083347/0.281450 | Value mean/std -0.080732/0.231064 | Adv std 1.457776e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.083347 | AvgAdv -0.000000 | AdvAfterStd 1.078623e-01
[Update 172] Samples 2048 | Reward mean/std -0.077542/0.166569 | Value mean/std -0.072543/0.104558 | Adv std 1.309847e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.077542 | AvgAdv 0.000000 | AdvAfterStd 9.140924e-02
[Update 173] Samples 2048 | Reward mean/std -0.084390/0.197156 | Value mean/std -0.085890/0.135784 | Adv std 1.510962e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.084390 | AvgAdv 0.000000 | AdvAfterStd 9.718022e-02
[Update 174] Samples 2048 | Reward mean/std -0.081794/0.192187 | Value mean/std -0.078192/0.146415 | Adv std 1.546157e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.081794 | AvgAdv 0.000000 | AdvAfterStd 1.158766e-01
[Update 175] Samples 2048 | Reward mean/std -0.079739/0.154923 | Value mean/std -0.073455/0.129479 | Adv std 1.394370e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.079739 | AvgAdv -0.000000 | AdvAfterStd 9.690591e-02
[Update 176] Samples 2048 | Reward mean/std -0.088334/0.271119 | Value mean/std -0.090241/0.182191 | Adv std 1.444564e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.088334 | AvgAdv -0.000000 | AdvAfterStd 9.677389e-02
[Update 177] Samples 2048 | Reward mean/std -0.081433/0.207060 | Value mean/std -0.081394/0.159079 | Adv std 1.352027e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.081433 | AvgAdv -0.000000 | AdvAfterStd 1.198407e-01
[Update 178] Samples 2048 | Reward mean/std -0.079191/0.173675 | Value mean/std -0.089948/0.128282 | Adv std 1.310083e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.079191 | AvgAdv -0.000000 | AdvAfterStd 1.006380e-01
[Update 179] Samples 2048 | Reward mean/std -0.073605/0.136247 | Value mean/std -0.056724/0.094104 | Adv std 1.257274e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.073605 | AvgAdv -0.000000 | AdvAfterStd 9.910363e-02
[Update 180] Samples 2048 | Reward mean/std -0.093771/0.241638 | Value mean/std -0.071425/0.117044 | Adv std 2.093072e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.093771 | AvgAdv 0.000000 | AdvAfterStd 1.211213e-01
[Update 181] Samples 2048 | Reward mean/std -0.074168/0.153815 | Value mean/std -0.096101/0.117379 | Adv std 1.314383e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.074168 | AvgAdv 0.000000 | AdvAfterStd 1.029785e-01
[Update 182] Samples 2048 | Reward mean/std -0.076757/0.148857 | Value mean/std -0.066626/0.123203 | Adv std 1.300076e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.076757 | AvgAdv -0.000000 | AdvAfterStd 9.778675e-02
[Update 183] Samples 2048 | Reward mean/std -0.085532/0.223633 | Value mean/std -0.088449/0.111028 | Adv std 1.891315e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.085532 | AvgAdv -0.000000 | AdvAfterStd 1.325949e-01
[Update 184] Samples 2048 | Reward mean/std -0.077431/0.182474 | Value mean/std -0.087178/0.133793 | Adv std 1.382710e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.077431 | AvgAdv 0.000000 | AdvAfterStd 1.077639e-01
[Update 185] Samples 2048 | Reward mean/std -0.079761/0.179944 | Value mean/std -0.070571/0.120606 | Adv std 1.478342e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.079761 | AvgAdv -0.000000 | AdvAfterStd 1.216539e-01
[Update 186] Samples 2048 | Reward mean/std -0.078679/0.179786 | Value mean/std -0.079917/0.154399 | Adv std 1.711012e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.078679 | AvgAdv -0.000000 | AdvAfterStd 9.622238e-02
[Update 187] Samples 2048 | Reward mean/std -0.078586/0.196197 | Value mean/std -0.078876/0.087664 | Adv std 1.669818e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.078586 | AvgAdv 0.000000 | AdvAfterStd 1.103414e-01
[Update 188] Samples 2048 | Reward mean/std -0.085386/0.174413 | Value mean/std -0.086640/0.124339 | Adv std 1.422703e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.085386 | AvgAdv 0.000000 | AdvAfterStd 1.113001e-01
[Update 189] Samples 2048 | Reward mean/std -0.094721/0.305530 | Value mean/std -0.089040/0.171478 | Adv std 2.272196e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.094721 | AvgAdv 0.000000 | AdvAfterStd 1.115776e-01
[Update 190] Samples 2048 | Reward mean/std -0.079086/0.161626 | Value mean/std -0.083162/0.141067 | Adv std 1.570151e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.079086 | AvgAdv 0.000000 | AdvAfterStd 1.069831e-01
[Update 191] Samples 2048 | Reward mean/std -0.082177/0.179230 | Value mean/std -0.080637/0.108578 | Adv std 1.510066e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.082177 | AvgAdv 0.000000 | AdvAfterStd 1.156945e-01
[Update 192] Samples 2048 | Reward mean/std -0.077604/0.191051 | Value mean/std -0.066187/0.118436 | Adv std 1.299951e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.077604 | AvgAdv -0.000000 | AdvAfterStd 9.005623e-02
[Update 193] Samples 2048 | Reward mean/std -0.082410/0.199830 | Value mean/std -0.088338/0.151027 | Adv std 1.274822e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.082410 | AvgAdv -0.000000 | AdvAfterStd 9.511897e-02
[Update 194] Samples 2048 | Reward mean/std -0.077579/0.162721 | Value mean/std -0.073372/0.101834 | Adv std 1.313087e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.077579 | AvgAdv -0.000000 | AdvAfterStd 1.085192e-01
[Update 195] Samples 2048 | Reward mean/std -0.073855/0.142340 | Value mean/std -0.083869/0.096044 | Adv std 1.156301e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.073855 | AvgAdv 0.000000 | AdvAfterStd 8.669910e-02
[Update 196] Samples 2048 | Reward mean/std -0.097334/0.449189 | Value mean/std -0.086037/0.212428 | Adv std 2.845711e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.097334 | AvgAdv 0.000000 | AdvAfterStd 1.245869e-01
[Update 197] Samples 2048 | Reward mean/std -0.078768/0.175258 | Value mean/std -0.077324/0.110062 | Adv std 1.229273e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.078768 | AvgAdv 0.000000 | AdvAfterStd 9.205102e-02
[Update 198] Samples 2048 | Reward mean/std -0.090453/0.241500 | Value mean/std -0.085937/0.232137 | Adv std 1.474076e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.090453 | AvgAdv -0.000000 | AdvAfterStd 1.045166e-01
[Update 199] Samples 2048 | Reward mean/std -0.087223/0.208347 | Value mean/std -0.086999/0.205586 | Adv std 1.472559e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.087223 | AvgAdv -0.000000 | AdvAfterStd 1.071213e-01
[Update 200] Samples 2048 | Reward mean/std -0.075651/0.160265 | Value mean/std -0.082762/0.150342 | Adv std 1.213949e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.075651 | AvgAdv 0.000000 | AdvAfterStd 9.987993e-02
Saved checkpoint at update 200
[Update 201] Samples 2048 | Reward mean/std -0.092451/0.302714 | Value mean/std -0.086218/0.195116 | Adv std 2.109250e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.092451 | AvgAdv 0.000000 | AdvAfterStd 1.181555e-01
[Update 202] Samples 2048 | Reward mean/std -0.087768/0.405247 | Value mean/std -0.089938/0.292100 | Adv std 2.669123e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.087768 | AvgAdv -0.000000 | AdvAfterStd 1.233746e-01
[Update 203] Samples 2048 | Reward mean/std -0.078404/0.172456 | Value mean/std -0.080159/0.165688 | Adv std 1.788008e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.078404 | AvgAdv -0.000000 | AdvAfterStd 9.611810e-02
[Update 204] Samples 2048 | Reward mean/std -0.077729/0.168904 | Value mean/std -0.069869/0.105519 | Adv std 1.185964e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.077729 | AvgAdv 0.000000 | AdvAfterStd 9.566765e-02
[Update 205] Samples 2048 | Reward mean/std -0.084966/0.315891 | Value mean/std -0.088302/0.189966 | Adv std 2.162030e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.084966 | AvgAdv 0.000000 | AdvAfterStd 9.356806e-02
[Update 206] Samples 2048 | Reward mean/std -0.075437/0.151843 | Value mean/std -0.083686/0.126762 | Adv std 1.286641e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.075437 | AvgAdv -0.000000 | AdvAfterStd 9.396119e-02
[Update 207] Samples 2048 | Reward mean/std -0.087273/0.288278 | Value mean/std -0.083207/0.177929 | Adv std 1.626156e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.087273 | AvgAdv 0.000000 | AdvAfterStd 9.653135e-02
[Update 208] Samples 2048 | Reward mean/std -0.073757/0.167914 | Value mean/std -0.084044/0.109395 | Adv std 1.347088e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.073757 | AvgAdv 0.000000 | AdvAfterStd 9.207800e-02
[Update 209] Samples 2048 | Reward mean/std -0.077444/0.204925 | Value mean/std -0.064151/0.162567 | Adv std 1.459765e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.077444 | AvgAdv 0.000000 | AdvAfterStd 1.183238e-01
[Update 210] Samples 2048 | Reward mean/std -0.072912/0.169855 | Value mean/std -0.068536/0.110893 | Adv std 1.257623e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.072912 | AvgAdv -0.000000 | AdvAfterStd 8.797655e-02
[Update 211] Samples 2048 | Reward mean/std -0.080008/0.210774 | Value mean/std -0.076549/0.201835 | Adv std 1.405089e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.080008 | AvgAdv -0.000000 | AdvAfterStd 1.090585e-01
[Update 212] Samples 2048 | Reward mean/std -0.072931/0.157062 | Value mean/std -0.079456/0.109417 | Adv std 1.342518e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.072931 | AvgAdv 0.000000 | AdvAfterStd 1.054964e-01
[Update 213] Samples 2048 | Reward mean/std -0.082192/0.199465 | Value mean/std -0.073903/0.168097 | Adv std 1.298031e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.082192 | AvgAdv -0.000000 | AdvAfterStd 1.048515e-01
[Update 214] Samples 2048 | Reward mean/std -0.075499/0.160334 | Value mean/std -0.079773/0.129396 | Adv std 1.165811e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.075499 | AvgAdv -0.000000 | AdvAfterStd 9.577430e-02
[Update 215] Samples 2048 | Reward mean/std -0.078324/0.173568 | Value mean/std -0.074418/0.122253 | Adv std 1.356194e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.078324 | AvgAdv 0.000000 | AdvAfterStd 9.127688e-02
[Update 216] Samples 2048 | Reward mean/std -0.086435/0.226848 | Value mean/std -0.085528/0.134725 | Adv std 1.600610e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.086435 | AvgAdv 0.000000 | AdvAfterStd 1.095384e-01
[Update 217] Samples 2048 | Reward mean/std -0.074286/0.202858 | Value mean/std -0.081421/0.184553 | Adv std 1.384724e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.074286 | AvgAdv 0.000000 | AdvAfterStd 7.804682e-02
[Update 218] Samples 2048 | Reward mean/std -0.077532/0.181157 | Value mean/std -0.079924/0.148269 | Adv std 1.208867e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.077532 | AvgAdv 0.000000 | AdvAfterStd 8.507568e-02
[Update 219] Samples 2048 | Reward mean/std -0.075022/0.174849 | Value mean/std -0.073447/0.137695 | Adv std 1.553822e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.075022 | AvgAdv 0.000000 | AdvAfterStd 9.843624e-02
[Update 220] Samples 2048 | Reward mean/std -0.083426/0.295878 | Value mean/std -0.080169/0.159095 | Adv std 2.339622e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.083426 | AvgAdv 0.000000 | AdvAfterStd 9.302662e-02
[Update 221] Samples 2048 | Reward mean/std -0.084364/0.264831 | Value mean/std -0.092658/0.320677 | Adv std 1.750873e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.084364 | AvgAdv 0.000000 | AdvAfterStd 1.144002e-01
[Update 222] Samples 2048 | Reward mean/std -0.076932/0.162968 | Value mean/std -0.069418/0.100599 | Adv std 1.348673e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.076932 | AvgAdv 0.000000 | AdvAfterStd 1.087758e-01
[Update 223] Samples 2048 | Reward mean/std -0.078985/0.202398 | Value mean/std -0.081884/0.140434 | Adv std 1.595400e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.078985 | AvgAdv 0.000000 | AdvAfterStd 9.789363e-02
[Update 224] Samples 2048 | Reward mean/std -0.085492/0.206063 | Value mean/std -0.072116/0.117839 | Adv std 1.475428e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.085492 | AvgAdv 0.000000 | AdvAfterStd 1.122441e-01
[Update 225] Samples 2048 | Reward mean/std -0.077863/0.170830 | Value mean/std -0.107821/0.167235 | Adv std 1.534870e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.077863 | AvgAdv 0.000000 | AdvAfterStd 9.755478e-02
[Update 226] Samples 2048 | Reward mean/std -0.092437/0.326198 | Value mean/std -0.081275/0.161632 | Adv std 2.214168e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.092437 | AvgAdv 0.000000 | AdvAfterStd 1.353856e-01
[Update 227] Samples 2048 | Reward mean/std -0.080107/0.293405 | Value mean/std -0.075515/0.314166 | Adv std 1.274211e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.080107 | AvgAdv 0.000000 | AdvAfterStd 1.048773e-01
[Update 228] Samples 2048 | Reward mean/std -0.074991/0.140754 | Value mean/std -0.073439/0.131892 | Adv std 9.363434e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.074991 | AvgAdv 0.000000 | AdvAfterStd 7.285302e-02
[Update 229] Samples 2048 | Reward mean/std -0.077176/0.298377 | Value mean/std -0.079404/0.284506 | Adv std 1.298430e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.077176 | AvgAdv -0.000000 | AdvAfterStd 8.544557e-02
[Update 230] Samples 2048 | Reward mean/std -0.077154/0.150715 | Value mean/std -0.069339/0.111817 | Adv std 1.199856e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.077154 | AvgAdv 0.000000 | AdvAfterStd 9.253155e-02
[Update 231] Samples 2048 | Reward mean/std -0.085635/0.234880 | Value mean/std -0.079125/0.179351 | Adv std 1.368967e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.085635 | AvgAdv -0.000000 | AdvAfterStd 1.099194e-01
[Update 232] Samples 2048 | Reward mean/std -0.076462/0.180892 | Value mean/std -0.082362/0.207343 | Adv std 1.938981e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.076462 | AvgAdv -0.000000 | AdvAfterStd 9.696233e-02
[Update 233] Samples 2048 | Reward mean/std -0.072454/0.243711 | Value mean/std -0.076511/0.163852 | Adv std 1.771769e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.072454 | AvgAdv -0.000000 | AdvAfterStd 8.663687e-02
[Update 234] Samples 2048 | Reward mean/std -0.090772/0.340965 | Value mean/std -0.078804/0.156991 | Adv std 2.281622e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.090772 | AvgAdv 0.000000 | AdvAfterStd 1.507300e-01
[Update 235] Samples 2048 | Reward mean/std -0.080707/0.260734 | Value mean/std -0.082536/0.250155 | Adv std 1.344292e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.080707 | AvgAdv 0.000000 | AdvAfterStd 1.115905e-01
[Update 236] Samples 2048 | Reward mean/std -0.081888/0.226778 | Value mean/std -0.089630/0.255625 | Adv std 1.253094e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.081888 | AvgAdv -0.000000 | AdvAfterStd 9.549048e-02
[Update 237] Samples 2048 | Reward mean/std -0.078591/0.190417 | Value mean/std -0.081834/0.123099 | Adv std 1.462506e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.078591 | AvgAdv 0.000000 | AdvAfterStd 1.077879e-01
[Update 238] Samples 2048 | Reward mean/std -0.095591/0.366268 | Value mean/std -0.080413/0.244105 | Adv std 1.835340e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.095591 | AvgAdv -0.000000 | AdvAfterStd 1.319799e-01
[Update 239] Samples 2048 | Reward mean/std -0.083512/0.289446 | Value mean/std -0.083497/0.226544 | Adv std 1.524467e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.083512 | AvgAdv -0.000000 | AdvAfterStd 1.108788e-01
[Update 240] Samples 2048 | Reward mean/std -0.084503/0.276342 | Value mean/std -0.100044/0.259609 | Adv std 1.280799e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.084503 | AvgAdv -0.000000 | AdvAfterStd 1.059152e-01
[Update 241] Samples 2048 | Reward mean/std -0.076888/0.175579 | Value mean/std -0.071005/0.151049 | Adv std 1.270910e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.076888 | AvgAdv -0.000000 | AdvAfterStd 9.588376e-02
[Update 242] Samples 2048 | Reward mean/std -0.079442/0.196347 | Value mean/std -0.087006/0.221535 | Adv std 2.003311e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.079442 | AvgAdv 0.000000 | AdvAfterStd 1.058553e-01
[Update 243] Samples 2048 | Reward mean/std -0.082017/0.205713 | Value mean/std -0.077829/0.128679 | Adv std 1.503222e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.082017 | AvgAdv 0.000000 | AdvAfterStd 1.027060e-01
[Update 244] Samples 2048 | Reward mean/std -0.075242/0.133529 | Value mean/std -0.081157/0.125197 | Adv std 1.414545e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.075242 | AvgAdv 0.000000 | AdvAfterStd 9.653830e-02
[Update 245] Samples 2048 | Reward mean/std -0.080075/0.216959 | Value mean/std -0.073415/0.133697 | Adv std 1.635733e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.080075 | AvgAdv -0.000000 | AdvAfterStd 1.081631e-01
[Update 246] Samples 2048 | Reward mean/std -0.082697/0.255666 | Value mean/std -0.080722/0.223931 | Adv std 1.421910e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.082697 | AvgAdv 0.000000 | AdvAfterStd 1.112356e-01
[Update 247] Samples 2048 | Reward mean/std -0.086849/0.263664 | Value mean/std -0.088802/0.175744 | Adv std 1.766571e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.086849 | AvgAdv 0.000000 | AdvAfterStd 1.034989e-01
[Update 248] Samples 2048 | Reward mean/std -0.083804/0.342529 | Value mean/std -0.083910/0.217391 | Adv std 2.065636e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.083804 | AvgAdv 0.000000 | AdvAfterStd 1.003695e-01
[Update 249] Samples 2048 | Reward mean/std -0.085177/0.182158 | Value mean/std -0.079032/0.169843 | Adv std 1.646221e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.085177 | AvgAdv -0.000000 | AdvAfterStd 9.716129e-02
[Update 250] Samples 2048 | Reward mean/std -0.077449/0.175275 | Value mean/std -0.079349/0.146274 | Adv std 1.136448e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.077449 | AvgAdv 0.000000 | AdvAfterStd 7.986119e-02
Saved checkpoint at update 250
[Update 251] Samples 2048 | Reward mean/std -0.078287/0.193641 | Value mean/std -0.075771/0.142295 | Adv std 1.324532e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.078287 | AvgAdv -0.000000 | AdvAfterStd 1.014877e-01
[Update 252] Samples 2048 | Reward mean/std -0.078680/0.151427 | Value mean/std -0.076095/0.103228 | Adv std 1.220939e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.078680 | AvgAdv -0.000000 | AdvAfterStd 9.286419e-02
[Update 253] Samples 2048 | Reward mean/std -0.083226/0.224924 | Value mean/std -0.086849/0.139294 | Adv std 1.723690e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.083226 | AvgAdv -0.000000 | AdvAfterStd 1.058640e-01
[Update 254] Samples 2048 | Reward mean/std -0.078932/0.184112 | Value mean/std -0.081926/0.163063 | Adv std 1.348873e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.078932 | AvgAdv -0.000000 | AdvAfterStd 9.740239e-02
[Update 255] Samples 2048 | Reward mean/std -0.077774/0.166566 | Value mean/std -0.079805/0.149083 | Adv std 1.309484e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.077774 | AvgAdv -0.000000 | AdvAfterStd 9.440930e-02
[Update 256] Samples 2048 | Reward mean/std -0.083645/0.222366 | Value mean/std -0.090732/0.176274 | Adv std 2.005208e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.083645 | AvgAdv 0.000000 | AdvAfterStd 1.179052e-01
[Update 257] Samples 2048 | Reward mean/std -0.074293/0.183696 | Value mean/std -0.063283/0.135272 | Adv std 1.223258e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.074293 | AvgAdv -0.000000 | AdvAfterStd 9.011485e-02
[Update 258] Samples 2048 | Reward mean/std -0.080338/0.209230 | Value mean/std -0.073014/0.120220 | Adv std 1.580565e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.080338 | AvgAdv 0.000000 | AdvAfterStd 1.039802e-01
[Update 259] Samples 2048 | Reward mean/std -0.083398/0.208290 | Value mean/std -0.089196/0.172564 | Adv std 1.522771e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.083398 | AvgAdv -0.000000 | AdvAfterStd 1.037376e-01
[Update 260] Samples 2048 | Reward mean/std -0.077672/0.199257 | Value mean/std -0.079194/0.139104 | Adv std 1.197448e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.077672 | AvgAdv 0.000000 | AdvAfterStd 9.068811e-02
[Update 261] Samples 2048 | Reward mean/std -0.085795/0.242251 | Value mean/std -0.086029/0.190461 | Adv std 1.471487e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.085795 | AvgAdv 0.000000 | AdvAfterStd 1.210197e-01
[Update 262] Samples 2048 | Reward mean/std -0.083033/0.195305 | Value mean/std -0.100675/0.174429 | Adv std 1.375764e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.083033 | AvgAdv 0.000000 | AdvAfterStd 9.603476e-02
[Update 263] Samples 2048 | Reward mean/std -0.080653/0.199836 | Value mean/std -0.086967/0.156608 | Adv std 1.261821e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.080653 | AvgAdv 0.000000 | AdvAfterStd 9.806575e-02
[Update 264] Samples 2048 | Reward mean/std -0.085223/0.224086 | Value mean/std -0.076425/0.159493 | Adv std 1.348675e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.085223 | AvgAdv -0.000000 | AdvAfterStd 1.004099e-01
[Update 265] Samples 2048 | Reward mean/std -0.078825/0.148219 | Value mean/std -0.080556/0.170721 | Adv std 1.512975e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.078825 | AvgAdv -0.000000 | AdvAfterStd 8.807336e-02
[Update 266] Samples 2048 | Reward mean/std -0.075366/0.140579 | Value mean/std -0.083166/0.137629 | Adv std 1.141736e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.075366 | AvgAdv 0.000000 | AdvAfterStd 8.064411e-02
[Update 267] Samples 2048 | Reward mean/std -0.075266/0.194420 | Value mean/std -0.072491/0.102423 | Adv std 1.714996e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.075266 | AvgAdv 0.000000 | AdvAfterStd 9.927550e-02
[Update 268] Samples 2048 | Reward mean/std -0.085488/0.324719 | Value mean/std -0.087597/0.302669 | Adv std 1.653985e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.085488 | AvgAdv -0.000000 | AdvAfterStd 8.539975e-02
[Update 269] Samples 2048 | Reward mean/std -0.083508/0.276013 | Value mean/std -0.084729/0.246667 | Adv std 1.300194e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.083508 | AvgAdv -0.000000 | AdvAfterStd 9.771519e-02
[Update 270] Samples 2048 | Reward mean/std -0.073427/0.195716 | Value mean/std -0.082627/0.140305 | Adv std 1.271362e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.073427 | AvgAdv 0.000000 | AdvAfterStd 8.767412e-02
[Update 271] Samples 2048 | Reward mean/std -0.077841/0.278592 | Value mean/std -0.075612/0.260637 | Adv std 1.259125e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.077841 | AvgAdv -0.000000 | AdvAfterStd 1.220826e-01
[Update 272] Samples 2048 | Reward mean/std -0.076177/0.166706 | Value mean/std -0.076819/0.162424 | Adv std 1.405374e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.076177 | AvgAdv -0.000000 | AdvAfterStd 1.063980e-01
[Update 273] Samples 2048 | Reward mean/std -0.075612/0.160567 | Value mean/std -0.078410/0.092322 | Adv std 1.216446e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.075612 | AvgAdv -0.000000 | AdvAfterStd 9.720523e-02
[Update 274] Samples 2048 | Reward mean/std -0.075335/0.148676 | Value mean/std -0.083372/0.107256 | Adv std 1.219436e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.075335 | AvgAdv -0.000000 | AdvAfterStd 9.718078e-02
[Update 275] Samples 2048 | Reward mean/std -0.074327/0.173252 | Value mean/std -0.086774/0.091061 | Adv std 1.441390e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.074327 | AvgAdv 0.000000 | AdvAfterStd 1.086093e-01
[Update 276] Samples 2048 | Reward mean/std -0.078180/0.211528 | Value mean/std -0.074833/0.132170 | Adv std 1.261718e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.078180 | AvgAdv 0.000000 | AdvAfterStd 8.044394e-02
[Update 277] Samples 2048 | Reward mean/std -0.073021/0.165792 | Value mean/std -0.074034/0.127966 | Adv std 1.053608e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.073021 | AvgAdv -0.000000 | AdvAfterStd 8.810671e-02
[Update 278] Samples 2048 | Reward mean/std -0.070782/0.192297 | Value mean/std -0.065872/0.115706 | Adv std 1.498915e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.070782 | AvgAdv 0.000000 | AdvAfterStd 9.462465e-02
[Update 279] Samples 2048 | Reward mean/std -0.080670/0.208331 | Value mean/std -0.076529/0.111722 | Adv std 1.724072e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.080670 | AvgAdv -0.000000 | AdvAfterStd 1.095982e-01
[Update 280] Samples 2048 | Reward mean/std -0.077270/0.180015 | Value mean/std -0.085080/0.148173 | Adv std 1.457887e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.077270 | AvgAdv 0.000000 | AdvAfterStd 9.674811e-02
[Update 281] Samples 2048 | Reward mean/std -0.070949/0.224359 | Value mean/std -0.074685/0.130721 | Adv std 1.604941e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.070949 | AvgAdv 0.000000 | AdvAfterStd 8.053621e-02
[Update 282] Samples 2048 | Reward mean/std -0.071276/0.130078 | Value mean/std -0.063500/0.108451 | Adv std 1.205018e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.071276 | AvgAdv -0.000000 | AdvAfterStd 8.893611e-02
[Update 283] Samples 2048 | Reward mean/std -0.073325/0.144211 | Value mean/std -0.066200/0.093951 | Adv std 1.142436e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.073325 | AvgAdv -0.000000 | AdvAfterStd 8.151244e-02
[Update 284] Samples 2048 | Reward mean/std -0.079738/0.248302 | Value mean/std -0.084519/0.176755 | Adv std 1.610088e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.079738 | AvgAdv -0.000000 | AdvAfterStd 9.103538e-02
[Update 285] Samples 2048 | Reward mean/std -0.069285/0.120999 | Value mean/std -0.074713/0.094264 | Adv std 1.080071e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.069285 | AvgAdv 0.000000 | AdvAfterStd 8.370774e-02
[Update 286] Samples 2048 | Reward mean/std -0.084574/0.291930 | Value mean/std -0.082080/0.106515 | Adv std 2.257354e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.084574 | AvgAdv 0.000000 | AdvAfterStd 1.155967e-01
[Update 287] Samples 2048 | Reward mean/std -0.079896/0.216851 | Value mean/std -0.072922/0.153224 | Adv std 1.497688e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.079896 | AvgAdv 0.000000 | AdvAfterStd 9.622578e-02
[Update 288] Samples 2048 | Reward mean/std -0.071188/0.144821 | Value mean/std -0.072309/0.109985 | Adv std 1.117186e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.071188 | AvgAdv 0.000000 | AdvAfterStd 8.618043e-02
[Update 289] Samples 2048 | Reward mean/std -0.080805/0.306871 | Value mean/std -0.082950/0.242359 | Adv std 2.667460e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.080805 | AvgAdv 0.000000 | AdvAfterStd 1.588242e-01
[Update 290] Samples 2048 | Reward mean/std -0.068250/0.195768 | Value mean/std -0.075843/0.135189 | Adv std 1.606044e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.068250 | AvgAdv -0.000000 | AdvAfterStd 1.075166e-01
[Update 291] Samples 2048 | Reward mean/std -0.087735/0.385790 | Value mean/std -0.085600/0.355286 | Adv std 1.504270e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.087735 | AvgAdv -0.000000 | AdvAfterStd 1.164642e-01
[Update 292] Samples 2048 | Reward mean/std -0.085597/0.336431 | Value mean/std -0.080599/0.335278 | Adv std 1.462336e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.085597 | AvgAdv -0.000000 | AdvAfterStd 1.703586e-01
[Update 293] Samples 2048 | Reward mean/std -0.076876/0.176310 | Value mean/std -0.080048/0.161711 | Adv std 1.227130e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.076876 | AvgAdv 0.000000 | AdvAfterStd 9.842578e-02
[Update 294] Samples 2048 | Reward mean/std -0.072255/0.159780 | Value mean/std -0.079585/0.115868 | Adv std 1.238747e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.072255 | AvgAdv 0.000000 | AdvAfterStd 8.382332e-02
[Update 295] Samples 2048 | Reward mean/std -0.074552/0.174542 | Value mean/std -0.068253/0.154758 | Adv std 1.146176e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.074552 | AvgAdv -0.000000 | AdvAfterStd 7.843078e-02
[Update 296] Samples 2048 | Reward mean/std -0.068758/0.177536 | Value mean/std -0.073974/0.125824 | Adv std 1.421840e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.068758 | AvgAdv -0.000000 | AdvAfterStd 1.103463e-01
[Update 297] Samples 2048 | Reward mean/std -0.076266/0.207198 | Value mean/std -0.072134/0.117095 | Adv std 1.522082e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.076266 | AvgAdv -0.000000 | AdvAfterStd 1.215767e-01
[Update 298] Samples 2048 | Reward mean/std -0.072615/0.181815 | Value mean/std -0.086923/0.129243 | Adv std 1.262148e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.072615 | AvgAdv 0.000000 | AdvAfterStd 9.327657e-02
[Update 299] Samples 2048 | Reward mean/std -0.085830/0.256957 | Value mean/std -0.072067/0.150547 | Adv std 1.593926e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.085830 | AvgAdv 0.000000 | AdvAfterStd 1.129407e-01
[Update 300] Samples 2048 | Reward mean/std -0.075222/0.184636 | Value mean/std -0.074656/0.141168 | Adv std 1.615943e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.075222 | AvgAdv 0.000000 | AdvAfterStd 1.055404e-01
Saved checkpoint at update 300
[Update 301] Samples 2048 | Reward mean/std -0.071676/0.150995 | Value mean/std -0.081288/0.128774 | Adv std 1.095828e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.071676 | AvgAdv -0.000000 | AdvAfterStd 8.350414e-02
[Update 302] Samples 2048 | Reward mean/std -0.080122/0.244491 | Value mean/std -0.086565/0.124813 | Adv std 1.726002e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.080122 | AvgAdv -0.000000 | AdvAfterStd 1.168663e-01
[Update 303] Samples 2048 | Reward mean/std -0.080676/0.258011 | Value mean/std -0.069397/0.187747 | Adv std 1.475513e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.080676 | AvgAdv 0.000000 | AdvAfterStd 9.775984e-02
[Update 304] Samples 2048 | Reward mean/std -0.071047/0.144766 | Value mean/std -0.080379/0.147810 | Adv std 1.240865e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.071047 | AvgAdv -0.000000 | AdvAfterStd 8.189695e-02
[Update 305] Samples 2048 | Reward mean/std -0.071857/0.150687 | Value mean/std -0.067032/0.101297 | Adv std 1.113298e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.071857 | AvgAdv 0.000000 | AdvAfterStd 7.811837e-02
[Update 306] Samples 2048 | Reward mean/std -0.073283/0.166490 | Value mean/std -0.076265/0.102841 | Adv std 1.286673e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.073283 | AvgAdv -0.000000 | AdvAfterStd 1.022712e-01
[Update 307] Samples 2048 | Reward mean/std -0.069883/0.169312 | Value mean/std -0.084117/0.112756 | Adv std 1.366547e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.069883 | AvgAdv -0.000000 | AdvAfterStd 8.873146e-02
[Update 308] Samples 2048 | Reward mean/std -0.073423/0.170334 | Value mean/std -0.076629/0.127402 | Adv std 1.195083e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.073423 | AvgAdv -0.000000 | AdvAfterStd 8.920034e-02
[Update 309] Samples 2048 | Reward mean/std -0.070123/0.150872 | Value mean/std -0.073574/0.111844 | Adv std 1.120782e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.070123 | AvgAdv -0.000000 | AdvAfterStd 8.534804e-02
[Update 310] Samples 2048 | Reward mean/std -0.072530/0.177293 | Value mean/std -0.073244/0.117691 | Adv std 1.128327e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.072530 | AvgAdv 0.000000 | AdvAfterStd 8.606333e-02
[Update 311] Samples 2048 | Reward mean/std -0.075136/0.205003 | Value mean/std -0.068908/0.175348 | Adv std 1.189016e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.075136 | AvgAdv -0.000000 | AdvAfterStd 1.010908e-01
[Update 312] Samples 2048 | Reward mean/std -0.074943/0.181766 | Value mean/std -0.073968/0.144601 | Adv std 1.593691e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.074943 | AvgAdv -0.000000 | AdvAfterStd 8.704273e-02
[Update 313] Samples 2048 | Reward mean/std -0.075055/0.168776 | Value mean/std -0.073020/0.165980 | Adv std 1.300456e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.075055 | AvgAdv -0.000000 | AdvAfterStd 9.590068e-02
[Update 314] Samples 2048 | Reward mean/std -0.071360/0.142365 | Value mean/std -0.077294/0.111994 | Adv std 1.155140e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.071360 | AvgAdv 0.000000 | AdvAfterStd 8.413513e-02
[Update 315] Samples 2048 | Reward mean/std -0.064370/0.129330 | Value mean/std -0.067207/0.091533 | Adv std 1.211627e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.064370 | AvgAdv -0.000000 | AdvAfterStd 8.557532e-02
[Update 316] Samples 2048 | Reward mean/std -0.078961/0.380400 | Value mean/std -0.069800/0.145717 | Adv std 2.556397e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.078961 | AvgAdv -0.000000 | AdvAfterStd 8.816282e-02
[Update 317] Samples 2048 | Reward mean/std -0.067678/0.121444 | Value mean/std -0.071790/0.138938 | Adv std 1.207910e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.067678 | AvgAdv 0.000000 | AdvAfterStd 7.962666e-02
[Update 318] Samples 2048 | Reward mean/std -0.073366/0.181095 | Value mean/std -0.062612/0.095866 | Adv std 1.379514e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.073366 | AvgAdv 0.000000 | AdvAfterStd 1.088372e-01
[Update 319] Samples 2048 | Reward mean/std -0.069636/0.151280 | Value mean/std -0.066123/0.096843 | Adv std 1.294748e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.069636 | AvgAdv 0.000000 | AdvAfterStd 9.027000e-02
[Update 320] Samples 2048 | Reward mean/std -0.071374/0.148562 | Value mean/std -0.080491/0.115178 | Adv std 1.226832e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.071374 | AvgAdv 0.000000 | AdvAfterStd 9.816784e-02
[Update 321] Samples 2048 | Reward mean/std -0.075007/0.189027 | Value mean/std -0.081046/0.116289 | Adv std 1.357472e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.075007 | AvgAdv 0.000000 | AdvAfterStd 8.585322e-02
[Update 322] Samples 2048 | Reward mean/std -0.073019/0.259746 | Value mean/std -0.069369/0.117355 | Adv std 1.990957e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.073019 | AvgAdv -0.000000 | AdvAfterStd 8.334039e-02
[Update 323] Samples 2048 | Reward mean/std -0.078861/0.250269 | Value mean/std -0.077479/0.144932 | Adv std 1.799498e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.078861 | AvgAdv -0.000000 | AdvAfterStd 9.823173e-02
[Update 324] Samples 2048 | Reward mean/std -0.071328/0.214758 | Value mean/std -0.079526/0.129627 | Adv std 1.438901e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.071328 | AvgAdv -0.000000 | AdvAfterStd 1.017710e-01
[Update 325] Samples 2048 | Reward mean/std -0.072852/0.197885 | Value mean/std -0.074049/0.142057 | Adv std 1.347156e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.072852 | AvgAdv -0.000000 | AdvAfterStd 1.143055e-01
[Update 326] Samples 2048 | Reward mean/std -0.072967/0.191935 | Value mean/std -0.076543/0.191654 | Adv std 1.473498e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.072967 | AvgAdv 0.000000 | AdvAfterStd 8.048669e-02
[Update 327] Samples 2048 | Reward mean/std -0.069940/0.161863 | Value mean/std -0.063981/0.135625 | Adv std 1.100987e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.069940 | AvgAdv 0.000000 | AdvAfterStd 1.017525e-01
[Update 328] Samples 2048 | Reward mean/std -0.079491/0.186894 | Value mean/std -0.070581/0.131433 | Adv std 1.422071e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.079491 | AvgAdv 0.000000 | AdvAfterStd 9.864804e-02
[Update 329] Samples 2048 | Reward mean/std -0.078427/0.189534 | Value mean/std -0.091114/0.147516 | Adv std 1.265637e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.078427 | AvgAdv -0.000000 | AdvAfterStd 8.931420e-02
[Update 330] Samples 2048 | Reward mean/std -0.081576/0.219141 | Value mean/std -0.076217/0.161025 | Adv std 1.424481e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.081576 | AvgAdv 0.000000 | AdvAfterStd 9.463645e-02
[Update 331] Samples 2048 | Reward mean/std -0.070027/0.138919 | Value mean/std -0.079026/0.137214 | Adv std 1.398291e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.070027 | AvgAdv -0.000000 | AdvAfterStd 9.560572e-02
[Update 332] Samples 2048 | Reward mean/std -0.073809/0.187805 | Value mean/std -0.076806/0.147445 | Adv std 1.145701e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.073809 | AvgAdv 0.000000 | AdvAfterStd 9.451710e-02
[Update 333] Samples 2048 | Reward mean/std -0.074609/0.192057 | Value mean/std -0.066515/0.114353 | Adv std 1.215955e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.074609 | AvgAdv 0.000000 | AdvAfterStd 8.385606e-02
[Update 334] Samples 2048 | Reward mean/std -0.073821/0.196004 | Value mean/std -0.091730/0.161145 | Adv std 1.311530e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.073821 | AvgAdv -0.000000 | AdvAfterStd 8.227883e-02
[Update 335] Samples 2048 | Reward mean/std -0.073163/0.314701 | Value mean/std -0.076057/0.151363 | Adv std 2.043273e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.073163 | AvgAdv -0.000000 | AdvAfterStd 8.764465e-02
[Update 336] Samples 2048 | Reward mean/std -0.069702/0.183237 | Value mean/std -0.070057/0.149404 | Adv std 1.240548e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.069702 | AvgAdv 0.000000 | AdvAfterStd 8.270051e-02
[Update 337] Samples 2048 | Reward mean/std -0.073052/0.156309 | Value mean/std -0.060020/0.104476 | Adv std 1.212366e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.073052 | AvgAdv -0.000000 | AdvAfterStd 8.643656e-02
[Update 338] Samples 2048 | Reward mean/std -0.080484/0.236688 | Value mean/std -0.072870/0.148320 | Adv std 1.459543e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.080484 | AvgAdv 0.000000 | AdvAfterStd 9.004984e-02
[Update 339] Samples 2048 | Reward mean/std -0.068476/0.158530 | Value mean/std -0.070088/0.182757 | Adv std 1.403148e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.068476 | AvgAdv -0.000000 | AdvAfterStd 8.144834e-02
[Update 340] Samples 2048 | Reward mean/std -0.076999/0.221941 | Value mean/std -0.069971/0.131361 | Adv std 1.501024e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.076999 | AvgAdv -0.000000 | AdvAfterStd 1.017349e-01
[Update 341] Samples 2048 | Reward mean/std -0.074247/0.182514 | Value mean/std -0.077636/0.143732 | Adv std 1.306211e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.074247 | AvgAdv 0.000000 | AdvAfterStd 9.518194e-02
[Update 342] Samples 2048 | Reward mean/std -0.069329/0.147280 | Value mean/std -0.064585/0.133543 | Adv std 1.101632e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.069329 | AvgAdv 0.000000 | AdvAfterStd 8.617901e-02
[Update 343] Samples 2048 | Reward mean/std -0.071598/0.171322 | Value mean/std -0.070372/0.128490 | Adv std 1.272826e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.071598 | AvgAdv -0.000000 | AdvAfterStd 9.098794e-02
[Update 344] Samples 2048 | Reward mean/std -0.070712/0.135535 | Value mean/std -0.061049/0.103093 | Adv std 1.261119e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.070712 | AvgAdv -0.000000 | AdvAfterStd 7.601468e-02
[Update 345] Samples 2048 | Reward mean/std -0.078474/0.227650 | Value mean/std -0.071834/0.117489 | Adv std 1.683143e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.078474 | AvgAdv -0.000000 | AdvAfterStd 1.005569e-01
[Update 346] Samples 2048 | Reward mean/std -0.077148/0.167103 | Value mean/std -0.084504/0.130498 | Adv std 1.512968e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.077148 | AvgAdv -0.000000 | AdvAfterStd 9.249771e-02
[Update 347] Samples 2048 | Reward mean/std -0.080746/0.199375 | Value mean/std -0.079057/0.115248 | Adv std 1.422316e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.080746 | AvgAdv 0.000000 | AdvAfterStd 1.060380e-01
[Update 348] Samples 2048 | Reward mean/std -0.070980/0.132307 | Value mean/std -0.077220/0.101940 | Adv std 1.144655e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.070980 | AvgAdv -0.000000 | AdvAfterStd 9.779099e-02
[Update 349] Samples 2048 | Reward mean/std -0.079618/0.203818 | Value mean/std -0.079764/0.155490 | Adv std 1.266730e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.079618 | AvgAdv 0.000000 | AdvAfterStd 9.257527e-02
[Update 350] Samples 2048 | Reward mean/std -0.074939/0.164712 | Value mean/std -0.075351/0.119092 | Adv std 1.120857e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.074939 | AvgAdv 0.000000 | AdvAfterStd 1.007833e-01
Saved checkpoint at update 350
[Update 351] Samples 2048 | Reward mean/std -0.074411/0.198685 | Value mean/std -0.082070/0.156784 | Adv std 1.320927e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.074411 | AvgAdv 0.000000 | AdvAfterStd 9.368619e-02
[Update 352] Samples 2048 | Reward mean/std -0.069793/0.145119 | Value mean/std -0.081449/0.166637 | Adv std 1.590451e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.069793 | AvgAdv -0.000000 | AdvAfterStd 9.099386e-02
[Update 353] Samples 2048 | Reward mean/std -0.070962/0.263980 | Value mean/std -0.059416/0.095280 | Adv std 2.292217e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.070962 | AvgAdv 0.000000 | AdvAfterStd 1.264238e-01
[Update 354] Samples 2048 | Reward mean/std -0.081125/0.267833 | Value mean/std -0.078131/0.160017 | Adv std 1.620901e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.081125 | AvgAdv -0.000000 | AdvAfterStd 9.381295e-02
[Update 355] Samples 2048 | Reward mean/std -0.083537/0.216305 | Value mean/std -0.082717/0.211380 | Adv std 1.621647e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.083537 | AvgAdv 0.000000 | AdvAfterStd 9.031206e-02
[Update 356] Samples 2048 | Reward mean/std -0.071660/0.155355 | Value mean/std -0.075772/0.121032 | Adv std 1.132193e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.071660 | AvgAdv -0.000000 | AdvAfterStd 9.082943e-02
[Update 357] Samples 2048 | Reward mean/std -0.082610/0.234664 | Value mean/std -0.090093/0.184358 | Adv std 1.517913e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.082610 | AvgAdv -0.000000 | AdvAfterStd 9.271235e-02
[Update 358] Samples 2048 | Reward mean/std -0.070999/0.147021 | Value mean/std -0.075285/0.118201 | Adv std 9.666009e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.070999 | AvgAdv 0.000000 | AdvAfterStd 7.898910e-02
[Update 359] Samples 2048 | Reward mean/std -0.075491/0.227636 | Value mean/std -0.074668/0.168480 | Adv std 1.367649e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.075491 | AvgAdv 0.000000 | AdvAfterStd 9.403659e-02
[Update 360] Samples 2048 | Reward mean/std -0.071111/0.143137 | Value mean/std -0.084078/0.094828 | Adv std 1.176477e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.071111 | AvgAdv 0.000000 | AdvAfterStd 9.452344e-02
[Update 361] Samples 2048 | Reward mean/std -0.077185/0.214193 | Value mean/std -0.072161/0.141297 | Adv std 1.707250e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.077185 | AvgAdv 0.000000 | AdvAfterStd 1.013112e-01
[Update 362] Samples 2048 | Reward mean/std -0.082329/0.203457 | Value mean/std -0.085394/0.142313 | Adv std 1.428728e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.082329 | AvgAdv -0.000000 | AdvAfterStd 8.144071e-02
[Update 363] Samples 2048 | Reward mean/std -0.079063/0.197362 | Value mean/std -0.075509/0.136268 | Adv std 1.333621e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.079063 | AvgAdv 0.000000 | AdvAfterStd 8.816326e-02
[Update 364] Samples 2048 | Reward mean/std -0.084639/0.256690 | Value mean/std -0.090763/0.208180 | Adv std 1.613068e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.084639 | AvgAdv 0.000000 | AdvAfterStd 1.096065e-01
[Update 365] Samples 2048 | Reward mean/std -0.084852/0.326211 | Value mean/std -0.086043/0.243967 | Adv std 1.715787e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.084852 | AvgAdv -0.000000 | AdvAfterStd 9.938677e-02
[Update 366] Samples 2048 | Reward mean/std -0.083181/0.256611 | Value mean/std -0.075059/0.219819 | Adv std 1.627628e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.083181 | AvgAdv 0.000000 | AdvAfterStd 1.043574e-01
[Update 367] Samples 2048 | Reward mean/std -0.077380/0.289815 | Value mean/std -0.082747/0.261277 | Adv std 1.041234e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.077380 | AvgAdv -0.000000 | AdvAfterStd 8.351590e-02
[Update 368] Samples 2048 | Reward mean/std -0.075841/0.220215 | Value mean/std -0.079178/0.150011 | Adv std 1.254944e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.075841 | AvgAdv 0.000000 | AdvAfterStd 8.880188e-02
[Update 369] Samples 2048 | Reward mean/std -0.077065/0.291593 | Value mean/std -0.082954/0.147562 | Adv std 1.954249e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.077065 | AvgAdv 0.000000 | AdvAfterStd 8.837581e-02
[Update 370] Samples 2048 | Reward mean/std -0.083014/0.296100 | Value mean/std -0.073754/0.291651 | Adv std 1.579060e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.083014 | AvgAdv -0.000000 | AdvAfterStd 1.044116e-01
[Update 371] Samples 2048 | Reward mean/std -0.074764/0.168691 | Value mean/std -0.086992/0.258218 | Adv std 1.934638e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.074764 | AvgAdv 0.000000 | AdvAfterStd 9.429237e-02
[Update 372] Samples 2048 | Reward mean/std -0.073546/0.245740 | Value mean/std -0.075600/0.206897 | Adv std 1.222102e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.073546 | AvgAdv 0.000000 | AdvAfterStd 8.768856e-02
[Update 373] Samples 2048 | Reward mean/std -0.069955/0.166662 | Value mean/std -0.077058/0.167502 | Adv std 1.246450e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.069955 | AvgAdv -0.000000 | AdvAfterStd 7.978665e-02
[Update 374] Samples 2048 | Reward mean/std -0.077926/0.184659 | Value mean/std -0.071678/0.140314 | Adv std 1.327192e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.077926 | AvgAdv -0.000000 | AdvAfterStd 1.146480e-01
[Update 375] Samples 2048 | Reward mean/std -0.073143/0.283023 | Value mean/std -0.063015/0.165935 | Adv std 1.775931e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.073143 | AvgAdv -0.000000 | AdvAfterStd 1.001011e-01
[Update 376] Samples 2048 | Reward mean/std -0.078811/0.235110 | Value mean/std -0.078939/0.175112 | Adv std 1.385821e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.078811 | AvgAdv -0.000000 | AdvAfterStd 9.691416e-02
[Update 377] Samples 2048 | Reward mean/std -0.072335/0.173957 | Value mean/std -0.075347/0.125862 | Adv std 1.243800e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.072335 | AvgAdv 0.000000 | AdvAfterStd 8.623245e-02
[Update 378] Samples 2048 | Reward mean/std -0.071577/0.150175 | Value mean/std -0.075416/0.117281 | Adv std 1.155091e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.071577 | AvgAdv 0.000000 | AdvAfterStd 8.300742e-02
[Update 379] Samples 2048 | Reward mean/std -0.066686/0.150504 | Value mean/std -0.069937/0.107709 | Adv std 1.068919e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.066686 | AvgAdv -0.000000 | AdvAfterStd 8.369809e-02
[Update 380] Samples 2048 | Reward mean/std -0.074037/0.165177 | Value mean/std -0.073085/0.178669 | Adv std 1.276097e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.074037 | AvgAdv 0.000000 | AdvAfterStd 8.123618e-02
[Update 381] Samples 2048 | Reward mean/std -0.071985/0.187645 | Value mean/std -0.072904/0.114158 | Adv std 1.249766e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.071985 | AvgAdv 0.000000 | AdvAfterStd 8.464655e-02
[Update 382] Samples 2048 | Reward mean/std -0.066572/0.126238 | Value mean/std -0.068501/0.102165 | Adv std 1.048216e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.066572 | AvgAdv -0.000000 | AdvAfterStd 7.853696e-02
[Update 383] Samples 2048 | Reward mean/std -0.075248/0.206994 | Value mean/std -0.069027/0.148969 | Adv std 1.239980e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.075248 | AvgAdv 0.000000 | AdvAfterStd 8.642868e-02
[Update 384] Samples 2048 | Reward mean/std -0.075945/0.179868 | Value mean/std -0.071343/0.121644 | Adv std 1.209048e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.075945 | AvgAdv 0.000000 | AdvAfterStd 9.207677e-02
[Update 385] Samples 2048 | Reward mean/std -0.081108/0.214095 | Value mean/std -0.073901/0.141006 | Adv std 1.393513e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.081108 | AvgAdv 0.000000 | AdvAfterStd 9.132059e-02
[Update 386] Samples 2048 | Reward mean/std -0.079695/0.289279 | Value mean/std -0.078584/0.252271 | Adv std 1.218074e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.079695 | AvgAdv -0.000000 | AdvAfterStd 8.979806e-02
[Update 387] Samples 2048 | Reward mean/std -0.072430/0.196951 | Value mean/std -0.078949/0.152769 | Adv std 1.135607e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.072430 | AvgAdv -0.000000 | AdvAfterStd 8.737034e-02
[Update 388] Samples 2048 | Reward mean/std -0.073859/0.183052 | Value mean/std -0.073812/0.163645 | Adv std 1.304235e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.073859 | AvgAdv -0.000000 | AdvAfterStd 9.310991e-02
[Update 389] Samples 2048 | Reward mean/std -0.077628/0.184939 | Value mean/std -0.084139/0.150576 | Adv std 1.159501e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.077628 | AvgAdv 0.000000 | AdvAfterStd 8.928119e-02
[Update 390] Samples 2048 | Reward mean/std -0.071698/0.168908 | Value mean/std -0.078622/0.118222 | Adv std 1.020186e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.071698 | AvgAdv 0.000000 | AdvAfterStd 8.532447e-02
[Update 391] Samples 2048 | Reward mean/std -0.074129/0.207226 | Value mean/std -0.078305/0.165125 | Adv std 1.097878e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.074129 | AvgAdv -0.000000 | AdvAfterStd 8.704760e-02
[Update 392] Samples 2048 | Reward mean/std -0.075062/0.271272 | Value mean/std -0.077773/0.166449 | Adv std 1.632805e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.075062 | AvgAdv -0.000000 | AdvAfterStd 8.181991e-02
[Update 393] Samples 2048 | Reward mean/std -0.079914/0.217424 | Value mean/std -0.070267/0.159093 | Adv std 1.415511e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.079914 | AvgAdv 0.000000 | AdvAfterStd 9.838244e-02
[Update 394] Samples 2048 | Reward mean/std -0.072679/0.177914 | Value mean/std -0.077038/0.181108 | Adv std 1.353763e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.072679 | AvgAdv -0.000000 | AdvAfterStd 8.486561e-02
[Update 395] Samples 2048 | Reward mean/std -0.075492/0.216064 | Value mean/std -0.074578/0.137728 | Adv std 1.523615e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.075492 | AvgAdv -0.000000 | AdvAfterStd 9.916447e-02
[Update 396] Samples 2048 | Reward mean/std -0.074493/0.216139 | Value mean/std -0.068178/0.155258 | Adv std 1.444978e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.074493 | AvgAdv -0.000000 | AdvAfterStd 8.441260e-02
[Update 397] Samples 2048 | Reward mean/std -0.075044/0.346309 | Value mean/std -0.083855/0.274930 | Adv std 1.385672e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.075044 | AvgAdv -0.000000 | AdvAfterStd 1.254199e-01
[Update 398] Samples 2048 | Reward mean/std -0.068385/0.129887 | Value mean/std -0.073620/0.119418 | Adv std 1.114491e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.068385 | AvgAdv 0.000000 | AdvAfterStd 7.632347e-02
[Update 399] Samples 2048 | Reward mean/std -0.078295/0.342696 | Value mean/std -0.080596/0.283610 | Adv std 1.603855e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.078295 | AvgAdv 0.000000 | AdvAfterStd 1.467073e-01
[Update 400] Samples 2048 | Reward mean/std -0.070559/0.166121 | Value mean/std -0.068305/0.184384 | Adv std 1.207350e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.070559 | AvgAdv -0.000000 | AdvAfterStd 7.737809e-02
Saved checkpoint at update 400
[Update 401] Samples 2048 | Reward mean/std -0.066814/0.146470 | Value mean/std -0.065965/0.079584 | Adv std 1.228765e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.066814 | AvgAdv 0.000000 | AdvAfterStd 9.377670e-02
[Update 402] Samples 2048 | Reward mean/std -0.073223/0.185781 | Value mean/std -0.067302/0.123673 | Adv std 1.136415e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.073223 | AvgAdv -0.000000 | AdvAfterStd 8.538973e-02
[Update 403] Samples 2048 | Reward mean/std -0.073251/0.260611 | Value mean/std -0.062947/0.127626 | Adv std 1.834943e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.073251 | AvgAdv 0.000000 | AdvAfterStd 8.872806e-02
[Update 404] Samples 2048 | Reward mean/std -0.075731/0.204567 | Value mean/std -0.086860/0.153059 | Adv std 1.161719e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.075731 | AvgAdv 0.000000 | AdvAfterStd 8.432021e-02
[Update 405] Samples 2048 | Reward mean/std -0.068837/0.162282 | Value mean/std -0.056223/0.127636 | Adv std 1.138275e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.068837 | AvgAdv 0.000000 | AdvAfterStd 8.022003e-02
[Update 406] Samples 2048 | Reward mean/std -0.073507/0.197556 | Value mean/std -0.066294/0.161652 | Adv std 1.072185e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.073507 | AvgAdv -0.000000 | AdvAfterStd 9.227883e-02
[Update 407] Samples 2048 | Reward mean/std -0.077192/0.351140 | Value mean/std -0.068940/0.249301 | Adv std 1.416707e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.077192 | AvgAdv 0.000000 | AdvAfterStd 9.842537e-02
[Update 408] Samples 2048 | Reward mean/std -0.076071/0.206964 | Value mean/std -0.072002/0.170168 | Adv std 1.882890e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.076071 | AvgAdv -0.000000 | AdvAfterStd 1.087478e-01
[Update 409] Samples 2048 | Reward mean/std -0.064181/0.126371 | Value mean/std -0.083389/0.101901 | Adv std 1.003970e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.064181 | AvgAdv -0.000000 | AdvAfterStd 7.724894e-02
[Update 410] Samples 2048 | Reward mean/std -0.072252/0.197366 | Value mean/std -0.075788/0.171391 | Adv std 1.123829e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.072252 | AvgAdv 0.000000 | AdvAfterStd 8.893317e-02
[Update 411] Samples 2048 | Reward mean/std -0.082413/0.393871 | Value mean/std -0.075850/0.277822 | Adv std 1.984292e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.082413 | AvgAdv -0.000000 | AdvAfterStd 1.309938e-01
[Update 412] Samples 2048 | Reward mean/std -0.078938/0.362213 | Value mean/std -0.085525/0.394498 | Adv std 1.369119e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.078938 | AvgAdv 0.000000 | AdvAfterStd 7.333614e-02
[Update 413] Samples 2048 | Reward mean/std -0.075123/0.217916 | Value mean/std -0.066680/0.146394 | Adv std 1.417159e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.075123 | AvgAdv 0.000000 | AdvAfterStd 9.120118e-02
[Update 414] Samples 2048 | Reward mean/std -0.070456/0.147958 | Value mean/std -0.075683/0.095081 | Adv std 1.192180e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.070456 | AvgAdv 0.000000 | AdvAfterStd 9.306545e-02
[Update 415] Samples 2048 | Reward mean/std -0.069269/0.179905 | Value mean/std -0.077875/0.122186 | Adv std 1.277958e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.069269 | AvgAdv -0.000000 | AdvAfterStd 8.809114e-02
[Update 416] Samples 2048 | Reward mean/std -0.080505/0.257713 | Value mean/std -0.080237/0.226681 | Adv std 1.244964e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.080505 | AvgAdv -0.000000 | AdvAfterStd 9.160981e-02
[Update 417] Samples 2048 | Reward mean/std -0.069262/0.142392 | Value mean/std -0.074875/0.145220 | Adv std 1.390423e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.069262 | AvgAdv 0.000000 | AdvAfterStd 8.118848e-02
[Update 418] Samples 2048 | Reward mean/std -0.077348/0.184594 | Value mean/std -0.077279/0.112774 | Adv std 1.359093e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.077348 | AvgAdv -0.000000 | AdvAfterStd 1.014198e-01
[Update 419] Samples 2048 | Reward mean/std -0.065940/0.135059 | Value mean/std -0.075546/0.138468 | Adv std 1.280020e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.065940 | AvgAdv 0.000000 | AdvAfterStd 9.465233e-02
[Update 420] Samples 2048 | Reward mean/std -0.073815/0.191932 | Value mean/std -0.070935/0.132950 | Adv std 1.056200e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.073815 | AvgAdv -0.000000 | AdvAfterStd 7.587899e-02
[Update 421] Samples 2048 | Reward mean/std -0.075882/0.241061 | Value mean/std -0.081134/0.199643 | Adv std 1.374429e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.075882 | AvgAdv 0.000000 | AdvAfterStd 1.064808e-01
[Update 422] Samples 2048 | Reward mean/std -0.069993/0.149478 | Value mean/std -0.078232/0.158387 | Adv std 1.318265e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.069993 | AvgAdv -0.000000 | AdvAfterStd 7.745250e-02
[Update 423] Samples 2048 | Reward mean/std -0.071527/0.172073 | Value mean/std -0.065224/0.119236 | Adv std 1.111700e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.071527 | AvgAdv -0.000000 | AdvAfterStd 7.883534e-02
[Update 424] Samples 2048 | Reward mean/std -0.071374/0.205190 | Value mean/std -0.077567/0.121909 | Adv std 1.705992e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.071374 | AvgAdv 0.000000 | AdvAfterStd 8.611307e-02
[Update 425] Samples 2048 | Reward mean/std -0.067690/0.161604 | Value mean/std -0.066909/0.160857 | Adv std 1.352385e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.067690 | AvgAdv 0.000000 | AdvAfterStd 8.682971e-02
[Update 426] Samples 2048 | Reward mean/std -0.071295/0.201173 | Value mean/std -0.074798/0.156896 | Adv std 1.325867e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.071295 | AvgAdv -0.000000 | AdvAfterStd 1.135215e-01
[Update 427] Samples 2048 | Reward mean/std -0.078296/0.282741 | Value mean/std -0.073052/0.173804 | Adv std 1.565346e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.078296 | AvgAdv -0.000000 | AdvAfterStd 9.491803e-02
[Update 428] Samples 2048 | Reward mean/std -0.067675/0.162735 | Value mean/std -0.071359/0.141505 | Adv std 1.263975e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.067675 | AvgAdv 0.000000 | AdvAfterStd 7.800756e-02
[Update 429] Samples 2048 | Reward mean/std -0.067868/0.145359 | Value mean/std -0.063475/0.087740 | Adv std 1.196030e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.067868 | AvgAdv 0.000000 | AdvAfterStd 9.736230e-02
[Update 430] Samples 2048 | Reward mean/std -0.068821/0.158974 | Value mean/std -0.066456/0.109450 | Adv std 1.236660e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.068821 | AvgAdv 0.000000 | AdvAfterStd 8.590265e-02
[Update 431] Samples 2048 | Reward mean/std -0.068235/0.171211 | Value mean/std -0.066528/0.144722 | Adv std 1.234528e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.068235 | AvgAdv 0.000000 | AdvAfterStd 8.610078e-02
[Update 432] Samples 2048 | Reward mean/std -0.078440/0.198859 | Value mean/std -0.073846/0.137466 | Adv std 1.471105e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.078440 | AvgAdv 0.000000 | AdvAfterStd 1.031295e-01
[Update 433] Samples 2048 | Reward mean/std -0.068030/0.140675 | Value mean/std -0.069669/0.113237 | Adv std 1.302684e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.068030 | AvgAdv 0.000000 | AdvAfterStd 9.046008e-02
[Update 434] Samples 2048 | Reward mean/std -0.071491/0.189378 | Value mean/std -0.069388/0.131958 | Adv std 1.250350e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.071491 | AvgAdv -0.000000 | AdvAfterStd 8.062644e-02
[Update 435] Samples 2048 | Reward mean/std -0.074880/0.238633 | Value mean/std -0.064860/0.143904 | Adv std 1.455361e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.074880 | AvgAdv -0.000000 | AdvAfterStd 9.377421e-02
[Update 436] Samples 2048 | Reward mean/std -0.068852/0.161373 | Value mean/std -0.066422/0.174856 | Adv std 1.131112e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.068852 | AvgAdv 0.000000 | AdvAfterStd 7.748791e-02
[Update 437] Samples 2048 | Reward mean/std -0.069091/0.189794 | Value mean/std -0.063468/0.114204 | Adv std 1.136828e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.069091 | AvgAdv 0.000000 | AdvAfterStd 7.747185e-02
[Update 438] Samples 2048 | Reward mean/std -0.081815/0.378780 | Value mean/std -0.081250/0.279603 | Adv std 1.723153e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.081815 | AvgAdv 0.000000 | AdvAfterStd 1.023765e-01
[Update 439] Samples 2048 | Reward mean/std -0.073532/0.175263 | Value mean/std -0.067773/0.114727 | Adv std 1.187991e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.073532 | AvgAdv 0.000000 | AdvAfterStd 8.739256e-02
[Update 440] Samples 2048 | Reward mean/std -0.073724/0.173464 | Value mean/std -0.080192/0.147602 | Adv std 1.193957e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.073724 | AvgAdv 0.000000 | AdvAfterStd 8.724684e-02
[Update 441] Samples 2048 | Reward mean/std -0.070471/0.154829 | Value mean/std -0.072027/0.134803 | Adv std 1.026031e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.070471 | AvgAdv 0.000000 | AdvAfterStd 7.419392e-02
[Update 442] Samples 2048 | Reward mean/std -0.075522/0.202479 | Value mean/std -0.073404/0.122890 | Adv std 1.516436e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.075522 | AvgAdv -0.000000 | AdvAfterStd 1.037174e-01
[Update 443] Samples 2048 | Reward mean/std -0.077159/0.238290 | Value mean/std -0.065287/0.126577 | Adv std 1.681470e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.077159 | AvgAdv 0.000000 | AdvAfterStd 9.721456e-02
[Update 444] Samples 2048 | Reward mean/std -0.070241/0.148061 | Value mean/std -0.072401/0.132687 | Adv std 1.169942e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.070241 | AvgAdv -0.000000 | AdvAfterStd 8.492443e-02
[Update 445] Samples 2048 | Reward mean/std -0.078169/0.216750 | Value mean/std -0.075190/0.166446 | Adv std 1.345186e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.078169 | AvgAdv -0.000000 | AdvAfterStd 9.352232e-02
[Update 446] Samples 2048 | Reward mean/std -0.076205/0.288436 | Value mean/std -0.072671/0.129709 | Adv std 2.088217e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.076205 | AvgAdv 0.000000 | AdvAfterStd 1.208845e-01
[Update 447] Samples 2048 | Reward mean/std -0.069656/0.173232 | Value mean/std -0.065538/0.191549 | Adv std 1.189735e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.069656 | AvgAdv 0.000000 | AdvAfterStd 8.355385e-02
[Update 448] Samples 2048 | Reward mean/std -0.070706/0.176195 | Value mean/std -0.067814/0.117875 | Adv std 1.314227e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.070706 | AvgAdv 0.000000 | AdvAfterStd 9.661825e-02
[Update 449] Samples 2048 | Reward mean/std -0.072600/0.156298 | Value mean/std -0.064810/0.132998 | Adv std 1.241981e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.072600 | AvgAdv -0.000000 | AdvAfterStd 9.118026e-02
[Update 450] Samples 2048 | Reward mean/std -0.067866/0.155094 | Value mean/std -0.072137/0.112043 | Adv std 1.092992e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.067866 | AvgAdv 0.000000 | AdvAfterStd 7.145143e-02
Saved checkpoint at update 450
[Update 451] Samples 2048 | Reward mean/std -0.068109/0.149643 | Value mean/std -0.075022/0.113627 | Adv std 1.147262e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.068109 | AvgAdv 0.000000 | AdvAfterStd 8.705720e-02
[Update 452] Samples 2048 | Reward mean/std -0.068738/0.162229 | Value mean/std -0.069288/0.114821 | Adv std 1.225541e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.068738 | AvgAdv 0.000000 | AdvAfterStd 9.165022e-02
[Update 453] Samples 2048 | Reward mean/std -0.071266/0.170467 | Value mean/std -0.070925/0.160937 | Adv std 1.333028e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.071266 | AvgAdv -0.000000 | AdvAfterStd 9.670837e-02
[Update 454] Samples 2048 | Reward mean/std -0.073263/0.188744 | Value mean/std -0.080729/0.124562 | Adv std 1.430213e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.073263 | AvgAdv 0.000000 | AdvAfterStd 8.024512e-02
[Update 455] Samples 2048 | Reward mean/std -0.072931/0.156314 | Value mean/std -0.070230/0.136131 | Adv std 1.341328e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.072931 | AvgAdv -0.000000 | AdvAfterStd 9.123772e-02
[Update 456] Samples 2048 | Reward mean/std -0.077302/0.183393 | Value mean/std -0.068084/0.120628 | Adv std 1.254986e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.077302 | AvgAdv 0.000000 | AdvAfterStd 8.668227e-02
[Update 457] Samples 2048 | Reward mean/std -0.069468/0.139444 | Value mean/std -0.079389/0.122049 | Adv std 1.092464e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.069468 | AvgAdv 0.000000 | AdvAfterStd 8.682340e-02
[Update 458] Samples 2048 | Reward mean/std -0.082387/0.243139 | Value mean/std -0.086446/0.158812 | Adv std 1.446335e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.082387 | AvgAdv -0.000000 | AdvAfterStd 8.783150e-02
[Update 459] Samples 2048 | Reward mean/std -0.068524/0.139522 | Value mean/std -0.066015/0.104800 | Adv std 1.048852e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.068524 | AvgAdv 0.000000 | AdvAfterStd 7.303160e-02
[Update 460] Samples 2048 | Reward mean/std -0.067528/0.149977 | Value mean/std -0.062893/0.102294 | Adv std 1.022331e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.067528 | AvgAdv -0.000000 | AdvAfterStd 7.479787e-02
[Update 461] Samples 2048 | Reward mean/std -0.071593/0.280035 | Value mean/std -0.061980/0.204062 | Adv std 1.223356e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.071593 | AvgAdv 0.000000 | AdvAfterStd 8.245295e-02
[Update 462] Samples 2048 | Reward mean/std -0.073517/0.205955 | Value mean/std -0.064449/0.109729 | Adv std 1.447796e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.073517 | AvgAdv -0.000000 | AdvAfterStd 9.004411e-02
[Update 463] Samples 2048 | Reward mean/std -0.063725/0.137524 | Value mean/std -0.065629/0.156459 | Adv std 1.084133e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.063725 | AvgAdv 0.000000 | AdvAfterStd 7.453726e-02
[Update 464] Samples 2048 | Reward mean/std -0.069188/0.257904 | Value mean/std -0.067023/0.105628 | Adv std 1.907651e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.069188 | AvgAdv -0.000000 | AdvAfterStd 6.989937e-02
[Update 465] Samples 2048 | Reward mean/std -0.069428/0.173240 | Value mean/std -0.076409/0.148227 | Adv std 1.286090e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.069428 | AvgAdv 0.000000 | AdvAfterStd 7.718287e-02
[Update 466] Samples 2048 | Reward mean/std -0.065499/0.126034 | Value mean/std -0.064835/0.082423 | Adv std 1.036063e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.065499 | AvgAdv -0.000000 | AdvAfterStd 8.292774e-02
[Update 467] Samples 2048 | Reward mean/std -0.072255/0.194078 | Value mean/std -0.064015/0.138259 | Adv std 1.128098e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.072255 | AvgAdv 0.000000 | AdvAfterStd 8.597874e-02
[Update 468] Samples 2048 | Reward mean/std -0.085434/0.472883 | Value mean/std -0.086007/0.317071 | Adv std 2.188746e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.085434 | AvgAdv -0.000000 | AdvAfterStd 1.187642e-01
[Update 469] Samples 2048 | Reward mean/std -0.073783/0.186488 | Value mean/std -0.059935/0.107118 | Adv std 1.281993e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.073783 | AvgAdv -0.000000 | AdvAfterStd 9.977718e-02
[Update 470] Samples 2048 | Reward mean/std -0.081782/0.254918 | Value mean/std -0.084570/0.222650 | Adv std 1.482124e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.081782 | AvgAdv 0.000000 | AdvAfterStd 1.036564e-01
[Update 471] Samples 2048 | Reward mean/std -0.071604/0.168384 | Value mean/std -0.069341/0.118726 | Adv std 1.277980e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.071604 | AvgAdv 0.000000 | AdvAfterStd 7.930202e-02
[Update 472] Samples 2048 | Reward mean/std -0.072421/0.210165 | Value mean/std -0.075800/0.186016 | Adv std 9.438457e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.072421 | AvgAdv 0.000000 | AdvAfterStd 7.279387e-02
[Update 473] Samples 2048 | Reward mean/std -0.075374/0.209471 | Value mean/std -0.067114/0.128430 | Adv std 1.520918e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.075374 | AvgAdv -0.000000 | AdvAfterStd 8.857535e-02
[Update 474] Samples 2048 | Reward mean/std -0.070006/0.188494 | Value mean/std -0.070369/0.147624 | Adv std 1.230011e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.070006 | AvgAdv 0.000000 | AdvAfterStd 8.766150e-02
[Update 475] Samples 2048 | Reward mean/std -0.073648/0.250517 | Value mean/std -0.070516/0.176602 | Adv std 1.404773e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.073648 | AvgAdv 0.000000 | AdvAfterStd 8.244092e-02
[Update 476] Samples 2048 | Reward mean/std -0.074641/0.221903 | Value mean/std -0.067026/0.187714 | Adv std 1.175389e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.074641 | AvgAdv 0.000000 | AdvAfterStd 8.907364e-02
[Update 477] Samples 2048 | Reward mean/std -0.068023/0.163777 | Value mean/std -0.075701/0.145252 | Adv std 1.060673e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.068023 | AvgAdv 0.000000 | AdvAfterStd 8.628318e-02
[Update 478] Samples 2048 | Reward mean/std -0.064514/0.147781 | Value mean/std -0.068803/0.105542 | Adv std 1.032228e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.064514 | AvgAdv -0.000000 | AdvAfterStd 7.749952e-02
[Update 479] Samples 2048 | Reward mean/std -0.071067/0.176370 | Value mean/std -0.067910/0.118549 | Adv std 1.279583e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.071067 | AvgAdv 0.000000 | AdvAfterStd 1.007232e-01
[Update 480] Samples 2048 | Reward mean/std -0.069322/0.165297 | Value mean/std -0.067202/0.150395 | Adv std 1.123365e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.069322 | AvgAdv 0.000000 | AdvAfterStd 7.755385e-02
[Update 481] Samples 2048 | Reward mean/std -0.076814/0.203127 | Value mean/std -0.076513/0.167429 | Adv std 1.329480e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.076814 | AvgAdv -0.000000 | AdvAfterStd 9.000501e-02
[Update 482] Samples 2048 | Reward mean/std -0.071207/0.299600 | Value mean/std -0.075837/0.187511 | Adv std 1.741559e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.071207 | AvgAdv 0.000000 | AdvAfterStd 8.584006e-02
[Update 483] Samples 2048 | Reward mean/std -0.081324/0.351726 | Value mean/std -0.080648/0.347186 | Adv std 1.854915e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.081324 | AvgAdv 0.000000 | AdvAfterStd 9.015052e-02
[Update 484] Samples 2048 | Reward mean/std -0.078627/0.287867 | Value mean/std -0.069256/0.281870 | Adv std 1.539407e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.078627 | AvgAdv -0.000000 | AdvAfterStd 1.021688e-01
[Update 485] Samples 2048 | Reward mean/std -0.073088/0.181852 | Value mean/std -0.071887/0.158970 | Adv std 1.223191e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.073088 | AvgAdv -0.000000 | AdvAfterStd 9.247822e-02
[Update 486] Samples 2048 | Reward mean/std -0.069527/0.158548 | Value mean/std -0.066462/0.139234 | Adv std 1.537676e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.069527 | AvgAdv 0.000000 | AdvAfterStd 9.021121e-02
[Update 487] Samples 2048 | Reward mean/std -0.070673/0.170240 | Value mean/std -0.075685/0.134414 | Adv std 1.287619e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.070673 | AvgAdv 0.000000 | AdvAfterStd 9.139653e-02
[Update 488] Samples 2048 | Reward mean/std -0.065735/0.151386 | Value mean/std -0.067115/0.114106 | Adv std 1.007531e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.065735 | AvgAdv 0.000000 | AdvAfterStd 7.300907e-02
[Update 489] Samples 2048 | Reward mean/std -0.071934/0.225454 | Value mean/std -0.071148/0.149034 | Adv std 1.567256e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.071934 | AvgAdv 0.000000 | AdvAfterStd 8.654754e-02
[Update 490] Samples 2048 | Reward mean/std -0.076400/0.283509 | Value mean/std -0.069641/0.219208 | Adv std 1.203414e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.076400 | AvgAdv -0.000000 | AdvAfterStd 8.099788e-02
[Update 491] Samples 2048 | Reward mean/std -0.069270/0.196936 | Value mean/std -0.074277/0.171557 | Adv std 1.441899e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.069270 | AvgAdv -0.000000 | AdvAfterStd 8.957382e-02
[Update 492] Samples 2048 | Reward mean/std -0.072627/0.176572 | Value mean/std -0.061827/0.150051 | Adv std 1.363812e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.072627 | AvgAdv 0.000000 | AdvAfterStd 9.005071e-02
[Update 493] Samples 2048 | Reward mean/std -0.074143/0.269735 | Value mean/std -0.074507/0.174922 | Adv std 1.822847e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.074143 | AvgAdv 0.000000 | AdvAfterStd 1.063591e-01
[Update 494] Samples 2048 | Reward mean/std -0.070262/0.184714 | Value mean/std -0.070831/0.143202 | Adv std 1.381222e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.070262 | AvgAdv 0.000000 | AdvAfterStd 8.901345e-02
[Update 495] Samples 2048 | Reward mean/std -0.071870/0.167985 | Value mean/std -0.076916/0.159532 | Adv std 1.156648e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.071870 | AvgAdv -0.000000 | AdvAfterStd 9.147032e-02
[Update 496] Samples 2048 | Reward mean/std -0.075396/0.196272 | Value mean/std -0.082382/0.171036 | Adv std 1.356033e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.075396 | AvgAdv 0.000000 | AdvAfterStd 9.267406e-02
[Update 497] Samples 2048 | Reward mean/std -0.069992/0.171971 | Value mean/std -0.074789/0.128437 | Adv std 1.077704e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.069992 | AvgAdv -0.000000 | AdvAfterStd 6.866903e-02
[Update 498] Samples 2048 | Reward mean/std -0.068587/0.151663 | Value mean/std -0.069849/0.114275 | Adv std 1.059675e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.068587 | AvgAdv 0.000000 | AdvAfterStd 7.907315e-02
[Update 499] Samples 2048 | Reward mean/std -0.072017/0.187695 | Value mean/std -0.081957/0.153619 | Adv std 1.152962e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.072017 | AvgAdv -0.000000 | AdvAfterStd 9.574193e-02
[Update 500] Samples 2048 | Reward mean/std -0.066652/0.165571 | Value mean/std -0.069107/0.118208 | Adv std 1.090310e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.066652 | AvgAdv -0.000000 | AdvAfterStd 7.918576e-02
Saved checkpoint at update 500
[Update 501] Samples 2048 | Reward mean/std -0.073242/0.173696 | Value mean/std -0.069602/0.144987 | Adv std 1.279441e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.073242 | AvgAdv 0.000000 | AdvAfterStd 9.553851e-02
[Update 502] Samples 2048 | Reward mean/std -0.069213/0.160920 | Value mean/std -0.070143/0.127594 | Adv std 1.246007e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.069213 | AvgAdv -0.000000 | AdvAfterStd 6.837142e-02
[Update 503] Samples 2048 | Reward mean/std -0.067413/0.145399 | Value mean/std -0.063550/0.112538 | Adv std 1.070681e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.067413 | AvgAdv -0.000000 | AdvAfterStd 8.273727e-02
[Update 504] Samples 2048 | Reward mean/std -0.068701/0.172622 | Value mean/std -0.063499/0.092986 | Adv std 1.320471e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.068701 | AvgAdv -0.000000 | AdvAfterStd 9.823409e-02
[Update 505] Samples 2048 | Reward mean/std -0.065811/0.163662 | Value mean/std -0.060982/0.135712 | Adv std 1.229073e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.065811 | AvgAdv -0.000000 | AdvAfterStd 8.678898e-02
[Update 506] Samples 2048 | Reward mean/std -0.065487/0.153362 | Value mean/std -0.064084/0.090247 | Adv std 1.062345e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.065487 | AvgAdv -0.000000 | AdvAfterStd 7.520815e-02
[Update 507] Samples 2048 | Reward mean/std -0.069064/0.191562 | Value mean/std -0.062790/0.106690 | Adv std 1.675695e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.069064 | AvgAdv -0.000000 | AdvAfterStd 9.262577e-02
[Update 508] Samples 2048 | Reward mean/std -0.074181/0.200087 | Value mean/std -0.068404/0.115169 | Adv std 1.233508e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.074181 | AvgAdv -0.000000 | AdvAfterStd 7.804634e-02
[Update 509] Samples 2048 | Reward mean/std -0.068880/0.225515 | Value mean/std -0.067658/0.156032 | Adv std 1.369955e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.068880 | AvgAdv -0.000000 | AdvAfterStd 9.323541e-02
[Update 510] Samples 2048 | Reward mean/std -0.077290/0.265173 | Value mean/std -0.065900/0.192753 | Adv std 1.552526e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.077290 | AvgAdv 0.000000 | AdvAfterStd 8.237865e-02
[Update 511] Samples 2048 | Reward mean/std -0.072717/0.199747 | Value mean/std -0.072756/0.163156 | Adv std 1.490443e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.072717 | AvgAdv 0.000000 | AdvAfterStd 9.911856e-02
[Update 512] Samples 2048 | Reward mean/std -0.071130/0.280630 | Value mean/std -0.065894/0.162337 | Adv std 1.532393e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.071130 | AvgAdv 0.000000 | AdvAfterStd 9.172253e-02
[Update 513] Samples 2048 | Reward mean/std -0.070331/0.275084 | Value mean/std -0.079493/0.276260 | Adv std 1.513897e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.070331 | AvgAdv 0.000000 | AdvAfterStd 6.994364e-02
[Update 514] Samples 2048 | Reward mean/std -0.061368/0.135780 | Value mean/std -0.061177/0.104272 | Adv std 9.851459e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.061368 | AvgAdv -0.000000 | AdvAfterStd 7.278956e-02
[Update 515] Samples 2048 | Reward mean/std -0.069020/0.227593 | Value mean/std -0.067181/0.224943 | Adv std 1.073288e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.069020 | AvgAdv 0.000000 | AdvAfterStd 8.392806e-02
[Update 516] Samples 2048 | Reward mean/std -0.065183/0.157823 | Value mean/std -0.050597/0.106057 | Adv std 1.122253e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.065183 | AvgAdv 0.000000 | AdvAfterStd 8.144247e-02
[Update 517] Samples 2048 | Reward mean/std -0.068941/0.151678 | Value mean/std -0.065391/0.120516 | Adv std 1.265136e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.068941 | AvgAdv 0.000000 | AdvAfterStd 9.125353e-02
[Update 518] Samples 2048 | Reward mean/std -0.073843/0.253937 | Value mean/std -0.071991/0.139568 | Adv std 1.742152e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.073843 | AvgAdv -0.000000 | AdvAfterStd 9.957352e-02
[Update 519] Samples 2048 | Reward mean/std -0.068356/0.207357 | Value mean/std -0.068183/0.229235 | Adv std 1.371592e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.068356 | AvgAdv 0.000000 | AdvAfterStd 8.556911e-02
[Update 520] Samples 2048 | Reward mean/std -0.068172/0.158111 | Value mean/std -0.072088/0.145812 | Adv std 1.425825e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.068172 | AvgAdv 0.000000 | AdvAfterStd 8.423705e-02
[Update 521] Samples 2048 | Reward mean/std -0.070173/0.156805 | Value mean/std -0.070512/0.104123 | Adv std 1.161933e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.070173 | AvgAdv 0.000000 | AdvAfterStd 8.739609e-02
[Update 522] Samples 2048 | Reward mean/std -0.070306/0.202339 | Value mean/std -0.064455/0.159867 | Adv std 1.113971e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.070306 | AvgAdv 0.000000 | AdvAfterStd 8.485989e-02
[Update 523] Samples 2048 | Reward mean/std -0.067713/0.150454 | Value mean/std -0.064047/0.097797 | Adv std 1.100542e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.067713 | AvgAdv -0.000000 | AdvAfterStd 8.299674e-02
[Update 524] Samples 2048 | Reward mean/std -0.062394/0.142672 | Value mean/std -0.072044/0.091284 | Adv std 9.866490e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.062394 | AvgAdv -0.000000 | AdvAfterStd 7.089274e-02
[Update 525] Samples 2048 | Reward mean/std -0.070481/0.168853 | Value mean/std -0.065777/0.116373 | Adv std 1.134939e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.070481 | AvgAdv -0.000000 | AdvAfterStd 8.507290e-02
[Update 526] Samples 2048 | Reward mean/std -0.069513/0.171234 | Value mean/std -0.067800/0.135714 | Adv std 1.151241e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.069513 | AvgAdv -0.000000 | AdvAfterStd 8.770100e-02
[Update 527] Samples 2048 | Reward mean/std -0.064591/0.129269 | Value mean/std -0.063533/0.108613 | Adv std 1.013687e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.064591 | AvgAdv -0.000000 | AdvAfterStd 7.381151e-02
[Update 528] Samples 2048 | Reward mean/std -0.065582/0.160710 | Value mean/std -0.063634/0.105509 | Adv std 1.084984e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.065582 | AvgAdv 0.000000 | AdvAfterStd 7.422820e-02
[Update 529] Samples 2048 | Reward mean/std -0.068607/0.169539 | Value mean/std -0.071017/0.136437 | Adv std 1.156655e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.068607 | AvgAdv 0.000000 | AdvAfterStd 7.575113e-02
[Update 530] Samples 2048 | Reward mean/std -0.073814/0.200505 | Value mean/std -0.075282/0.155779 | Adv std 1.171876e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.073814 | AvgAdv 0.000000 | AdvAfterStd 8.602816e-02
[Update 531] Samples 2048 | Reward mean/std -0.069102/0.191045 | Value mean/std -0.072371/0.227614 | Adv std 1.304500e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.069102 | AvgAdv -0.000000 | AdvAfterStd 7.367826e-02
[Update 532] Samples 2048 | Reward mean/std -0.069753/0.155540 | Value mean/std -0.069275/0.113990 | Adv std 1.060073e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.069753 | AvgAdv 0.000000 | AdvAfterStd 8.815338e-02
[Update 533] Samples 2048 | Reward mean/std -0.071242/0.202119 | Value mean/std -0.074874/0.160840 | Adv std 1.170612e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.071242 | AvgAdv 0.000000 | AdvAfterStd 8.150064e-02
[Update 534] Samples 2048 | Reward mean/std -0.086156/0.374583 | Value mean/std -0.082131/0.214475 | Adv std 2.139749e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.086156 | AvgAdv 0.000000 | AdvAfterStd 1.295216e-01
[Update 535] Samples 2048 | Reward mean/std -0.075889/0.200081 | Value mean/std -0.081086/0.202080 | Adv std 1.258086e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.075889 | AvgAdv -0.000000 | AdvAfterStd 9.414551e-02
[Update 536] Samples 2048 | Reward mean/std -0.080024/0.283251 | Value mean/std -0.077295/0.227607 | Adv std 1.367741e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.080024 | AvgAdv 0.000000 | AdvAfterStd 9.210895e-02
[Update 537] Samples 2048 | Reward mean/std -0.075174/0.280427 | Value mean/std -0.076418/0.175485 | Adv std 1.912409e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.075174 | AvgAdv -0.000000 | AdvAfterStd 9.767769e-02
[Update 538] Samples 2048 | Reward mean/std -0.078075/0.181804 | Value mean/std -0.070502/0.136942 | Adv std 1.275575e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.078075 | AvgAdv 0.000000 | AdvAfterStd 9.561451e-02
[Update 539] Samples 2048 | Reward mean/std -0.068020/0.160949 | Value mean/std -0.065772/0.132184 | Adv std 1.150037e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.068020 | AvgAdv -0.000000 | AdvAfterStd 8.157340e-02
[Update 540] Samples 2048 | Reward mean/std -0.073627/0.238450 | Value mean/std -0.068073/0.133945 | Adv std 1.588217e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.073627 | AvgAdv 0.000000 | AdvAfterStd 7.775034e-02
[Update 541] Samples 2048 | Reward mean/std -0.069921/0.200285 | Value mean/std -0.074898/0.195479 | Adv std 1.043832e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.069921 | AvgAdv 0.000000 | AdvAfterStd 8.540870e-02
[Update 542] Samples 2048 | Reward mean/std -0.071398/0.161398 | Value mean/std -0.068386/0.139971 | Adv std 1.075147e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.071398 | AvgAdv -0.000000 | AdvAfterStd 7.494637e-02
[Update 543] Samples 2048 | Reward mean/std -0.079892/0.236164 | Value mean/std -0.072977/0.248701 | Adv std 1.331226e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.079892 | AvgAdv -0.000000 | AdvAfterStd 1.034964e-01
[Update 544] Samples 2048 | Reward mean/std -0.072251/0.172432 | Value mean/std -0.072009/0.107167 | Adv std 1.161333e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.072251 | AvgAdv -0.000000 | AdvAfterStd 8.287095e-02
[Update 545] Samples 2048 | Reward mean/std -0.074811/0.184464 | Value mean/std -0.079895/0.160668 | Adv std 1.277500e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.074811 | AvgAdv 0.000000 | AdvAfterStd 9.081196e-02
[Update 546] Samples 2048 | Reward mean/std -0.074350/0.179806 | Value mean/std -0.071895/0.122880 | Adv std 1.186965e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.074350 | AvgAdv 0.000000 | AdvAfterStd 1.077947e-01
[Update 547] Samples 2048 | Reward mean/std -0.067850/0.157135 | Value mean/std -0.070556/0.118497 | Adv std 1.000469e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.067850 | AvgAdv -0.000000 | AdvAfterStd 7.875479e-02
[Update 548] Samples 2048 | Reward mean/std -0.073064/0.232978 | Value mean/std -0.077510/0.174431 | Adv std 1.316447e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.073064 | AvgAdv -0.000000 | AdvAfterStd 1.056813e-01
[Update 549] Samples 2048 | Reward mean/std -0.082027/0.304887 | Value mean/std -0.077992/0.165265 | Adv std 1.983946e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.082027 | AvgAdv -0.000000 | AdvAfterStd 1.189381e-01
[Update 550] Samples 2048 | Reward mean/std -0.073628/0.185107 | Value mean/std -0.075768/0.180669 | Adv std 1.401420e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.073628 | AvgAdv -0.000000 | AdvAfterStd 8.919714e-02
Saved checkpoint at update 550
[Update 551] Samples 2048 | Reward mean/std -0.067899/0.167411 | Value mean/std -0.069353/0.143431 | Adv std 1.012664e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.067899 | AvgAdv -0.000000 | AdvAfterStd 7.209700e-02
[Update 552] Samples 2048 | Reward mean/std -0.065000/0.143141 | Value mean/std -0.071418/0.135646 | Adv std 1.080874e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.065000 | AvgAdv 0.000000 | AdvAfterStd 7.034517e-02
[Update 553] Samples 2048 | Reward mean/std -0.073336/0.178100 | Value mean/std -0.067300/0.109050 | Adv std 1.294921e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.073336 | AvgAdv -0.000000 | AdvAfterStd 8.439361e-02
[Update 554] Samples 2048 | Reward mean/std -0.079700/0.358874 | Value mean/std -0.081741/0.198260 | Adv std 2.350805e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.079700 | AvgAdv -0.000000 | AdvAfterStd 9.258872e-02
[Update 555] Samples 2048 | Reward mean/std -0.066274/0.171347 | Value mean/std -0.064022/0.178631 | Adv std 1.418239e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.066274 | AvgAdv -0.000000 | AdvAfterStd 7.992186e-02
[Update 556] Samples 2048 | Reward mean/std -0.085178/0.308691 | Value mean/std -0.067211/0.186722 | Adv std 1.801566e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.085178 | AvgAdv -0.000000 | AdvAfterStd 9.073927e-02
[Update 557] Samples 2048 | Reward mean/std -0.077755/0.245520 | Value mean/std -0.074219/0.172966 | Adv std 1.472412e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.077755 | AvgAdv 0.000000 | AdvAfterStd 1.155296e-01
[Update 558] Samples 2048 | Reward mean/std -0.062948/0.127761 | Value mean/std -0.065892/0.155545 | Adv std 1.092013e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.062948 | AvgAdv 0.000000 | AdvAfterStd 9.066588e-02
[Update 559] Samples 2048 | Reward mean/std -0.077575/0.197100 | Value mean/std -0.068381/0.150555 | Adv std 1.235769e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.077575 | AvgAdv -0.000000 | AdvAfterStd 8.664179e-02
[Update 560] Samples 2048 | Reward mean/std -0.066036/0.137516 | Value mean/std -0.063529/0.131458 | Adv std 1.168182e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.066036 | AvgAdv 0.000000 | AdvAfterStd 8.013657e-02
[Update 561] Samples 2048 | Reward mean/std -0.079848/0.345518 | Value mean/std -0.075298/0.236145 | Adv std 1.561569e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.079848 | AvgAdv 0.000000 | AdvAfterStd 9.401011e-02
[Update 562] Samples 2048 | Reward mean/std -0.063090/0.129249 | Value mean/std -0.070386/0.118347 | Adv std 1.113909e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.063090 | AvgAdv -0.000000 | AdvAfterStd 7.115921e-02
[Update 563] Samples 2048 | Reward mean/std -0.067761/0.170783 | Value mean/std -0.067620/0.099334 | Adv std 1.400970e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.067761 | AvgAdv 0.000000 | AdvAfterStd 9.091461e-02
[Update 564] Samples 2048 | Reward mean/std -0.063552/0.126963 | Value mean/std -0.072116/0.108475 | Adv std 9.733169e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.063552 | AvgAdv 0.000000 | AdvAfterStd 7.346172e-02
[Update 565] Samples 2048 | Reward mean/std -0.068234/0.194566 | Value mean/std -0.059582/0.120089 | Adv std 1.392750e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.068234 | AvgAdv -0.000000 | AdvAfterStd 9.226405e-02
[Update 566] Samples 2048 | Reward mean/std -0.065982/0.175991 | Value mean/std -0.064766/0.127806 | Adv std 1.283545e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.065982 | AvgAdv 0.000000 | AdvAfterStd 7.725962e-02
[Update 567] Samples 2048 | Reward mean/std -0.065462/0.163770 | Value mean/std -0.071148/0.160085 | Adv std 1.467157e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.065462 | AvgAdv -0.000000 | AdvAfterStd 7.130473e-02
[Update 568] Samples 2048 | Reward mean/std -0.067300/0.178815 | Value mean/std -0.065372/0.117488 | Adv std 1.250172e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.067300 | AvgAdv 0.000000 | AdvAfterStd 7.555869e-02
[Update 569] Samples 2048 | Reward mean/std -0.069994/0.222644 | Value mean/std -0.067887/0.190199 | Adv std 1.256098e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.069994 | AvgAdv -0.000000 | AdvAfterStd 8.436491e-02
[Update 570] Samples 2048 | Reward mean/std -0.068533/0.165953 | Value mean/std -0.071925/0.134634 | Adv std 1.148836e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.068533 | AvgAdv 0.000000 | AdvAfterStd 7.984812e-02
[Update 571] Samples 2048 | Reward mean/std -0.060172/0.117925 | Value mean/std -0.056719/0.079304 | Adv std 9.949943e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.060172 | AvgAdv -0.000000 | AdvAfterStd 7.561374e-02
[Update 572] Samples 2048 | Reward mean/std -0.069397/0.177587 | Value mean/std -0.064313/0.114991 | Adv std 1.037465e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.069397 | AvgAdv 0.000000 | AdvAfterStd 7.911532e-02
[Update 573] Samples 2048 | Reward mean/std -0.071254/0.196427 | Value mean/std -0.071265/0.196524 | Adv std 1.426563e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.071254 | AvgAdv 0.000000 | AdvAfterStd 9.517050e-02
[Update 574] Samples 2048 | Reward mean/std -0.079952/0.294791 | Value mean/std -0.068972/0.201303 | Adv std 1.450594e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.079952 | AvgAdv 0.000000 | AdvAfterStd 9.416275e-02
[Update 575] Samples 2048 | Reward mean/std -0.070041/0.194522 | Value mean/std -0.068396/0.156134 | Adv std 1.161972e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.070041 | AvgAdv -0.000000 | AdvAfterStd 9.284221e-02
[Update 576] Samples 2048 | Reward mean/std -0.071110/0.168868 | Value mean/std -0.071312/0.171079 | Adv std 1.180916e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.071110 | AvgAdv 0.000000 | AdvAfterStd 7.726210e-02
[Update 577] Samples 2048 | Reward mean/std -0.065275/0.158247 | Value mean/std -0.067660/0.168147 | Adv std 1.348001e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.065275 | AvgAdv 0.000000 | AdvAfterStd 9.196496e-02
[Update 578] Samples 2048 | Reward mean/std -0.064679/0.150641 | Value mean/std -0.067623/0.092239 | Adv std 1.066632e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.064679 | AvgAdv 0.000000 | AdvAfterStd 7.819553e-02
[Update 579] Samples 2048 | Reward mean/std -0.071178/0.256746 | Value mean/std -0.069534/0.207141 | Adv std 1.212749e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.071178 | AvgAdv 0.000000 | AdvAfterStd 9.508175e-02
[Update 580] Samples 2048 | Reward mean/std -0.064106/0.159480 | Value mean/std -0.066496/0.126581 | Adv std 1.149037e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.064106 | AvgAdv -0.000000 | AdvAfterStd 8.506505e-02
[Update 581] Samples 2048 | Reward mean/std -0.064388/0.152156 | Value mean/std -0.064161/0.099448 | Adv std 1.040800e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.064388 | AvgAdv -0.000000 | AdvAfterStd 8.074789e-02
[Update 582] Samples 2048 | Reward mean/std -0.062883/0.138476 | Value mean/std -0.071320/0.119977 | Adv std 9.234533e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.062883 | AvgAdv 0.000000 | AdvAfterStd 8.599085e-02
[Update 583] Samples 2048 | Reward mean/std -0.068624/0.160303 | Value mean/std -0.059386/0.091202 | Adv std 1.369004e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.068624 | AvgAdv 0.000000 | AdvAfterStd 9.258547e-02
[Update 584] Samples 2048 | Reward mean/std -0.069727/0.166170 | Value mean/std -0.078478/0.114009 | Adv std 1.333009e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.069727 | AvgAdv 0.000000 | AdvAfterStd 9.291299e-02
[Update 585] Samples 2048 | Reward mean/std -0.066660/0.163013 | Value mean/std -0.062256/0.152416 | Adv std 1.530759e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.066660 | AvgAdv 0.000000 | AdvAfterStd 7.876778e-02
[Update 586] Samples 2048 | Reward mean/std -0.067495/0.165474 | Value mean/std -0.064791/0.123033 | Adv std 1.058596e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.067495 | AvgAdv -0.000000 | AdvAfterStd 7.360077e-02
[Update 587] Samples 2048 | Reward mean/std -0.069716/0.159042 | Value mean/std -0.064830/0.118653 | Adv std 1.070411e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.069716 | AvgAdv 0.000000 | AdvAfterStd 8.615841e-02
[Update 588] Samples 2048 | Reward mean/std -0.072599/0.227653 | Value mean/std -0.069998/0.164791 | Adv std 1.325833e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.072599 | AvgAdv -0.000000 | AdvAfterStd 8.093393e-02
[Update 589] Samples 2048 | Reward mean/std -0.069369/0.195029 | Value mean/std -0.065723/0.163788 | Adv std 9.918445e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.069369 | AvgAdv -0.000000 | AdvAfterStd 8.481175e-02
[Update 590] Samples 2048 | Reward mean/std -0.074208/0.202057 | Value mean/std -0.072001/0.173055 | Adv std 1.447651e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.074208 | AvgAdv 0.000000 | AdvAfterStd 7.861369e-02
[Update 591] Samples 2048 | Reward mean/std -0.072178/0.242059 | Value mean/std -0.070362/0.113723 | Adv std 1.858777e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.072178 | AvgAdv 0.000000 | AdvAfterStd 9.575839e-02
[Update 592] Samples 2048 | Reward mean/std -0.069871/0.209181 | Value mean/std -0.063466/0.193837 | Adv std 1.205722e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.069871 | AvgAdv -0.000000 | AdvAfterStd 8.466294e-02
[Update 593] Samples 2048 | Reward mean/std -0.067374/0.149361 | Value mean/std -0.064319/0.166908 | Adv std 1.427919e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.067374 | AvgAdv -0.000000 | AdvAfterStd 8.698972e-02
[Update 594] Samples 2048 | Reward mean/std -0.074926/0.277523 | Value mean/std -0.069064/0.142353 | Adv std 1.929252e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.074926 | AvgAdv -0.000000 | AdvAfterStd 7.505274e-02
[Update 595] Samples 2048 | Reward mean/std -0.075112/0.183586 | Value mean/std -0.081644/0.181137 | Adv std 1.278340e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.075112 | AvgAdv 0.000000 | AdvAfterStd 8.730677e-02
[Update 596] Samples 2048 | Reward mean/std -0.075880/0.287799 | Value mean/std -0.074460/0.210643 | Adv std 1.525652e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.075880 | AvgAdv 0.000000 | AdvAfterStd 7.533863e-02
[Update 597] Samples 2048 | Reward mean/std -0.071495/0.189429 | Value mean/std -0.069221/0.161568 | Adv std 1.116582e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.071495 | AvgAdv -0.000000 | AdvAfterStd 8.786575e-02
[Update 598] Samples 2048 | Reward mean/std -0.069620/0.178602 | Value mean/std -0.075783/0.162490 | Adv std 1.437009e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.069620 | AvgAdv 0.000000 | AdvAfterStd 8.657327e-02
[Update 599] Samples 2048 | Reward mean/std -0.071509/0.195986 | Value mean/std -0.067966/0.164797 | Adv std 1.277178e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.071509 | AvgAdv 0.000000 | AdvAfterStd 8.545945e-02
[Update 600] Samples 2048 | Reward mean/std -0.069269/0.179584 | Value mean/std -0.067001/0.125222 | Adv std 1.311046e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.069269 | AvgAdv -0.000000 | AdvAfterStd 9.418379e-02
Saved checkpoint at update 600
[Update 601] Samples 2048 | Reward mean/std -0.077120/0.217379 | Value mean/std -0.082877/0.140833 | Adv std 1.602638e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.077120 | AvgAdv 0.000000 | AdvAfterStd 9.658273e-02
[Update 602] Samples 2048 | Reward mean/std -0.064646/0.148826 | Value mean/std -0.066375/0.104623 | Adv std 1.055455e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.064646 | AvgAdv -0.000000 | AdvAfterStd 7.978521e-02
[Update 603] Samples 2048 | Reward mean/std -0.076427/0.306535 | Value mean/std -0.065700/0.217794 | Adv std 1.394319e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.076427 | AvgAdv -0.000000 | AdvAfterStd 9.007188e-02
[Update 604] Samples 2048 | Reward mean/std -0.072927/0.214991 | Value mean/std -0.067541/0.161390 | Adv std 1.356582e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.072927 | AvgAdv 0.000000 | AdvAfterStd 8.758558e-02
[Update 605] Samples 2048 | Reward mean/std -0.071751/0.193329 | Value mean/std -0.074574/0.186025 | Adv std 1.228039e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.071751 | AvgAdv -0.000000 | AdvAfterStd 8.051419e-02
[Update 606] Samples 2048 | Reward mean/std -0.067269/0.151419 | Value mean/std -0.062927/0.108405 | Adv std 1.021850e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.067269 | AvgAdv -0.000000 | AdvAfterStd 8.604120e-02
[Update 607] Samples 2048 | Reward mean/std -0.069908/0.164010 | Value mean/std -0.072521/0.139289 | Adv std 1.180769e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.069908 | AvgAdv -0.000000 | AdvAfterStd 8.480171e-02
[Update 608] Samples 2048 | Reward mean/std -0.069638/0.215424 | Value mean/std -0.064576/0.143714 | Adv std 1.211946e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.069638 | AvgAdv 0.000000 | AdvAfterStd 7.350079e-02
[Update 609] Samples 2048 | Reward mean/std -0.069019/0.161169 | Value mean/std -0.062684/0.103157 | Adv std 1.101199e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.069019 | AvgAdv 0.000000 | AdvAfterStd 7.433289e-02
[Update 610] Samples 2048 | Reward mean/std -0.066623/0.162544 | Value mean/std -0.075790/0.148937 | Adv std 1.093879e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.066623 | AvgAdv -0.000000 | AdvAfterStd 7.579959e-02
[Update 611] Samples 2048 | Reward mean/std -0.073578/0.240897 | Value mean/std -0.071360/0.133851 | Adv std 1.818571e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.073578 | AvgAdv -0.000000 | AdvAfterStd 9.063475e-02
[Update 612] Samples 2048 | Reward mean/std -0.074246/0.236426 | Value mean/std -0.075694/0.178197 | Adv std 1.446504e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.074246 | AvgAdv 0.000000 | AdvAfterStd 9.612856e-02
[Update 613] Samples 2048 | Reward mean/std -0.069979/0.200420 | Value mean/std -0.069165/0.137394 | Adv std 1.162222e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.069979 | AvgAdv 0.000000 | AdvAfterStd 8.205847e-02
[Update 614] Samples 2048 | Reward mean/std -0.080863/0.272306 | Value mean/std -0.078156/0.200879 | Adv std 1.727212e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.080863 | AvgAdv 0.000000 | AdvAfterStd 8.276941e-02
[Update 615] Samples 2048 | Reward mean/std -0.073287/0.196005 | Value mean/std -0.073649/0.208417 | Adv std 1.443176e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.073287 | AvgAdv -0.000000 | AdvAfterStd 1.037225e-01
[Update 616] Samples 2048 | Reward mean/std -0.074900/0.249373 | Value mean/std -0.076099/0.179675 | Adv std 1.628159e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.074900 | AvgAdv 0.000000 | AdvAfterStd 8.950043e-02
[Update 617] Samples 2048 | Reward mean/std -0.082724/0.294863 | Value mean/std -0.079344/0.212311 | Adv std 1.525326e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.082724 | AvgAdv 0.000000 | AdvAfterStd 9.112908e-02
[Update 618] Samples 2048 | Reward mean/std -0.078939/0.314522 | Value mean/std -0.069971/0.239260 | Adv std 1.501602e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.078939 | AvgAdv -0.000000 | AdvAfterStd 1.157695e-01
[Update 619] Samples 2048 | Reward mean/std -0.065970/0.155706 | Value mean/std -0.067798/0.117759 | Adv std 1.097046e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.065970 | AvgAdv 0.000000 | AdvAfterStd 8.073480e-02
[Update 620] Samples 2048 | Reward mean/std -0.072091/0.185013 | Value mean/std -0.064935/0.179779 | Adv std 1.161765e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.072091 | AvgAdv 0.000000 | AdvAfterStd 7.779560e-02
[Update 621] Samples 2048 | Reward mean/std -0.066105/0.151286 | Value mean/std -0.070617/0.134029 | Adv std 1.116441e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.066105 | AvgAdv -0.000000 | AdvAfterStd 9.713791e-02
[Update 622] Samples 2048 | Reward mean/std -0.067910/0.127329 | Value mean/std -0.066592/0.115354 | Adv std 1.016788e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.067910 | AvgAdv 0.000000 | AdvAfterStd 7.974181e-02
[Update 623] Samples 2048 | Reward mean/std -0.068005/0.149549 | Value mean/std -0.064197/0.097563 | Adv std 1.068783e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.068005 | AvgAdv 0.000000 | AdvAfterStd 8.555467e-02
[Update 624] Samples 2048 | Reward mean/std -0.067690/0.169350 | Value mean/std -0.067621/0.128530 | Adv std 1.179323e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.067690 | AvgAdv 0.000000 | AdvAfterStd 7.656491e-02
[Update 625] Samples 2048 | Reward mean/std -0.073636/0.211102 | Value mean/std -0.067204/0.162537 | Adv std 1.283418e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.073636 | AvgAdv -0.000000 | AdvAfterStd 8.305763e-02
[Update 626] Samples 2048 | Reward mean/std -0.065910/0.152402 | Value mean/std -0.064139/0.105579 | Adv std 1.121473e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.065910 | AvgAdv 0.000000 | AdvAfterStd 7.822662e-02
[Update 627] Samples 2048 | Reward mean/std -0.070805/0.190804 | Value mean/std -0.073598/0.136692 | Adv std 1.244611e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.070805 | AvgAdv -0.000000 | AdvAfterStd 7.512688e-02
[Update 628] Samples 2048 | Reward mean/std -0.068377/0.145592 | Value mean/std -0.072360/0.122841 | Adv std 1.016250e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.068377 | AvgAdv 0.000000 | AdvAfterStd 7.527580e-02
[Update 629] Samples 2048 | Reward mean/std -0.071252/0.209558 | Value mean/std -0.067733/0.126975 | Adv std 1.336007e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.071252 | AvgAdv -0.000000 | AdvAfterStd 8.512004e-02
[Update 630] Samples 2048 | Reward mean/std -0.077732/0.288406 | Value mean/std -0.075200/0.217213 | Adv std 1.353793e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.077732 | AvgAdv 0.000000 | AdvAfterStd 8.587337e-02
[Update 631] Samples 2048 | Reward mean/std -0.065009/0.151518 | Value mean/std -0.079585/0.156432 | Adv std 1.019768e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.065009 | AvgAdv -0.000000 | AdvAfterStd 7.560587e-02
[Update 632] Samples 2048 | Reward mean/std -0.073125/0.192830 | Value mean/std -0.073137/0.192021 | Adv std 1.302398e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.073125 | AvgAdv 0.000000 | AdvAfterStd 8.726070e-02
[Update 633] Samples 2048 | Reward mean/std -0.068039/0.224754 | Value mean/std -0.067810/0.174209 | Adv std 1.065143e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.068039 | AvgAdv -0.000000 | AdvAfterStd 7.484116e-02
[Update 634] Samples 2048 | Reward mean/std -0.067977/0.160055 | Value mean/std -0.067311/0.134639 | Adv std 9.947684e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.067977 | AvgAdv 0.000000 | AdvAfterStd 7.440013e-02
[Update 635] Samples 2048 | Reward mean/std -0.064993/0.145142 | Value mean/std -0.064833/0.110227 | Adv std 1.047841e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.064993 | AvgAdv -0.000000 | AdvAfterStd 7.661022e-02
[Update 636] Samples 2048 | Reward mean/std -0.067175/0.157186 | Value mean/std -0.065600/0.112200 | Adv std 1.194863e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.067175 | AvgAdv 0.000000 | AdvAfterStd 9.145744e-02
[Update 637] Samples 2048 | Reward mean/std -0.065367/0.144142 | Value mean/std -0.058861/0.093351 | Adv std 1.099568e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.065367 | AvgAdv -0.000000 | AdvAfterStd 7.552277e-02
[Update 638] Samples 2048 | Reward mean/std -0.069902/0.172300 | Value mean/std -0.068196/0.146732 | Adv std 1.062806e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.069902 | AvgAdv 0.000000 | AdvAfterStd 7.679456e-02
[Update 639] Samples 2048 | Reward mean/std -0.074133/0.277666 | Value mean/std -0.074381/0.257878 | Adv std 1.183224e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.074133 | AvgAdv 0.000000 | AdvAfterStd 8.230019e-02
[Update 640] Samples 2048 | Reward mean/std -0.073339/0.212130 | Value mean/std -0.073431/0.209329 | Adv std 1.445698e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.073339 | AvgAdv -0.000000 | AdvAfterStd 8.411492e-02
[Update 641] Samples 2048 | Reward mean/std -0.065505/0.169195 | Value mean/std -0.065914/0.152445 | Adv std 1.013669e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.065505 | AvgAdv -0.000000 | AdvAfterStd 7.612426e-02
[Update 642] Samples 2048 | Reward mean/std -0.074264/0.217659 | Value mean/std -0.067989/0.162232 | Adv std 1.693742e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.074264 | AvgAdv 0.000000 | AdvAfterStd 8.978793e-02
[Update 643] Samples 2048 | Reward mean/std -0.060059/0.130744 | Value mean/std -0.065885/0.118423 | Adv std 8.965513e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.060059 | AvgAdv 0.000000 | AdvAfterStd 7.370711e-02
[Update 644] Samples 2048 | Reward mean/std -0.061812/0.118680 | Value mean/std -0.061005/0.127279 | Adv std 1.058682e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.061812 | AvgAdv 0.000000 | AdvAfterStd 6.517338e-02
[Update 645] Samples 2048 | Reward mean/std -0.064837/0.174010 | Value mean/std -0.066962/0.107789 | Adv std 1.235239e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.064837 | AvgAdv 0.000000 | AdvAfterStd 7.595383e-02
[Update 646] Samples 2048 | Reward mean/std -0.064390/0.128365 | Value mean/std -0.069602/0.104534 | Adv std 1.006577e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.064390 | AvgAdv -0.000000 | AdvAfterStd 7.279966e-02
[Update 647] Samples 2048 | Reward mean/std -0.063455/0.141125 | Value mean/std -0.063026/0.117873 | Adv std 1.097366e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.063455 | AvgAdv -0.000000 | AdvAfterStd 8.154462e-02
[Update 648] Samples 2048 | Reward mean/std -0.068606/0.162455 | Value mean/std -0.069276/0.105160 | Adv std 1.085825e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.068606 | AvgAdv -0.000000 | AdvAfterStd 8.474469e-02
[Update 649] Samples 2048 | Reward mean/std -0.068844/0.180000 | Value mean/std -0.064691/0.104649 | Adv std 1.297949e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.068844 | AvgAdv 0.000000 | AdvAfterStd 9.118445e-02
[Update 650] Samples 2048 | Reward mean/std -0.069449/0.161210 | Value mean/std -0.070729/0.153385 | Adv std 1.111971e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.069449 | AvgAdv 0.000000 | AdvAfterStd 8.477291e-02
Saved checkpoint at update 650
[Update 651] Samples 2048 | Reward mean/std -0.063650/0.149019 | Value mean/std -0.062641/0.111635 | Adv std 9.233217e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.063650 | AvgAdv 0.000000 | AdvAfterStd 7.929622e-02
[Update 652] Samples 2048 | Reward mean/std -0.064259/0.154664 | Value mean/std -0.063097/0.094834 | Adv std 1.016949e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.064259 | AvgAdv 0.000000 | AdvAfterStd 7.992744e-02
[Update 653] Samples 2048 | Reward mean/std -0.065902/0.175513 | Value mean/std -0.073367/0.129843 | Adv std 1.259916e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.065902 | AvgAdv 0.000000 | AdvAfterStd 8.232709e-02
[Update 654] Samples 2048 | Reward mean/std -0.070686/0.230967 | Value mean/std -0.073626/0.190320 | Adv std 1.319203e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.070686 | AvgAdv -0.000000 | AdvAfterStd 9.421554e-02
[Update 655] Samples 2048 | Reward mean/std -0.067648/0.137527 | Value mean/std -0.063266/0.092978 | Adv std 1.092011e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.067648 | AvgAdv 0.000000 | AdvAfterStd 8.743677e-02
[Update 656] Samples 2048 | Reward mean/std -0.066090/0.152302 | Value mean/std -0.066044/0.113647 | Adv std 1.072038e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.066090 | AvgAdv -0.000000 | AdvAfterStd 8.302870e-02
[Update 657] Samples 2048 | Reward mean/std -0.062417/0.121269 | Value mean/std -0.061611/0.085419 | Adv std 9.997182e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.062417 | AvgAdv 0.000000 | AdvAfterStd 7.912989e-02
[Update 658] Samples 2048 | Reward mean/std -0.070143/0.168948 | Value mean/std -0.070938/0.126923 | Adv std 1.144957e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.070143 | AvgAdv -0.000000 | AdvAfterStd 8.614910e-02
[Update 659] Samples 2048 | Reward mean/std -0.076198/0.229644 | Value mean/std -0.063631/0.157756 | Adv std 1.367292e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.076198 | AvgAdv 0.000000 | AdvAfterStd 9.056410e-02
[Update 660] Samples 2048 | Reward mean/std -0.070837/0.196122 | Value mean/std -0.071302/0.160495 | Adv std 1.323186e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.070837 | AvgAdv 0.000000 | AdvAfterStd 8.501158e-02
[Update 661] Samples 2048 | Reward mean/std -0.064828/0.162993 | Value mean/std -0.058710/0.117425 | Adv std 1.161352e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.064828 | AvgAdv -0.000000 | AdvAfterStd 9.285092e-02
[Update 662] Samples 2048 | Reward mean/std -0.078047/0.363165 | Value mean/std -0.068480/0.220990 | Adv std 1.882634e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.078047 | AvgAdv -0.000000 | AdvAfterStd 1.087037e-01
[Update 663] Samples 2048 | Reward mean/std -0.071621/0.174916 | Value mean/std -0.082804/0.151242 | Adv std 1.252433e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.071621 | AvgAdv 0.000000 | AdvAfterStd 8.374598e-02
[Update 664] Samples 2048 | Reward mean/std -0.079910/0.220844 | Value mean/std -0.070790/0.161846 | Adv std 1.240099e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.079910 | AvgAdv -0.000000 | AdvAfterStd 9.150139e-02
[Update 665] Samples 2048 | Reward mean/std -0.073300/0.200684 | Value mean/std -0.076012/0.186878 | Adv std 1.027248e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.073300 | AvgAdv -0.000000 | AdvAfterStd 8.236057e-02
[Update 666] Samples 2048 | Reward mean/std -0.064380/0.140183 | Value mean/std -0.064960/0.125890 | Adv std 1.035953e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.064380 | AvgAdv -0.000000 | AdvAfterStd 7.858267e-02
[Update 667] Samples 2048 | Reward mean/std -0.073754/0.195811 | Value mean/std -0.072968/0.136503 | Adv std 1.250498e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.073754 | AvgAdv -0.000000 | AdvAfterStd 8.359281e-02
[Update 668] Samples 2048 | Reward mean/std -0.072221/0.180816 | Value mean/std -0.077066/0.129714 | Adv std 1.202548e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.072221 | AvgAdv 0.000000 | AdvAfterStd 8.398693e-02
[Update 669] Samples 2048 | Reward mean/std -0.076631/0.449217 | Value mean/std -0.079048/0.350858 | Adv std 1.608791e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.076631 | AvgAdv -0.000000 | AdvAfterStd 1.113250e-01
[Update 670] Samples 2048 | Reward mean/std -0.067715/0.175805 | Value mean/std -0.069271/0.129367 | Adv std 1.082485e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.067715 | AvgAdv 0.000000 | AdvAfterStd 7.606149e-02
[Update 671] Samples 2048 | Reward mean/std -0.071768/0.235983 | Value mean/std -0.068946/0.185341 | Adv std 1.277485e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.071768 | AvgAdv -0.000000 | AdvAfterStd 8.208305e-02
[Update 672] Samples 2048 | Reward mean/std -0.064342/0.160204 | Value mean/std -0.061780/0.143585 | Adv std 9.931787e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.064342 | AvgAdv 0.000000 | AdvAfterStd 8.178127e-02
[Update 673] Samples 2048 | Reward mean/std -0.071914/0.193622 | Value mean/std -0.071592/0.156173 | Adv std 1.260690e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.071914 | AvgAdv -0.000000 | AdvAfterStd 9.309264e-02
[Update 674] Samples 2048 | Reward mean/std -0.070685/0.217135 | Value mean/std -0.060548/0.163781 | Adv std 1.130363e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.070685 | AvgAdv 0.000000 | AdvAfterStd 7.337216e-02
[Update 675] Samples 2048 | Reward mean/std -0.075114/0.345717 | Value mean/std -0.071476/0.310399 | Adv std 1.146326e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.075114 | AvgAdv 0.000000 | AdvAfterStd 7.727987e-02
[Update 676] Samples 2048 | Reward mean/std -0.077674/0.290316 | Value mean/std -0.070966/0.172928 | Adv std 2.290979e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.077674 | AvgAdv 0.000000 | AdvAfterStd 1.187792e-01
[Update 677] Samples 2048 | Reward mean/std -0.066890/0.155734 | Value mean/std -0.072136/0.150813 | Adv std 1.047623e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.066890 | AvgAdv 0.000000 | AdvAfterStd 6.795249e-02
[Update 678] Samples 2048 | Reward mean/std -0.067267/0.261568 | Value mean/std -0.070320/0.267044 | Adv std 1.090042e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.067267 | AvgAdv -0.000000 | AdvAfterStd 8.186232e-02
[Update 679] Samples 2048 | Reward mean/std -0.065904/0.140055 | Value mean/std -0.054284/0.094534 | Adv std 1.057718e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.065904 | AvgAdv 0.000000 | AdvAfterStd 8.962119e-02
[Update 680] Samples 2048 | Reward mean/std -0.078930/0.309586 | Value mean/std -0.071910/0.283517 | Adv std 1.389641e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.078930 | AvgAdv -0.000000 | AdvAfterStd 8.506320e-02
[Update 681] Samples 2048 | Reward mean/std -0.071940/0.178422 | Value mean/std -0.076412/0.165981 | Adv std 1.071134e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.071940 | AvgAdv -0.000000 | AdvAfterStd 7.945255e-02
[Update 682] Samples 2048 | Reward mean/std -0.073237/0.345860 | Value mean/std -0.069006/0.247934 | Adv std 1.419739e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.073237 | AvgAdv 0.000000 | AdvAfterStd 7.279211e-02
[Update 683] Samples 2048 | Reward mean/std -0.072626/0.201762 | Value mean/std -0.064125/0.157013 | Adv std 1.455092e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.072626 | AvgAdv -0.000000 | AdvAfterStd 8.359749e-02
[Update 684] Samples 2048 | Reward mean/std -0.066182/0.167402 | Value mean/std -0.066078/0.142372 | Adv std 1.112683e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.066182 | AvgAdv -0.000000 | AdvAfterStd 7.061423e-02
[Update 685] Samples 2048 | Reward mean/std -0.073433/0.235450 | Value mean/std -0.071287/0.218492 | Adv std 1.564524e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.073433 | AvgAdv -0.000000 | AdvAfterStd 8.423815e-02
[Update 686] Samples 2048 | Reward mean/std -0.074478/0.271181 | Value mean/std -0.073355/0.181496 | Adv std 1.839305e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.074478 | AvgAdv 0.000000 | AdvAfterStd 7.528005e-02
[Update 687] Samples 2048 | Reward mean/std -0.067056/0.198834 | Value mean/std -0.066365/0.161083 | Adv std 1.135257e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.067056 | AvgAdv 0.000000 | AdvAfterStd 7.586475e-02
[Update 688] Samples 2048 | Reward mean/std -0.077955/0.385318 | Value mean/std -0.072974/0.312206 | Adv std 1.403432e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.077955 | AvgAdv -0.000000 | AdvAfterStd 1.035045e-01
[Update 689] Samples 2048 | Reward mean/std -0.070759/0.171332 | Value mean/std -0.064972/0.115866 | Adv std 1.232264e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.070759 | AvgAdv 0.000000 | AdvAfterStd 8.385989e-02
[Update 690] Samples 2048 | Reward mean/std -0.078778/0.394506 | Value mean/std -0.071380/0.311785 | Adv std 1.564668e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.078778 | AvgAdv 0.000000 | AdvAfterStd 8.295120e-02
[Update 691] Samples 2048 | Reward mean/std -0.069706/0.236021 | Value mean/std -0.062823/0.137061 | Adv std 1.511430e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.069706 | AvgAdv -0.000000 | AdvAfterStd 8.822130e-02
[Update 692] Samples 2048 | Reward mean/std -0.068026/0.166929 | Value mean/std -0.065101/0.115050 | Adv std 1.232288e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.068026 | AvgAdv -0.000000 | AdvAfterStd 8.146044e-02
[Update 693] Samples 2048 | Reward mean/std -0.065059/0.251498 | Value mean/std -0.069888/0.243444 | Adv std 1.166837e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.065059 | AvgAdv -0.000000 | AdvAfterStd 6.981053e-02
[Update 694] Samples 2048 | Reward mean/std -0.061175/0.176286 | Value mean/std -0.065993/0.197432 | Adv std 1.143764e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.061175 | AvgAdv 0.000000 | AdvAfterStd 7.165772e-02
[Update 695] Samples 2048 | Reward mean/std -0.063235/0.151923 | Value mean/std -0.064648/0.162128 | Adv std 1.280073e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.063235 | AvgAdv 0.000000 | AdvAfterStd 7.437452e-02
[Update 696] Samples 2048 | Reward mean/std -0.075569/0.310285 | Value mean/std -0.066481/0.244368 | Adv std 1.288924e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.075569 | AvgAdv 0.000000 | AdvAfterStd 7.879458e-02
[Update 697] Samples 2048 | Reward mean/std -0.060932/0.146313 | Value mean/std -0.066653/0.174042 | Adv std 1.329285e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.060932 | AvgAdv 0.000000 | AdvAfterStd 8.474268e-02
[Update 698] Samples 2048 | Reward mean/std -0.069007/0.245516 | Value mean/std -0.056375/0.134000 | Adv std 2.040778e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.069007 | AvgAdv 0.000000 | AdvAfterStd 7.778116e-02
[Update 699] Samples 2048 | Reward mean/std -0.088365/0.389727 | Value mean/std -0.086422/0.283786 | Adv std 1.953417e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.088365 | AvgAdv -0.000000 | AdvAfterStd 7.123000e-02
[Update 700] Samples 2048 | Reward mean/std -0.077297/0.305225 | Value mean/std -0.074674/0.266150 | Adv std 1.378393e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.077297 | AvgAdv 0.000000 | AdvAfterStd 1.054234e-01
Saved checkpoint at update 700
[Update 701] Samples 2048 | Reward mean/std -0.070791/0.252406 | Value mean/std -0.074017/0.301272 | Adv std 1.840213e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.070791 | AvgAdv 0.000000 | AdvAfterStd 1.333957e-01
[Update 702] Samples 2048 | Reward mean/std -0.075821/0.220699 | Value mean/std -0.069093/0.193610 | Adv std 1.419993e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.075821 | AvgAdv -0.000000 | AdvAfterStd 9.925126e-02
[Update 703] Samples 2048 | Reward mean/std -0.072317/0.229086 | Value mean/std -0.071546/0.183843 | Adv std 1.140758e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.072317 | AvgAdv 0.000000 | AdvAfterStd 8.715615e-02
[Update 704] Samples 2048 | Reward mean/std -0.064876/0.158300 | Value mean/std -0.070666/0.163600 | Adv std 1.127682e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.064876 | AvgAdv -0.000000 | AdvAfterStd 8.456215e-02
[Update 705] Samples 2048 | Reward mean/std -0.066513/0.198262 | Value mean/std -0.072928/0.155077 | Adv std 1.219549e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.066513 | AvgAdv -0.000000 | AdvAfterStd 7.296457e-02
[Update 706] Samples 2048 | Reward mean/std -0.069050/0.205501 | Value mean/std -0.064939/0.145784 | Adv std 1.588147e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.069050 | AvgAdv -0.000000 | AdvAfterStd 9.917896e-02
[Update 707] Samples 2048 | Reward mean/std -0.069279/0.222965 | Value mean/std -0.069398/0.218511 | Adv std 1.254044e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.069279 | AvgAdv -0.000000 | AdvAfterStd 8.161768e-02
[Update 708] Samples 2048 | Reward mean/std -0.065124/0.153486 | Value mean/std -0.059786/0.127509 | Adv std 1.333279e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.065124 | AvgAdv -0.000000 | AdvAfterStd 7.640467e-02
[Update 709] Samples 2048 | Reward mean/std -0.066769/0.179842 | Value mean/std -0.063224/0.128674 | Adv std 1.049814e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.066769 | AvgAdv 0.000000 | AdvAfterStd 8.415178e-02
[Update 710] Samples 2048 | Reward mean/std -0.070770/0.251698 | Value mean/std -0.074407/0.192401 | Adv std 1.253267e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.070770 | AvgAdv 0.000000 | AdvAfterStd 8.005433e-02
[Update 711] Samples 2048 | Reward mean/std -0.063399/0.221105 | Value mean/std -0.073408/0.254416 | Adv std 1.211342e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.063399 | AvgAdv 0.000000 | AdvAfterStd 6.775554e-02
[Update 712] Samples 2048 | Reward mean/std -0.072613/0.224267 | Value mean/std -0.068793/0.185350 | Adv std 1.284610e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.072613 | AvgAdv 0.000000 | AdvAfterStd 8.939799e-02
[Update 713] Samples 2048 | Reward mean/std -0.065283/0.281801 | Value mean/std -0.055065/0.143039 | Adv std 1.963948e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.065283 | AvgAdv -0.000000 | AdvAfterStd 7.452931e-02
[Update 714] Samples 2048 | Reward mean/std -0.060355/0.147409 | Value mean/std -0.060598/0.147734 | Adv std 9.396628e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.060355 | AvgAdv 0.000000 | AdvAfterStd 6.367778e-02
[Update 715] Samples 2048 | Reward mean/std -0.063631/0.185829 | Value mean/std -0.059301/0.099372 | Adv std 1.414867e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.063631 | AvgAdv 0.000000 | AdvAfterStd 8.415093e-02
[Update 716] Samples 2048 | Reward mean/std -0.062640/0.147615 | Value mean/std -0.070498/0.180355 | Adv std 1.291310e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.062640 | AvgAdv 0.000000 | AdvAfterStd 7.476347e-02
[Update 717] Samples 2048 | Reward mean/std -0.068832/0.265133 | Value mean/std -0.066575/0.172510 | Adv std 1.326807e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.068832 | AvgAdv 0.000000 | AdvAfterStd 7.103433e-02
[Update 718] Samples 2048 | Reward mean/std -0.064475/0.151869 | Value mean/std -0.061410/0.141960 | Adv std 1.096315e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.064475 | AvgAdv 0.000000 | AdvAfterStd 8.210340e-02
[Update 719] Samples 2048 | Reward mean/std -0.061738/0.150828 | Value mean/std -0.063890/0.146265 | Adv std 1.334131e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.061738 | AvgAdv 0.000000 | AdvAfterStd 8.039629e-02
[Update 720] Samples 2048 | Reward mean/std -0.067752/0.169606 | Value mean/std -0.063446/0.118142 | Adv std 1.051540e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.067752 | AvgAdv 0.000000 | AdvAfterStd 8.636999e-02
[Update 721] Samples 2048 | Reward mean/std -0.064310/0.187147 | Value mean/std -0.066829/0.190533 | Adv std 1.017733e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.064310 | AvgAdv 0.000000 | AdvAfterStd 7.166956e-02
[Update 722] Samples 2048 | Reward mean/std -0.067498/0.211217 | Value mean/std -0.058579/0.111648 | Adv std 1.321863e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.067498 | AvgAdv -0.000000 | AdvAfterStd 7.627556e-02
[Update 723] Samples 2048 | Reward mean/std -0.065936/0.171395 | Value mean/std -0.063906/0.144411 | Adv std 1.026412e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.065936 | AvgAdv 0.000000 | AdvAfterStd 8.224774e-02
[Update 724] Samples 2048 | Reward mean/std -0.065674/0.142627 | Value mean/std -0.062007/0.101602 | Adv std 9.949480e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.065674 | AvgAdv 0.000000 | AdvAfterStd 8.009689e-02
[Update 725] Samples 2048 | Reward mean/std -0.065796/0.215139 | Value mean/std -0.065695/0.201159 | Adv std 1.129894e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.065796 | AvgAdv -0.000000 | AdvAfterStd 7.159656e-02
[Update 726] Samples 2048 | Reward mean/std -0.075671/0.314523 | Value mean/std -0.069395/0.220838 | Adv std 1.701476e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.075671 | AvgAdv -0.000000 | AdvAfterStd 7.774735e-02
[Update 727] Samples 2048 | Reward mean/std -0.063410/0.137773 | Value mean/std -0.056440/0.098944 | Adv std 1.021184e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.063410 | AvgAdv -0.000000 | AdvAfterStd 7.610953e-02
[Update 728] Samples 2048 | Reward mean/std -0.064919/0.147275 | Value mean/std -0.060094/0.113883 | Adv std 9.723122e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.064919 | AvgAdv 0.000000 | AdvAfterStd 6.922996e-02
[Update 729] Samples 2048 | Reward mean/std -0.065446/0.159353 | Value mean/std -0.060290/0.116978 | Adv std 1.135930e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.065446 | AvgAdv -0.000000 | AdvAfterStd 7.910150e-02
[Update 730] Samples 2048 | Reward mean/std -0.069927/0.206163 | Value mean/std -0.070179/0.130974 | Adv std 1.209973e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.069927 | AvgAdv 0.000000 | AdvAfterStd 8.794559e-02
[Update 731] Samples 2048 | Reward mean/std -0.061913/0.213710 | Value mean/std -0.070291/0.175674 | Adv std 9.018882e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.061913 | AvgAdv -0.000000 | AdvAfterStd 6.654441e-02
[Update 732] Samples 2048 | Reward mean/std -0.069912/0.159390 | Value mean/std -0.066821/0.099149 | Adv std 1.136572e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.069912 | AvgAdv 0.000000 | AdvAfterStd 9.127734e-02
[Update 733] Samples 2048 | Reward mean/std -0.066069/0.157080 | Value mean/std -0.080071/0.154162 | Adv std 1.093981e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.066069 | AvgAdv 0.000000 | AdvAfterStd 6.535476e-02
[Update 734] Samples 2048 | Reward mean/std -0.057017/0.123390 | Value mean/std -0.049914/0.082890 | Adv std 9.104677e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.057017 | AvgAdv 0.000000 | AdvAfterStd 7.212143e-02
[Update 735] Samples 2048 | Reward mean/std -0.064590/0.146553 | Value mean/std -0.066864/0.145762 | Adv std 1.162554e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.064590 | AvgAdv 0.000000 | AdvAfterStd 7.966975e-02
[Update 736] Samples 2048 | Reward mean/std -0.070404/0.231839 | Value mean/std -0.067835/0.148914 | Adv std 1.326836e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.070404 | AvgAdv -0.000000 | AdvAfterStd 7.764665e-02
[Update 737] Samples 2048 | Reward mean/std -0.066171/0.163770 | Value mean/std -0.068882/0.151966 | Adv std 1.132053e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.066171 | AvgAdv 0.000000 | AdvAfterStd 8.014811e-02
[Update 738] Samples 2048 | Reward mean/std -0.068771/0.286192 | Value mean/std -0.072065/0.237504 | Adv std 9.853827e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.068771 | AvgAdv 0.000000 | AdvAfterStd 8.433854e-02
[Update 739] Samples 2048 | Reward mean/std -0.062597/0.154517 | Value mean/std -0.067109/0.089139 | Adv std 1.056746e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.062597 | AvgAdv -0.000000 | AdvAfterStd 7.601094e-02
[Update 740] Samples 2048 | Reward mean/std -0.063660/0.144686 | Value mean/std -0.064864/0.141084 | Adv std 1.143664e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.063660 | AvgAdv -0.000000 | AdvAfterStd 7.532620e-02
[Update 741] Samples 2048 | Reward mean/std -0.071231/0.196970 | Value mean/std -0.068090/0.140884 | Adv std 1.175268e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.071231 | AvgAdv -0.000000 | AdvAfterStd 7.633606e-02
[Update 742] Samples 2048 | Reward mean/std -0.069577/0.292696 | Value mean/std -0.069088/0.267620 | Adv std 1.692994e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.069577 | AvgAdv 0.000000 | AdvAfterStd 8.288280e-02
[Update 743] Samples 2048 | Reward mean/std -0.064918/0.186704 | Value mean/std -0.068920/0.163756 | Adv std 1.115250e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.064918 | AvgAdv 0.000000 | AdvAfterStd 8.082834e-02
[Update 744] Samples 2048 | Reward mean/std -0.066822/0.161224 | Value mean/std -0.064689/0.145204 | Adv std 1.246590e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.066822 | AvgAdv 0.000000 | AdvAfterStd 7.991025e-02
[Update 745] Samples 2048 | Reward mean/std -0.062341/0.143761 | Value mean/std -0.060689/0.121803 | Adv std 1.020770e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.062341 | AvgAdv 0.000000 | AdvAfterStd 7.469882e-02
[Update 746] Samples 2048 | Reward mean/std -0.061902/0.153172 | Value mean/std -0.063697/0.138145 | Adv std 1.450204e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.061902 | AvgAdv -0.000000 | AdvAfterStd 7.536036e-02
[Update 747] Samples 2048 | Reward mean/std -0.071030/0.211240 | Value mean/std -0.069003/0.193861 | Adv std 1.270313e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.071030 | AvgAdv -0.000000 | AdvAfterStd 8.846144e-02
[Update 748] Samples 2048 | Reward mean/std -0.065669/0.184413 | Value mean/std -0.068263/0.144721 | Adv std 1.142905e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.065669 | AvgAdv 0.000000 | AdvAfterStd 8.249582e-02
[Update 749] Samples 2048 | Reward mean/std -0.066089/0.183760 | Value mean/std -0.064015/0.182070 | Adv std 1.038618e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.066089 | AvgAdv 0.000000 | AdvAfterStd 8.144468e-02
[Update 750] Samples 2048 | Reward mean/std -0.072117/0.187637 | Value mean/std -0.064809/0.133792 | Adv std 1.214438e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.072117 | AvgAdv -0.000000 | AdvAfterStd 9.873675e-02
Saved checkpoint at update 750
[Update 751] Samples 2048 | Reward mean/std -0.070046/0.200933 | Value mean/std -0.072070/0.151789 | Adv std 1.212029e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.070046 | AvgAdv -0.000000 | AdvAfterStd 7.779123e-02
[Update 752] Samples 2048 | Reward mean/std -0.064144/0.164987 | Value mean/std -0.069822/0.187907 | Adv std 1.247939e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.064144 | AvgAdv -0.000000 | AdvAfterStd 8.504418e-02
[Update 753] Samples 2048 | Reward mean/std -0.072782/0.300825 | Value mean/std -0.075497/0.218019 | Adv std 1.331066e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.072782 | AvgAdv -0.000000 | AdvAfterStd 8.836474e-02
[Update 754] Samples 2048 | Reward mean/std -0.060652/0.158847 | Value mean/std -0.060701/0.097327 | Adv std 1.240019e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.060652 | AvgAdv 0.000000 | AdvAfterStd 7.534661e-02
[Update 755] Samples 2048 | Reward mean/std -0.073947/0.185143 | Value mean/std -0.066447/0.148953 | Adv std 1.119336e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.073947 | AvgAdv 0.000000 | AdvAfterStd 8.481046e-02
[Update 756] Samples 2048 | Reward mean/std -0.067014/0.213898 | Value mean/std -0.065679/0.152165 | Adv std 1.337327e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.067014 | AvgAdv 0.000000 | AdvAfterStd 7.757828e-02
[Update 757] Samples 2048 | Reward mean/std -0.063755/0.191569 | Value mean/std -0.062657/0.180236 | Adv std 1.024259e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.063755 | AvgAdv -0.000000 | AdvAfterStd 7.657317e-02
[Update 758] Samples 2048 | Reward mean/std -0.067940/0.194078 | Value mean/std -0.073249/0.255762 | Adv std 1.586440e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.067940 | AvgAdv -0.000000 | AdvAfterStd 8.194939e-02
[Update 759] Samples 2048 | Reward mean/std -0.064755/0.153585 | Value mean/std -0.062385/0.113712 | Adv std 1.141291e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.064755 | AvgAdv 0.000000 | AdvAfterStd 8.280331e-02
[Update 760] Samples 2048 | Reward mean/std -0.065279/0.151365 | Value mean/std -0.061214/0.105936 | Adv std 1.207045e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.065279 | AvgAdv 0.000000 | AdvAfterStd 8.446676e-02
[Update 761] Samples 2048 | Reward mean/std -0.068821/0.178887 | Value mean/std -0.072744/0.146415 | Adv std 1.352307e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.068821 | AvgAdv 0.000000 | AdvAfterStd 9.884481e-02
[Update 762] Samples 2048 | Reward mean/std -0.069180/0.169486 | Value mean/std -0.073171/0.134858 | Adv std 1.139288e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.069180 | AvgAdv 0.000000 | AdvAfterStd 7.929940e-02
[Update 763] Samples 2048 | Reward mean/std -0.063127/0.147061 | Value mean/std -0.065597/0.107520 | Adv std 1.152947e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.063127 | AvgAdv 0.000000 | AdvAfterStd 8.255090e-02
[Update 764] Samples 2048 | Reward mean/std -0.065514/0.153333 | Value mean/std -0.067960/0.104573 | Adv std 1.033415e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.065514 | AvgAdv 0.000000 | AdvAfterStd 6.747673e-02
[Update 765] Samples 2048 | Reward mean/std -0.068607/0.161166 | Value mean/std -0.067560/0.110781 | Adv std 1.068993e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.068607 | AvgAdv 0.000000 | AdvAfterStd 8.776104e-02
[Update 766] Samples 2048 | Reward mean/std -0.067678/0.160180 | Value mean/std -0.059534/0.136802 | Adv std 9.647819e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.067678 | AvgAdv -0.000000 | AdvAfterStd 7.517751e-02
[Update 767] Samples 2048 | Reward mean/std -0.065325/0.152430 | Value mean/std -0.064864/0.125814 | Adv std 1.022538e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.065325 | AvgAdv -0.000000 | AdvAfterStd 8.360790e-02
[Update 768] Samples 2048 | Reward mean/std -0.064577/0.158010 | Value mean/std -0.066484/0.114281 | Adv std 9.644340e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.064577 | AvgAdv 0.000000 | AdvAfterStd 7.091375e-02
[Update 769] Samples 2048 | Reward mean/std -0.057064/0.131326 | Value mean/std -0.061699/0.102261 | Adv std 9.036570e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.057064 | AvgAdv 0.000000 | AdvAfterStd 6.421354e-02
[Update 770] Samples 2048 | Reward mean/std -0.065117/0.173239 | Value mean/std -0.058436/0.099695 | Adv std 1.252978e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.065117 | AvgAdv 0.000000 | AdvAfterStd 8.483417e-02
[Update 771] Samples 2048 | Reward mean/std -0.068312/0.172159 | Value mean/std -0.074088/0.167928 | Adv std 1.290943e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.068312 | AvgAdv -0.000000 | AdvAfterStd 7.874306e-02
[Update 772] Samples 2048 | Reward mean/std -0.065610/0.181028 | Value mean/std -0.062858/0.122261 | Adv std 1.160384e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.065610 | AvgAdv 0.000000 | AdvAfterStd 7.200621e-02
[Update 773] Samples 2048 | Reward mean/std -0.062511/0.140722 | Value mean/std -0.066542/0.127026 | Adv std 1.094873e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.062511 | AvgAdv -0.000000 | AdvAfterStd 7.573307e-02
[Update 774] Samples 2048 | Reward mean/std -0.070442/0.157204 | Value mean/std -0.062576/0.139933 | Adv std 1.281921e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.070442 | AvgAdv -0.000000 | AdvAfterStd 8.021814e-02
[Update 775] Samples 2048 | Reward mean/std -0.064573/0.164848 | Value mean/std -0.069216/0.097438 | Adv std 1.363971e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.064573 | AvgAdv 0.000000 | AdvAfterStd 8.248183e-02
[Update 776] Samples 2048 | Reward mean/std -0.064533/0.174639 | Value mean/std -0.059438/0.118755 | Adv std 1.137755e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.064533 | AvgAdv 0.000000 | AdvAfterStd 7.501616e-02
[Update 777] Samples 2048 | Reward mean/std -0.063166/0.154466 | Value mean/std -0.059381/0.118553 | Adv std 1.028308e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.063166 | AvgAdv -0.000000 | AdvAfterStd 7.868853e-02
[Update 778] Samples 2048 | Reward mean/std -0.068792/0.243665 | Value mean/std -0.072900/0.228716 | Adv std 1.224409e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.068792 | AvgAdv -0.000000 | AdvAfterStd 8.330507e-02
[Update 779] Samples 2048 | Reward mean/std -0.065722/0.160539 | Value mean/std -0.067152/0.182238 | Adv std 1.536967e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.065722 | AvgAdv 0.000000 | AdvAfterStd 8.112852e-02
[Update 780] Samples 2048 | Reward mean/std -0.066680/0.175227 | Value mean/std -0.064808/0.113863 | Adv std 1.217579e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.066680 | AvgAdv -0.000000 | AdvAfterStd 7.882488e-02
[Update 781] Samples 2048 | Reward mean/std -0.066067/0.181594 | Value mean/std -0.071289/0.179432 | Adv std 1.140079e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.066067 | AvgAdv -0.000000 | AdvAfterStd 6.595068e-02
[Update 782] Samples 2048 | Reward mean/std -0.071928/0.177761 | Value mean/std -0.070666/0.132719 | Adv std 1.293978e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.071928 | AvgAdv -0.000000 | AdvAfterStd 7.395492e-02
[Update 783] Samples 2048 | Reward mean/std -0.066952/0.153649 | Value mean/std -0.073034/0.124857 | Adv std 1.232406e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.066952 | AvgAdv -0.000000 | AdvAfterStd 8.346602e-02
[Update 784] Samples 2048 | Reward mean/std -0.066375/0.152183 | Value mean/std -0.066091/0.115561 | Adv std 1.150060e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.066375 | AvgAdv -0.000000 | AdvAfterStd 8.076432e-02
[Update 785] Samples 2048 | Reward mean/std -0.065041/0.154028 | Value mean/std -0.068049/0.118553 | Adv std 1.080089e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.065041 | AvgAdv 0.000000 | AdvAfterStd 8.080085e-02
[Update 786] Samples 2048 | Reward mean/std -0.066396/0.145748 | Value mean/std -0.064515/0.120326 | Adv std 9.572695e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.066396 | AvgAdv 0.000000 | AdvAfterStd 8.221554e-02
[Update 787] Samples 2048 | Reward mean/std -0.066735/0.158921 | Value mean/std -0.062967/0.110994 | Adv std 1.099662e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.066735 | AvgAdv 0.000000 | AdvAfterStd 8.439416e-02
[Update 788] Samples 2048 | Reward mean/std -0.070950/0.192982 | Value mean/std -0.062644/0.133169 | Adv std 1.257541e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.070950 | AvgAdv 0.000000 | AdvAfterStd 8.848991e-02
[Update 789] Samples 2048 | Reward mean/std -0.068079/0.194291 | Value mean/std -0.067596/0.138045 | Adv std 1.232291e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.068079 | AvgAdv 0.000000 | AdvAfterStd 8.966593e-02
[Update 790] Samples 2048 | Reward mean/std -0.070196/0.169649 | Value mean/std -0.068356/0.182137 | Adv std 1.313071e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.070196 | AvgAdv 0.000000 | AdvAfterStd 8.749100e-02
[Update 791] Samples 2048 | Reward mean/std -0.073268/0.189336 | Value mean/std -0.066382/0.163671 | Adv std 1.295636e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.073268 | AvgAdv 0.000000 | AdvAfterStd 8.160776e-02
[Update 792] Samples 2048 | Reward mean/std -0.069427/0.174635 | Value mean/std -0.063874/0.132667 | Adv std 1.294071e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.069427 | AvgAdv -0.000000 | AdvAfterStd 9.105520e-02
[Update 793] Samples 2048 | Reward mean/std -0.067405/0.179660 | Value mean/std -0.073608/0.110174 | Adv std 1.135681e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.067405 | AvgAdv -0.000000 | AdvAfterStd 9.408476e-02
[Update 794] Samples 2048 | Reward mean/std -0.068427/0.190566 | Value mean/std -0.068243/0.137487 | Adv std 1.423692e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.068427 | AvgAdv -0.000000 | AdvAfterStd 7.546940e-02
[Update 795] Samples 2048 | Reward mean/std -0.059897/0.129375 | Value mean/std -0.062600/0.135358 | Adv std 1.091773e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.059897 | AvgAdv -0.000000 | AdvAfterStd 6.995684e-02
[Update 796] Samples 2048 | Reward mean/std -0.066360/0.148993 | Value mean/std -0.061409/0.131441 | Adv std 9.868617e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.066360 | AvgAdv 0.000000 | AdvAfterStd 8.020896e-02
[Update 797] Samples 2048 | Reward mean/std -0.066679/0.157240 | Value mean/std -0.072528/0.110204 | Adv std 1.062377e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.066679 | AvgAdv 0.000000 | AdvAfterStd 9.041796e-02
[Update 798] Samples 2048 | Reward mean/std -0.067436/0.170702 | Value mean/std -0.072869/0.124582 | Adv std 1.027239e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.067436 | AvgAdv -0.000000 | AdvAfterStd 7.579211e-02
[Update 799] Samples 2048 | Reward mean/std -0.062733/0.162901 | Value mean/std -0.060466/0.137934 | Adv std 1.121319e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.062733 | AvgAdv 0.000000 | AdvAfterStd 6.678020e-02
[Update 800] Samples 2048 | Reward mean/std -0.063140/0.181037 | Value mean/std -0.064356/0.137078 | Adv std 1.050470e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.063140 | AvgAdv -0.000000 | AdvAfterStd 7.251507e-02
Saved checkpoint at update 800
[Update 801] Samples 2048 | Reward mean/std -0.069838/0.234384 | Value mean/std -0.069113/0.209289 | Adv std 1.152855e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.069838 | AvgAdv 0.000000 | AdvAfterStd 8.914451e-02
[Update 802] Samples 2048 | Reward mean/std -0.062888/0.181143 | Value mean/std -0.059829/0.103025 | Adv std 1.287913e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.062888 | AvgAdv -0.000000 | AdvAfterStd 7.497619e-02
[Update 803] Samples 2048 | Reward mean/std -0.075131/0.202042 | Value mean/std -0.074320/0.197899 | Adv std 1.245843e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.075131 | AvgAdv 0.000000 | AdvAfterStd 7.922633e-02
[Update 804] Samples 2048 | Reward mean/std -0.063597/0.125487 | Value mean/std -0.066994/0.118989 | Adv std 1.086734e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.063597 | AvgAdv -0.000000 | AdvAfterStd 7.818190e-02
[Update 805] Samples 2048 | Reward mean/std -0.067826/0.206155 | Value mean/std -0.072954/0.172824 | Adv std 1.239297e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.067826 | AvgAdv -0.000000 | AdvAfterStd 7.685696e-02
[Update 806] Samples 2048 | Reward mean/std -0.064504/0.154565 | Value mean/std -0.056773/0.110795 | Adv std 9.992622e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.064504 | AvgAdv -0.000000 | AdvAfterStd 8.078096e-02
[Update 807] Samples 2048 | Reward mean/std -0.070141/0.238046 | Value mean/std -0.071423/0.164479 | Adv std 1.747383e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.070141 | AvgAdv 0.000000 | AdvAfterStd 1.429836e-01
[Update 808] Samples 2048 | Reward mean/std -0.064046/0.154057 | Value mean/std -0.058392/0.127220 | Adv std 9.787047e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.064046 | AvgAdv -0.000000 | AdvAfterStd 8.523209e-02
[Update 809] Samples 2048 | Reward mean/std -0.071049/0.248565 | Value mean/std -0.071782/0.172891 | Adv std 1.527317e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.071049 | AvgAdv 0.000000 | AdvAfterStd 9.429158e-02
[Update 810] Samples 2048 | Reward mean/std -0.063361/0.166728 | Value mean/std -0.070142/0.135097 | Adv std 1.210856e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.063361 | AvgAdv -0.000000 | AdvAfterStd 8.227823e-02
[Update 811] Samples 2048 | Reward mean/std -0.065989/0.219382 | Value mean/std -0.065329/0.208879 | Adv std 1.471801e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.065989 | AvgAdv 0.000000 | AdvAfterStd 7.321264e-02
[Update 812] Samples 2048 | Reward mean/std -0.059318/0.141878 | Value mean/std -0.065608/0.122049 | Adv std 1.156955e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.059318 | AvgAdv 0.000000 | AdvAfterStd 7.434833e-02
[Update 813] Samples 2048 | Reward mean/std -0.060107/0.152390 | Value mean/std -0.063231/0.144360 | Adv std 1.119787e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.060107 | AvgAdv 0.000000 | AdvAfterStd 8.442028e-02
[Update 814] Samples 2048 | Reward mean/std -0.064567/0.164807 | Value mean/std -0.065831/0.151264 | Adv std 1.177996e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.064567 | AvgAdv 0.000000 | AdvAfterStd 7.761911e-02
[Update 815] Samples 2048 | Reward mean/std -0.068916/0.213817 | Value mean/std -0.063196/0.185121 | Adv std 1.168994e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.068916 | AvgAdv -0.000000 | AdvAfterStd 8.324151e-02
[Update 816] Samples 2048 | Reward mean/std -0.063991/0.164222 | Value mean/std -0.066342/0.144004 | Adv std 1.079827e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.063991 | AvgAdv 0.000000 | AdvAfterStd 7.969777e-02
[Update 817] Samples 2048 | Reward mean/std -0.070576/0.225890 | Value mean/std -0.060163/0.152159 | Adv std 1.324047e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.070576 | AvgAdv 0.000000 | AdvAfterStd 7.436635e-02
[Update 818] Samples 2048 | Reward mean/std -0.066532/0.153556 | Value mean/std -0.071822/0.136798 | Adv std 1.007913e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.066532 | AvgAdv -0.000000 | AdvAfterStd 7.828719e-02
[Update 819] Samples 2048 | Reward mean/std -0.076339/0.260422 | Value mean/std -0.073838/0.169020 | Adv std 1.408283e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.076339 | AvgAdv 0.000000 | AdvAfterStd 8.473634e-02
[Update 820] Samples 2048 | Reward mean/std -0.066156/0.184027 | Value mean/std -0.068589/0.173257 | Adv std 1.099252e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.066156 | AvgAdv -0.000000 | AdvAfterStd 8.515158e-02
[Update 821] Samples 2048 | Reward mean/std -0.072556/0.253582 | Value mean/std -0.073666/0.179160 | Adv std 1.330467e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.072556 | AvgAdv -0.000000 | AdvAfterStd 1.055740e-01
[Update 822] Samples 2048 | Reward mean/std -0.073687/0.226310 | Value mean/std -0.071688/0.298498 | Adv std 1.998795e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.073687 | AvgAdv 0.000000 | AdvAfterStd 9.647074e-02
[Update 823] Samples 2048 | Reward mean/std -0.058518/0.162173 | Value mean/std -0.069284/0.157025 | Adv std 9.480526e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.058518 | AvgAdv -0.000000 | AdvAfterStd 7.075094e-02
[Update 824] Samples 2048 | Reward mean/std -0.059797/0.154467 | Value mean/std -0.055627/0.109167 | Adv std 9.587068e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.059797 | AvgAdv -0.000000 | AdvAfterStd 6.791314e-02
[Update 825] Samples 2048 | Reward mean/std -0.058250/0.125795 | Value mean/std -0.060068/0.089619 | Adv std 9.838047e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.058250 | AvgAdv -0.000000 | AdvAfterStd 8.208671e-02
[Update 826] Samples 2048 | Reward mean/std -0.061476/0.135441 | Value mean/std -0.059590/0.107917 | Adv std 1.169180e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.061476 | AvgAdv -0.000000 | AdvAfterStd 8.196768e-02
[Update 827] Samples 2048 | Reward mean/std -0.066205/0.178395 | Value mean/std -0.067059/0.117131 | Adv std 1.126877e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.066205 | AvgAdv 0.000000 | AdvAfterStd 8.353623e-02
[Update 828] Samples 2048 | Reward mean/std -0.073594/0.193752 | Value mean/std -0.064072/0.132660 | Adv std 1.316386e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.073594 | AvgAdv 0.000000 | AdvAfterStd 8.850230e-02
[Update 829] Samples 2048 | Reward mean/std -0.065484/0.164057 | Value mean/std -0.067545/0.126634 | Adv std 1.162136e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.065484 | AvgAdv -0.000000 | AdvAfterStd 8.120257e-02
[Update 830] Samples 2048 | Reward mean/std -0.067116/0.180039 | Value mean/std -0.067503/0.124458 | Adv std 1.357334e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.067116 | AvgAdv 0.000000 | AdvAfterStd 8.006440e-02
[Update 831] Samples 2048 | Reward mean/std -0.067722/0.206186 | Value mean/std -0.063230/0.166366 | Adv std 1.253933e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.067722 | AvgAdv 0.000000 | AdvAfterStd 1.002866e-01
[Update 832] Samples 2048 | Reward mean/std -0.073074/0.206541 | Value mean/std -0.060621/0.138696 | Adv std 1.335359e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.073074 | AvgAdv 0.000000 | AdvAfterStd 8.635421e-02
[Update 833] Samples 2048 | Reward mean/std -0.062515/0.168312 | Value mean/std -0.074443/0.188698 | Adv std 1.269245e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.062515 | AvgAdv 0.000000 | AdvAfterStd 7.362978e-02
[Update 834] Samples 2048 | Reward mean/std -0.059377/0.133770 | Value mean/std -0.056743/0.114465 | Adv std 8.651519e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.059377 | AvgAdv 0.000000 | AdvAfterStd 6.702090e-02
[Update 835] Samples 2048 | Reward mean/std -0.073563/0.247362 | Value mean/std -0.065361/0.174443 | Adv std 1.461331e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.073563 | AvgAdv -0.000000 | AdvAfterStd 8.284780e-02
[Update 836] Samples 2048 | Reward mean/std -0.066130/0.200962 | Value mean/std -0.065019/0.185629 | Adv std 1.059599e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.066130 | AvgAdv 0.000000 | AdvAfterStd 8.612042e-02
[Update 837] Samples 2048 | Reward mean/std -0.065420/0.177755 | Value mean/std -0.071483/0.137617 | Adv std 9.897728e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.065420 | AvgAdv -0.000000 | AdvAfterStd 7.477460e-02
[Update 838] Samples 2048 | Reward mean/std -0.061983/0.136643 | Value mean/std -0.063809/0.096510 | Adv std 9.735621e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.061983 | AvgAdv -0.000000 | AdvAfterStd 8.399889e-02
[Update 839] Samples 2048 | Reward mean/std -0.062295/0.131406 | Value mean/std -0.056041/0.096924 | Adv std 1.010542e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.062295 | AvgAdv -0.000000 | AdvAfterStd 7.424018e-02
[Update 840] Samples 2048 | Reward mean/std -0.059434/0.140909 | Value mean/std -0.055674/0.101167 | Adv std 8.626315e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.059434 | AvgAdv 0.000000 | AdvAfterStd 6.644881e-02
[Update 841] Samples 2048 | Reward mean/std -0.066661/0.176767 | Value mean/std -0.063196/0.140969 | Adv std 1.013428e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.066661 | AvgAdv -0.000000 | AdvAfterStd 7.932521e-02
[Update 842] Samples 2048 | Reward mean/std -0.070441/0.213788 | Value mean/std -0.061556/0.127805 | Adv std 1.471084e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.070441 | AvgAdv -0.000000 | AdvAfterStd 1.080338e-01
[Update 843] Samples 2048 | Reward mean/std -0.071643/0.216240 | Value mean/std -0.066490/0.134057 | Adv std 1.435865e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.071643 | AvgAdv 0.000000 | AdvAfterStd 1.061576e-01
[Update 844] Samples 2048 | Reward mean/std -0.064303/0.181218 | Value mean/std -0.072899/0.158068 | Adv std 1.531906e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.064303 | AvgAdv -0.000000 | AdvAfterStd 9.308860e-02
[Update 845] Samples 2048 | Reward mean/std -0.060768/0.142592 | Value mean/std -0.054572/0.131375 | Adv std 9.861095e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.060768 | AvgAdv -0.000000 | AdvAfterStd 7.668856e-02
[Update 846] Samples 2048 | Reward mean/std -0.065171/0.151165 | Value mean/std -0.055603/0.111130 | Adv std 1.167419e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.065171 | AvgAdv -0.000000 | AdvAfterStd 8.673475e-02
[Update 847] Samples 2048 | Reward mean/std -0.061544/0.151551 | Value mean/std -0.070766/0.108990 | Adv std 1.228166e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.061544 | AvgAdv -0.000000 | AdvAfterStd 7.031798e-02
[Update 848] Samples 2048 | Reward mean/std -0.068848/0.246305 | Value mean/std -0.069553/0.177477 | Adv std 1.331531e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.068848 | AvgAdv 0.000000 | AdvAfterStd 6.773674e-02
[Update 849] Samples 2048 | Reward mean/std -0.066609/0.170837 | Value mean/std -0.065756/0.151757 | Adv std 1.180570e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.066609 | AvgAdv -0.000000 | AdvAfterStd 8.452130e-02
[Update 850] Samples 2048 | Reward mean/std -0.066409/0.147422 | Value mean/std -0.067263/0.143770 | Adv std 1.289205e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.066409 | AvgAdv -0.000000 | AdvAfterStd 7.914986e-02
Saved checkpoint at update 850
[Update 851] Samples 2048 | Reward mean/std -0.069096/0.192941 | Value mean/std -0.061133/0.118554 | Adv std 1.413598e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.069096 | AvgAdv 0.000000 | AdvAfterStd 7.899792e-02
[Update 852] Samples 2048 | Reward mean/std -0.068688/0.175905 | Value mean/std -0.069607/0.166950 | Adv std 1.163505e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.068688 | AvgAdv -0.000000 | AdvAfterStd 9.004709e-02
[Update 853] Samples 2048 | Reward mean/std -0.064769/0.155342 | Value mean/std -0.068436/0.100189 | Adv std 1.176280e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.064769 | AvgAdv -0.000000 | AdvAfterStd 8.655658e-02
[Update 854] Samples 2048 | Reward mean/std -0.062781/0.172448 | Value mean/std -0.067990/0.138314 | Adv std 1.216204e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.062781 | AvgAdv 0.000000 | AdvAfterStd 7.285323e-02
[Update 855] Samples 2048 | Reward mean/std -0.067156/0.177640 | Value mean/std -0.061796/0.136033 | Adv std 1.286445e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.067156 | AvgAdv 0.000000 | AdvAfterStd 9.578443e-02
[Update 856] Samples 2048 | Reward mean/std -0.069750/0.225067 | Value mean/std -0.071651/0.170747 | Adv std 1.500550e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.069750 | AvgAdv 0.000000 | AdvAfterStd 1.040191e-01
[Update 857] Samples 2048 | Reward mean/std -0.069677/0.217661 | Value mean/std -0.063580/0.133834 | Adv std 1.701377e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.069677 | AvgAdv 0.000000 | AdvAfterStd 1.160949e-01
[Update 858] Samples 2048 | Reward mean/std -0.067602/0.184835 | Value mean/std -0.066390/0.152727 | Adv std 1.245732e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.067602 | AvgAdv -0.000000 | AdvAfterStd 8.691485e-02
[Update 859] Samples 2048 | Reward mean/std -0.067452/0.234709 | Value mean/std -0.071434/0.195249 | Adv std 1.191897e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.067452 | AvgAdv -0.000000 | AdvAfterStd 7.780084e-02
[Update 860] Samples 2048 | Reward mean/std -0.063844/0.187934 | Value mean/std -0.066322/0.160503 | Adv std 1.165572e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.063844 | AvgAdv -0.000000 | AdvAfterStd 8.375259e-02
[Update 861] Samples 2048 | Reward mean/std -0.068818/0.212264 | Value mean/std -0.061861/0.176829 | Adv std 9.788701e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.068818 | AvgAdv -0.000000 | AdvAfterStd 7.640520e-02
[Update 862] Samples 2048 | Reward mean/std -0.067319/0.182189 | Value mean/std -0.072505/0.145262 | Adv std 1.325593e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.067319 | AvgAdv 0.000000 | AdvAfterStd 8.733933e-02
[Update 863] Samples 2048 | Reward mean/std -0.070389/0.265108 | Value mean/std -0.062660/0.143462 | Adv std 1.627694e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.070389 | AvgAdv -0.000000 | AdvAfterStd 1.008444e-01
[Update 864] Samples 2048 | Reward mean/std -0.065943/0.178860 | Value mean/std -0.063246/0.125993 | Adv std 1.339409e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.065943 | AvgAdv 0.000000 | AdvAfterStd 8.997066e-02
[Update 865] Samples 2048 | Reward mean/std -0.063731/0.161554 | Value mean/std -0.061087/0.116913 | Adv std 1.178692e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.063731 | AvgAdv -0.000000 | AdvAfterStd 8.583770e-02
[Update 866] Samples 2048 | Reward mean/std -0.060616/0.138696 | Value mean/std -0.065099/0.118048 | Adv std 9.218356e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.060616 | AvgAdv 0.000000 | AdvAfterStd 6.905410e-02
[Update 867] Samples 2048 | Reward mean/std -0.063577/0.189410 | Value mean/std -0.059480/0.129575 | Adv std 1.224672e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.063577 | AvgAdv -0.000000 | AdvAfterStd 7.964569e-02
[Update 868] Samples 2048 | Reward mean/std -0.069233/0.207418 | Value mean/std -0.062567/0.194439 | Adv std 1.462525e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.069233 | AvgAdv 0.000000 | AdvAfterStd 9.926505e-02
[Update 869] Samples 2048 | Reward mean/std -0.064471/0.212926 | Value mean/std -0.070771/0.177809 | Adv std 1.024904e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.064471 | AvgAdv 0.000000 | AdvAfterStd 9.637605e-02
[Update 870] Samples 2048 | Reward mean/std -0.070288/0.192470 | Value mean/std -0.070283/0.158817 | Adv std 1.332908e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.070288 | AvgAdv 0.000000 | AdvAfterStd 9.560257e-02
[Update 871] Samples 2048 | Reward mean/std -0.068542/0.165521 | Value mean/std -0.065940/0.132933 | Adv std 1.160661e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.068542 | AvgAdv 0.000000 | AdvAfterStd 8.585294e-02
[Update 872] Samples 2048 | Reward mean/std -0.069060/0.201090 | Value mean/std -0.065481/0.137498 | Adv std 1.227954e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.069060 | AvgAdv 0.000000 | AdvAfterStd 8.217768e-02
[Update 873] Samples 2048 | Reward mean/std -0.066630/0.194938 | Value mean/std -0.067784/0.185482 | Adv std 1.340110e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.066630 | AvgAdv 0.000000 | AdvAfterStd 8.396608e-02
[Update 874] Samples 2048 | Reward mean/std -0.073276/0.234373 | Value mean/std -0.075473/0.176578 | Adv std 1.493636e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.073276 | AvgAdv -0.000000 | AdvAfterStd 8.699760e-02
[Update 875] Samples 2048 | Reward mean/std -0.068401/0.267601 | Value mean/std -0.073307/0.234005 | Adv std 1.270141e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.068401 | AvgAdv -0.000000 | AdvAfterStd 7.529554e-02
[Update 876] Samples 2048 | Reward mean/std -0.064817/0.192619 | Value mean/std -0.070398/0.176764 | Adv std 9.288210e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.064817 | AvgAdv 0.000000 | AdvAfterStd 7.343535e-02
[Update 877] Samples 2048 | Reward mean/std -0.064724/0.167268 | Value mean/std -0.062496/0.105686 | Adv std 1.343597e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.064724 | AvgAdv 0.000000 | AdvAfterStd 9.111264e-02
[Update 878] Samples 2048 | Reward mean/std -0.059227/0.124240 | Value mean/std -0.056810/0.135819 | Adv std 9.168511e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.059227 | AvgAdv 0.000000 | AdvAfterStd 6.998401e-02
[Update 879] Samples 2048 | Reward mean/std -0.069701/0.253646 | Value mean/std -0.068568/0.154438 | Adv std 2.247045e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.069701 | AvgAdv -0.000000 | AdvAfterStd 1.043558e-01
[Update 880] Samples 2048 | Reward mean/std -0.074214/0.214025 | Value mean/std -0.074935/0.284909 | Adv std 2.210025e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.074214 | AvgAdv 0.000000 | AdvAfterStd 1.045930e-01
[Update 881] Samples 2048 | Reward mean/std -0.064218/0.158281 | Value mean/std -0.067934/0.135147 | Adv std 1.062704e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.064218 | AvgAdv 0.000000 | AdvAfterStd 7.573137e-02
[Update 882] Samples 2048 | Reward mean/std -0.061423/0.152109 | Value mean/std -0.056155/0.134438 | Adv std 1.043160e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.061423 | AvgAdv 0.000000 | AdvAfterStd 8.367104e-02
[Update 883] Samples 2048 | Reward mean/std -0.068546/0.184736 | Value mean/std -0.062624/0.123930 | Adv std 1.293483e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.068546 | AvgAdv -0.000000 | AdvAfterStd 9.360219e-02
[Update 884] Samples 2048 | Reward mean/std -0.067810/0.202851 | Value mean/std -0.068534/0.174269 | Adv std 1.103454e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.067810 | AvgAdv 0.000000 | AdvAfterStd 9.296946e-02
[Update 885] Samples 2048 | Reward mean/std -0.065102/0.198808 | Value mean/std -0.065179/0.196527 | Adv std 1.141329e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.065102 | AvgAdv 0.000000 | AdvAfterStd 7.944403e-02
[Update 886] Samples 2048 | Reward mean/std -0.061557/0.182029 | Value mean/std -0.056958/0.126096 | Adv std 1.040296e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.061557 | AvgAdv 0.000000 | AdvAfterStd 8.248943e-02
[Update 887] Samples 2048 | Reward mean/std -0.068412/0.180646 | Value mean/std -0.062395/0.124072 | Adv std 1.062472e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.068412 | AvgAdv 0.000000 | AdvAfterStd 7.739977e-02
[Update 888] Samples 2048 | Reward mean/std -0.064107/0.165082 | Value mean/std -0.062763/0.120323 | Adv std 1.186620e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.064107 | AvgAdv 0.000000 | AdvAfterStd 8.109687e-02
[Update 889] Samples 2048 | Reward mean/std -0.073795/0.225192 | Value mean/std -0.072577/0.227230 | Adv std 1.888057e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.073795 | AvgAdv 0.000000 | AdvAfterStd 8.306912e-02
[Update 890] Samples 2048 | Reward mean/std -0.061366/0.159475 | Value mean/std -0.069054/0.146429 | Adv std 1.016450e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.061366 | AvgAdv 0.000000 | AdvAfterStd 7.123990e-02
[Update 891] Samples 2048 | Reward mean/std -0.063203/0.209178 | Value mean/std -0.062963/0.120965 | Adv std 1.216512e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.063203 | AvgAdv 0.000000 | AdvAfterStd 7.442655e-02
[Update 892] Samples 2048 | Reward mean/std -0.062850/0.229796 | Value mean/std -0.063714/0.155633 | Adv std 1.203696e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.062850 | AvgAdv 0.000000 | AdvAfterStd 7.492930e-02
[Update 893] Samples 2048 | Reward mean/std -0.064910/0.155430 | Value mean/std -0.059174/0.125792 | Adv std 1.048995e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.064910 | AvgAdv -0.000000 | AdvAfterStd 7.739963e-02
[Update 894] Samples 2048 | Reward mean/std -0.071764/0.196885 | Value mean/std -0.068333/0.166792 | Adv std 1.184782e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.071764 | AvgAdv 0.000000 | AdvAfterStd 8.485125e-02
[Update 895] Samples 2048 | Reward mean/std -0.074140/0.232315 | Value mean/std -0.073802/0.178702 | Adv std 1.182860e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.074140 | AvgAdv 0.000000 | AdvAfterStd 9.874952e-02
[Update 896] Samples 2048 | Reward mean/std -0.061687/0.151423 | Value mean/std -0.063098/0.151595 | Adv std 1.071769e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.061687 | AvgAdv 0.000000 | AdvAfterStd 6.956635e-02
[Update 897] Samples 2048 | Reward mean/std -0.070564/0.218821 | Value mean/std -0.065685/0.156730 | Adv std 1.278489e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.070564 | AvgAdv 0.000000 | AdvAfterStd 7.289729e-02
[Update 898] Samples 2048 | Reward mean/std -0.059638/0.134292 | Value mean/std -0.061547/0.124313 | Adv std 1.000525e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.059638 | AvgAdv -0.000000 | AdvAfterStd 7.687557e-02
[Update 899] Samples 2048 | Reward mean/std -0.069616/0.247711 | Value mean/std -0.060710/0.202706 | Adv std 1.119181e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.069616 | AvgAdv 0.000000 | AdvAfterStd 6.654898e-02
[Update 900] Samples 2048 | Reward mean/std -0.066275/0.206793 | Value mean/std -0.070972/0.207645 | Adv std 1.075887e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.066275 | AvgAdv 0.000000 | AdvAfterStd 7.316254e-02
Saved checkpoint at update 900
[Update 901] Samples 2048 | Reward mean/std -0.058019/0.146731 | Value mean/std -0.054817/0.156304 | Adv std 9.306908e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.058019 | AvgAdv -0.000000 | AdvAfterStd 6.367681e-02
[Update 902] Samples 2048 | Reward mean/std -0.066976/0.217377 | Value mean/std -0.059610/0.127687 | Adv std 1.439024e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.066976 | AvgAdv -0.000000 | AdvAfterStd 7.210472e-02
[Update 903] Samples 2048 | Reward mean/std -0.063487/0.183319 | Value mean/std -0.064803/0.143803 | Adv std 1.025505e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.063487 | AvgAdv -0.000000 | AdvAfterStd 7.855210e-02
[Update 904] Samples 2048 | Reward mean/std -0.059376/0.146773 | Value mean/std -0.055537/0.103809 | Adv std 8.950619e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.059376 | AvgAdv -0.000000 | AdvAfterStd 6.709101e-02
[Update 905] Samples 2048 | Reward mean/std -0.064331/0.190765 | Value mean/std -0.059709/0.127283 | Adv std 1.144096e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.064331 | AvgAdv 0.000000 | AdvAfterStd 6.806323e-02
[Update 906] Samples 2048 | Reward mean/std -0.064974/0.231841 | Value mean/std -0.065653/0.202391 | Adv std 9.579610e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.064974 | AvgAdv 0.000000 | AdvAfterStd 6.358999e-02
[Update 907] Samples 2048 | Reward mean/std -0.064261/0.210046 | Value mean/std -0.068621/0.199010 | Adv std 1.113145e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.064261 | AvgAdv 0.000000 | AdvAfterStd 7.841498e-02
[Update 908] Samples 2048 | Reward mean/std -0.067524/0.163014 | Value mean/std -0.062254/0.144302 | Adv std 1.244204e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.067524 | AvgAdv 0.000000 | AdvAfterStd 7.941935e-02
[Update 909] Samples 2048 | Reward mean/std -0.057549/0.127019 | Value mean/std -0.063146/0.124138 | Adv std 1.010956e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.057549 | AvgAdv 0.000000 | AdvAfterStd 7.828482e-02
[Update 910] Samples 2048 | Reward mean/std -0.062561/0.172413 | Value mean/std -0.062235/0.123927 | Adv std 1.001725e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.062561 | AvgAdv -0.000000 | AdvAfterStd 7.250360e-02
[Update 911] Samples 2048 | Reward mean/std -0.064303/0.162477 | Value mean/std -0.066777/0.152643 | Adv std 9.307745e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.064303 | AvgAdv 0.000000 | AdvAfterStd 6.886844e-02
[Update 912] Samples 2048 | Reward mean/std -0.063352/0.161051 | Value mean/std -0.054157/0.130127 | Adv std 9.935039e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.063352 | AvgAdv 0.000000 | AdvAfterStd 8.598569e-02
[Update 913] Samples 2048 | Reward mean/std -0.073423/0.203804 | Value mean/std -0.082749/0.217058 | Adv std 1.789585e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.073423 | AvgAdv -0.000000 | AdvAfterStd 8.697184e-02
[Update 914] Samples 2048 | Reward mean/std -0.057298/0.140393 | Value mean/std -0.062045/0.108456 | Adv std 1.076111e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.057298 | AvgAdv -0.000000 | AdvAfterStd 6.869248e-02
[Update 915] Samples 2048 | Reward mean/std -0.060973/0.157637 | Value mean/std -0.054645/0.108161 | Adv std 1.065175e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.060973 | AvgAdv 0.000000 | AdvAfterStd 7.658962e-02
[Update 916] Samples 2048 | Reward mean/std -0.062095/0.144565 | Value mean/std -0.057195/0.103334 | Adv std 8.816482e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.062095 | AvgAdv -0.000000 | AdvAfterStd 7.508695e-02
[Update 917] Samples 2048 | Reward mean/std -0.069911/0.165871 | Value mean/std -0.071747/0.139661 | Adv std 1.163701e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.069911 | AvgAdv -0.000000 | AdvAfterStd 7.235815e-02
[Update 918] Samples 2048 | Reward mean/std -0.061215/0.158148 | Value mean/std -0.060018/0.119799 | Adv std 1.088511e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.061215 | AvgAdv -0.000000 | AdvAfterStd 8.407588e-02
[Update 919] Samples 2048 | Reward mean/std -0.066720/0.197203 | Value mean/std -0.069311/0.148018 | Adv std 1.095879e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.066720 | AvgAdv -0.000000 | AdvAfterStd 7.250788e-02
[Update 920] Samples 2048 | Reward mean/std -0.067226/0.183005 | Value mean/std -0.060694/0.161582 | Adv std 9.598010e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.067226 | AvgAdv 0.000000 | AdvAfterStd 7.593410e-02
[Update 921] Samples 2048 | Reward mean/std -0.065963/0.156339 | Value mean/std -0.065640/0.159185 | Adv std 1.191844e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.065963 | AvgAdv 0.000000 | AdvAfterStd 8.389006e-02
[Update 922] Samples 2048 | Reward mean/std -0.067537/0.220979 | Value mean/std -0.066923/0.120395 | Adv std 1.829996e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.067537 | AvgAdv 0.000000 | AdvAfterStd 8.862673e-02
[Update 923] Samples 2048 | Reward mean/std -0.062468/0.143119 | Value mean/std -0.063271/0.104999 | Adv std 9.959314e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.062468 | AvgAdv -0.000000 | AdvAfterStd 7.520323e-02
[Update 924] Samples 2048 | Reward mean/std -0.060319/0.132103 | Value mean/std -0.064273/0.114652 | Adv std 9.614144e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.060319 | AvgAdv 0.000000 | AdvAfterStd 6.914079e-02
[Update 925] Samples 2048 | Reward mean/std -0.064749/0.153447 | Value mean/std -0.057694/0.112801 | Adv std 1.012721e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.064749 | AvgAdv -0.000000 | AdvAfterStd 8.033898e-02
[Update 926] Samples 2048 | Reward mean/std -0.057720/0.130274 | Value mean/std -0.067954/0.087552 | Adv std 9.280708e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.057720 | AvgAdv 0.000000 | AdvAfterStd 7.564837e-02
[Update 927] Samples 2048 | Reward mean/std -0.063213/0.185890 | Value mean/std -0.064586/0.140799 | Adv std 9.982413e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.063213 | AvgAdv 0.000000 | AdvAfterStd 7.684073e-02
[Update 928] Samples 2048 | Reward mean/std -0.069626/0.195711 | Value mean/std -0.065399/0.158001 | Adv std 1.092835e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.069626 | AvgAdv 0.000000 | AdvAfterStd 8.175036e-02
[Update 929] Samples 2048 | Reward mean/std -0.070248/0.210781 | Value mean/std -0.068694/0.176802 | Adv std 1.338730e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.070248 | AvgAdv 0.000000 | AdvAfterStd 7.991235e-02
[Update 930] Samples 2048 | Reward mean/std -0.070040/0.185961 | Value mean/std -0.070716/0.167269 | Adv std 1.195489e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.070040 | AvgAdv 0.000000 | AdvAfterStd 7.794813e-02
[Update 931] Samples 2048 | Reward mean/std -0.070274/0.207618 | Value mean/std -0.072373/0.169938 | Adv std 1.186006e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.070274 | AvgAdv -0.000000 | AdvAfterStd 7.481197e-02
[Update 932] Samples 2048 | Reward mean/std -0.066649/0.178052 | Value mean/std -0.065002/0.143999 | Adv std 8.991913e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.066649 | AvgAdv -0.000000 | AdvAfterStd 7.803336e-02
[Update 933] Samples 2048 | Reward mean/std -0.070928/0.219900 | Value mean/std -0.065875/0.150617 | Adv std 1.313028e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.070928 | AvgAdv -0.000000 | AdvAfterStd 8.663061e-02
[Update 934] Samples 2048 | Reward mean/std -0.074536/0.257559 | Value mean/std -0.077345/0.242936 | Adv std 1.559114e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.074536 | AvgAdv 0.000000 | AdvAfterStd 8.609855e-02
[Update 935] Samples 2048 | Reward mean/std -0.059985/0.129732 | Value mean/std -0.063485/0.121462 | Adv std 1.028462e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.059985 | AvgAdv 0.000000 | AdvAfterStd 7.575250e-02
[Update 936] Samples 2048 | Reward mean/std -0.072223/0.269456 | Value mean/std -0.061792/0.137523 | Adv std 1.864368e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.072223 | AvgAdv -0.000000 | AdvAfterStd 8.519671e-02
[Update 937] Samples 2048 | Reward mean/std -0.061668/0.162666 | Value mean/std -0.063497/0.121542 | Adv std 1.126649e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.061668 | AvgAdv -0.000000 | AdvAfterStd 8.833410e-02
[Update 938] Samples 2048 | Reward mean/std -0.071812/0.232538 | Value mean/std -0.063890/0.197774 | Adv std 1.295850e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.071812 | AvgAdv -0.000000 | AdvAfterStd 8.678170e-02
[Update 939] Samples 2048 | Reward mean/std -0.059499/0.150846 | Value mean/std -0.070296/0.158516 | Adv std 1.022678e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.059499 | AvgAdv -0.000000 | AdvAfterStd 6.876954e-02
[Update 940] Samples 2048 | Reward mean/std -0.057231/0.133934 | Value mean/std -0.055426/0.099987 | Adv std 9.257500e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.057231 | AvgAdv -0.000000 | AdvAfterStd 7.228985e-02
[Update 941] Samples 2048 | Reward mean/std -0.065197/0.175912 | Value mean/std -0.063515/0.128185 | Adv std 1.208253e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.065197 | AvgAdv -0.000000 | AdvAfterStd 8.299176e-02
[Update 942] Samples 2048 | Reward mean/std -0.062246/0.156359 | Value mean/std -0.067019/0.155339 | Adv std 9.505319e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.062246 | AvgAdv 0.000000 | AdvAfterStd 6.660616e-02
[Update 943] Samples 2048 | Reward mean/std -0.062154/0.197167 | Value mean/std -0.061947/0.151773 | Adv std 9.809344e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.062154 | AvgAdv -0.000000 | AdvAfterStd 6.892873e-02
[Update 944] Samples 2048 | Reward mean/std -0.064586/0.194905 | Value mean/std -0.064569/0.130921 | Adv std 1.394958e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.064586 | AvgAdv -0.000000 | AdvAfterStd 1.049223e-01
[Update 945] Samples 2048 | Reward mean/std -0.069154/0.211963 | Value mean/std -0.063734/0.147468 | Adv std 1.183915e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.069154 | AvgAdv -0.000000 | AdvAfterStd 1.009659e-01
[Update 946] Samples 2048 | Reward mean/std -0.062041/0.199025 | Value mean/std -0.066448/0.147304 | Adv std 1.182863e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.062041 | AvgAdv -0.000000 | AdvAfterStd 7.372575e-02
[Update 947] Samples 2048 | Reward mean/std -0.068808/0.244281 | Value mean/std -0.057337/0.156550 | Adv std 1.368610e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.068808 | AvgAdv -0.000000 | AdvAfterStd 9.136397e-02
[Update 948] Samples 2048 | Reward mean/std -0.068867/0.195687 | Value mean/std -0.074179/0.168695 | Adv std 1.112334e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.068867 | AvgAdv 0.000000 | AdvAfterStd 8.457708e-02
[Update 949] Samples 2048 | Reward mean/std -0.064098/0.150929 | Value mean/std -0.062357/0.113766 | Adv std 1.020039e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.064098 | AvgAdv 0.000000 | AdvAfterStd 8.240899e-02
[Update 950] Samples 2048 | Reward mean/std -0.066259/0.193963 | Value mean/std -0.073859/0.205672 | Adv std 1.434638e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.066259 | AvgAdv 0.000000 | AdvAfterStd 7.809260e-02
Saved checkpoint at update 950
[Update 951] Samples 2048 | Reward mean/std -0.065341/0.279305 | Value mean/std -0.070532/0.150723 | Adv std 1.663543e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.065341 | AvgAdv 0.000000 | AdvAfterStd 6.780838e-02
[Update 952] Samples 2048 | Reward mean/std -0.062592/0.153579 | Value mean/std -0.054139/0.120529 | Adv std 1.043191e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.062592 | AvgAdv -0.000000 | AdvAfterStd 8.374779e-02
[Update 953] Samples 2048 | Reward mean/std -0.058221/0.133511 | Value mean/std -0.061173/0.108653 | Adv std 9.541903e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.058221 | AvgAdv 0.000000 | AdvAfterStd 6.732209e-02
[Update 954] Samples 2048 | Reward mean/std -0.059792/0.152085 | Value mean/std -0.053413/0.143923 | Adv std 1.079317e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.059792 | AvgAdv -0.000000 | AdvAfterStd 7.024702e-02
[Update 955] Samples 2048 | Reward mean/std -0.061940/0.142240 | Value mean/std -0.061063/0.102003 | Adv std 9.132146e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.061940 | AvgAdv 0.000000 | AdvAfterStd 7.552378e-02
[Update 956] Samples 2048 | Reward mean/std -0.064572/0.216209 | Value mean/std -0.067794/0.228175 | Adv std 9.645087e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.064572 | AvgAdv 0.000000 | AdvAfterStd 8.755637e-02
[Update 957] Samples 2048 | Reward mean/std -0.059665/0.172054 | Value mean/std -0.064484/0.131840 | Adv std 9.323286e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.059665 | AvgAdv -0.000000 | AdvAfterStd 7.039203e-02
[Update 958] Samples 2048 | Reward mean/std -0.060581/0.217110 | Value mean/std -0.066710/0.212630 | Adv std 1.020760e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.060581 | AvgAdv 0.000000 | AdvAfterStd 7.693255e-02
[Update 959] Samples 2048 | Reward mean/std -0.064520/0.164045 | Value mean/std -0.062668/0.130336 | Adv std 1.035021e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.064520 | AvgAdv 0.000000 | AdvAfterStd 7.696691e-02
[Update 960] Samples 2048 | Reward mean/std -0.061800/0.174900 | Value mean/std -0.059975/0.154927 | Adv std 1.618175e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.061800 | AvgAdv 0.000000 | AdvAfterStd 7.582556e-02
[Update 961] Samples 2048 | Reward mean/std -0.061509/0.143199 | Value mean/std -0.065487/0.102497 | Adv std 1.014500e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.061509 | AvgAdv -0.000000 | AdvAfterStd 7.667772e-02
[Update 962] Samples 2048 | Reward mean/std -0.062804/0.155662 | Value mean/std -0.059680/0.115010 | Adv std 1.008312e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.062804 | AvgAdv 0.000000 | AdvAfterStd 8.059949e-02
[Update 963] Samples 2048 | Reward mean/std -0.069494/0.193663 | Value mean/std -0.060322/0.121594 | Adv std 1.288393e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.069494 | AvgAdv 0.000000 | AdvAfterStd 8.472035e-02
[Update 964] Samples 2048 | Reward mean/std -0.066650/0.184031 | Value mean/std -0.070898/0.137904 | Adv std 1.139565e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.066650 | AvgAdv 0.000000 | AdvAfterStd 8.346639e-02
[Update 965] Samples 2048 | Reward mean/std -0.071008/0.185207 | Value mean/std -0.066135/0.161206 | Adv std 1.059333e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.071008 | AvgAdv 0.000000 | AdvAfterStd 8.817668e-02
[Update 966] Samples 2048 | Reward mean/std -0.078571/0.339064 | Value mean/std -0.075499/0.196318 | Adv std 1.945381e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.078571 | AvgAdv 0.000000 | AdvAfterStd 1.066180e-01
[Update 967] Samples 2048 | Reward mean/std -0.067681/0.188223 | Value mean/std -0.070576/0.164119 | Adv std 9.893311e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.067681 | AvgAdv -0.000000 | AdvAfterStd 7.555169e-02
[Update 968] Samples 2048 | Reward mean/std -0.063389/0.162321 | Value mean/std -0.059416/0.139982 | Adv std 1.043919e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.063389 | AvgAdv 0.000000 | AdvAfterStd 7.079180e-02
[Update 969] Samples 2048 | Reward mean/std -0.059601/0.152431 | Value mean/std -0.059509/0.120912 | Adv std 9.114584e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.059601 | AvgAdv -0.000000 | AdvAfterStd 7.333200e-02
[Update 970] Samples 2048 | Reward mean/std -0.064739/0.158710 | Value mean/std -0.058119/0.110432 | Adv std 1.127663e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.064739 | AvgAdv 0.000000 | AdvAfterStd 8.352366e-02
[Update 971] Samples 2048 | Reward mean/std -0.063728/0.173233 | Value mean/std -0.064395/0.148789 | Adv std 9.748403e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.063728 | AvgAdv -0.000000 | AdvAfterStd 7.636455e-02
[Update 972] Samples 2048 | Reward mean/std -0.075603/0.255193 | Value mean/std -0.066376/0.178981 | Adv std 1.506696e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.075603 | AvgAdv 0.000000 | AdvAfterStd 9.300211e-02
[Update 973] Samples 2048 | Reward mean/std -0.066383/0.210448 | Value mean/std -0.064155/0.137655 | Adv std 1.334210e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.066383 | AvgAdv -0.000000 | AdvAfterStd 7.748218e-02
[Update 974] Samples 2048 | Reward mean/std -0.059812/0.152018 | Value mean/std -0.060072/0.120129 | Adv std 9.755400e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.059812 | AvgAdv 0.000000 | AdvAfterStd 6.961199e-02
[Update 975] Samples 2048 | Reward mean/std -0.065146/0.154165 | Value mean/std -0.062744/0.139406 | Adv std 1.134010e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.065146 | AvgAdv -0.000000 | AdvAfterStd 8.758282e-02
[Update 976] Samples 2048 | Reward mean/std -0.069011/0.232623 | Value mean/std -0.067807/0.178658 | Adv std 1.238483e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.069011 | AvgAdv 0.000000 | AdvAfterStd 8.053332e-02
[Update 977] Samples 2048 | Reward mean/std -0.063932/0.176150 | Value mean/std -0.061099/0.115674 | Adv std 1.107662e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.063932 | AvgAdv 0.000000 | AdvAfterStd 7.496765e-02
[Update 978] Samples 2048 | Reward mean/std -0.066806/0.170943 | Value mean/std -0.068201/0.135959 | Adv std 1.185770e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.066806 | AvgAdv -0.000000 | AdvAfterStd 8.300000e-02
[Update 979] Samples 2048 | Reward mean/std -0.068709/0.245794 | Value mean/std -0.065605/0.166443 | Adv std 1.494190e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.068709 | AvgAdv -0.000000 | AdvAfterStd 7.958267e-02
[Update 980] Samples 2048 | Reward mean/std -0.059912/0.140781 | Value mean/std -0.067388/0.112354 | Adv std 9.595460e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.059912 | AvgAdv 0.000000 | AdvAfterStd 7.101911e-02
[Update 981] Samples 2048 | Reward mean/std -0.061059/0.162211 | Value mean/std -0.058711/0.131270 | Adv std 1.219328e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.061059 | AvgAdv 0.000000 | AdvAfterStd 7.127938e-02
[Update 982] Samples 2048 | Reward mean/std -0.065842/0.195661 | Value mean/std -0.061517/0.161881 | Adv std 1.146491e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.065842 | AvgAdv -0.000000 | AdvAfterStd 9.152153e-02
[Update 983] Samples 2048 | Reward mean/std -0.058556/0.138045 | Value mean/std -0.053294/0.091660 | Adv std 9.693343e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.058556 | AvgAdv 0.000000 | AdvAfterStd 7.747254e-02
[Update 984] Samples 2048 | Reward mean/std -0.063101/0.166016 | Value mean/std -0.055723/0.113177 | Adv std 1.165266e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.063101 | AvgAdv -0.000000 | AdvAfterStd 8.858488e-02
[Update 985] Samples 2048 | Reward mean/std -0.067179/0.183993 | Value mean/std -0.072368/0.177850 | Adv std 1.066823e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.067179 | AvgAdv -0.000000 | AdvAfterStd 8.039685e-02
[Update 986] Samples 2048 | Reward mean/std -0.059259/0.168856 | Value mean/std -0.061048/0.134092 | Adv std 1.071810e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.059259 | AvgAdv -0.000000 | AdvAfterStd 8.085049e-02
[Update 987] Samples 2048 | Reward mean/std -0.066928/0.175088 | Value mean/std -0.068845/0.149932 | Adv std 9.670455e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.066928 | AvgAdv 0.000000 | AdvAfterStd 8.029102e-02
[Update 988] Samples 2048 | Reward mean/std -0.061445/0.154481 | Value mean/std -0.060508/0.131908 | Adv std 9.990319e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.061445 | AvgAdv -0.000000 | AdvAfterStd 7.605830e-02
[Update 989] Samples 2048 | Reward mean/std -0.065201/0.164645 | Value mean/std -0.062999/0.134918 | Adv std 1.174711e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.065201 | AvgAdv -0.000000 | AdvAfterStd 8.869420e-02
[Update 990] Samples 2048 | Reward mean/std -0.066622/0.186822 | Value mean/std -0.064981/0.156043 | Adv std 1.277230e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.066622 | AvgAdv 0.000000 | AdvAfterStd 9.101824e-02
[Update 991] Samples 2048 | Reward mean/std -0.063161/0.157635 | Value mean/std -0.069484/0.144707 | Adv std 1.054855e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.063161 | AvgAdv -0.000000 | AdvAfterStd 7.237066e-02
[Update 992] Samples 2048 | Reward mean/std -0.059384/0.131553 | Value mean/std -0.064357/0.112480 | Adv std 8.908757e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.059384 | AvgAdv -0.000000 | AdvAfterStd 7.346856e-02
[Update 993] Samples 2048 | Reward mean/std -0.068346/0.182547 | Value mean/std -0.057849/0.110607 | Adv std 1.209068e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.068346 | AvgAdv 0.000000 | AdvAfterStd 8.077396e-02
[Update 994] Samples 2048 | Reward mean/std -0.066585/0.184698 | Value mean/std -0.072898/0.160046 | Adv std 1.450013e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.066585 | AvgAdv -0.000000 | AdvAfterStd 8.002806e-02
[Update 995] Samples 2048 | Reward mean/std -0.081171/0.263384 | Value mean/std -0.071468/0.180695 | Adv std 1.416992e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.081171 | AvgAdv 0.000000 | AdvAfterStd 1.031284e-01
[Update 996] Samples 2048 | Reward mean/std -0.067603/0.171544 | Value mean/std -0.072430/0.149386 | Adv std 1.240525e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.067603 | AvgAdv -0.000000 | AdvAfterStd 8.326524e-02
[Update 997] Samples 2048 | Reward mean/std -0.068389/0.229481 | Value mean/std -0.068515/0.248838 | Adv std 1.256984e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.068389 | AvgAdv 0.000000 | AdvAfterStd 8.767498e-02
[Update 998] Samples 2048 | Reward mean/std -0.066673/0.265432 | Value mean/std -0.066109/0.219528 | Adv std 9.726425e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.066673 | AvgAdv 0.000000 | AdvAfterStd 7.535663e-02
[Update 999] Samples 2048 | Reward mean/std -0.071119/0.221074 | Value mean/std -0.071642/0.253217 | Adv std 1.336472e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.071119 | AvgAdv 0.000000 | AdvAfterStd 8.781879e-02
[Update 1000] Samples 2048 | Reward mean/std -0.066996/0.205041 | Value mean/std -0.065059/0.175513 | Adv std 9.989947e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.066996 | AvgAdv -0.000000 | AdvAfterStd 7.864024e-02
Saved checkpoint at update 1000
[Update 1001] Samples 2048 | Reward mean/std -0.067531/0.219253 | Value mean/std -0.072111/0.211355 | Adv std 1.146497e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.067531 | AvgAdv 0.000000 | AdvAfterStd 8.869642e-02
[Update 1002] Samples 2048 | Reward mean/std -0.061025/0.149404 | Value mean/std -0.061408/0.132638 | Adv std 9.348729e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.061025 | AvgAdv -0.000000 | AdvAfterStd 6.701922e-02
[Update 1003] Samples 2048 | Reward mean/std -0.068603/0.194749 | Value mean/std -0.066232/0.143311 | Adv std 1.084165e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.068603 | AvgAdv -0.000000 | AdvAfterStd 7.361142e-02
[Update 1004] Samples 2048 | Reward mean/std -0.065700/0.222467 | Value mean/std -0.063593/0.165434 | Adv std 1.062793e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.065700 | AvgAdv 0.000000 | AdvAfterStd 8.343901e-02
[Update 1005] Samples 2048 | Reward mean/std -0.061625/0.159866 | Value mean/std -0.068300/0.118428 | Adv std 1.021753e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.061625 | AvgAdv -0.000000 | AdvAfterStd 7.279335e-02
[Update 1006] Samples 2048 | Reward mean/std -0.065703/0.192267 | Value mean/std -0.061172/0.150446 | Adv std 1.008144e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.065703 | AvgAdv -0.000000 | AdvAfterStd 8.833692e-02
[Update 1007] Samples 2048 | Reward mean/std -0.066874/0.189389 | Value mean/std -0.068423/0.206534 | Adv std 1.228672e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.066874 | AvgAdv -0.000000 | AdvAfterStd 9.931108e-02
[Update 1008] Samples 2048 | Reward mean/std -0.059013/0.133905 | Value mean/std -0.060017/0.095683 | Adv std 9.978398e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.059013 | AvgAdv -0.000000 | AdvAfterStd 7.913910e-02
[Update 1009] Samples 2048 | Reward mean/std -0.080532/0.241953 | Value mean/std -0.076467/0.175417 | Adv std 1.378325e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.080532 | AvgAdv 0.000000 | AdvAfterStd 9.132302e-02
[Update 1010] Samples 2048 | Reward mean/std -0.064662/0.221501 | Value mean/std -0.073351/0.153739 | Adv std 1.572898e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.064662 | AvgAdv -0.000000 | AdvAfterStd 7.958069e-02
[Update 1011] Samples 2048 | Reward mean/std -0.061796/0.156362 | Value mean/std -0.059215/0.157690 | Adv std 9.451409e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.061796 | AvgAdv 0.000000 | AdvAfterStd 7.229476e-02
[Update 1012] Samples 2048 | Reward mean/std -0.071642/0.217315 | Value mean/std -0.077339/0.262423 | Adv std 1.448562e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.071642 | AvgAdv 0.000000 | AdvAfterStd 8.228988e-02
[Update 1013] Samples 2048 | Reward mean/std -0.064250/0.168416 | Value mean/std -0.058436/0.122775 | Adv std 1.166747e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.064250 | AvgAdv 0.000000 | AdvAfterStd 7.745159e-02
[Update 1014] Samples 2048 | Reward mean/std -0.070064/0.216067 | Value mean/std -0.070001/0.191071 | Adv std 1.121525e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.070064 | AvgAdv 0.000000 | AdvAfterStd 8.099417e-02
[Update 1015] Samples 2048 | Reward mean/std -0.060595/0.220819 | Value mean/std -0.062816/0.175676 | Adv std 8.690030e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.060595 | AvgAdv 0.000000 | AdvAfterStd 6.212056e-02
[Update 1016] Samples 2048 | Reward mean/std -0.063501/0.156398 | Value mean/std -0.060237/0.113917 | Adv std 1.122272e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.063501 | AvgAdv -0.000000 | AdvAfterStd 8.461874e-02
[Update 1017] Samples 2048 | Reward mean/std -0.061062/0.146338 | Value mean/std -0.063116/0.139130 | Adv std 9.402920e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.061062 | AvgAdv -0.000000 | AdvAfterStd 6.744821e-02
[Update 1018] Samples 2048 | Reward mean/std -0.072398/0.222264 | Value mean/std -0.064120/0.152176 | Adv std 1.219106e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.072398 | AvgAdv -0.000000 | AdvAfterStd 9.091776e-02
[Update 1019] Samples 2048 | Reward mean/std -0.062680/0.156308 | Value mean/std -0.064131/0.105828 | Adv std 1.214122e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.062680 | AvgAdv 0.000000 | AdvAfterStd 9.095837e-02
[Update 1020] Samples 2048 | Reward mean/std -0.061030/0.149265 | Value mean/std -0.066006/0.110564 | Adv std 1.168666e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.061030 | AvgAdv -0.000000 | AdvAfterStd 8.450519e-02
[Update 1021] Samples 2048 | Reward mean/std -0.065547/0.201421 | Value mean/std -0.071457/0.176080 | Adv std 1.113733e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.065547 | AvgAdv 0.000000 | AdvAfterStd 8.171416e-02
[Update 1022] Samples 2048 | Reward mean/std -0.061591/0.143580 | Value mean/std -0.061594/0.108899 | Adv std 9.314512e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.061591 | AvgAdv 0.000000 | AdvAfterStd 6.497615e-02
[Update 1023] Samples 2048 | Reward mean/std -0.062037/0.199090 | Value mean/std -0.059161/0.163180 | Adv std 9.865208e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.062037 | AvgAdv 0.000000 | AdvAfterStd 7.179760e-02
[Update 1024] Samples 2048 | Reward mean/std -0.060203/0.148457 | Value mean/std -0.064746/0.123236 | Adv std 9.159655e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.060203 | AvgAdv 0.000000 | AdvAfterStd 6.658003e-02
[Update 1025] Samples 2048 | Reward mean/std -0.064175/0.178574 | Value mean/std -0.065737/0.149885 | Adv std 1.068919e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.064175 | AvgAdv 0.000000 | AdvAfterStd 7.102952e-02
[Update 1026] Samples 2048 | Reward mean/std -0.069536/0.191284 | Value mean/std -0.064753/0.147092 | Adv std 1.090392e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.069536 | AvgAdv 0.000000 | AdvAfterStd 7.222100e-02
[Update 1027] Samples 2048 | Reward mean/std -0.065739/0.230759 | Value mean/std -0.065458/0.153354 | Adv std 1.504839e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.065739 | AvgAdv -0.000000 | AdvAfterStd 7.938486e-02
[Update 1028] Samples 2048 | Reward mean/std -0.054892/0.127986 | Value mean/std -0.056507/0.157345 | Adv std 1.298929e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.054892 | AvgAdv 0.000000 | AdvAfterStd 7.579637e-02
[Update 1029] Samples 2048 | Reward mean/std -0.066263/0.208583 | Value mean/std -0.064262/0.149738 | Adv std 1.269154e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.066263 | AvgAdv -0.000000 | AdvAfterStd 1.225418e-01
[Update 1030] Samples 2048 | Reward mean/std -0.068628/0.226707 | Value mean/std -0.064056/0.140504 | Adv std 1.452846e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.068628 | AvgAdv -0.000000 | AdvAfterStd 8.551546e-02
[Update 1031] Samples 2048 | Reward mean/std -0.062960/0.177247 | Value mean/std -0.064110/0.152554 | Adv std 1.129817e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.062960 | AvgAdv 0.000000 | AdvAfterStd 8.034774e-02
[Update 1032] Samples 2048 | Reward mean/std -0.073720/0.299144 | Value mean/std -0.066504/0.191340 | Adv std 1.536178e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.073720 | AvgAdv -0.000000 | AdvAfterStd 9.752417e-02
[Update 1033] Samples 2048 | Reward mean/std -0.067004/0.209364 | Value mean/std -0.071304/0.222607 | Adv std 1.112674e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.067004 | AvgAdv 0.000000 | AdvAfterStd 1.007188e-01
[Update 1034] Samples 2048 | Reward mean/std -0.059321/0.162567 | Value mean/std -0.063608/0.153886 | Adv std 7.962907e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.059321 | AvgAdv -0.000000 | AdvAfterStd 6.681023e-02
[Update 1035] Samples 2048 | Reward mean/std -0.061854/0.160563 | Value mean/std -0.051568/0.113762 | Adv std 1.121176e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.061854 | AvgAdv -0.000000 | AdvAfterStd 7.259207e-02
[Update 1036] Samples 2048 | Reward mean/std -0.069522/0.307809 | Value mean/std -0.069132/0.225483 | Adv std 1.261274e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.069522 | AvgAdv 0.000000 | AdvAfterStd 8.007607e-02
[Update 1037] Samples 2048 | Reward mean/std -0.067024/0.303576 | Value mean/std -0.065880/0.250607 | Adv std 1.127130e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.067024 | AvgAdv 0.000000 | AdvAfterStd 8.494562e-02
[Update 1038] Samples 2048 | Reward mean/std -0.066495/0.195968 | Value mean/std -0.070003/0.156121 | Adv std 1.004140e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.066495 | AvgAdv 0.000000 | AdvAfterStd 8.338834e-02
[Update 1039] Samples 2048 | Reward mean/std -0.064376/0.164038 | Value mean/std -0.059259/0.131063 | Adv std 1.068695e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.064376 | AvgAdv 0.000000 | AdvAfterStd 8.186626e-02
[Update 1040] Samples 2048 | Reward mean/std -0.069072/0.259964 | Value mean/std -0.080514/0.259875 | Adv std 1.178499e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.069072 | AvgAdv 0.000000 | AdvAfterStd 8.155330e-02
[Update 1041] Samples 2048 | Reward mean/std -0.059748/0.154589 | Value mean/std -0.058729/0.115454 | Adv std 9.426480e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.059748 | AvgAdv -0.000000 | AdvAfterStd 7.003922e-02
[Update 1042] Samples 2048 | Reward mean/std -0.058831/0.178125 | Value mean/std -0.064943/0.233068 | Adv std 1.265497e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.058831 | AvgAdv 0.000000 | AdvAfterStd 6.627861e-02
[Update 1043] Samples 2048 | Reward mean/std -0.058113/0.159581 | Value mean/std -0.052670/0.085064 | Adv std 1.207616e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.058113 | AvgAdv -0.000000 | AdvAfterStd 7.879159e-02
[Update 1044] Samples 2048 | Reward mean/std -0.065185/0.198284 | Value mean/std -0.058064/0.115619 | Adv std 1.290304e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.065185 | AvgAdv 0.000000 | AdvAfterStd 8.723963e-02
[Update 1045] Samples 2048 | Reward mean/std -0.065762/0.229710 | Value mean/std -0.068658/0.227242 | Adv std 1.208366e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.065762 | AvgAdv 0.000000 | AdvAfterStd 8.751889e-02
[Update 1046] Samples 2048 | Reward mean/std -0.060451/0.143008 | Value mean/std -0.057225/0.105939 | Adv std 8.816120e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.060451 | AvgAdv 0.000000 | AdvAfterStd 7.650648e-02
[Update 1047] Samples 2048 | Reward mean/std -0.062986/0.151405 | Value mean/std -0.063943/0.101544 | Adv std 1.065436e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.062986 | AvgAdv -0.000000 | AdvAfterStd 8.710638e-02
[Update 1048] Samples 2048 | Reward mean/std -0.063816/0.158677 | Value mean/std -0.067550/0.138457 | Adv std 9.912866e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.063816 | AvgAdv -0.000000 | AdvAfterStd 8.486934e-02
[Update 1049] Samples 2048 | Reward mean/std -0.058602/0.138120 | Value mean/std -0.061561/0.110735 | Adv std 8.940111e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.058602 | AvgAdv -0.000000 | AdvAfterStd 6.987226e-02
[Update 1050] Samples 2048 | Reward mean/std -0.069880/0.229107 | Value mean/std -0.068797/0.175747 | Adv std 1.092222e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.069880 | AvgAdv 0.000000 | AdvAfterStd 9.754384e-02
Saved checkpoint at update 1050
[Update 1051] Samples 2048 | Reward mean/std -0.063589/0.175312 | Value mean/std -0.056806/0.123054 | Adv std 1.101635e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.063589 | AvgAdv -0.000000 | AdvAfterStd 9.390445e-02
[Update 1052] Samples 2048 | Reward mean/std -0.067181/0.182603 | Value mean/std -0.069924/0.148793 | Adv std 1.181610e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.067181 | AvgAdv 0.000000 | AdvAfterStd 9.042322e-02
[Update 1053] Samples 2048 | Reward mean/std -0.063627/0.189969 | Value mean/std -0.061771/0.133056 | Adv std 1.146437e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.063627 | AvgAdv 0.000000 | AdvAfterStd 7.962886e-02
[Update 1054] Samples 2048 | Reward mean/std -0.065673/0.223002 | Value mean/std -0.069123/0.176902 | Adv std 1.096227e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.065673 | AvgAdv 0.000000 | AdvAfterStd 9.157902e-02
[Update 1055] Samples 2048 | Reward mean/std -0.071294/0.190981 | Value mean/std -0.068146/0.131931 | Adv std 1.338882e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.071294 | AvgAdv 0.000000 | AdvAfterStd 8.053353e-02
[Update 1056] Samples 2048 | Reward mean/std -0.067467/0.212564 | Value mean/std -0.072157/0.235457 | Adv std 1.381128e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.067467 | AvgAdv -0.000000 | AdvAfterStd 8.410893e-02
[Update 1057] Samples 2048 | Reward mean/std -0.070457/0.200755 | Value mean/std -0.064714/0.141817 | Adv std 1.462986e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.070457 | AvgAdv 0.000000 | AdvAfterStd 8.951821e-02
[Update 1058] Samples 2048 | Reward mean/std -0.061281/0.188060 | Value mean/std -0.073793/0.214781 | Adv std 1.135928e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.061281 | AvgAdv -0.000000 | AdvAfterStd 8.381750e-02
[Update 1059] Samples 2048 | Reward mean/std -0.071211/0.265874 | Value mean/std -0.072682/0.222095 | Adv std 1.071467e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.071211 | AvgAdv 0.000000 | AdvAfterStd 7.906260e-02
[Update 1060] Samples 2048 | Reward mean/std -0.064256/0.214053 | Value mean/std -0.060313/0.160294 | Adv std 1.108983e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.064256 | AvgAdv -0.000000 | AdvAfterStd 8.595397e-02
[Update 1061] Samples 2048 | Reward mean/std -0.066409/0.169545 | Value mean/std -0.065538/0.152365 | Adv std 1.050376e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.066409 | AvgAdv 0.000000 | AdvAfterStd 8.243130e-02
[Update 1062] Samples 2048 | Reward mean/std -0.069140/0.189763 | Value mean/std -0.067433/0.165391 | Adv std 1.291173e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.069140 | AvgAdv -0.000000 | AdvAfterStd 8.043077e-02
[Update 1063] Samples 2048 | Reward mean/std -0.067710/0.273295 | Value mean/std -0.070477/0.231095 | Adv std 1.362184e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.067710 | AvgAdv 0.000000 | AdvAfterStd 9.420634e-02
[Update 1064] Samples 2048 | Reward mean/std -0.060522/0.149450 | Value mean/std -0.060056/0.111572 | Adv std 1.037234e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.060522 | AvgAdv 0.000000 | AdvAfterStd 8.220945e-02
[Update 1065] Samples 2048 | Reward mean/std -0.065127/0.179633 | Value mean/std -0.068212/0.164487 | Adv std 1.128404e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.065127 | AvgAdv 0.000000 | AdvAfterStd 9.175447e-02
[Update 1066] Samples 2048 | Reward mean/std -0.065045/0.163709 | Value mean/std -0.059935/0.122361 | Adv std 1.094589e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.065045 | AvgAdv 0.000000 | AdvAfterStd 8.980054e-02
[Update 1067] Samples 2048 | Reward mean/std -0.072932/0.301667 | Value mean/std -0.066225/0.247270 | Adv std 1.762354e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.072932 | AvgAdv -0.000000 | AdvAfterStd 8.357441e-02
[Update 1068] Samples 2048 | Reward mean/std -0.062820/0.171580 | Value mean/std -0.066879/0.139480 | Adv std 9.356669e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.062820 | AvgAdv -0.000000 | AdvAfterStd 7.390425e-02
[Update 1069] Samples 2048 | Reward mean/std -0.077743/0.365340 | Value mean/std -0.069893/0.315413 | Adv std 1.239309e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.077743 | AvgAdv -0.000000 | AdvAfterStd 9.016682e-02
[Update 1070] Samples 2048 | Reward mean/std -0.063743/0.170160 | Value mean/std -0.057535/0.119124 | Adv std 1.185045e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.063743 | AvgAdv 0.000000 | AdvAfterStd 8.084385e-02
[Update 1071] Samples 2048 | Reward mean/std -0.067546/0.188849 | Value mean/std -0.061732/0.142982 | Adv std 1.038237e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.067546 | AvgAdv -0.000000 | AdvAfterStd 7.389202e-02
[Update 1072] Samples 2048 | Reward mean/std -0.063601/0.173170 | Value mean/std -0.064951/0.195209 | Adv std 1.233540e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.063601 | AvgAdv 0.000000 | AdvAfterStd 8.147216e-02
[Update 1073] Samples 2048 | Reward mean/std -0.064597/0.225208 | Value mean/std -0.068997/0.198989 | Adv std 1.269650e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.064597 | AvgAdv 0.000000 | AdvAfterStd 6.683087e-02
[Update 1074] Samples 2048 | Reward mean/std -0.063654/0.204670 | Value mean/std -0.072598/0.206778 | Adv std 1.234608e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.063654 | AvgAdv 0.000000 | AdvAfterStd 7.882698e-02
[Update 1075] Samples 2048 | Reward mean/std -0.054715/0.138707 | Value mean/std -0.053124/0.122726 | Adv std 9.073100e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.054715 | AvgAdv 0.000000 | AdvAfterStd 6.559413e-02
[Update 1076] Samples 2048 | Reward mean/std -0.068179/0.177349 | Value mean/std -0.059953/0.135441 | Adv std 1.030113e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.068179 | AvgAdv 0.000000 | AdvAfterStd 7.737356e-02
[Update 1077] Samples 2048 | Reward mean/std -0.062800/0.177805 | Value mean/std -0.062519/0.113551 | Adv std 1.315321e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.062800 | AvgAdv -0.000000 | AdvAfterStd 6.558384e-02
[Update 1078] Samples 2048 | Reward mean/std -0.058802/0.160693 | Value mean/std -0.059156/0.145864 | Adv std 9.464060e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.058802 | AvgAdv 0.000000 | AdvAfterStd 6.639463e-02
[Update 1079] Samples 2048 | Reward mean/std -0.057133/0.139835 | Value mean/std -0.057498/0.112927 | Adv std 9.631532e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.057133 | AvgAdv 0.000000 | AdvAfterStd 7.039674e-02
[Update 1080] Samples 2048 | Reward mean/std -0.058945/0.169068 | Value mean/std -0.059125/0.133347 | Adv std 9.463231e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.058945 | AvgAdv -0.000000 | AdvAfterStd 7.014140e-02
[Update 1081] Samples 2048 | Reward mean/std -0.068991/0.197693 | Value mean/std -0.060433/0.142512 | Adv std 1.160829e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.068991 | AvgAdv -0.000000 | AdvAfterStd 8.571985e-02
[Update 1082] Samples 2048 | Reward mean/std -0.061968/0.171451 | Value mean/std -0.068775/0.156241 | Adv std 1.148953e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.061968 | AvgAdv 0.000000 | AdvAfterStd 8.169838e-02
[Update 1083] Samples 2048 | Reward mean/std -0.060338/0.134681 | Value mean/std -0.056834/0.106524 | Adv std 1.079158e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.060338 | AvgAdv 0.000000 | AdvAfterStd 7.321610e-02
[Update 1084] Samples 2048 | Reward mean/std -0.059367/0.154582 | Value mean/std -0.060659/0.099203 | Adv std 1.217342e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.059367 | AvgAdv 0.000000 | AdvAfterStd 8.405483e-02
[Update 1085] Samples 2048 | Reward mean/std -0.065261/0.221440 | Value mean/std -0.067537/0.171638 | Adv std 1.548686e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.065261 | AvgAdv -0.000000 | AdvAfterStd 1.298647e-01
[Update 1086] Samples 2048 | Reward mean/std -0.061062/0.266640 | Value mean/std -0.062656/0.171377 | Adv std 1.767724e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.061062 | AvgAdv 0.000000 | AdvAfterStd 7.970246e-02
[Update 1087] Samples 2048 | Reward mean/std -0.063963/0.205289 | Value mean/std -0.061890/0.134453 | Adv std 1.699550e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.063963 | AvgAdv 0.000000 | AdvAfterStd 8.749854e-02
[Update 1088] Samples 2048 | Reward mean/std -0.064512/0.233305 | Value mean/std -0.065196/0.226689 | Adv std 1.204522e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.064512 | AvgAdv -0.000000 | AdvAfterStd 1.220654e-01
[Update 1089] Samples 2048 | Reward mean/std -0.066494/0.242439 | Value mean/std -0.062263/0.171196 | Adv std 1.254883e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.066494 | AvgAdv 0.000000 | AdvAfterStd 9.077530e-02
[Update 1090] Samples 2048 | Reward mean/std -0.065546/0.197197 | Value mean/std -0.062579/0.147318 | Adv std 1.214087e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.065546 | AvgAdv 0.000000 | AdvAfterStd 8.506968e-02
[Update 1091] Samples 2048 | Reward mean/std -0.064378/0.186464 | Value mean/std -0.062972/0.145458 | Adv std 1.181903e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.064378 | AvgAdv 0.000000 | AdvAfterStd 7.786731e-02
[Update 1092] Samples 2048 | Reward mean/std -0.059000/0.138663 | Value mean/std -0.063929/0.103636 | Adv std 9.342430e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.059000 | AvgAdv -0.000000 | AdvAfterStd 7.525005e-02
[Update 1093] Samples 2048 | Reward mean/std -0.064509/0.169558 | Value mean/std -0.058728/0.126575 | Adv std 1.106685e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.064509 | AvgAdv -0.000000 | AdvAfterStd 8.630273e-02
[Update 1094] Samples 2048 | Reward mean/std -0.058435/0.151980 | Value mean/std -0.057567/0.125198 | Adv std 1.067395e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.058435 | AvgAdv 0.000000 | AdvAfterStd 7.978740e-02
[Update 1095] Samples 2048 | Reward mean/std -0.053835/0.124174 | Value mean/std -0.055618/0.093139 | Adv std 9.733317e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.053835 | AvgAdv -0.000000 | AdvAfterStd 7.437522e-02
[Update 1096] Samples 2048 | Reward mean/std -0.066330/0.250343 | Value mean/std -0.059000/0.204236 | Adv std 1.198349e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.066330 | AvgAdv 0.000000 | AdvAfterStd 9.383719e-02
[Update 1097] Samples 2048 | Reward mean/std -0.065228/0.211200 | Value mean/std -0.071489/0.233924 | Adv std 1.283462e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.065228 | AvgAdv 0.000000 | AdvAfterStd 7.740749e-02
[Update 1098] Samples 2048 | Reward mean/std -0.058676/0.155341 | Value mean/std -0.053719/0.106528 | Adv std 1.011435e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.058676 | AvgAdv 0.000000 | AdvAfterStd 7.719508e-02
[Update 1099] Samples 2048 | Reward mean/std -0.064533/0.172229 | Value mean/std -0.055284/0.121499 | Adv std 1.178268e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.064533 | AvgAdv -0.000000 | AdvAfterStd 8.487649e-02
[Update 1100] Samples 2048 | Reward mean/std -0.060514/0.151587 | Value mean/std -0.063127/0.141161 | Adv std 1.066654e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.060514 | AvgAdv -0.000000 | AdvAfterStd 8.333644e-02
Saved checkpoint at update 1100
[Update 1101] Samples 2048 | Reward mean/std -0.062534/0.158495 | Value mean/std -0.059390/0.101630 | Adv std 1.057790e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.062534 | AvgAdv -0.000000 | AdvAfterStd 8.913875e-02
[Update 1102] Samples 2048 | Reward mean/std -0.058655/0.167910 | Value mean/std -0.062638/0.122585 | Adv std 1.077344e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.058655 | AvgAdv -0.000000 | AdvAfterStd 7.679127e-02
[Update 1103] Samples 2048 | Reward mean/std -0.054526/0.121351 | Value mean/std -0.048734/0.097858 | Adv std 9.505665e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.054526 | AvgAdv 0.000000 | AdvAfterStd 7.309314e-02
[Update 1104] Samples 2048 | Reward mean/std -0.063404/0.180990 | Value mean/std -0.058541/0.187022 | Adv std 1.151296e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.063404 | AvgAdv 0.000000 | AdvAfterStd 8.024111e-02
[Update 1105] Samples 2048 | Reward mean/std -0.062938/0.157719 | Value mean/std -0.070354/0.144607 | Adv std 9.582284e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.062938 | AvgAdv 0.000000 | AdvAfterStd 7.941569e-02
[Update 1106] Samples 2048 | Reward mean/std -0.067719/0.212654 | Value mean/std -0.058702/0.128157 | Adv std 1.388278e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.067719 | AvgAdv 0.000000 | AdvAfterStd 8.779304e-02
[Update 1107] Samples 2048 | Reward mean/std -0.057076/0.156689 | Value mean/std -0.063665/0.117845 | Adv std 9.673062e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.057076 | AvgAdv 0.000000 | AdvAfterStd 6.025597e-02
[Update 1108] Samples 2048 | Reward mean/std -0.064828/0.242192 | Value mean/std -0.060855/0.181782 | Adv std 1.183096e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.064828 | AvgAdv 0.000000 | AdvAfterStd 7.095454e-02
[Update 1109] Samples 2048 | Reward mean/std -0.071852/0.281799 | Value mean/std -0.065961/0.228863 | Adv std 1.332439e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.071852 | AvgAdv -0.000000 | AdvAfterStd 8.958527e-02
[Update 1110] Samples 2048 | Reward mean/std -0.059323/0.140063 | Value mean/std -0.069862/0.161011 | Adv std 1.029650e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.059323 | AvgAdv -0.000000 | AdvAfterStd 6.557237e-02
[Update 1111] Samples 2048 | Reward mean/std -0.059209/0.142741 | Value mean/std -0.053576/0.101342 | Adv std 9.145965e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.059209 | AvgAdv -0.000000 | AdvAfterStd 7.396030e-02
[Update 1112] Samples 2048 | Reward mean/std -0.059353/0.133475 | Value mean/std -0.059077/0.111658 | Adv std 1.054704e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.059353 | AvgAdv -0.000000 | AdvAfterStd 7.978297e-02
[Update 1113] Samples 2048 | Reward mean/std -0.059977/0.145330 | Value mean/std -0.056621/0.102475 | Adv std 9.632140e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.059977 | AvgAdv 0.000000 | AdvAfterStd 7.959139e-02
[Update 1114] Samples 2048 | Reward mean/std -0.064455/0.173560 | Value mean/std -0.059373/0.124443 | Adv std 1.148964e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.064455 | AvgAdv 0.000000 | AdvAfterStd 7.692539e-02
[Update 1115] Samples 2048 | Reward mean/std -0.072698/0.187126 | Value mean/std -0.067257/0.144587 | Adv std 1.299913e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.072698 | AvgAdv 0.000000 | AdvAfterStd 9.820312e-02
[Update 1116] Samples 2048 | Reward mean/std -0.071277/0.284620 | Value mean/std -0.078396/0.193900 | Adv std 1.452105e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.071277 | AvgAdv 0.000000 | AdvAfterStd 7.665495e-02
[Update 1117] Samples 2048 | Reward mean/std -0.059404/0.170378 | Value mean/std -0.057710/0.109044 | Adv std 1.380941e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.059404 | AvgAdv 0.000000 | AdvAfterStd 7.963518e-02
[Update 1118] Samples 2048 | Reward mean/std -0.058357/0.134019 | Value mean/std -0.057269/0.130719 | Adv std 1.135921e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.058357 | AvgAdv -0.000000 | AdvAfterStd 7.067890e-02
[Update 1119] Samples 2048 | Reward mean/std -0.059662/0.140909 | Value mean/std -0.050615/0.078647 | Adv std 1.117565e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.059662 | AvgAdv 0.000000 | AdvAfterStd 7.763886e-02
[Update 1120] Samples 2048 | Reward mean/std -0.061896/0.148482 | Value mean/std -0.064099/0.130608 | Adv std 1.139545e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.061896 | AvgAdv 0.000000 | AdvAfterStd 7.228570e-02
[Update 1121] Samples 2048 | Reward mean/std -0.065346/0.276697 | Value mean/std -0.063490/0.147385 | Adv std 1.663077e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.065346 | AvgAdv -0.000000 | AdvAfterStd 9.090929e-02
[Update 1122] Samples 2048 | Reward mean/std -0.066636/0.253783 | Value mean/std -0.067809/0.305359 | Adv std 1.395014e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.066636 | AvgAdv 0.000000 | AdvAfterStd 1.039666e-01
[Update 1123] Samples 2048 | Reward mean/std -0.060311/0.162724 | Value mean/std -0.059416/0.156563 | Adv std 1.028890e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.060311 | AvgAdv -0.000000 | AdvAfterStd 6.736761e-02
[Update 1124] Samples 2048 | Reward mean/std -0.059351/0.156597 | Value mean/std -0.060740/0.151191 | Adv std 1.008931e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.059351 | AvgAdv 0.000000 | AdvAfterStd 6.957538e-02
[Update 1125] Samples 2048 | Reward mean/std -0.060948/0.141750 | Value mean/std -0.058622/0.105861 | Adv std 1.069017e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.060948 | AvgAdv -0.000000 | AdvAfterStd 7.798293e-02
[Update 1126] Samples 2048 | Reward mean/std -0.060927/0.156989 | Value mean/std -0.056309/0.104704 | Adv std 1.101369e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.060927 | AvgAdv -0.000000 | AdvAfterStd 8.176571e-02
[Update 1127] Samples 2048 | Reward mean/std -0.051423/0.125042 | Value mean/std -0.053288/0.115193 | Adv std 7.876072e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.051423 | AvgAdv -0.000000 | AdvAfterStd 6.464997e-02
[Update 1128] Samples 2048 | Reward mean/std -0.057843/0.179725 | Value mean/std -0.049541/0.099796 | Adv std 1.274146e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.057843 | AvgAdv 0.000000 | AdvAfterStd 8.024903e-02
[Update 1129] Samples 2048 | Reward mean/std -0.058734/0.160047 | Value mean/std -0.062140/0.148754 | Adv std 1.298578e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.058734 | AvgAdv 0.000000 | AdvAfterStd 8.681926e-02
[Update 1130] Samples 2048 | Reward mean/std -0.058474/0.148952 | Value mean/std -0.062941/0.143233 | Adv std 1.208009e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.058474 | AvgAdv -0.000000 | AdvAfterStd 7.211424e-02
[Update 1131] Samples 2048 | Reward mean/std -0.059783/0.175815 | Value mean/std -0.054971/0.135812 | Adv std 1.130895e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.059783 | AvgAdv 0.000000 | AdvAfterStd 7.247306e-02
[Update 1132] Samples 2048 | Reward mean/std -0.062896/0.193188 | Value mean/std -0.060843/0.141908 | Adv std 1.210185e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.062896 | AvgAdv -0.000000 | AdvAfterStd 8.968998e-02
[Update 1133] Samples 2048 | Reward mean/std -0.063330/0.201742 | Value mean/std -0.055633/0.146461 | Adv std 1.506504e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.063330 | AvgAdv -0.000000 | AdvAfterStd 9.049653e-02
[Update 1134] Samples 2048 | Reward mean/std -0.064227/0.186815 | Value mean/std -0.059911/0.148167 | Adv std 1.182888e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.064227 | AvgAdv 0.000000 | AdvAfterStd 8.371423e-02
[Update 1135] Samples 2048 | Reward mean/std -0.064281/0.245881 | Value mean/std -0.061704/0.206166 | Adv std 1.458333e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.064281 | AvgAdv 0.000000 | AdvAfterStd 7.992107e-02
[Update 1136] Samples 2048 | Reward mean/std -0.067600/0.308410 | Value mean/std -0.061869/0.236520 | Adv std 1.319671e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.067600 | AvgAdv -0.000000 | AdvAfterStd 1.031264e-01
[Update 1137] Samples 2048 | Reward mean/std -0.067362/0.223901 | Value mean/std -0.074743/0.168843 | Adv std 1.402981e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.067362 | AvgAdv 0.000000 | AdvAfterStd 9.049947e-02
[Update 1138] Samples 2048 | Reward mean/std -0.053307/0.138748 | Value mean/std -0.058464/0.168291 | Adv std 9.740730e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.053307 | AvgAdv -0.000000 | AdvAfterStd 6.202578e-02
[Update 1139] Samples 2048 | Reward mean/std -0.058660/0.175502 | Value mean/std -0.054895/0.137955 | Adv std 1.043526e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.058660 | AvgAdv 0.000000 | AdvAfterStd 6.443530e-02
[Update 1140] Samples 2048 | Reward mean/std -0.064581/0.201458 | Value mean/std -0.054168/0.172520 | Adv std 1.219589e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.064581 | AvgAdv -0.000000 | AdvAfterStd 7.760582e-02
[Update 1141] Samples 2048 | Reward mean/std -0.055897/0.134575 | Value mean/std -0.059384/0.118404 | Adv std 8.790661e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.055897 | AvgAdv -0.000000 | AdvAfterStd 7.647725e-02
[Update 1142] Samples 2048 | Reward mean/std -0.056957/0.134650 | Value mean/std -0.058861/0.142373 | Adv std 1.056825e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.056957 | AvgAdv 0.000000 | AdvAfterStd 7.181296e-02
[Update 1143] Samples 2048 | Reward mean/std -0.061166/0.195075 | Value mean/std -0.059805/0.109834 | Adv std 1.444814e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.061166 | AvgAdv -0.000000 | AdvAfterStd 8.425060e-02
[Update 1144] Samples 2048 | Reward mean/std -0.059002/0.147032 | Value mean/std -0.055219/0.109761 | Adv std 1.147046e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.059002 | AvgAdv 0.000000 | AdvAfterStd 8.129700e-02
[Update 1145] Samples 2048 | Reward mean/std -0.063292/0.236922 | Value mean/std -0.056218/0.167725 | Adv std 1.148656e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.063292 | AvgAdv -0.000000 | AdvAfterStd 7.239258e-02
[Update 1146] Samples 2048 | Reward mean/std -0.068913/0.217858 | Value mean/std -0.071244/0.240032 | Adv std 1.472482e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.068913 | AvgAdv 0.000000 | AdvAfterStd 8.846755e-02
[Update 1147] Samples 2048 | Reward mean/std -0.058617/0.141105 | Value mean/std -0.059436/0.099883 | Adv std 1.059436e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.058617 | AvgAdv -0.000000 | AdvAfterStd 7.471394e-02
[Update 1148] Samples 2048 | Reward mean/std -0.066538/0.241357 | Value mean/std -0.059514/0.212496 | Adv std 1.042622e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.066538 | AvgAdv -0.000000 | AdvAfterStd 8.841294e-02
[Update 1149] Samples 2048 | Reward mean/std -0.059968/0.171341 | Value mean/std -0.059674/0.118711 | Adv std 1.115778e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.059968 | AvgAdv -0.000000 | AdvAfterStd 8.782362e-02
[Update 1150] Samples 2048 | Reward mean/std -0.065605/0.170462 | Value mean/std -0.058806/0.123817 | Adv std 1.157298e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.065605 | AvgAdv 0.000000 | AdvAfterStd 8.070176e-02
Saved checkpoint at update 1150
[Update 1151] Samples 2048 | Reward mean/std -0.064801/0.183311 | Value mean/std -0.058905/0.162299 | Adv std 1.112586e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.064801 | AvgAdv -0.000000 | AdvAfterStd 8.033538e-02
[Update 1152] Samples 2048 | Reward mean/std -0.062382/0.195682 | Value mean/std -0.066740/0.155696 | Adv std 1.160753e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.062382 | AvgAdv 0.000000 | AdvAfterStd 7.951806e-02
[Update 1153] Samples 2048 | Reward mean/std -0.065136/0.278602 | Value mean/std -0.062662/0.191632 | Adv std 1.339181e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.065136 | AvgAdv 0.000000 | AdvAfterStd 7.073855e-02
[Update 1154] Samples 2048 | Reward mean/std -0.061900/0.168067 | Value mean/std -0.062422/0.135972 | Adv std 9.414973e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.061900 | AvgAdv -0.000000 | AdvAfterStd 7.123904e-02
[Update 1155] Samples 2048 | Reward mean/std -0.069932/0.195698 | Value mean/std -0.063289/0.158516 | Adv std 1.234247e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.069932 | AvgAdv -0.000000 | AdvAfterStd 1.009613e-01
[Update 1156] Samples 2048 | Reward mean/std -0.070422/0.300893 | Value mean/std -0.068326/0.254007 | Adv std 1.484624e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.070422 | AvgAdv -0.000000 | AdvAfterStd 7.220723e-02
[Update 1157] Samples 2048 | Reward mean/std -0.067728/0.231480 | Value mean/std -0.072031/0.237377 | Adv std 1.150084e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.067728 | AvgAdv -0.000000 | AdvAfterStd 7.898419e-02
[Update 1158] Samples 2048 | Reward mean/std -0.057222/0.145157 | Value mean/std -0.056425/0.109422 | Adv std 9.359188e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.057222 | AvgAdv -0.000000 | AdvAfterStd 6.960306e-02
[Update 1159] Samples 2048 | Reward mean/std -0.062842/0.167993 | Value mean/std -0.062621/0.141044 | Adv std 1.034775e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.062842 | AvgAdv -0.000000 | AdvAfterStd 8.053011e-02
[Update 1160] Samples 2048 | Reward mean/std -0.063198/0.149952 | Value mean/std -0.062635/0.118627 | Adv std 1.077626e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.063198 | AvgAdv -0.000000 | AdvAfterStd 9.067712e-02
[Update 1161] Samples 2048 | Reward mean/std -0.063284/0.162686 | Value mean/std -0.067880/0.119539 | Adv std 1.036567e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.063284 | AvgAdv -0.000000 | AdvAfterStd 8.135560e-02
[Update 1162] Samples 2048 | Reward mean/std -0.064016/0.234805 | Value mean/std -0.066835/0.183543 | Adv std 1.020947e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.064016 | AvgAdv 0.000000 | AdvAfterStd 7.867683e-02
[Update 1163] Samples 2048 | Reward mean/std -0.066386/0.218684 | Value mean/std -0.066324/0.190035 | Adv std 9.165602e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.066386 | AvgAdv 0.000000 | AdvAfterStd 7.398481e-02
[Update 1164] Samples 2048 | Reward mean/std -0.064304/0.199997 | Value mean/std -0.063163/0.126889 | Adv std 1.357942e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.064304 | AvgAdv -0.000000 | AdvAfterStd 7.209132e-02
[Update 1165] Samples 2048 | Reward mean/std -0.066284/0.186692 | Value mean/std -0.059123/0.162872 | Adv std 1.247795e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.066284 | AvgAdv 0.000000 | AdvAfterStd 7.505961e-02
[Update 1166] Samples 2048 | Reward mean/std -0.063195/0.222574 | Value mean/std -0.066385/0.252333 | Adv std 1.819309e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.063195 | AvgAdv -0.000000 | AdvAfterStd 8.491915e-02
[Update 1167] Samples 2048 | Reward mean/std -0.064950/0.189106 | Value mean/std -0.065818/0.172220 | Adv std 1.222460e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.064950 | AvgAdv -0.000000 | AdvAfterStd 7.214864e-02
[Update 1168] Samples 2048 | Reward mean/std -0.052735/0.134632 | Value mean/std -0.049556/0.110342 | Adv std 1.006477e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.052735 | AvgAdv 0.000000 | AdvAfterStd 6.900131e-02
[Update 1169] Samples 2048 | Reward mean/std -0.064489/0.236495 | Value mean/std -0.056453/0.193615 | Adv std 1.118675e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.064489 | AvgAdv 0.000000 | AdvAfterStd 9.098872e-02
[Update 1170] Samples 2048 | Reward mean/std -0.057165/0.132990 | Value mean/std -0.052612/0.085062 | Adv std 9.658901e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.057165 | AvgAdv -0.000000 | AdvAfterStd 8.171389e-02
[Update 1171] Samples 2048 | Reward mean/std -0.064148/0.186663 | Value mean/std -0.061122/0.174410 | Adv std 1.021274e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.064148 | AvgAdv -0.000000 | AdvAfterStd 7.054970e-02
[Update 1172] Samples 2048 | Reward mean/std -0.059847/0.145132 | Value mean/std -0.058481/0.106536 | Adv std 9.114463e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.059847 | AvgAdv 0.000000 | AdvAfterStd 7.940912e-02
[Update 1173] Samples 2048 | Reward mean/std -0.066329/0.255600 | Value mean/std -0.065138/0.233241 | Adv std 1.210576e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.066329 | AvgAdv -0.000000 | AdvAfterStd 7.974629e-02
[Update 1174] Samples 2048 | Reward mean/std -0.068378/0.226727 | Value mean/std -0.071111/0.230307 | Adv std 1.236399e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.068378 | AvgAdv -0.000000 | AdvAfterStd 8.926308e-02
[Update 1175] Samples 2048 | Reward mean/std -0.063909/0.174648 | Value mean/std -0.061601/0.119432 | Adv std 1.065009e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.063909 | AvgAdv 0.000000 | AdvAfterStd 7.839240e-02
[Update 1176] Samples 2048 | Reward mean/std -0.061780/0.134280 | Value mean/std -0.062056/0.114325 | Adv std 9.257793e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.061780 | AvgAdv -0.000000 | AdvAfterStd 6.808768e-02
[Update 1177] Samples 2048 | Reward mean/std -0.055817/0.125593 | Value mean/std -0.059815/0.095280 | Adv std 9.377448e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.055817 | AvgAdv 0.000000 | AdvAfterStd 6.703964e-02
[Update 1178] Samples 2048 | Reward mean/std -0.067964/0.257426 | Value mean/std -0.060912/0.184217 | Adv std 1.179097e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.067964 | AvgAdv 0.000000 | AdvAfterStd 6.787615e-02
[Update 1179] Samples 2048 | Reward mean/std -0.059991/0.132057 | Value mean/std -0.057963/0.091287 | Adv std 1.102505e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.059991 | AvgAdv 0.000000 | AdvAfterStd 8.860344e-02
[Update 1180] Samples 2048 | Reward mean/std -0.060347/0.186840 | Value mean/std -0.063207/0.147343 | Adv std 1.061627e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.060347 | AvgAdv 0.000000 | AdvAfterStd 7.780843e-02
[Update 1181] Samples 2048 | Reward mean/std -0.071089/0.295325 | Value mean/std -0.066680/0.259479 | Adv std 1.337864e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.071089 | AvgAdv -0.000000 | AdvAfterStd 9.703273e-02
[Update 1182] Samples 2048 | Reward mean/std -0.069346/0.249876 | Value mean/std -0.065357/0.148092 | Adv std 1.563188e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.069346 | AvgAdv -0.000000 | AdvAfterStd 8.054634e-02
[Update 1183] Samples 2048 | Reward mean/std -0.067283/0.239730 | Value mean/std -0.073390/0.235237 | Adv std 1.178482e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.067283 | AvgAdv 0.000000 | AdvAfterStd 8.002629e-02
[Update 1184] Samples 2048 | Reward mean/std -0.061763/0.238792 | Value mean/std -0.062267/0.219451 | Adv std 9.714333e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.061763 | AvgAdv 0.000000 | AdvAfterStd 7.962129e-02
[Update 1185] Samples 2048 | Reward mean/std -0.056664/0.161121 | Value mean/std -0.055849/0.114915 | Adv std 7.922391e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.056664 | AvgAdv -0.000000 | AdvAfterStd 6.202118e-02
[Update 1186] Samples 2048 | Reward mean/std -0.060233/0.154579 | Value mean/std -0.058021/0.130401 | Adv std 1.132313e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.060233 | AvgAdv 0.000000 | AdvAfterStd 7.958741e-02
[Update 1187] Samples 2048 | Reward mean/std -0.060091/0.155366 | Value mean/std -0.055203/0.111583 | Adv std 1.037664e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.060091 | AvgAdv 0.000000 | AdvAfterStd 7.752096e-02
[Update 1188] Samples 2048 | Reward mean/std -0.069926/0.218937 | Value mean/std -0.064779/0.187344 | Adv std 1.067192e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.069926 | AvgAdv -0.000000 | AdvAfterStd 7.820114e-02
[Update 1189] Samples 2048 | Reward mean/std -0.065357/0.153241 | Value mean/std -0.061920/0.126766 | Adv std 1.159602e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.065357 | AvgAdv -0.000000 | AdvAfterStd 7.512463e-02
[Update 1190] Samples 2048 | Reward mean/std -0.063317/0.173622 | Value mean/std -0.065980/0.136007 | Adv std 1.127429e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.063317 | AvgAdv 0.000000 | AdvAfterStd 8.092646e-02
[Update 1191] Samples 2048 | Reward mean/std -0.062856/0.160623 | Value mean/std -0.060689/0.126122 | Adv std 1.025476e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.062856 | AvgAdv 0.000000 | AdvAfterStd 7.202738e-02
[Update 1192] Samples 2048 | Reward mean/std -0.067727/0.285927 | Value mean/std -0.064595/0.201468 | Adv std 1.291714e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.067727 | AvgAdv -0.000000 | AdvAfterStd 7.414299e-02
[Update 1193] Samples 2048 | Reward mean/std -0.058553/0.149911 | Value mean/std -0.060188/0.132915 | Adv std 1.039511e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.058553 | AvgAdv -0.000000 | AdvAfterStd 7.312629e-02
[Update 1194] Samples 2048 | Reward mean/std -0.071629/0.207875 | Value mean/std -0.069524/0.155307 | Adv std 1.072529e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.071629 | AvgAdv 0.000000 | AdvAfterStd 9.293487e-02
[Update 1195] Samples 2048 | Reward mean/std -0.060788/0.157429 | Value mean/std -0.057416/0.125605 | Adv std 8.931430e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.060788 | AvgAdv -0.000000 | AdvAfterStd 7.445142e-02
[Update 1196] Samples 2048 | Reward mean/std -0.071788/0.286024 | Value mean/std -0.070837/0.283083 | Adv std 1.283769e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.071788 | AvgAdv 0.000000 | AdvAfterStd 7.366380e-02
[Update 1197] Samples 2048 | Reward mean/std -0.064037/0.224362 | Value mean/std -0.067393/0.198907 | Adv std 1.015154e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.064037 | AvgAdv 0.000000 | AdvAfterStd 7.986046e-02
[Update 1198] Samples 2048 | Reward mean/std -0.059349/0.138111 | Value mean/std -0.061850/0.110136 | Adv std 9.281589e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.059349 | AvgAdv 0.000000 | AdvAfterStd 7.393164e-02
[Update 1199] Samples 2048 | Reward mean/std -0.056805/0.140839 | Value mean/std -0.065651/0.116294 | Adv std 9.582720e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.056805 | AvgAdv 0.000000 | AdvAfterStd 6.966168e-02
[Update 1200] Samples 2048 | Reward mean/std -0.060137/0.141339 | Value mean/std -0.058230/0.110318 | Adv std 9.177085e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.060137 | AvgAdv -0.000000 | AdvAfterStd 7.741766e-02
Saved checkpoint at update 1200
[Update 1201] Samples 2048 | Reward mean/std -0.057192/0.128048 | Value mean/std -0.058375/0.113407 | Adv std 8.184649e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.057192 | AvgAdv 0.000000 | AdvAfterStd 6.384075e-02
[Update 1202] Samples 2048 | Reward mean/std -0.062907/0.249691 | Value mean/std -0.057305/0.133483 | Adv std 1.608789e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.062907 | AvgAdv 0.000000 | AdvAfterStd 8.046314e-02
[Update 1203] Samples 2048 | Reward mean/std -0.063429/0.171669 | Value mean/std -0.064264/0.184018 | Adv std 1.251486e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.063429 | AvgAdv 0.000000 | AdvAfterStd 8.114944e-02
[Update 1204] Samples 2048 | Reward mean/std -0.063802/0.177238 | Value mean/std -0.063677/0.136307 | Adv std 1.015233e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.063802 | AvgAdv 0.000000 | AdvAfterStd 7.279682e-02
[Update 1205] Samples 2048 | Reward mean/std -0.064062/0.155807 | Value mean/std -0.067449/0.120188 | Adv std 1.140189e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.064062 | AvgAdv -0.000000 | AdvAfterStd 8.653527e-02
[Update 1206] Samples 2048 | Reward mean/std -0.061593/0.144526 | Value mean/std -0.059812/0.108865 | Adv std 9.598466e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.061593 | AvgAdv 0.000000 | AdvAfterStd 7.912765e-02
[Update 1207] Samples 2048 | Reward mean/std -0.075707/0.277727 | Value mean/std -0.071014/0.192077 | Adv std 1.537840e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.075707 | AvgAdv 0.000000 | AdvAfterStd 9.976620e-02
[Update 1208] Samples 2048 | Reward mean/std -0.060737/0.154336 | Value mean/std -0.063832/0.128030 | Adv std 9.901265e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.060737 | AvgAdv 0.000000 | AdvAfterStd 7.581127e-02
[Update 1209] Samples 2048 | Reward mean/std -0.058421/0.198357 | Value mean/std -0.058947/0.199438 | Adv std 8.685753e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.058421 | AvgAdv 0.000000 | AdvAfterStd 6.469399e-02
[Update 1210] Samples 2048 | Reward mean/std -0.062970/0.157660 | Value mean/std -0.064104/0.145678 | Adv std 1.170667e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.062970 | AvgAdv -0.000000 | AdvAfterStd 7.271605e-02
[Update 1211] Samples 2048 | Reward mean/std -0.057082/0.139696 | Value mean/std -0.052363/0.106898 | Adv std 8.969017e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.057082 | AvgAdv 0.000000 | AdvAfterStd 6.535465e-02
[Update 1212] Samples 2048 | Reward mean/std -0.064074/0.169411 | Value mean/std -0.062480/0.114176 | Adv std 1.078599e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.064074 | AvgAdv 0.000000 | AdvAfterStd 7.420190e-02
[Update 1213] Samples 2048 | Reward mean/std -0.055080/0.199362 | Value mean/std -0.056760/0.204622 | Adv std 7.175339e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.055080 | AvgAdv -0.000000 | AdvAfterStd 6.131458e-02
[Update 1214] Samples 2048 | Reward mean/std -0.061797/0.188186 | Value mean/std -0.059071/0.178207 | Adv std 8.585902e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.061797 | AvgAdv 0.000000 | AdvAfterStd 7.310744e-02
[Update 1215] Samples 2048 | Reward mean/std -0.062606/0.171659 | Value mean/std -0.066379/0.123326 | Adv std 9.929081e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.062606 | AvgAdv -0.000000 | AdvAfterStd 7.272648e-02
[Update 1216] Samples 2048 | Reward mean/std -0.055264/0.131323 | Value mean/std -0.058755/0.133386 | Adv std 8.589533e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.055264 | AvgAdv 0.000000 | AdvAfterStd 6.929840e-02
[Update 1217] Samples 2048 | Reward mean/std -0.062075/0.167335 | Value mean/std -0.062112/0.131471 | Adv std 1.109730e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.062075 | AvgAdv -0.000000 | AdvAfterStd 8.166107e-02
[Update 1218] Samples 2048 | Reward mean/std -0.057028/0.200684 | Value mean/std -0.058177/0.102141 | Adv std 1.416471e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.057028 | AvgAdv -0.000000 | AdvAfterStd 6.261523e-02
[Update 1219] Samples 2048 | Reward mean/std -0.059851/0.162378 | Value mean/std -0.058402/0.125153 | Adv std 1.053643e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.059851 | AvgAdv 0.000000 | AdvAfterStd 7.936291e-02
[Update 1220] Samples 2048 | Reward mean/std -0.062764/0.238453 | Value mean/std -0.071972/0.224643 | Adv std 1.072822e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.062764 | AvgAdv 0.000000 | AdvAfterStd 7.499445e-02
[Update 1221] Samples 2048 | Reward mean/std -0.060650/0.158172 | Value mean/std -0.048818/0.101395 | Adv std 1.096235e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.060650 | AvgAdv 0.000000 | AdvAfterStd 7.489723e-02
[Update 1222] Samples 2048 | Reward mean/std -0.064488/0.187434 | Value mean/std -0.055428/0.123149 | Adv std 1.261834e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.064488 | AvgAdv -0.000000 | AdvAfterStd 8.379037e-02
[Update 1223] Samples 2048 | Reward mean/std -0.059230/0.179158 | Value mean/std -0.059594/0.246131 | Adv std 1.585803e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.059230 | AvgAdv 0.000000 | AdvAfterStd 7.110611e-02
[Update 1224] Samples 2048 | Reward mean/std -0.062958/0.175626 | Value mean/std -0.065996/0.117003 | Adv std 1.108550e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.062958 | AvgAdv 0.000000 | AdvAfterStd 7.414164e-02
[Update 1225] Samples 2048 | Reward mean/std -0.064030/0.206120 | Value mean/std -0.060587/0.164887 | Adv std 1.125634e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.064030 | AvgAdv 0.000000 | AdvAfterStd 8.626354e-02
[Update 1226] Samples 2048 | Reward mean/std -0.064211/0.191042 | Value mean/std -0.061109/0.160096 | Adv std 1.250874e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.064211 | AvgAdv -0.000000 | AdvAfterStd 7.510038e-02
[Update 1227] Samples 2048 | Reward mean/std -0.061889/0.188698 | Value mean/std -0.065878/0.172010 | Adv std 1.226566e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.061889 | AvgAdv -0.000000 | AdvAfterStd 9.501599e-02
[Update 1228] Samples 2048 | Reward mean/std -0.056697/0.142171 | Value mean/std -0.055454/0.110479 | Adv std 9.597696e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.056697 | AvgAdv 0.000000 | AdvAfterStd 7.033177e-02
[Update 1229] Samples 2048 | Reward mean/std -0.053668/0.130759 | Value mean/std -0.060696/0.156798 | Adv std 1.105952e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.053668 | AvgAdv 0.000000 | AdvAfterStd 7.369201e-02
[Update 1230] Samples 2048 | Reward mean/std -0.058759/0.145029 | Value mean/std -0.059840/0.109630 | Adv std 9.890208e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.058759 | AvgAdv -0.000000 | AdvAfterStd 8.182739e-02
[Update 1231] Samples 2048 | Reward mean/std -0.057916/0.157612 | Value mean/std -0.053966/0.099817 | Adv std 1.163426e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.057916 | AvgAdv 0.000000 | AdvAfterStd 9.707107e-02
[Update 1232] Samples 2048 | Reward mean/std -0.061167/0.149830 | Value mean/std -0.066388/0.139987 | Adv std 9.559281e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.061167 | AvgAdv 0.000000 | AdvAfterStd 7.666343e-02
[Update 1233] Samples 2048 | Reward mean/std -0.060670/0.175189 | Value mean/std -0.059786/0.115940 | Adv std 1.173227e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.060670 | AvgAdv -0.000000 | AdvAfterStd 8.315939e-02
[Update 1234] Samples 2048 | Reward mean/std -0.061892/0.182344 | Value mean/std -0.055971/0.123218 | Adv std 1.072313e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.061892 | AvgAdv 0.000000 | AdvAfterStd 8.203813e-02
[Update 1235] Samples 2048 | Reward mean/std -0.060222/0.147602 | Value mean/std -0.055727/0.132957 | Adv std 1.181658e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.060222 | AvgAdv -0.000000 | AdvAfterStd 8.768348e-02
[Update 1236] Samples 2048 | Reward mean/std -0.076816/0.334133 | Value mean/std -0.066297/0.222567 | Adv std 1.615514e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.076816 | AvgAdv 0.000000 | AdvAfterStd 8.447287e-02
[Update 1237] Samples 2048 | Reward mean/std -0.065299/0.159016 | Value mean/std -0.063553/0.130630 | Adv std 1.266824e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.065299 | AvgAdv 0.000000 | AdvAfterStd 9.188035e-02
[Update 1238] Samples 2048 | Reward mean/std -0.066279/0.238849 | Value mean/std -0.069404/0.203288 | Adv std 1.154403e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.066279 | AvgAdv 0.000000 | AdvAfterStd 8.939992e-02
[Update 1239] Samples 2048 | Reward mean/std -0.061047/0.163453 | Value mean/std -0.061311/0.170519 | Adv std 1.104620e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.061047 | AvgAdv -0.000000 | AdvAfterStd 8.179307e-02
[Update 1240] Samples 2048 | Reward mean/std -0.066066/0.180308 | Value mean/std -0.074179/0.154829 | Adv std 1.127128e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.066066 | AvgAdv 0.000000 | AdvAfterStd 7.505706e-02
[Update 1241] Samples 2048 | Reward mean/std -0.061459/0.239310 | Value mean/std -0.061756/0.188354 | Adv std 1.030581e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.061459 | AvgAdv 0.000000 | AdvAfterStd 6.681238e-02
[Update 1242] Samples 2048 | Reward mean/std -0.063504/0.171393 | Value mean/std -0.066060/0.114081 | Adv std 1.193473e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.063504 | AvgAdv 0.000000 | AdvAfterStd 7.726031e-02
[Update 1243] Samples 2048 | Reward mean/std -0.070295/0.309793 | Value mean/std -0.066913/0.247295 | Adv std 1.395143e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.070295 | AvgAdv 0.000000 | AdvAfterStd 9.014641e-02
[Update 1244] Samples 2048 | Reward mean/std -0.057094/0.131993 | Value mean/std -0.055400/0.095519 | Adv std 8.663891e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.057094 | AvgAdv -0.000000 | AdvAfterStd 6.991570e-02
[Update 1245] Samples 2048 | Reward mean/std -0.060361/0.139547 | Value mean/std -0.065466/0.121139 | Adv std 9.977596e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.060361 | AvgAdv 0.000000 | AdvAfterStd 8.180708e-02
[Update 1246] Samples 2048 | Reward mean/std -0.058162/0.199190 | Value mean/std -0.058108/0.104002 | Adv std 1.495158e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.058162 | AvgAdv 0.000000 | AdvAfterStd 6.961930e-02
[Update 1247] Samples 2048 | Reward mean/std -0.059035/0.176818 | Value mean/std -0.052886/0.169947 | Adv std 1.353735e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.059035 | AvgAdv -0.000000 | AdvAfterStd 8.386575e-02
[Update 1248] Samples 2048 | Reward mean/std -0.060569/0.202438 | Value mean/std -0.061531/0.194398 | Adv std 8.968855e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.060569 | AvgAdv -0.000000 | AdvAfterStd 6.672031e-02
[Update 1249] Samples 2048 | Reward mean/std -0.056327/0.143935 | Value mean/std -0.055807/0.123489 | Adv std 1.166772e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.056327 | AvgAdv 0.000000 | AdvAfterStd 7.649501e-02
[Update 1250] Samples 2048 | Reward mean/std -0.059391/0.193973 | Value mean/std -0.060429/0.150581 | Adv std 1.077657e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.059391 | AvgAdv -0.000000 | AdvAfterStd 7.598440e-02
Saved checkpoint at update 1250
[Update 1251] Samples 2048 | Reward mean/std -0.058834/0.162419 | Value mean/std -0.059700/0.156952 | Adv std 1.046275e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.058834 | AvgAdv 0.000000 | AdvAfterStd 7.694890e-02
[Update 1252] Samples 2048 | Reward mean/std -0.054687/0.140089 | Value mean/std -0.053005/0.085264 | Adv std 1.026580e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.054687 | AvgAdv 0.000000 | AdvAfterStd 7.260298e-02
[Update 1253] Samples 2048 | Reward mean/std -0.068120/0.213517 | Value mean/std -0.066751/0.179491 | Adv std 1.216831e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.068120 | AvgAdv 0.000000 | AdvAfterStd 8.570048e-02
[Update 1254] Samples 2048 | Reward mean/std -0.061044/0.238241 | Value mean/std -0.069319/0.193789 | Adv std 1.239187e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.061044 | AvgAdv -0.000000 | AdvAfterStd 7.384738e-02
[Update 1255] Samples 2048 | Reward mean/std -0.064077/0.238891 | Value mean/std -0.056617/0.188473 | Adv std 1.094208e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.064077 | AvgAdv -0.000000 | AdvAfterStd 9.600379e-02
[Update 1256] Samples 2048 | Reward mean/std -0.057883/0.155205 | Value mean/std -0.062286/0.140545 | Adv std 9.333105e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.057883 | AvgAdv 0.000000 | AdvAfterStd 7.114961e-02
[Update 1257] Samples 2048 | Reward mean/std -0.059819/0.140625 | Value mean/std -0.055640/0.096241 | Adv std 1.007343e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.059819 | AvgAdv -0.000000 | AdvAfterStd 8.174778e-02
[Update 1258] Samples 2048 | Reward mean/std -0.056796/0.211755 | Value mean/std -0.063692/0.282995 | Adv std 1.294612e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.056796 | AvgAdv 0.000000 | AdvAfterStd 7.373948e-02
[Update 1259] Samples 2048 | Reward mean/std -0.068418/0.252074 | Value mean/std -0.057444/0.172610 | Adv std 1.347356e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.068418 | AvgAdv -0.000000 | AdvAfterStd 7.424261e-02
[Update 1260] Samples 2048 | Reward mean/std -0.065995/0.309955 | Value mean/std -0.065416/0.246079 | Adv std 1.322715e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.065995 | AvgAdv -0.000000 | AdvAfterStd 7.385048e-02
[Update 1261] Samples 2048 | Reward mean/std -0.062516/0.223725 | Value mean/std -0.066549/0.179074 | Adv std 1.204932e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.062516 | AvgAdv 0.000000 | AdvAfterStd 8.048148e-02
[Update 1262] Samples 2048 | Reward mean/std -0.060475/0.192062 | Value mean/std -0.058776/0.193609 | Adv std 9.194794e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.060475 | AvgAdv -0.000000 | AdvAfterStd 6.695978e-02
[Update 1263] Samples 2048 | Reward mean/std -0.065188/0.209626 | Value mean/std -0.062970/0.195594 | Adv std 9.762552e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.065188 | AvgAdv -0.000000 | AdvAfterStd 7.091630e-02
[Update 1264] Samples 2048 | Reward mean/std -0.057775/0.159614 | Value mean/std -0.046934/0.129555 | Adv std 9.638892e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.057775 | AvgAdv -0.000000 | AdvAfterStd 7.695892e-02
[Update 1265] Samples 2048 | Reward mean/std -0.070156/0.241992 | Value mean/std -0.074399/0.216273 | Adv std 1.254017e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.070156 | AvgAdv 0.000000 | AdvAfterStd 7.867894e-02
[Update 1266] Samples 2048 | Reward mean/std -0.062928/0.216446 | Value mean/std -0.059503/0.156494 | Adv std 1.359504e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.062928 | AvgAdv 0.000000 | AdvAfterStd 7.271355e-02
[Update 1267] Samples 2048 | Reward mean/std -0.052018/0.134703 | Value mean/std -0.054938/0.178707 | Adv std 1.125521e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.052018 | AvgAdv -0.000000 | AdvAfterStd 5.951094e-02
[Update 1268] Samples 2048 | Reward mean/std -0.066747/0.284112 | Value mean/std -0.059395/0.232580 | Adv std 1.099074e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.066747 | AvgAdv -0.000000 | AdvAfterStd 8.097494e-02
[Update 1269] Samples 2048 | Reward mean/std -0.052561/0.114563 | Value mean/std -0.053201/0.092773 | Adv std 8.066635e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.052561 | AvgAdv -0.000000 | AdvAfterStd 6.336979e-02
[Update 1270] Samples 2048 | Reward mean/std -0.065488/0.176342 | Value mean/std -0.054951/0.122016 | Adv std 1.227792e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.065488 | AvgAdv -0.000000 | AdvAfterStd 8.580302e-02
[Update 1271] Samples 2048 | Reward mean/std -0.064321/0.186252 | Value mean/std -0.060697/0.133050 | Adv std 1.118907e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.064321 | AvgAdv -0.000000 | AdvAfterStd 9.131000e-02
[Update 1272] Samples 2048 | Reward mean/std -0.059150/0.146659 | Value mean/std -0.068115/0.124154 | Adv std 9.794957e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.059150 | AvgAdv -0.000000 | AdvAfterStd 6.773865e-02
[Update 1273] Samples 2048 | Reward mean/std -0.057085/0.137851 | Value mean/std -0.057633/0.102945 | Adv std 9.657971e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.057085 | AvgAdv 0.000000 | AdvAfterStd 7.213005e-02
[Update 1274] Samples 2048 | Reward mean/std -0.058138/0.149964 | Value mean/std -0.061490/0.120209 | Adv std 1.002916e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.058138 | AvgAdv -0.000000 | AdvAfterStd 7.467900e-02
[Update 1275] Samples 2048 | Reward mean/std -0.064846/0.186142 | Value mean/std -0.057980/0.124627 | Adv std 1.104280e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.064846 | AvgAdv 0.000000 | AdvAfterStd 7.738896e-02
[Update 1276] Samples 2048 | Reward mean/std -0.061403/0.186600 | Value mean/std -0.063237/0.153314 | Adv std 1.366568e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.061403 | AvgAdv 0.000000 | AdvAfterStd 9.219927e-02
[Update 1277] Samples 2048 | Reward mean/std -0.065100/0.190009 | Value mean/std -0.056547/0.140569 | Adv std 1.204818e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.065100 | AvgAdv 0.000000 | AdvAfterStd 8.081115e-02
[Update 1278] Samples 2048 | Reward mean/std -0.065041/0.242094 | Value mean/std -0.071431/0.155025 | Adv std 1.555469e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.065041 | AvgAdv 0.000000 | AdvAfterStd 7.691402e-02
[Update 1279] Samples 2048 | Reward mean/std -0.058334/0.164498 | Value mean/std -0.057696/0.126659 | Adv std 1.016001e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.058334 | AvgAdv 0.000000 | AdvAfterStd 8.020185e-02
[Update 1280] Samples 2048 | Reward mean/std -0.059231/0.186032 | Value mean/std -0.059266/0.234678 | Adv std 1.195721e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.059231 | AvgAdv -0.000000 | AdvAfterStd 7.188677e-02
[Update 1281] Samples 2048 | Reward mean/std -0.060600/0.146512 | Value mean/std -0.054142/0.096941 | Adv std 1.124916e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.060600 | AvgAdv -0.000000 | AdvAfterStd 8.627778e-02
[Update 1282] Samples 2048 | Reward mean/std -0.067932/0.241870 | Value mean/std -0.062919/0.163762 | Adv std 1.351582e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.067932 | AvgAdv 0.000000 | AdvAfterStd 9.840745e-02
[Update 1283] Samples 2048 | Reward mean/std -0.061774/0.165878 | Value mean/std -0.063039/0.147712 | Adv std 1.250274e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.061774 | AvgAdv 0.000000 | AdvAfterStd 8.541665e-02
[Update 1284] Samples 2048 | Reward mean/std -0.056881/0.142082 | Value mean/std -0.059561/0.143289 | Adv std 7.862610e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.056881 | AvgAdv -0.000000 | AdvAfterStd 6.381278e-02
[Update 1285] Samples 2048 | Reward mean/std -0.055745/0.140970 | Value mean/std -0.050516/0.086085 | Adv std 8.853682e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.055745 | AvgAdv -0.000000 | AdvAfterStd 6.491417e-02
[Update 1286] Samples 2048 | Reward mean/std -0.058894/0.170918 | Value mean/std -0.058818/0.143550 | Adv std 8.401444e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.058894 | AvgAdv 0.000000 | AdvAfterStd 7.031906e-02
[Update 1287] Samples 2048 | Reward mean/std -0.062596/0.175584 | Value mean/std -0.063068/0.171798 | Adv std 1.015591e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.062596 | AvgAdv 0.000000 | AdvAfterStd 7.234875e-02
[Update 1288] Samples 2048 | Reward mean/std -0.058682/0.153064 | Value mean/std -0.062197/0.120113 | Adv std 8.656085e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.058682 | AvgAdv -0.000000 | AdvAfterStd 6.315364e-02
[Update 1289] Samples 2048 | Reward mean/std -0.061696/0.169581 | Value mean/std -0.057621/0.129021 | Adv std 1.089310e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.061696 | AvgAdv 0.000000 | AdvAfterStd 8.151610e-02
[Update 1290] Samples 2048 | Reward mean/std -0.056702/0.179113 | Value mean/std -0.060838/0.124465 | Adv std 1.036734e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.056702 | AvgAdv -0.000000 | AdvAfterStd 7.056455e-02
[Update 1291] Samples 2048 | Reward mean/std -0.056675/0.141239 | Value mean/std -0.055363/0.113548 | Adv std 8.772524e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.056675 | AvgAdv -0.000000 | AdvAfterStd 7.172807e-02
[Update 1292] Samples 2048 | Reward mean/std -0.058627/0.166235 | Value mean/std -0.055413/0.110827 | Adv std 9.312949e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.058627 | AvgAdv 0.000000 | AdvAfterStd 7.077518e-02
[Update 1293] Samples 2048 | Reward mean/std -0.061114/0.186378 | Value mean/std -0.059163/0.133648 | Adv std 1.226893e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.061114 | AvgAdv -0.000000 | AdvAfterStd 7.858026e-02
[Update 1294] Samples 2048 | Reward mean/std -0.063806/0.214475 | Value mean/std -0.062802/0.177200 | Adv std 1.050768e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.063806 | AvgAdv -0.000000 | AdvAfterStd 7.323647e-02
[Update 1295] Samples 2048 | Reward mean/std -0.062354/0.253590 | Value mean/std -0.067158/0.246252 | Adv std 1.022094e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.062354 | AvgAdv -0.000000 | AdvAfterStd 7.138779e-02
[Update 1296] Samples 2048 | Reward mean/std -0.057965/0.156956 | Value mean/std -0.058258/0.130047 | Adv std 9.346440e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.057965 | AvgAdv 0.000000 | AdvAfterStd 6.790265e-02
[Update 1297] Samples 2048 | Reward mean/std -0.061878/0.237575 | Value mean/std -0.055377/0.217497 | Adv std 1.094553e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.061878 | AvgAdv 0.000000 | AdvAfterStd 7.377493e-02
[Update 1298] Samples 2048 | Reward mean/std -0.065209/0.257430 | Value mean/std -0.062678/0.228697 | Adv std 1.095749e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.065209 | AvgAdv -0.000000 | AdvAfterStd 7.065775e-02
[Update 1299] Samples 2048 | Reward mean/std -0.067827/0.206600 | Value mean/std -0.057848/0.153884 | Adv std 1.343905e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.067827 | AvgAdv 0.000000 | AdvAfterStd 8.158186e-02
[Update 1300] Samples 2048 | Reward mean/std -0.057789/0.174420 | Value mean/std -0.069445/0.180045 | Adv std 1.112444e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.057789 | AvgAdv 0.000000 | AdvAfterStd 6.994327e-02
Saved checkpoint at update 1300
[Update 1301] Samples 2048 | Reward mean/std -0.063553/0.260791 | Value mean/std -0.057443/0.191206 | Adv std 1.156520e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.063553 | AvgAdv 0.000000 | AdvAfterStd 7.312577e-02
[Update 1302] Samples 2048 | Reward mean/std -0.058815/0.173184 | Value mean/std -0.062244/0.153375 | Adv std 8.769938e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.058815 | AvgAdv 0.000000 | AdvAfterStd 8.466593e-02
[Update 1303] Samples 2048 | Reward mean/std -0.065074/0.214193 | Value mean/std -0.061059/0.151584 | Adv std 1.313197e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.065074 | AvgAdv 0.000000 | AdvAfterStd 8.271656e-02
[Update 1304] Samples 2048 | Reward mean/std -0.052840/0.138946 | Value mean/std -0.052592/0.115640 | Adv std 9.077387e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.052840 | AvgAdv 0.000000 | AdvAfterStd 6.661499e-02
[Update 1305] Samples 2048 | Reward mean/std -0.059960/0.244676 | Value mean/std -0.059382/0.223236 | Adv std 8.826955e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.059960 | AvgAdv -0.000000 | AdvAfterStd 6.748124e-02
[Update 1306] Samples 2048 | Reward mean/std -0.061240/0.225252 | Value mean/std -0.059772/0.189652 | Adv std 8.824725e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.061240 | AvgAdv 0.000000 | AdvAfterStd 7.124876e-02
[Update 1307] Samples 2048 | Reward mean/std -0.058938/0.213080 | Value mean/std -0.061674/0.208884 | Adv std 8.883795e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.058938 | AvgAdv 0.000000 | AdvAfterStd 6.861573e-02
[Update 1308] Samples 2048 | Reward mean/std -0.051758/0.149129 | Value mean/std -0.048531/0.069006 | Adv std 1.384230e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.051758 | AvgAdv -0.000000 | AdvAfterStd 9.455204e-02
[Update 1309] Samples 2048 | Reward mean/std -0.059097/0.199706 | Value mean/std -0.054463/0.150914 | Adv std 9.946191e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.059097 | AvgAdv 0.000000 | AdvAfterStd 6.891740e-02
[Update 1310] Samples 2048 | Reward mean/std -0.065756/0.257732 | Value mean/std -0.066683/0.252039 | Adv std 1.408323e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.065756 | AvgAdv 0.000000 | AdvAfterStd 7.758445e-02
[Update 1311] Samples 2048 | Reward mean/std -0.056471/0.148895 | Value mean/std -0.063607/0.160856 | Adv std 1.152831e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.056471 | AvgAdv -0.000000 | AdvAfterStd 7.041989e-02
[Update 1312] Samples 2048 | Reward mean/std -0.067126/0.226341 | Value mean/std -0.065009/0.184787 | Adv std 1.119023e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.067126 | AvgAdv -0.000000 | AdvAfterStd 8.726875e-02
[Update 1313] Samples 2048 | Reward mean/std -0.059884/0.161034 | Value mean/std -0.063723/0.134897 | Adv std 9.867448e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.059884 | AvgAdv -0.000000 | AdvAfterStd 6.912686e-02
[Update 1314] Samples 2048 | Reward mean/std -0.061052/0.179890 | Value mean/std -0.054511/0.138005 | Adv std 1.051608e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.061052 | AvgAdv 0.000000 | AdvAfterStd 7.989576e-02
[Update 1315] Samples 2048 | Reward mean/std -0.059399/0.175181 | Value mean/std -0.063784/0.123581 | Adv std 9.785967e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.059399 | AvgAdv -0.000000 | AdvAfterStd 6.535496e-02
[Update 1316] Samples 2048 | Reward mean/std -0.053991/0.130353 | Value mean/std -0.053229/0.080881 | Adv std 1.111042e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.053991 | AvgAdv 0.000000 | AdvAfterStd 7.798905e-02
[Update 1317] Samples 2048 | Reward mean/std -0.055427/0.138402 | Value mean/std -0.060491/0.196658 | Adv std 1.353328e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.055427 | AvgAdv -0.000000 | AdvAfterStd 6.284006e-02
[Update 1318] Samples 2048 | Reward mean/std -0.060799/0.202041 | Value mean/std -0.059078/0.152435 | Adv std 1.046632e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.060799 | AvgAdv -0.000000 | AdvAfterStd 6.960696e-02
[Update 1319] Samples 2048 | Reward mean/std -0.061645/0.144101 | Value mean/std -0.058954/0.137703 | Adv std 1.172746e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.061645 | AvgAdv -0.000000 | AdvAfterStd 8.425040e-02
[Update 1320] Samples 2048 | Reward mean/std -0.067888/0.267022 | Value mean/std -0.071743/0.201371 | Adv std 1.263135e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.067888 | AvgAdv 0.000000 | AdvAfterStd 7.965092e-02
[Update 1321] Samples 2048 | Reward mean/std -0.063246/0.175968 | Value mean/std -0.057485/0.125790 | Adv std 1.179658e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.063246 | AvgAdv 0.000000 | AdvAfterStd 9.902347e-02
[Update 1322] Samples 2048 | Reward mean/std -0.055848/0.126556 | Value mean/std -0.067125/0.104827 | Adv std 9.468392e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.055848 | AvgAdv 0.000000 | AdvAfterStd 7.240888e-02
[Update 1323] Samples 2048 | Reward mean/std -0.070325/0.259782 | Value mean/std -0.070698/0.188229 | Adv std 1.454717e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.070325 | AvgAdv -0.000000 | AdvAfterStd 1.025158e-01
[Update 1324] Samples 2048 | Reward mean/std -0.057608/0.139315 | Value mean/std -0.052219/0.117767 | Adv std 1.014626e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.057608 | AvgAdv 0.000000 | AdvAfterStd 8.144463e-02
[Update 1325] Samples 2048 | Reward mean/std -0.066819/0.313257 | Value mean/std -0.064120/0.201749 | Adv std 1.857149e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.066819 | AvgAdv -0.000000 | AdvAfterStd 1.461536e-01
[Update 1326] Samples 2048 | Reward mean/std -0.064066/0.277870 | Value mean/std -0.065130/0.293254 | Adv std 1.256810e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.064066 | AvgAdv -0.000000 | AdvAfterStd 9.348676e-02
[Update 1327] Samples 2048 | Reward mean/std -0.058347/0.153721 | Value mean/std -0.058934/0.113599 | Adv std 1.211277e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.058347 | AvgAdv -0.000000 | AdvAfterStd 7.709946e-02
[Update 1328] Samples 2048 | Reward mean/std -0.052688/0.157391 | Value mean/std -0.056396/0.152962 | Adv std 9.550659e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.052688 | AvgAdv 0.000000 | AdvAfterStd 6.820940e-02
[Update 1329] Samples 2048 | Reward mean/std -0.056727/0.125881 | Value mean/std -0.051556/0.088632 | Adv std 8.808222e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.056727 | AvgAdv 0.000000 | AdvAfterStd 7.587579e-02
[Update 1330] Samples 2048 | Reward mean/std -0.059146/0.146695 | Value mean/std -0.057117/0.104793 | Adv std 9.997405e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.059146 | AvgAdv -0.000000 | AdvAfterStd 8.004928e-02
[Update 1331] Samples 2048 | Reward mean/std -0.064183/0.277743 | Value mean/std -0.065499/0.149553 | Adv std 1.623520e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.064183 | AvgAdv 0.000000 | AdvAfterStd 8.882704e-02
[Update 1332] Samples 2048 | Reward mean/std -0.069913/0.271979 | Value mean/std -0.069327/0.236979 | Adv std 1.096209e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.069913 | AvgAdv -0.000000 | AdvAfterStd 8.968297e-02
[Update 1333] Samples 2048 | Reward mean/std -0.061672/0.195282 | Value mean/std -0.065028/0.169916 | Adv std 1.074940e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.061672 | AvgAdv -0.000000 | AdvAfterStd 8.358306e-02
[Update 1334] Samples 2048 | Reward mean/std -0.065516/0.200390 | Value mean/std -0.061241/0.152070 | Adv std 1.199856e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.065516 | AvgAdv 0.000000 | AdvAfterStd 8.161785e-02
[Update 1335] Samples 2048 | Reward mean/std -0.058104/0.145736 | Value mean/std -0.056992/0.120618 | Adv std 9.626921e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.058104 | AvgAdv 0.000000 | AdvAfterStd 7.340469e-02
[Update 1336] Samples 2048 | Reward mean/std -0.066150/0.259575 | Value mean/std -0.070264/0.261784 | Adv std 1.010212e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.066150 | AvgAdv 0.000000 | AdvAfterStd 8.273958e-02
[Update 1337] Samples 2048 | Reward mean/std -0.064342/0.274245 | Value mean/std -0.065275/0.249828 | Adv std 1.285165e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.064342 | AvgAdv -0.000000 | AdvAfterStd 9.775171e-02
[Update 1338] Samples 2048 | Reward mean/std -0.056177/0.134334 | Value mean/std -0.059310/0.120393 | Adv std 9.683869e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.056177 | AvgAdv 0.000000 | AdvAfterStd 7.613017e-02
[Update 1339] Samples 2048 | Reward mean/std -0.061607/0.154767 | Value mean/std -0.055475/0.114304 | Adv std 9.985600e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.061607 | AvgAdv -0.000000 | AdvAfterStd 7.828774e-02
[Update 1340] Samples 2048 | Reward mean/std -0.062359/0.222415 | Value mean/std -0.059953/0.122777 | Adv std 1.518571e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.062359 | AvgAdv 0.000000 | AdvAfterStd 9.764802e-02
[Update 1341] Samples 2048 | Reward mean/std -0.059158/0.186518 | Value mean/std -0.057891/0.129284 | Adv std 1.096193e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.059158 | AvgAdv -0.000000 | AdvAfterStd 7.804627e-02
[Update 1342] Samples 2048 | Reward mean/std -0.061652/0.170690 | Value mean/std -0.060634/0.132175 | Adv std 1.124143e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.061652 | AvgAdv -0.000000 | AdvAfterStd 7.289821e-02
[Update 1343] Samples 2048 | Reward mean/std -0.057477/0.172768 | Value mean/std -0.061479/0.136247 | Adv std 1.080706e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.057477 | AvgAdv -0.000000 | AdvAfterStd 8.471599e-02
[Update 1344] Samples 2048 | Reward mean/std -0.061491/0.233068 | Value mean/std -0.056896/0.123154 | Adv std 1.528572e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.061491 | AvgAdv 0.000000 | AdvAfterStd 9.197087e-02
[Update 1345] Samples 2048 | Reward mean/std -0.055047/0.173312 | Value mean/std -0.057120/0.173153 | Adv std 8.933917e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.055047 | AvgAdv 0.000000 | AdvAfterStd 6.995133e-02
[Update 1346] Samples 2048 | Reward mean/std -0.060633/0.205741 | Value mean/std -0.058958/0.129029 | Adv std 1.093409e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.060633 | AvgAdv -0.000000 | AdvAfterStd 9.666144e-02
[Update 1347] Samples 2048 | Reward mean/std -0.058671/0.190676 | Value mean/std -0.063282/0.202291 | Adv std 1.114937e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.058671 | AvgAdv 0.000000 | AdvAfterStd 7.484103e-02
[Update 1348] Samples 2048 | Reward mean/std -0.061326/0.158079 | Value mean/std -0.061015/0.226420 | Adv std 1.626647e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.061326 | AvgAdv -0.000000 | AdvAfterStd 8.133803e-02
[Update 1349] Samples 2048 | Reward mean/std -0.063837/0.204462 | Value mean/std -0.064193/0.166184 | Adv std 1.087914e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.063837 | AvgAdv -0.000000 | AdvAfterStd 7.382249e-02
[Update 1350] Samples 2048 | Reward mean/std -0.063676/0.203063 | Value mean/std -0.066668/0.176066 | Adv std 1.010281e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.063676 | AvgAdv -0.000000 | AdvAfterStd 7.498369e-02
Saved checkpoint at update 1350
[Update 1351] Samples 2048 | Reward mean/std -0.057588/0.195145 | Value mean/std -0.053085/0.142744 | Adv std 9.929445e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.057588 | AvgAdv 0.000000 | AdvAfterStd 6.582937e-02
[Update 1352] Samples 2048 | Reward mean/std -0.068352/0.248239 | Value mean/std -0.071981/0.248293 | Adv std 1.074699e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.068352 | AvgAdv -0.000000 | AdvAfterStd 7.751931e-02
[Update 1353] Samples 2048 | Reward mean/std -0.060430/0.153619 | Value mean/std -0.059233/0.117651 | Adv std 1.032191e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.060430 | AvgAdv 0.000000 | AdvAfterStd 7.842191e-02
[Update 1354] Samples 2048 | Reward mean/std -0.076224/0.360448 | Value mean/std -0.071213/0.268806 | Adv std 1.454190e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.076224 | AvgAdv -0.000000 | AdvAfterStd 1.225744e-01
[Update 1355] Samples 2048 | Reward mean/std -0.056736/0.164222 | Value mean/std -0.057139/0.122093 | Adv std 9.775478e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.056736 | AvgAdv -0.000000 | AdvAfterStd 6.173983e-02
[Update 1356] Samples 2048 | Reward mean/std -0.069821/0.242662 | Value mean/std -0.069390/0.179940 | Adv std 1.213666e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.069821 | AvgAdv -0.000000 | AdvAfterStd 7.866395e-02
[Update 1357] Samples 2048 | Reward mean/std -0.051630/0.154501 | Value mean/std -0.059027/0.155436 | Adv std 7.567037e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.051630 | AvgAdv -0.000000 | AdvAfterStd 6.252420e-02
[Update 1358] Samples 2048 | Reward mean/std -0.057325/0.192871 | Value mean/std -0.054725/0.156409 | Adv std 9.306872e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.057325 | AvgAdv -0.000000 | AdvAfterStd 6.970850e-02
[Update 1359] Samples 2048 | Reward mean/std -0.058872/0.152752 | Value mean/std -0.060734/0.164356 | Adv std 1.159115e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.058872 | AvgAdv 0.000000 | AdvAfterStd 8.178164e-02
[Update 1360] Samples 2048 | Reward mean/std -0.051308/0.115370 | Value mean/std -0.053536/0.147123 | Adv std 1.209603e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.051308 | AvgAdv -0.000000 | AdvAfterStd 6.864452e-02
[Update 1361] Samples 2048 | Reward mean/std -0.064456/0.212276 | Value mean/std -0.051751/0.119599 | Adv std 1.362171e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.064456 | AvgAdv 0.000000 | AdvAfterStd 8.277341e-02
[Update 1362] Samples 2048 | Reward mean/std -0.055093/0.155547 | Value mean/std -0.057980/0.112203 | Adv std 9.928981e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.055093 | AvgAdv -0.000000 | AdvAfterStd 7.431622e-02
[Update 1363] Samples 2048 | Reward mean/std -0.065477/0.227701 | Value mean/std -0.064793/0.193977 | Adv std 1.092310e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.065477 | AvgAdv 0.000000 | AdvAfterStd 7.321801e-02
[Update 1364] Samples 2048 | Reward mean/std -0.061966/0.191956 | Value mean/std -0.062044/0.167551 | Adv std 1.014429e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.061966 | AvgAdv 0.000000 | AdvAfterStd 7.577495e-02
[Update 1365] Samples 2048 | Reward mean/std -0.057453/0.173122 | Value mean/std -0.057897/0.151959 | Adv std 8.854574e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.057453 | AvgAdv -0.000000 | AdvAfterStd 6.899039e-02
[Update 1366] Samples 2048 | Reward mean/std -0.059918/0.193007 | Value mean/std -0.058758/0.149061 | Adv std 1.087985e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.059918 | AvgAdv 0.000000 | AdvAfterStd 7.273685e-02
[Update 1367] Samples 2048 | Reward mean/std -0.060933/0.183983 | Value mean/std -0.056553/0.154083 | Adv std 9.155500e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.060933 | AvgAdv -0.000000 | AdvAfterStd 6.360187e-02
[Update 1368] Samples 2048 | Reward mean/std -0.060202/0.215318 | Value mean/std -0.062128/0.156036 | Adv std 1.197710e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.060202 | AvgAdv -0.000000 | AdvAfterStd 7.571964e-02
[Update 1369] Samples 2048 | Reward mean/std -0.063536/0.211079 | Value mean/std -0.059577/0.171204 | Adv std 1.048647e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.063536 | AvgAdv -0.000000 | AdvAfterStd 6.789594e-02
[Update 1370] Samples 2048 | Reward mean/std -0.063826/0.226976 | Value mean/std -0.064390/0.178269 | Adv std 1.085600e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.063826 | AvgAdv -0.000000 | AdvAfterStd 9.327387e-02
[Update 1371] Samples 2048 | Reward mean/std -0.054076/0.128457 | Value mean/std -0.056155/0.102132 | Adv std 7.899679e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.054076 | AvgAdv -0.000000 | AdvAfterStd 6.993918e-02
[Update 1372] Samples 2048 | Reward mean/std -0.060161/0.180306 | Value mean/std -0.063638/0.133949 | Adv std 9.857193e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.060161 | AvgAdv 0.000000 | AdvAfterStd 7.542904e-02
[Update 1373] Samples 2048 | Reward mean/std -0.055603/0.135461 | Value mean/std -0.062232/0.166863 | Adv std 1.344199e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.055603 | AvgAdv -0.000000 | AdvAfterStd 7.315592e-02
[Update 1374] Samples 2048 | Reward mean/std -0.066237/0.228313 | Value mean/std -0.058956/0.186155 | Adv std 1.126643e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.066237 | AvgAdv -0.000000 | AdvAfterStd 8.349749e-02
[Update 1375] Samples 2048 | Reward mean/std -0.058749/0.233589 | Value mean/std -0.058502/0.210013 | Adv std 9.368515e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.058749 | AvgAdv -0.000000 | AdvAfterStd 8.302803e-02
[Update 1376] Samples 2048 | Reward mean/std -0.067530/0.224811 | Value mean/std -0.057265/0.196296 | Adv std 1.160961e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.067530 | AvgAdv 0.000000 | AdvAfterStd 8.563270e-02
[Update 1377] Samples 2048 | Reward mean/std -0.058467/0.174407 | Value mean/std -0.058107/0.110004 | Adv std 1.171402e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.058467 | AvgAdv -0.000000 | AdvAfterStd 8.119444e-02
[Update 1378] Samples 2048 | Reward mean/std -0.060571/0.144634 | Value mean/std -0.055038/0.106597 | Adv std 1.062419e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.060571 | AvgAdv -0.000000 | AdvAfterStd 8.769536e-02
[Update 1379] Samples 2048 | Reward mean/std -0.053550/0.135522 | Value mean/std -0.054101/0.109583 | Adv std 8.757105e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.053550 | AvgAdv 0.000000 | AdvAfterStd 5.742026e-02
[Update 1380] Samples 2048 | Reward mean/std -0.062903/0.168841 | Value mean/std -0.058022/0.146223 | Adv std 1.041902e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.062903 | AvgAdv -0.000000 | AdvAfterStd 7.424191e-02
[Update 1381] Samples 2048 | Reward mean/std -0.066658/0.258969 | Value mean/std -0.065987/0.221249 | Adv std 9.314213e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.066658 | AvgAdv -0.000000 | AdvAfterStd 6.658636e-02
[Update 1382] Samples 2048 | Reward mean/std -0.060087/0.181385 | Value mean/std -0.063199/0.159927 | Adv std 9.347804e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.060087 | AvgAdv 0.000000 | AdvAfterStd 7.422253e-02
[Update 1383] Samples 2048 | Reward mean/std -0.053844/0.163699 | Value mean/std -0.053559/0.117670 | Adv std 8.884636e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.053844 | AvgAdv 0.000000 | AdvAfterStd 5.829894e-02
[Update 1384] Samples 2048 | Reward mean/std -0.060928/0.242508 | Value mean/std -0.057286/0.231002 | Adv std 8.417140e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.060928 | AvgAdv -0.000000 | AdvAfterStd 7.496480e-02
[Update 1385] Samples 2048 | Reward mean/std -0.064148/0.174303 | Value mean/std -0.055397/0.143236 | Adv std 1.141104e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.064148 | AvgAdv -0.000000 | AdvAfterStd 8.688156e-02
[Update 1386] Samples 2048 | Reward mean/std -0.069250/0.245683 | Value mean/std -0.066591/0.229783 | Adv std 1.087917e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.069250 | AvgAdv 0.000000 | AdvAfterStd 8.442784e-02
[Update 1387] Samples 2048 | Reward mean/std -0.060730/0.147300 | Value mean/std -0.063772/0.154003 | Adv std 1.014838e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.060730 | AvgAdv 0.000000 | AdvAfterStd 7.235872e-02
[Update 1388] Samples 2048 | Reward mean/std -0.068817/0.250293 | Value mean/std -0.059099/0.172338 | Adv std 1.203004e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.068817 | AvgAdv -0.000000 | AdvAfterStd 7.429721e-02
[Update 1389] Samples 2048 | Reward mean/std -0.055396/0.124455 | Value mean/std -0.052832/0.117956 | Adv std 9.778824e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.055396 | AvgAdv 0.000000 | AdvAfterStd 6.792635e-02
[Update 1390] Samples 2048 | Reward mean/std -0.068145/0.201574 | Value mean/std -0.067009/0.162280 | Adv std 1.064301e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.068145 | AvgAdv 0.000000 | AdvAfterStd 8.340213e-02
[Update 1391] Samples 2048 | Reward mean/std -0.066702/0.206083 | Value mean/std -0.064039/0.169439 | Adv std 9.752186e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.066702 | AvgAdv 0.000000 | AdvAfterStd 7.560974e-02
[Update 1392] Samples 2048 | Reward mean/std -0.058939/0.133008 | Value mean/std -0.059479/0.118783 | Adv std 8.444173e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.058939 | AvgAdv 0.000000 | AdvAfterStd 6.767862e-02
[Update 1393] Samples 2048 | Reward mean/std -0.067239/0.198998 | Value mean/std -0.063590/0.184404 | Adv std 1.184092e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.067239 | AvgAdv 0.000000 | AdvAfterStd 7.964315e-02
[Update 1394] Samples 2048 | Reward mean/std -0.071801/0.310595 | Value mean/std -0.070298/0.222844 | Adv std 1.333477e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.071801 | AvgAdv -0.000000 | AdvAfterStd 1.028174e-01
[Update 1395] Samples 2048 | Reward mean/std -0.073634/0.312718 | Value mean/std -0.068683/0.282410 | Adv std 1.291467e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.073634 | AvgAdv -0.000000 | AdvAfterStd 9.593014e-02
[Update 1396] Samples 2048 | Reward mean/std -0.064088/0.165923 | Value mean/std -0.066196/0.155995 | Adv std 1.238400e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.064088 | AvgAdv -0.000000 | AdvAfterStd 8.677328e-02
[Update 1397] Samples 2048 | Reward mean/std -0.069754/0.212842 | Value mean/std -0.067139/0.177359 | Adv std 1.198606e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.069754 | AvgAdv 0.000000 | AdvAfterStd 9.000703e-02
[Update 1398] Samples 2048 | Reward mean/std -0.059720/0.174058 | Value mean/std -0.060630/0.148879 | Adv std 9.828119e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.059720 | AvgAdv 0.000000 | AdvAfterStd 7.653467e-02
[Update 1399] Samples 2048 | Reward mean/std -0.057310/0.141106 | Value mean/std -0.056222/0.139775 | Adv std 9.334004e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.057310 | AvgAdv 0.000000 | AdvAfterStd 7.090251e-02
[Update 1400] Samples 2048 | Reward mean/std -0.062624/0.182310 | Value mean/std -0.059370/0.137693 | Adv std 9.840358e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.062624 | AvgAdv 0.000000 | AdvAfterStd 7.762448e-02
Saved checkpoint at update 1400
[Update 1401] Samples 2048 | Reward mean/std -0.067144/0.284145 | Value mean/std -0.066214/0.283848 | Adv std 8.924796e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.067144 | AvgAdv -0.000000 | AdvAfterStd 6.626549e-02
[Update 1402] Samples 2048 | Reward mean/std -0.068614/0.245747 | Value mean/std -0.056161/0.188180 | Adv std 1.120706e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.068614 | AvgAdv 0.000000 | AdvAfterStd 8.409321e-02
[Update 1403] Samples 2048 | Reward mean/std -0.059918/0.190966 | Value mean/std -0.063530/0.218476 | Adv std 1.353414e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.059918 | AvgAdv 0.000000 | AdvAfterStd 7.350114e-02
[Update 1404] Samples 2048 | Reward mean/std -0.059179/0.165193 | Value mean/std -0.063819/0.165035 | Adv std 1.038380e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.059179 | AvgAdv 0.000000 | AdvAfterStd 7.433616e-02
[Update 1405] Samples 2048 | Reward mean/std -0.060525/0.170361 | Value mean/std -0.058303/0.127541 | Adv std 1.028115e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.060525 | AvgAdv -0.000000 | AdvAfterStd 8.206657e-02
[Update 1406] Samples 2048 | Reward mean/std -0.063051/0.188546 | Value mean/std -0.063304/0.133711 | Adv std 1.053877e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.063051 | AvgAdv 0.000000 | AdvAfterStd 7.993042e-02
[Update 1407] Samples 2048 | Reward mean/std -0.063109/0.204732 | Value mean/std -0.061600/0.156339 | Adv std 1.126614e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.063109 | AvgAdv -0.000000 | AdvAfterStd 6.763730e-02
[Update 1408] Samples 2048 | Reward mean/std -0.070351/0.276656 | Value mean/std -0.068708/0.238826 | Adv std 1.087404e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.070351 | AvgAdv 0.000000 | AdvAfterStd 8.957073e-02
[Update 1409] Samples 2048 | Reward mean/std -0.065041/0.194633 | Value mean/std -0.059035/0.139145 | Adv std 1.209706e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.065041 | AvgAdv 0.000000 | AdvAfterStd 7.674042e-02
[Update 1410] Samples 2048 | Reward mean/std -0.059890/0.161260 | Value mean/std -0.062261/0.123395 | Adv std 9.814027e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.059890 | AvgAdv -0.000000 | AdvAfterStd 7.793196e-02
[Update 1411] Samples 2048 | Reward mean/std -0.060159/0.156900 | Value mean/std -0.058159/0.124124 | Adv std 1.065257e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.060159 | AvgAdv -0.000000 | AdvAfterStd 9.805564e-02
[Update 1412] Samples 2048 | Reward mean/std -0.057675/0.163214 | Value mean/std -0.056563/0.140075 | Adv std 8.149032e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.057675 | AvgAdv -0.000000 | AdvAfterStd 6.135089e-02
[Update 1413] Samples 2048 | Reward mean/std -0.056554/0.152896 | Value mean/std -0.054220/0.107480 | Adv std 8.729818e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.056554 | AvgAdv 0.000000 | AdvAfterStd 6.939343e-02
[Update 1414] Samples 2048 | Reward mean/std -0.063213/0.201717 | Value mean/std -0.062386/0.165819 | Adv std 9.131820e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.063213 | AvgAdv -0.000000 | AdvAfterStd 7.457679e-02
[Update 1415] Samples 2048 | Reward mean/std -0.064893/0.182337 | Value mean/std -0.056591/0.122333 | Adv std 1.210062e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.064893 | AvgAdv 0.000000 | AdvAfterStd 9.737273e-02
[Update 1416] Samples 2048 | Reward mean/std -0.057085/0.170288 | Value mean/std -0.068186/0.170005 | Adv std 9.731211e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.057085 | AvgAdv -0.000000 | AdvAfterStd 7.264371e-02
[Update 1417] Samples 2048 | Reward mean/std -0.056422/0.144261 | Value mean/std -0.054349/0.112427 | Adv std 1.073219e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.056422 | AvgAdv 0.000000 | AdvAfterStd 7.755651e-02
[Update 1418] Samples 2048 | Reward mean/std -0.056242/0.142977 | Value mean/std -0.052281/0.089684 | Adv std 1.138192e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.056242 | AvgAdv -0.000000 | AdvAfterStd 7.377367e-02
[Update 1419] Samples 2048 | Reward mean/std -0.066892/0.195147 | Value mean/std -0.057357/0.142399 | Adv std 1.080917e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.066892 | AvgAdv -0.000000 | AdvAfterStd 7.509421e-02
[Update 1420] Samples 2048 | Reward mean/std -0.060481/0.209671 | Value mean/std -0.066476/0.170539 | Adv std 1.128803e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.060481 | AvgAdv 0.000000 | AdvAfterStd 8.618218e-02
[Update 1421] Samples 2048 | Reward mean/std -0.065725/0.222918 | Value mean/std -0.057065/0.165394 | Adv std 1.265550e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.065725 | AvgAdv -0.000000 | AdvAfterStd 8.062410e-02
[Update 1422] Samples 2048 | Reward mean/std -0.060186/0.175973 | Value mean/std -0.057335/0.154327 | Adv std 8.502045e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.060186 | AvgAdv -0.000000 | AdvAfterStd 6.550541e-02
[Update 1423] Samples 2048 | Reward mean/std -0.066311/0.281215 | Value mean/std -0.069068/0.262460 | Adv std 1.015853e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.066311 | AvgAdv 0.000000 | AdvAfterStd 8.201445e-02
[Update 1424] Samples 2048 | Reward mean/std -0.058764/0.153257 | Value mean/std -0.061019/0.136558 | Adv std 9.261866e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.058764 | AvgAdv 0.000000 | AdvAfterStd 6.393927e-02
[Update 1425] Samples 2048 | Reward mean/std -0.063440/0.257085 | Value mean/std -0.064757/0.286788 | Adv std 9.883218e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.063440 | AvgAdv -0.000000 | AdvAfterStd 6.937905e-02
[Update 1426] Samples 2048 | Reward mean/std -0.061806/0.258997 | Value mean/std -0.056557/0.218183 | Adv std 9.464908e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.061806 | AvgAdv 0.000000 | AdvAfterStd 7.060526e-02
[Update 1427] Samples 2048 | Reward mean/std -0.065457/0.175867 | Value mean/std -0.054737/0.131829 | Adv std 1.076335e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.065457 | AvgAdv 0.000000 | AdvAfterStd 8.242620e-02
[Update 1428] Samples 2048 | Reward mean/std -0.062214/0.183092 | Value mean/std -0.070267/0.157671 | Adv std 9.886893e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.062214 | AvgAdv 0.000000 | AdvAfterStd 7.128724e-02
[Update 1429] Samples 2048 | Reward mean/std -0.063963/0.181983 | Value mean/std -0.058362/0.159272 | Adv std 9.787294e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.063963 | AvgAdv -0.000000 | AdvAfterStd 8.091690e-02
[Update 1430] Samples 2048 | Reward mean/std -0.060106/0.245471 | Value mean/std -0.063553/0.217553 | Adv std 1.058008e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.060106 | AvgAdv 0.000000 | AdvAfterStd 6.326734e-02
[Update 1431] Samples 2048 | Reward mean/std -0.066285/0.194112 | Value mean/std -0.063402/0.168635 | Adv std 1.119904e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.066285 | AvgAdv -0.000000 | AdvAfterStd 8.192716e-02
[Update 1432] Samples 2048 | Reward mean/std -0.067534/0.246686 | Value mean/std -0.070647/0.197377 | Adv std 1.205264e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.067534 | AvgAdv 0.000000 | AdvAfterStd 9.987151e-02
[Update 1433] Samples 2048 | Reward mean/std -0.056421/0.136547 | Value mean/std -0.060793/0.150448 | Adv std 1.213922e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.056421 | AvgAdv -0.000000 | AdvAfterStd 7.158615e-02
[Update 1434] Samples 2048 | Reward mean/std -0.060803/0.155232 | Value mean/std -0.056957/0.112324 | Adv std 9.316138e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.060803 | AvgAdv 0.000000 | AdvAfterStd 8.097653e-02
[Update 1435] Samples 2048 | Reward mean/std -0.055572/0.139104 | Value mean/std -0.059633/0.116461 | Adv std 9.741533e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.055572 | AvgAdv 0.000000 | AdvAfterStd 7.022094e-02
[Update 1436] Samples 2048 | Reward mean/std -0.055637/0.139096 | Value mean/std -0.063028/0.115108 | Adv std 7.724038e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.055637 | AvgAdv -0.000000 | AdvAfterStd 6.058463e-02
[Update 1437] Samples 2048 | Reward mean/std -0.065437/0.219952 | Value mean/std -0.055684/0.152087 | Adv std 1.111807e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.065437 | AvgAdv 0.000000 | AdvAfterStd 7.561719e-02
[Update 1438] Samples 2048 | Reward mean/std -0.061129/0.169466 | Value mean/std -0.060096/0.133224 | Adv std 1.009200e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.061129 | AvgAdv 0.000000 | AdvAfterStd 7.644105e-02
[Update 1439] Samples 2048 | Reward mean/std -0.076258/0.271806 | Value mean/std -0.073624/0.178702 | Adv std 1.591639e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.076258 | AvgAdv -0.000000 | AdvAfterStd 1.053597e-01
[Update 1440] Samples 2048 | Reward mean/std -0.057544/0.169814 | Value mean/std -0.063828/0.171754 | Adv std 1.080104e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.057544 | AvgAdv -0.000000 | AdvAfterStd 7.083515e-02
[Update 1441] Samples 2048 | Reward mean/std -0.056692/0.142703 | Value mean/std -0.054429/0.119131 | Adv std 8.818536e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.056692 | AvgAdv 0.000000 | AdvAfterStd 6.890868e-02
[Update 1442] Samples 2048 | Reward mean/std -0.072856/0.305360 | Value mean/std -0.066970/0.215843 | Adv std 1.476163e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.072856 | AvgAdv 0.000000 | AdvAfterStd 8.810790e-02
[Update 1443] Samples 2048 | Reward mean/std -0.066569/0.253285 | Value mean/std -0.061512/0.150331 | Adv std 1.916799e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.066569 | AvgAdv 0.000000 | AdvAfterStd 7.516581e-02
[Update 1444] Samples 2048 | Reward mean/std -0.065893/0.215956 | Value mean/std -0.068365/0.187690 | Adv std 1.437040e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.065893 | AvgAdv -0.000000 | AdvAfterStd 8.839649e-02
[Update 1445] Samples 2048 | Reward mean/std -0.067795/0.295847 | Value mean/std -0.057096/0.146395 | Adv std 2.027854e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.067795 | AvgAdv -0.000000 | AdvAfterStd 8.259650e-02
[Update 1446] Samples 2048 | Reward mean/std -0.065910/0.195393 | Value mean/std -0.072538/0.204113 | Adv std 1.572301e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.065910 | AvgAdv 0.000000 | AdvAfterStd 9.061156e-02
[Update 1447] Samples 2048 | Reward mean/std -0.064719/0.260191 | Value mean/std -0.062373/0.276171 | Adv std 1.155905e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.064719 | AvgAdv 0.000000 | AdvAfterStd 1.002947e-01
[Update 1448] Samples 2048 | Reward mean/std -0.069062/0.251045 | Value mean/std -0.068268/0.241247 | Adv std 1.234160e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.069062 | AvgAdv -0.000000 | AdvAfterStd 7.816432e-02
[Update 1449] Samples 2048 | Reward mean/std -0.065977/0.217726 | Value mean/std -0.071737/0.193690 | Adv std 1.009868e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.065977 | AvgAdv 0.000000 | AdvAfterStd 8.054187e-02
[Update 1450] Samples 2048 | Reward mean/std -0.062158/0.175214 | Value mean/std -0.064596/0.146467 | Adv std 1.126585e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.062158 | AvgAdv 0.000000 | AdvAfterStd 8.161649e-02
Saved checkpoint at update 1450
[Update 1451] Samples 2048 | Reward mean/std -0.061952/0.188947 | Value mean/std -0.053803/0.120475 | Adv std 1.366287e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.061952 | AvgAdv 0.000000 | AdvAfterStd 1.073144e-01
[Update 1452] Samples 2048 | Reward mean/std -0.064402/0.203868 | Value mean/std -0.066797/0.191503 | Adv std 1.378810e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.064402 | AvgAdv 0.000000 | AdvAfterStd 8.430586e-02
[Update 1453] Samples 2048 | Reward mean/std -0.063992/0.209684 | Value mean/std -0.064243/0.168232 | Adv std 1.017100e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.063992 | AvgAdv -0.000000 | AdvAfterStd 7.974251e-02
[Update 1454] Samples 2048 | Reward mean/std -0.065814/0.193992 | Value mean/std -0.056579/0.150311 | Adv std 1.069367e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.065814 | AvgAdv -0.000000 | AdvAfterStd 8.736738e-02
[Update 1455] Samples 2048 | Reward mean/std -0.065442/0.164668 | Value mean/std -0.067288/0.152668 | Adv std 1.068683e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.065442 | AvgAdv -0.000000 | AdvAfterStd 7.769844e-02
[Update 1456] Samples 2048 | Reward mean/std -0.067030/0.205670 | Value mean/std -0.063240/0.162987 | Adv std 1.186534e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.067030 | AvgAdv -0.000000 | AdvAfterStd 7.675681e-02
[Update 1457] Samples 2048 | Reward mean/std -0.060369/0.165250 | Value mean/std -0.062696/0.147156 | Adv std 9.504715e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.060369 | AvgAdv -0.000000 | AdvAfterStd 6.882299e-02
[Update 1458] Samples 2048 | Reward mean/std -0.070274/0.264115 | Value mean/std -0.064662/0.230879 | Adv std 9.710245e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.070274 | AvgAdv 0.000000 | AdvAfterStd 8.193022e-02
[Update 1459] Samples 2048 | Reward mean/std -0.053580/0.123333 | Value mean/std -0.060676/0.108118 | Adv std 8.781302e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.053580 | AvgAdv -0.000000 | AdvAfterStd 7.540575e-02
[Update 1460] Samples 2048 | Reward mean/std -0.067994/0.322510 | Value mean/std -0.066937/0.303906 | Adv std 1.131828e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.067994 | AvgAdv 0.000000 | AdvAfterStd 8.436229e-02
[Update 1461] Samples 2048 | Reward mean/std -0.052385/0.125245 | Value mean/std -0.054168/0.083094 | Adv std 9.198543e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.052385 | AvgAdv 0.000000 | AdvAfterStd 7.629699e-02
[Update 1462] Samples 2048 | Reward mean/std -0.052171/0.112068 | Value mean/std -0.057490/0.094119 | Adv std 7.616523e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.052171 | AvgAdv -0.000000 | AdvAfterStd 6.502533e-02
[Update 1463] Samples 2048 | Reward mean/std -0.064697/0.188660 | Value mean/std -0.058008/0.169280 | Adv std 1.005932e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.064697 | AvgAdv -0.000000 | AdvAfterStd 7.690581e-02
[Update 1464] Samples 2048 | Reward mean/std -0.064690/0.216694 | Value mean/std -0.067744/0.172428 | Adv std 1.329553e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.064690 | AvgAdv 0.000000 | AdvAfterStd 7.468587e-02
[Update 1465] Samples 2048 | Reward mean/std -0.059285/0.154145 | Value mean/std -0.059955/0.127486 | Adv std 9.565125e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.059285 | AvgAdv -0.000000 | AdvAfterStd 7.123890e-02
[Update 1466] Samples 2048 | Reward mean/std -0.060295/0.197722 | Value mean/std -0.063216/0.180239 | Adv std 9.382197e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.060295 | AvgAdv 0.000000 | AdvAfterStd 7.301504e-02
[Update 1467] Samples 2048 | Reward mean/std -0.059820/0.130778 | Value mean/std -0.059431/0.107538 | Adv std 1.060526e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.059820 | AvgAdv -0.000000 | AdvAfterStd 8.805069e-02
[Update 1468] Samples 2048 | Reward mean/std -0.054825/0.134567 | Value mean/std -0.060977/0.119147 | Adv std 8.233796e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.054825 | AvgAdv -0.000000 | AdvAfterStd 6.485032e-02
[Update 1469] Samples 2048 | Reward mean/std -0.065109/0.191149 | Value mean/std -0.059322/0.134571 | Adv std 1.192572e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.065109 | AvgAdv 0.000000 | AdvAfterStd 8.032151e-02
[Update 1470] Samples 2048 | Reward mean/std -0.062760/0.190452 | Value mean/std -0.059790/0.142250 | Adv std 1.058620e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.062760 | AvgAdv -0.000000 | AdvAfterStd 8.525471e-02
[Update 1471] Samples 2048 | Reward mean/std -0.066598/0.207052 | Value mean/std -0.064126/0.157114 | Adv std 1.013414e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.066598 | AvgAdv -0.000000 | AdvAfterStd 8.377681e-02
[Update 1472] Samples 2048 | Reward mean/std -0.067964/0.210992 | Value mean/std -0.064147/0.157049 | Adv std 1.268734e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.067964 | AvgAdv -0.000000 | AdvAfterStd 7.754300e-02
[Update 1473] Samples 2048 | Reward mean/std -0.068467/0.205193 | Value mean/std -0.070788/0.180137 | Adv std 1.197662e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.068467 | AvgAdv 0.000000 | AdvAfterStd 9.537762e-02
[Update 1474] Samples 2048 | Reward mean/std -0.058890/0.180332 | Value mean/std -0.062145/0.134926 | Adv std 1.141436e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.058890 | AvgAdv 0.000000 | AdvAfterStd 7.439822e-02
[Update 1475] Samples 2048 | Reward mean/std -0.066243/0.183089 | Value mean/std -0.059409/0.141949 | Adv std 1.110817e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.066243 | AvgAdv 0.000000 | AdvAfterStd 8.942474e-02
[Update 1476] Samples 2048 | Reward mean/std -0.060662/0.164326 | Value mean/std -0.062244/0.131455 | Adv std 1.035234e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.060662 | AvgAdv -0.000000 | AdvAfterStd 8.203698e-02
[Update 1477] Samples 2048 | Reward mean/std -0.063696/0.190164 | Value mean/std -0.059978/0.162367 | Adv std 1.061911e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.063696 | AvgAdv -0.000000 | AdvAfterStd 8.225793e-02
[Update 1478] Samples 2048 | Reward mean/std -0.058059/0.143498 | Value mean/std -0.055734/0.110795 | Adv std 9.132809e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.058059 | AvgAdv 0.000000 | AdvAfterStd 7.885486e-02
[Update 1479] Samples 2048 | Reward mean/std -0.055469/0.158796 | Value mean/std -0.058303/0.112302 | Adv std 1.078988e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.055469 | AvgAdv -0.000000 | AdvAfterStd 6.765077e-02
[Update 1480] Samples 2048 | Reward mean/std -0.068263/0.218315 | Value mean/std -0.067507/0.181680 | Adv std 1.088207e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.068263 | AvgAdv 0.000000 | AdvAfterStd 8.755390e-02
[Update 1481] Samples 2048 | Reward mean/std -0.063929/0.166257 | Value mean/std -0.067145/0.156342 | Adv std 1.030712e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.063929 | AvgAdv -0.000000 | AdvAfterStd 7.107265e-02
[Update 1482] Samples 2048 | Reward mean/std -0.065318/0.244185 | Value mean/std -0.067914/0.173914 | Adv std 1.506875e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.065318 | AvgAdv -0.000000 | AdvAfterStd 1.232103e-01
[Update 1483] Samples 2048 | Reward mean/std -0.064105/0.164730 | Value mean/std -0.060945/0.136644 | Adv std 1.198913e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.064105 | AvgAdv -0.000000 | AdvAfterStd 9.705700e-02
[Update 1484] Samples 2048 | Reward mean/std -0.074779/0.227950 | Value mean/std -0.073107/0.168540 | Adv std 1.335765e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.074779 | AvgAdv 0.000000 | AdvAfterStd 9.062892e-02
[Update 1485] Samples 2048 | Reward mean/std -0.060199/0.166245 | Value mean/std -0.060220/0.130036 | Adv std 9.068656e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.060199 | AvgAdv 0.000000 | AdvAfterStd 7.024948e-02
[Update 1486] Samples 2048 | Reward mean/std -0.062328/0.186693 | Value mean/std -0.067664/0.168381 | Adv std 9.098045e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.062328 | AvgAdv -0.000000 | AdvAfterStd 7.145569e-02
[Update 1487] Samples 2048 | Reward mean/std -0.064983/0.241898 | Value mean/std -0.066644/0.197654 | Adv std 1.071688e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.064983 | AvgAdv -0.000000 | AdvAfterStd 8.649380e-02
[Update 1488] Samples 2048 | Reward mean/std -0.060468/0.172603 | Value mean/std -0.060163/0.119002 | Adv std 1.049040e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.060468 | AvgAdv -0.000000 | AdvAfterStd 6.956358e-02
[Update 1489] Samples 2048 | Reward mean/std -0.069658/0.289259 | Value mean/std -0.065837/0.191400 | Adv std 1.372053e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.069658 | AvgAdv -0.000000 | AdvAfterStd 9.977042e-02
[Update 1490] Samples 2048 | Reward mean/std -0.065259/0.199963 | Value mean/std -0.058456/0.146610 | Adv std 1.090814e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.065259 | AvgAdv -0.000000 | AdvAfterStd 7.936064e-02
[Update 1491] Samples 2048 | Reward mean/std -0.060518/0.169387 | Value mean/std -0.070355/0.169443 | Adv std 9.698160e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.060518 | AvgAdv 0.000000 | AdvAfterStd 6.699281e-02
[Update 1492] Samples 2048 | Reward mean/std -0.054398/0.137551 | Value mean/std -0.053706/0.141849 | Adv std 1.225694e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.054398 | AvgAdv -0.000000 | AdvAfterStd 8.242783e-02
[Update 1493] Samples 2048 | Reward mean/std -0.065264/0.199401 | Value mean/std -0.064006/0.170623 | Adv std 9.848202e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.065264 | AvgAdv 0.000000 | AdvAfterStd 7.429342e-02
[Update 1494] Samples 2048 | Reward mean/std -0.062246/0.221613 | Value mean/std -0.055119/0.152245 | Adv std 1.245099e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.062246 | AvgAdv -0.000000 | AdvAfterStd 8.305508e-02
[Update 1495] Samples 2048 | Reward mean/std -0.062248/0.165067 | Value mean/std -0.055985/0.127914 | Adv std 1.045057e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.062248 | AvgAdv -0.000000 | AdvAfterStd 7.726255e-02
[Update 1496] Samples 2048 | Reward mean/std -0.065419/0.173498 | Value mean/std -0.063093/0.132945 | Adv std 1.017032e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.065419 | AvgAdv -0.000000 | AdvAfterStd 7.719606e-02
[Update 1497] Samples 2048 | Reward mean/std -0.059540/0.169636 | Value mean/std -0.057901/0.136702 | Adv std 9.930104e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.059540 | AvgAdv 0.000000 | AdvAfterStd 7.023851e-02
[Update 1498] Samples 2048 | Reward mean/std -0.057118/0.161360 | Value mean/std -0.064567/0.169847 | Adv std 8.650055e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.057118 | AvgAdv 0.000000 | AdvAfterStd 7.221708e-02
[Update 1499] Samples 2048 | Reward mean/std -0.066122/0.244963 | Value mean/std -0.065822/0.151170 | Adv std 1.586462e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.066122 | AvgAdv 0.000000 | AdvAfterStd 7.019162e-02
[Update 1500] Samples 2048 | Reward mean/std -0.058564/0.154572 | Value mean/std -0.062676/0.144047 | Adv std 8.871973e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.058564 | AvgAdv -0.000000 | AdvAfterStd 7.087565e-02
Saved checkpoint at update 1500
[Update 1501] Samples 2048 | Reward mean/std -0.067300/0.174038 | Value mean/std -0.061125/0.125030 | Adv std 1.036816e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.067300 | AvgAdv -0.000000 | AdvAfterStd 9.033847e-02
[Update 1502] Samples 2048 | Reward mean/std -0.059662/0.159946 | Value mean/std -0.067488/0.118331 | Adv std 1.101840e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.059662 | AvgAdv 0.000000 | AdvAfterStd 8.281379e-02
[Update 1503] Samples 2048 | Reward mean/std -0.061697/0.163017 | Value mean/std -0.064066/0.152039 | Adv std 8.986196e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.061697 | AvgAdv 0.000000 | AdvAfterStd 7.131611e-02
[Update 1504] Samples 2048 | Reward mean/std -0.059020/0.161079 | Value mean/std -0.061590/0.142227 | Adv std 9.006080e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.059020 | AvgAdv 0.000000 | AdvAfterStd 7.532012e-02
[Update 1505] Samples 2048 | Reward mean/std -0.061901/0.167590 | Value mean/std -0.058477/0.125574 | Adv std 1.018432e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.061901 | AvgAdv 0.000000 | AdvAfterStd 7.874564e-02
[Update 1506] Samples 2048 | Reward mean/std -0.055581/0.158735 | Value mean/std -0.056664/0.113886 | Adv std 9.051920e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.055581 | AvgAdv 0.000000 | AdvAfterStd 7.219611e-02
[Update 1507] Samples 2048 | Reward mean/std -0.060444/0.197151 | Value mean/std -0.059408/0.148391 | Adv std 1.244631e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.060444 | AvgAdv 0.000000 | AdvAfterStd 9.599349e-02
[Update 1508] Samples 2048 | Reward mean/std -0.061635/0.183629 | Value mean/std -0.059743/0.162275 | Adv std 1.177056e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.061635 | AvgAdv 0.000000 | AdvAfterStd 7.107048e-02
[Update 1509] Samples 2048 | Reward mean/std -0.058025/0.213785 | Value mean/std -0.053217/0.127257 | Adv std 1.204922e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.058025 | AvgAdv 0.000000 | AdvAfterStd 7.125129e-02
[Update 1510] Samples 2048 | Reward mean/std -0.058219/0.180080 | Value mean/std -0.060075/0.145904 | Adv std 9.277556e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.058219 | AvgAdv 0.000000 | AdvAfterStd 8.871724e-02
[Update 1511] Samples 2048 | Reward mean/std -0.067124/0.248758 | Value mean/std -0.065084/0.239838 | Adv std 1.048050e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.067124 | AvgAdv -0.000000 | AdvAfterStd 8.105726e-02
[Update 1512] Samples 2048 | Reward mean/std -0.060898/0.170609 | Value mean/std -0.056976/0.139513 | Adv std 1.132302e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.060898 | AvgAdv 0.000000 | AdvAfterStd 7.659162e-02
[Update 1513] Samples 2048 | Reward mean/std -0.065315/0.187670 | Value mean/std -0.060515/0.156932 | Adv std 1.163630e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.065315 | AvgAdv -0.000000 | AdvAfterStd 9.088900e-02
[Update 1514] Samples 2048 | Reward mean/std -0.052853/0.127276 | Value mean/std -0.060653/0.113667 | Adv std 8.902986e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.052853 | AvgAdv -0.000000 | AdvAfterStd 7.159048e-02
[Update 1515] Samples 2048 | Reward mean/std -0.061820/0.187844 | Value mean/std -0.059526/0.131134 | Adv std 1.196868e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.061820 | AvgAdv -0.000000 | AdvAfterStd 7.953999e-02
[Update 1516] Samples 2048 | Reward mean/std -0.060870/0.160367 | Value mean/std -0.062024/0.164791 | Adv std 1.093228e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.060870 | AvgAdv -0.000000 | AdvAfterStd 7.728855e-02
[Update 1517] Samples 2048 | Reward mean/std -0.067509/0.205170 | Value mean/std -0.069738/0.209856 | Adv std 1.397283e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.067509 | AvgAdv -0.000000 | AdvAfterStd 9.009679e-02
[Update 1518] Samples 2048 | Reward mean/std -0.058143/0.177659 | Value mean/std -0.063789/0.148686 | Adv std 1.002818e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.058143 | AvgAdv 0.000000 | AdvAfterStd 7.910371e-02
[Update 1519] Samples 2048 | Reward mean/std -0.064122/0.198916 | Value mean/std -0.062239/0.163769 | Adv std 1.216098e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.064122 | AvgAdv -0.000000 | AdvAfterStd 9.366215e-02
[Update 1520] Samples 2048 | Reward mean/std -0.060906/0.224348 | Value mean/std -0.058687/0.149354 | Adv std 1.439222e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.060906 | AvgAdv 0.000000 | AdvAfterStd 6.573613e-02
[Update 1521] Samples 2048 | Reward mean/std -0.065410/0.201179 | Value mean/std -0.067072/0.215053 | Adv std 1.470246e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.065410 | AvgAdv 0.000000 | AdvAfterStd 8.582583e-02
[Update 1522] Samples 2048 | Reward mean/std -0.058937/0.166396 | Value mean/std -0.055375/0.114143 | Adv std 1.082014e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.058937 | AvgAdv 0.000000 | AdvAfterStd 6.746505e-02
[Update 1523] Samples 2048 | Reward mean/std -0.063602/0.216017 | Value mean/std -0.066834/0.169317 | Adv std 1.318085e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.063602 | AvgAdv 0.000000 | AdvAfterStd 6.899025e-02
[Update 1524] Samples 2048 | Reward mean/std -0.073004/0.275086 | Value mean/std -0.068707/0.257685 | Adv std 1.181155e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.073004 | AvgAdv -0.000000 | AdvAfterStd 9.288526e-02
[Update 1525] Samples 2048 | Reward mean/std -0.062164/0.172719 | Value mean/std -0.065824/0.162534 | Adv std 1.193803e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.062164 | AvgAdv -0.000000 | AdvAfterStd 7.291859e-02
[Update 1526] Samples 2048 | Reward mean/std -0.057912/0.178923 | Value mean/std -0.053016/0.120043 | Adv std 1.110130e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.057912 | AvgAdv -0.000000 | AdvAfterStd 7.110362e-02
[Update 1527] Samples 2048 | Reward mean/std -0.061305/0.166740 | Value mean/std -0.065290/0.150703 | Adv std 1.033740e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.061305 | AvgAdv 0.000000 | AdvAfterStd 9.054958e-02
[Update 1528] Samples 2048 | Reward mean/std -0.061304/0.152476 | Value mean/std -0.059943/0.130960 | Adv std 9.261779e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.061304 | AvgAdv 0.000000 | AdvAfterStd 8.095030e-02
[Update 1529] Samples 2048 | Reward mean/std -0.057326/0.161279 | Value mean/std -0.057877/0.129623 | Adv std 8.738589e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.057326 | AvgAdv 0.000000 | AdvAfterStd 6.461583e-02
[Update 1530] Samples 2048 | Reward mean/std -0.060601/0.163631 | Value mean/std -0.062113/0.150492 | Adv std 9.450960e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.060601 | AvgAdv -0.000000 | AdvAfterStd 8.339814e-02
[Update 1531] Samples 2048 | Reward mean/std -0.060902/0.187592 | Value mean/std -0.055115/0.128891 | Adv std 1.035301e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.060902 | AvgAdv -0.000000 | AdvAfterStd 7.930215e-02
[Update 1532] Samples 2048 | Reward mean/std -0.063452/0.223853 | Value mean/std -0.074836/0.207342 | Adv std 1.014044e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.063452 | AvgAdv 0.000000 | AdvAfterStd 7.057032e-02
[Update 1533] Samples 2048 | Reward mean/std -0.060252/0.181555 | Value mean/std -0.050739/0.120289 | Adv std 1.133484e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.060252 | AvgAdv 0.000000 | AdvAfterStd 6.492415e-02
[Update 1534] Samples 2048 | Reward mean/std -0.051401/0.121720 | Value mean/std -0.054961/0.152236 | Adv std 1.061526e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.051401 | AvgAdv 0.000000 | AdvAfterStd 6.171854e-02
[Update 1535] Samples 2048 | Reward mean/std -0.059922/0.173518 | Value mean/std -0.061794/0.140999 | Adv std 9.892376e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.059922 | AvgAdv -0.000000 | AdvAfterStd 7.497986e-02
[Update 1536] Samples 2048 | Reward mean/std -0.056218/0.159634 | Value mean/std -0.056017/0.160784 | Adv std 1.017758e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.056218 | AvgAdv 0.000000 | AdvAfterStd 7.582194e-02
[Update 1537] Samples 2048 | Reward mean/std -0.059862/0.178921 | Value mean/std -0.060862/0.146551 | Adv std 9.433502e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.059862 | AvgAdv 0.000000 | AdvAfterStd 8.348752e-02
[Update 1538] Samples 2048 | Reward mean/std -0.056286/0.140454 | Value mean/std -0.053856/0.117014 | Adv std 8.790403e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.056286 | AvgAdv -0.000000 | AdvAfterStd 7.209399e-02
[Update 1539] Samples 2048 | Reward mean/std -0.054688/0.137346 | Value mean/std -0.053777/0.111381 | Adv std 7.813118e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.054688 | AvgAdv -0.000000 | AdvAfterStd 5.891021e-02
[Update 1540] Samples 2048 | Reward mean/std -0.055014/0.129064 | Value mean/std -0.047967/0.084969 | Adv std 9.329052e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.055014 | AvgAdv 0.000000 | AdvAfterStd 7.484482e-02
[Update 1541] Samples 2048 | Reward mean/std -0.067536/0.242492 | Value mean/std -0.061526/0.230371 | Adv std 9.300372e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.067536 | AvgAdv -0.000000 | AdvAfterStd 7.388176e-02
[Update 1542] Samples 2048 | Reward mean/std -0.060027/0.167023 | Value mean/std -0.061893/0.166149 | Adv std 9.890451e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.060027 | AvgAdv -0.000000 | AdvAfterStd 6.556328e-02
[Update 1543] Samples 2048 | Reward mean/std -0.059866/0.182339 | Value mean/std -0.051143/0.138268 | Adv std 1.027053e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.059866 | AvgAdv 0.000000 | AdvAfterStd 8.272704e-02
[Update 1544] Samples 2048 | Reward mean/std -0.067700/0.275722 | Value mean/std -0.064125/0.226842 | Adv std 1.065058e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.067700 | AvgAdv 0.000000 | AdvAfterStd 7.929869e-02
[Update 1545] Samples 2048 | Reward mean/std -0.065225/0.195718 | Value mean/std -0.061103/0.169309 | Adv std 9.965453e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.065225 | AvgAdv 0.000000 | AdvAfterStd 8.130603e-02
[Update 1546] Samples 2048 | Reward mean/std -0.063115/0.171869 | Value mean/std -0.060749/0.136591 | Adv std 1.087781e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.063115 | AvgAdv 0.000000 | AdvAfterStd 7.924134e-02
[Update 1547] Samples 2048 | Reward mean/std -0.057495/0.167872 | Value mean/std -0.065093/0.153557 | Adv std 9.654710e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.057495 | AvgAdv 0.000000 | AdvAfterStd 7.020408e-02
[Update 1548] Samples 2048 | Reward mean/std -0.071175/0.296735 | Value mean/std -0.057382/0.193732 | Adv std 1.632434e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.071175 | AvgAdv 0.000000 | AdvAfterStd 7.677035e-02
[Update 1549] Samples 2048 | Reward mean/std -0.069640/0.231468 | Value mean/std -0.069441/0.219787 | Adv std 1.145350e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.069640 | AvgAdv 0.000000 | AdvAfterStd 8.821876e-02
[Update 1550] Samples 2048 | Reward mean/std -0.064269/0.220903 | Value mean/std -0.062533/0.183767 | Adv std 1.372028e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.064269 | AvgAdv 0.000000 | AdvAfterStd 7.252359e-02
Saved checkpoint at update 1550
[Update 1551] Samples 2048 | Reward mean/std -0.057368/0.186038 | Value mean/std -0.057453/0.149129 | Adv std 1.209898e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.057368 | AvgAdv 0.000000 | AdvAfterStd 6.290885e-02
[Update 1552] Samples 2048 | Reward mean/std -0.061274/0.175394 | Value mean/std -0.062741/0.144919 | Adv std 1.028064e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.061274 | AvgAdv 0.000000 | AdvAfterStd 8.174524e-02
[Update 1553] Samples 2048 | Reward mean/std -0.062469/0.200846 | Value mean/std -0.059477/0.180832 | Adv std 1.200788e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.062469 | AvgAdv 0.000000 | AdvAfterStd 7.429977e-02
[Update 1554] Samples 2048 | Reward mean/std -0.060962/0.170766 | Value mean/std -0.060896/0.145175 | Adv std 8.631968e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.060962 | AvgAdv -0.000000 | AdvAfterStd 7.180102e-02
[Update 1555] Samples 2048 | Reward mean/std -0.057415/0.161976 | Value mean/std -0.054679/0.116546 | Adv std 9.948042e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.057415 | AvgAdv 0.000000 | AdvAfterStd 6.938292e-02
[Update 1556] Samples 2048 | Reward mean/std -0.058136/0.152662 | Value mean/std -0.060989/0.144780 | Adv std 8.586917e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.058136 | AvgAdv 0.000000 | AdvAfterStd 5.920000e-02
[Update 1557] Samples 2048 | Reward mean/std -0.059903/0.200820 | Value mean/std -0.065119/0.226660 | Adv std 1.030232e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.059903 | AvgAdv 0.000000 | AdvAfterStd 6.567594e-02
[Update 1558] Samples 2048 | Reward mean/std -0.063766/0.235341 | Value mean/std -0.061831/0.180555 | Adv std 1.378943e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.063766 | AvgAdv -0.000000 | AdvAfterStd 7.838871e-02
[Update 1559] Samples 2048 | Reward mean/std -0.068583/0.220588 | Value mean/std -0.068003/0.182696 | Adv std 1.199268e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.068583 | AvgAdv 0.000000 | AdvAfterStd 8.661696e-02
[Update 1560] Samples 2048 | Reward mean/std -0.061367/0.171810 | Value mean/std -0.063597/0.155010 | Adv std 1.200480e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.061367 | AvgAdv 0.000000 | AdvAfterStd 9.167559e-02
[Update 1561] Samples 2048 | Reward mean/std -0.062113/0.212898 | Value mean/std -0.052849/0.121243 | Adv std 1.610257e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.062113 | AvgAdv 0.000000 | AdvAfterStd 1.560535e-01
[Update 1562] Samples 2048 | Reward mean/std -0.065978/0.212954 | Value mean/std -0.072608/0.182982 | Adv std 1.054203e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.065978 | AvgAdv -0.000000 | AdvAfterStd 8.058437e-02
[Update 1563] Samples 2048 | Reward mean/std -0.059813/0.183641 | Value mean/std -0.060336/0.153871 | Adv std 1.020033e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.059813 | AvgAdv 0.000000 | AdvAfterStd 7.801664e-02
[Update 1564] Samples 2048 | Reward mean/std -0.068048/0.232849 | Value mean/std -0.064037/0.175959 | Adv std 1.380621e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.068048 | AvgAdv 0.000000 | AdvAfterStd 7.870075e-02
[Update 1565] Samples 2048 | Reward mean/std -0.067954/0.230803 | Value mean/std -0.066380/0.202352 | Adv std 1.204636e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.067954 | AvgAdv 0.000000 | AdvAfterStd 8.035682e-02
[Update 1566] Samples 2048 | Reward mean/std -0.059617/0.172646 | Value mean/std -0.064096/0.173559 | Adv std 1.064973e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.059617 | AvgAdv 0.000000 | AdvAfterStd 7.658179e-02
[Update 1567] Samples 2048 | Reward mean/std -0.063409/0.184584 | Value mean/std -0.057126/0.153009 | Adv std 1.004279e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.063409 | AvgAdv 0.000000 | AdvAfterStd 8.649454e-02
[Update 1568] Samples 2048 | Reward mean/std -0.064249/0.217495 | Value mean/std -0.065425/0.136143 | Adv std 1.436626e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.064249 | AvgAdv 0.000000 | AdvAfterStd 6.963848e-02
[Update 1569] Samples 2048 | Reward mean/std -0.058839/0.147571 | Value mean/std -0.058270/0.121619 | Adv std 1.001466e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.058839 | AvgAdv -0.000000 | AdvAfterStd 6.900062e-02
[Update 1570] Samples 2048 | Reward mean/std -0.060820/0.213346 | Value mean/std -0.052128/0.114053 | Adv std 1.741765e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.060820 | AvgAdv 0.000000 | AdvAfterStd 9.673575e-02
[Update 1571] Samples 2048 | Reward mean/std -0.062405/0.191074 | Value mean/std -0.073145/0.235521 | Adv std 1.450635e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.062405 | AvgAdv -0.000000 | AdvAfterStd 8.547670e-02
[Update 1572] Samples 2048 | Reward mean/std -0.063137/0.195295 | Value mean/std -0.060377/0.128629 | Adv std 1.086507e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.063137 | AvgAdv -0.000000 | AdvAfterStd 7.665660e-02
[Update 1573] Samples 2048 | Reward mean/std -0.059565/0.188877 | Value mean/std -0.061224/0.188340 | Adv std 1.141248e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.059565 | AvgAdv -0.000000 | AdvAfterStd 6.139849e-02
[Update 1574] Samples 2048 | Reward mean/std -0.063736/0.188439 | Value mean/std -0.067353/0.183182 | Adv std 1.467368e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.063736 | AvgAdv -0.000000 | AdvAfterStd 9.732304e-02
[Update 1575] Samples 2048 | Reward mean/std -0.061815/0.169290 | Value mean/std -0.066197/0.137307 | Adv std 9.306566e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.061815 | AvgAdv -0.000000 | AdvAfterStd 7.348081e-02
[Update 1576] Samples 2048 | Reward mean/std -0.061427/0.236858 | Value mean/std -0.059706/0.230553 | Adv std 8.686212e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.061427 | AvgAdv -0.000000 | AdvAfterStd 6.143460e-02
[Update 1577] Samples 2048 | Reward mean/std -0.062457/0.220278 | Value mean/std -0.057964/0.141142 | Adv std 1.452060e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.062457 | AvgAdv -0.000000 | AdvAfterStd 8.586237e-02
[Update 1578] Samples 2048 | Reward mean/std -0.069154/0.268448 | Value mean/std -0.069015/0.225917 | Adv std 1.029394e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.069154 | AvgAdv -0.000000 | AdvAfterStd 7.147804e-02
[Update 1579] Samples 2048 | Reward mean/std -0.053441/0.141459 | Value mean/std -0.055130/0.141213 | Adv std 9.096746e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.053441 | AvgAdv 0.000000 | AdvAfterStd 6.226029e-02
[Update 1580] Samples 2048 | Reward mean/std -0.060769/0.156931 | Value mean/std -0.057462/0.138066 | Adv std 1.015390e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.060769 | AvgAdv 0.000000 | AdvAfterStd 7.113207e-02
[Update 1581] Samples 2048 | Reward mean/std -0.054848/0.163146 | Value mean/std -0.059350/0.156883 | Adv std 8.784175e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.054848 | AvgAdv -0.000000 | AdvAfterStd 7.145573e-02
[Update 1582] Samples 2048 | Reward mean/std -0.069853/0.271446 | Value mean/std -0.063319/0.182404 | Adv std 1.651938e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.069853 | AvgAdv -0.000000 | AdvAfterStd 9.292424e-02
[Update 1583] Samples 2048 | Reward mean/std -0.058158/0.204057 | Value mean/std -0.059904/0.216738 | Adv std 8.583437e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.058158 | AvgAdv -0.000000 | AdvAfterStd 6.017755e-02
[Update 1584] Samples 2048 | Reward mean/std -0.057421/0.183722 | Value mean/std -0.053176/0.108069 | Adv std 1.185123e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.057421 | AvgAdv 0.000000 | AdvAfterStd 7.022016e-02
[Update 1585] Samples 2048 | Reward mean/std -0.060482/0.164396 | Value mean/std -0.059902/0.187834 | Adv std 1.190163e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.060482 | AvgAdv 0.000000 | AdvAfterStd 7.738665e-02
[Update 1586] Samples 2048 | Reward mean/std -0.062784/0.225553 | Value mean/std -0.057976/0.182377 | Adv std 1.314449e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.062784 | AvgAdv 0.000000 | AdvAfterStd 7.217351e-02
[Update 1587] Samples 2048 | Reward mean/std -0.062088/0.198637 | Value mean/std -0.058448/0.199630 | Adv std 9.417771e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.062088 | AvgAdv 0.000000 | AdvAfterStd 7.308529e-02
[Update 1588] Samples 2048 | Reward mean/std -0.052285/0.128259 | Value mean/std -0.053694/0.111036 | Adv std 9.003862e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.052285 | AvgAdv 0.000000 | AdvAfterStd 6.776164e-02
[Update 1589] Samples 2048 | Reward mean/std -0.063272/0.220783 | Value mean/std -0.060978/0.156243 | Adv std 1.177834e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.063272 | AvgAdv 0.000000 | AdvAfterStd 9.598443e-02
[Update 1590] Samples 2048 | Reward mean/std -0.058107/0.177276 | Value mean/std -0.058622/0.155950 | Adv std 1.012921e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.058107 | AvgAdv -0.000000 | AdvAfterStd 7.978830e-02
[Update 1591] Samples 2048 | Reward mean/std -0.063540/0.183882 | Value mean/std -0.065580/0.163694 | Adv std 1.071384e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.063540 | AvgAdv 0.000000 | AdvAfterStd 7.193664e-02
[Update 1592] Samples 2048 | Reward mean/std -0.057949/0.188564 | Value mean/std -0.062595/0.229589 | Adv std 1.451436e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.057949 | AvgAdv 0.000000 | AdvAfterStd 8.345611e-02
[Update 1593] Samples 2048 | Reward mean/std -0.061069/0.199287 | Value mean/std -0.061071/0.146983 | Adv std 1.372972e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.061069 | AvgAdv 0.000000 | AdvAfterStd 8.725898e-02
[Update 1594] Samples 2048 | Reward mean/std -0.071692/0.346352 | Value mean/std -0.067027/0.220702 | Adv std 2.399620e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.071692 | AvgAdv 0.000000 | AdvAfterStd 8.490630e-02
[Update 1595] Samples 2048 | Reward mean/std -0.060792/0.173598 | Value mean/std -0.066033/0.286902 | Adv std 2.251889e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.060792 | AvgAdv 0.000000 | AdvAfterStd 6.536990e-02
[Update 1596] Samples 2048 | Reward mean/std -0.054763/0.168526 | Value mean/std -0.061782/0.174667 | Adv std 1.027482e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.054763 | AvgAdv -0.000000 | AdvAfterStd 6.856813e-02
[Update 1597] Samples 2048 | Reward mean/std -0.058113/0.149008 | Value mean/std -0.051410/0.103042 | Adv std 1.087536e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.058113 | AvgAdv 0.000000 | AdvAfterStd 6.831330e-02
[Update 1598] Samples 2048 | Reward mean/std -0.070552/0.246473 | Value mean/std -0.066100/0.230093 | Adv std 1.218081e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.070552 | AvgAdv 0.000000 | AdvAfterStd 9.062738e-02
[Update 1599] Samples 2048 | Reward mean/std -0.059726/0.169915 | Value mean/std -0.065924/0.165050 | Adv std 1.102499e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.059726 | AvgAdv -0.000000 | AdvAfterStd 7.748614e-02
[Update 1600] Samples 2048 | Reward mean/std -0.058001/0.177098 | Value mean/std -0.057803/0.118990 | Adv std 1.149504e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.058001 | AvgAdv 0.000000 | AdvAfterStd 5.946218e-02
Saved checkpoint at update 1600
[Update 1601] Samples 2048 | Reward mean/std -0.064926/0.196036 | Value mean/std -0.064072/0.186006 | Adv std 1.307899e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.064926 | AvgAdv -0.000000 | AdvAfterStd 8.096201e-02
[Update 1602] Samples 2048 | Reward mean/std -0.059911/0.175677 | Value mean/std -0.058857/0.168850 | Adv std 8.983351e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.059911 | AvgAdv 0.000000 | AdvAfterStd 7.125819e-02
[Update 1603] Samples 2048 | Reward mean/std -0.058374/0.197125 | Value mean/std -0.061612/0.186130 | Adv std 8.512937e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.058374 | AvgAdv -0.000000 | AdvAfterStd 6.722163e-02
[Update 1604] Samples 2048 | Reward mean/std -0.060472/0.152208 | Value mean/std -0.053749/0.126907 | Adv std 8.889182e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.060472 | AvgAdv -0.000000 | AdvAfterStd 7.582568e-02
[Update 1605] Samples 2048 | Reward mean/std -0.060791/0.274229 | Value mean/std -0.061235/0.150885 | Adv std 1.699759e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.060791 | AvgAdv -0.000000 | AdvAfterStd 8.912074e-02
[Update 1606] Samples 2048 | Reward mean/std -0.064023/0.179801 | Value mean/std -0.057954/0.136301 | Adv std 1.219331e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.064023 | AvgAdv -0.000000 | AdvAfterStd 8.899921e-02
[Update 1607] Samples 2048 | Reward mean/std -0.053174/0.149782 | Value mean/std -0.052762/0.119638 | Adv std 9.470864e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.053174 | AvgAdv 0.000000 | AdvAfterStd 6.548905e-02
[Update 1608] Samples 2048 | Reward mean/std -0.061494/0.177000 | Value mean/std -0.055662/0.145609 | Adv std 1.023533e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.061494 | AvgAdv 0.000000 | AdvAfterStd 7.707070e-02
[Update 1609] Samples 2048 | Reward mean/std -0.059130/0.179594 | Value mean/std -0.069040/0.158222 | Adv std 1.039031e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.059130 | AvgAdv 0.000000 | AdvAfterStd 7.079694e-02
[Update 1610] Samples 2048 | Reward mean/std -0.058866/0.160230 | Value mean/std -0.051362/0.116753 | Adv std 1.039189e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.058866 | AvgAdv -0.000000 | AdvAfterStd 7.705355e-02
[Update 1611] Samples 2048 | Reward mean/std -0.061763/0.200537 | Value mean/std -0.062968/0.206837 | Adv std 9.302459e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.061763 | AvgAdv -0.000000 | AdvAfterStd 8.166084e-02
[Update 1612] Samples 2048 | Reward mean/std -0.058726/0.200680 | Value mean/std -0.056083/0.150335 | Adv std 1.210553e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.058726 | AvgAdv -0.000000 | AdvAfterStd 6.502756e-02
[Update 1613] Samples 2048 | Reward mean/std -0.057606/0.192893 | Value mean/std -0.058651/0.179258 | Adv std 1.039599e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.057606 | AvgAdv 0.000000 | AdvAfterStd 9.103497e-02
[Update 1614] Samples 2048 | Reward mean/std -0.056535/0.150885 | Value mean/std -0.051385/0.111591 | Adv std 1.020890e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.056535 | AvgAdv 0.000000 | AdvAfterStd 8.553894e-02
[Update 1615] Samples 2048 | Reward mean/std -0.060126/0.188454 | Value mean/std -0.060244/0.125557 | Adv std 1.131795e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.060126 | AvgAdv -0.000000 | AdvAfterStd 8.385687e-02
[Update 1616] Samples 2048 | Reward mean/std -0.060040/0.216217 | Value mean/std -0.058042/0.127302 | Adv std 1.494229e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.060040 | AvgAdv -0.000000 | AdvAfterStd 8.697852e-02
[Update 1617] Samples 2048 | Reward mean/std -0.063407/0.183438 | Value mean/std -0.068343/0.225115 | Adv std 1.526139e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.063407 | AvgAdv 0.000000 | AdvAfterStd 7.344584e-02
[Update 1618] Samples 2048 | Reward mean/std -0.056261/0.153341 | Value mean/std -0.060018/0.171842 | Adv std 8.033872e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.056261 | AvgAdv 0.000000 | AdvAfterStd 6.716312e-02
[Update 1619] Samples 2048 | Reward mean/std -0.061903/0.174881 | Value mean/std -0.054982/0.135252 | Adv std 9.219096e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.061903 | AvgAdv 0.000000 | AdvAfterStd 7.012292e-02
[Update 1620] Samples 2048 | Reward mean/std -0.058802/0.141450 | Value mean/std -0.061227/0.134004 | Adv std 1.132601e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.058802 | AvgAdv 0.000000 | AdvAfterStd 7.370033e-02
[Update 1621] Samples 2048 | Reward mean/std -0.053417/0.133757 | Value mean/std -0.054273/0.118149 | Adv std 8.036567e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.053417 | AvgAdv -0.000000 | AdvAfterStd 7.050372e-02
[Update 1622] Samples 2048 | Reward mean/std -0.071330/0.258541 | Value mean/std -0.066015/0.195662 | Adv std 1.235501e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.071330 | AvgAdv 0.000000 | AdvAfterStd 9.236055e-02
[Update 1623] Samples 2048 | Reward mean/std -0.070124/0.254687 | Value mean/std -0.072266/0.221424 | Adv std 1.713721e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.070124 | AvgAdv -0.000000 | AdvAfterStd 9.029819e-02
[Update 1624] Samples 2048 | Reward mean/std -0.066962/0.311463 | Value mean/std -0.067242/0.213566 | Adv std 1.533981e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.066962 | AvgAdv -0.000000 | AdvAfterStd 8.482644e-02
[Update 1625] Samples 2048 | Reward mean/std -0.070625/0.236018 | Value mean/std -0.072790/0.312389 | Adv std 1.736917e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.070625 | AvgAdv -0.000000 | AdvAfterStd 1.102120e-01
[Update 1626] Samples 2048 | Reward mean/std -0.075272/0.249457 | Value mean/std -0.074683/0.214611 | Adv std 1.173785e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.075272 | AvgAdv 0.000000 | AdvAfterStd 9.264006e-02
[Update 1627] Samples 2048 | Reward mean/std -0.069128/0.263648 | Value mean/std -0.074269/0.276615 | Adv std 1.122292e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.069128 | AvgAdv -0.000000 | AdvAfterStd 7.755924e-02
[Update 1628] Samples 2048 | Reward mean/std -0.058684/0.150007 | Value mean/std -0.057691/0.111804 | Adv std 1.011849e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.058684 | AvgAdv -0.000000 | AdvAfterStd 7.788996e-02
[Update 1629] Samples 2048 | Reward mean/std -0.066193/0.290007 | Value mean/std -0.063742/0.212302 | Adv std 1.371728e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.066193 | AvgAdv 0.000000 | AdvAfterStd 8.319905e-02
[Update 1630] Samples 2048 | Reward mean/std -0.071756/0.261653 | Value mean/std -0.070932/0.313408 | Adv std 1.395883e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.071756 | AvgAdv 0.000000 | AdvAfterStd 7.923695e-02
[Update 1631] Samples 2048 | Reward mean/std -0.066117/0.237007 | Value mean/std -0.066659/0.196202 | Adv std 1.009496e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.066117 | AvgAdv -0.000000 | AdvAfterStd 7.579001e-02
[Update 1632] Samples 2048 | Reward mean/std -0.063963/0.177462 | Value mean/std -0.063877/0.180250 | Adv std 1.065063e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.063963 | AvgAdv 0.000000 | AdvAfterStd 7.678275e-02
[Update 1633] Samples 2048 | Reward mean/std -0.072599/0.290207 | Value mean/std -0.073500/0.213820 | Adv std 1.359758e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.072599 | AvgAdv 0.000000 | AdvAfterStd 8.393390e-02
[Update 1634] Samples 2048 | Reward mean/std -0.063270/0.187985 | Value mean/std -0.064922/0.150576 | Adv std 9.256789e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.063270 | AvgAdv -0.000000 | AdvAfterStd 7.697277e-02
[Update 1635] Samples 2048 | Reward mean/std -0.056445/0.169272 | Value mean/std -0.060136/0.142941 | Adv std 8.977427e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.056445 | AvgAdv 0.000000 | AdvAfterStd 7.539288e-02
[Update 1636] Samples 2048 | Reward mean/std -0.064729/0.213760 | Value mean/std -0.068062/0.204100 | Adv std 1.048688e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.064729 | AvgAdv 0.000000 | AdvAfterStd 7.412232e-02
[Update 1637] Samples 2048 | Reward mean/std -0.068210/0.237528 | Value mean/std -0.065947/0.230511 | Adv std 1.107898e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.068210 | AvgAdv -0.000000 | AdvAfterStd 7.953197e-02
[Update 1638] Samples 2048 | Reward mean/std -0.057709/0.166654 | Value mean/std -0.066304/0.181337 | Adv std 1.104138e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.057709 | AvgAdv -0.000000 | AdvAfterStd 8.027390e-02
[Update 1639] Samples 2048 | Reward mean/std -0.056308/0.144921 | Value mean/std -0.053358/0.125955 | Adv std 8.779684e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.056308 | AvgAdv -0.000000 | AdvAfterStd 7.373236e-02
[Update 1640] Samples 2048 | Reward mean/std -0.059548/0.172303 | Value mean/std -0.062664/0.138904 | Adv std 9.922483e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.059548 | AvgAdv -0.000000 | AdvAfterStd 7.581546e-02
[Update 1641] Samples 2048 | Reward mean/std -0.064026/0.193153 | Value mean/std -0.061426/0.152072 | Adv std 9.941548e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.064026 | AvgAdv -0.000000 | AdvAfterStd 7.728962e-02
[Update 1642] Samples 2048 | Reward mean/std -0.057169/0.146017 | Value mean/std -0.057410/0.131818 | Adv std 8.775824e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.057169 | AvgAdv -0.000000 | AdvAfterStd 6.889111e-02
[Update 1643] Samples 2048 | Reward mean/std -0.065174/0.256027 | Value mean/std -0.061689/0.186879 | Adv std 1.120437e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.065174 | AvgAdv 0.000000 | AdvAfterStd 9.778655e-02
[Update 1644] Samples 2048 | Reward mean/std -0.059837/0.153220 | Value mean/std -0.059926/0.125647 | Adv std 1.005865e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.059837 | AvgAdv 0.000000 | AdvAfterStd 7.852121e-02
[Update 1645] Samples 2048 | Reward mean/std -0.058423/0.202767 | Value mean/std -0.068134/0.238936 | Adv std 1.026411e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.058423 | AvgAdv 0.000000 | AdvAfterStd 6.801782e-02
[Update 1646] Samples 2048 | Reward mean/std -0.059906/0.181733 | Value mean/std -0.058561/0.149884 | Adv std 9.201934e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.059906 | AvgAdv -0.000000 | AdvAfterStd 7.637776e-02
[Update 1647] Samples 2048 | Reward mean/std -0.055212/0.148210 | Value mean/std -0.055849/0.125328 | Adv std 8.244251e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.055212 | AvgAdv 0.000000 | AdvAfterStd 5.980983e-02
[Update 1648] Samples 2048 | Reward mean/std -0.061221/0.191196 | Value mean/std -0.054986/0.168394 | Adv std 8.979208e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.061221 | AvgAdv -0.000000 | AdvAfterStd 6.795685e-02
[Update 1649] Samples 2048 | Reward mean/std -0.067125/0.200895 | Value mean/std -0.064482/0.143977 | Adv std 1.201815e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.067125 | AvgAdv -0.000000 | AdvAfterStd 8.302677e-02
[Update 1650] Samples 2048 | Reward mean/std -0.060410/0.157779 | Value mean/std -0.058739/0.141628 | Adv std 8.470461e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.060410 | AvgAdv -0.000000 | AdvAfterStd 7.877283e-02
Saved checkpoint at update 1650
[Update 1651] Samples 2048 | Reward mean/std -0.064170/0.265490 | Value mean/std -0.059393/0.181898 | Adv std 1.112837e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.064170 | AvgAdv -0.000000 | AdvAfterStd 7.402840e-02
[Update 1652] Samples 2048 | Reward mean/std -0.058654/0.168842 | Value mean/std -0.055918/0.116554 | Adv std 1.055658e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.058654 | AvgAdv -0.000000 | AdvAfterStd 6.791788e-02
[Update 1653] Samples 2048 | Reward mean/std -0.057468/0.182491 | Value mean/std -0.057943/0.141061 | Adv std 9.091682e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.057468 | AvgAdv -0.000000 | AdvAfterStd 6.959929e-02
[Update 1654] Samples 2048 | Reward mean/std -0.054376/0.150045 | Value mean/std -0.056201/0.139432 | Adv std 9.016623e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.054376 | AvgAdv -0.000000 | AdvAfterStd 7.319894e-02
[Update 1655] Samples 2048 | Reward mean/std -0.057317/0.174412 | Value mean/std -0.060059/0.151607 | Adv std 8.661772e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.057317 | AvgAdv -0.000000 | AdvAfterStd 5.663233e-02
[Update 1656] Samples 2048 | Reward mean/std -0.056640/0.169634 | Value mean/std -0.056553/0.131618 | Adv std 1.139294e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.056640 | AvgAdv -0.000000 | AdvAfterStd 6.828367e-02
[Update 1657] Samples 2048 | Reward mean/std -0.061538/0.190239 | Value mean/std -0.058658/0.165593 | Adv std 1.014297e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.061538 | AvgAdv -0.000000 | AdvAfterStd 8.319975e-02
[Update 1658] Samples 2048 | Reward mean/std -0.062792/0.207401 | Value mean/std -0.062842/0.166208 | Adv std 1.169908e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.062792 | AvgAdv -0.000000 | AdvAfterStd 6.621854e-02
[Update 1659] Samples 2048 | Reward mean/std -0.059048/0.162402 | Value mean/std -0.056708/0.142093 | Adv std 1.085376e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.059048 | AvgAdv -0.000000 | AdvAfterStd 6.808022e-02
[Update 1660] Samples 2048 | Reward mean/std -0.062072/0.223365 | Value mean/std -0.059865/0.155362 | Adv std 1.330043e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.062072 | AvgAdv 0.000000 | AdvAfterStd 7.280380e-02
[Update 1661] Samples 2048 | Reward mean/std -0.052173/0.133319 | Value mean/std -0.053069/0.120854 | Adv std 7.491411e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.052173 | AvgAdv 0.000000 | AdvAfterStd 6.361232e-02
[Update 1662] Samples 2048 | Reward mean/std -0.059235/0.164060 | Value mean/std -0.053107/0.139445 | Adv std 1.025434e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.059235 | AvgAdv 0.000000 | AdvAfterStd 6.514954e-02
[Update 1663] Samples 2048 | Reward mean/std -0.061675/0.208376 | Value mean/std -0.053723/0.166241 | Adv std 9.306427e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.061675 | AvgAdv 0.000000 | AdvAfterStd 7.141115e-02
[Update 1664] Samples 2048 | Reward mean/std -0.059305/0.217367 | Value mean/std -0.060445/0.146684 | Adv std 1.080228e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.059305 | AvgAdv -0.000000 | AdvAfterStd 7.539158e-02
[Update 1665] Samples 2048 | Reward mean/std -0.059684/0.211419 | Value mean/std -0.066346/0.249427 | Adv std 1.607359e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.059684 | AvgAdv 0.000000 | AdvAfterStd 6.509364e-02
[Update 1666] Samples 2048 | Reward mean/std -0.060092/0.240022 | Value mean/std -0.056282/0.212020 | Adv std 9.557603e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.060092 | AvgAdv 0.000000 | AdvAfterStd 7.112411e-02
[Update 1667] Samples 2048 | Reward mean/std -0.057195/0.151780 | Value mean/std -0.059466/0.150544 | Adv std 7.440266e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.057195 | AvgAdv 0.000000 | AdvAfterStd 7.290516e-02
[Update 1668] Samples 2048 | Reward mean/std -0.050806/0.116204 | Value mean/std -0.048361/0.090540 | Adv std 6.978361e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.050806 | AvgAdv 0.000000 | AdvAfterStd 5.904819e-02
[Update 1669] Samples 2048 | Reward mean/std -0.056449/0.179889 | Value mean/std -0.053974/0.111708 | Adv std 1.119418e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.056449 | AvgAdv 0.000000 | AdvAfterStd 6.800225e-02
[Update 1670] Samples 2048 | Reward mean/std -0.057818/0.170519 | Value mean/std -0.060345/0.133331 | Adv std 1.058347e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.057818 | AvgAdv 0.000000 | AdvAfterStd 7.452365e-02
[Update 1671] Samples 2048 | Reward mean/std -0.053689/0.150293 | Value mean/std -0.050377/0.120829 | Adv std 9.305421e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.053689 | AvgAdv -0.000000 | AdvAfterStd 7.107013e-02
[Update 1672] Samples 2048 | Reward mean/std -0.056998/0.186874 | Value mean/std -0.053253/0.123020 | Adv std 1.121055e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.056998 | AvgAdv 0.000000 | AdvAfterStd 7.627697e-02
[Update 1673] Samples 2048 | Reward mean/std -0.055570/0.145949 | Value mean/std -0.057770/0.139312 | Adv std 1.191304e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.055570 | AvgAdv 0.000000 | AdvAfterStd 6.589875e-02
[Update 1674] Samples 2048 | Reward mean/std -0.057395/0.143309 | Value mean/std -0.061260/0.128067 | Adv std 8.991358e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.057395 | AvgAdv -0.000000 | AdvAfterStd 6.975888e-02
[Update 1675] Samples 2048 | Reward mean/std -0.060741/0.183736 | Value mean/std -0.055737/0.135370 | Adv std 9.966535e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.060741 | AvgAdv 0.000000 | AdvAfterStd 7.271791e-02
[Update 1676] Samples 2048 | Reward mean/std -0.052571/0.127388 | Value mean/std -0.052133/0.089453 | Adv std 9.413508e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.052571 | AvgAdv -0.000000 | AdvAfterStd 7.475639e-02
[Update 1677] Samples 2048 | Reward mean/std -0.058173/0.220442 | Value mean/std -0.059573/0.160094 | Adv std 1.146856e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.058173 | AvgAdv 0.000000 | AdvAfterStd 7.702289e-02
[Update 1678] Samples 2048 | Reward mean/std -0.067209/0.205711 | Value mean/std -0.063254/0.179248 | Adv std 1.037612e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.067209 | AvgAdv 0.000000 | AdvAfterStd 1.005112e-01
[Update 1679] Samples 2048 | Reward mean/std -0.062539/0.171405 | Value mean/std -0.055316/0.139655 | Adv std 1.114642e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.062539 | AvgAdv 0.000000 | AdvAfterStd 8.786569e-02
[Update 1680] Samples 2048 | Reward mean/std -0.061137/0.186780 | Value mean/std -0.058721/0.150635 | Adv std 1.092002e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.061137 | AvgAdv 0.000000 | AdvAfterStd 8.632488e-02
[Update 1681] Samples 2048 | Reward mean/std -0.061395/0.167537 | Value mean/std -0.058599/0.126674 | Adv std 1.019371e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.061395 | AvgAdv -0.000000 | AdvAfterStd 8.706619e-02
[Update 1682] Samples 2048 | Reward mean/std -0.070382/0.252841 | Value mean/std -0.066496/0.201409 | Adv std 1.270699e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.070382 | AvgAdv 0.000000 | AdvAfterStd 9.066162e-02
[Update 1683] Samples 2048 | Reward mean/std -0.061432/0.190471 | Value mean/std -0.060067/0.182485 | Adv std 1.016626e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.061432 | AvgAdv 0.000000 | AdvAfterStd 7.537945e-02
[Update 1684] Samples 2048 | Reward mean/std -0.058833/0.206095 | Value mean/std -0.061035/0.198635 | Adv std 1.235798e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.058833 | AvgAdv -0.000000 | AdvAfterStd 7.435629e-02
[Update 1685] Samples 2048 | Reward mean/std -0.059345/0.196588 | Value mean/std -0.059933/0.169654 | Adv std 9.044986e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.059345 | AvgAdv 0.000000 | AdvAfterStd 6.804499e-02
[Update 1686] Samples 2048 | Reward mean/std -0.059355/0.195196 | Value mean/std -0.059269/0.144035 | Adv std 1.214615e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.059355 | AvgAdv 0.000000 | AdvAfterStd 6.628515e-02
[Update 1687] Samples 2048 | Reward mean/std -0.059866/0.174490 | Value mean/std -0.056101/0.120194 | Adv std 1.130764e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.059866 | AvgAdv 0.000000 | AdvAfterStd 8.546545e-02
[Update 1688] Samples 2048 | Reward mean/std -0.060688/0.187341 | Value mean/std -0.061182/0.173851 | Adv std 1.110293e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.060688 | AvgAdv -0.000000 | AdvAfterStd 8.827156e-02
[Update 1689] Samples 2048 | Reward mean/std -0.061237/0.204311 | Value mean/std -0.059450/0.181622 | Adv std 7.922793e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.061237 | AvgAdv 0.000000 | AdvAfterStd 7.163537e-02
[Update 1690] Samples 2048 | Reward mean/std -0.069416/0.269702 | Value mean/std -0.066627/0.182102 | Adv std 1.596831e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.069416 | AvgAdv -0.000000 | AdvAfterStd 1.037370e-01
[Update 1691] Samples 2048 | Reward mean/std -0.063700/0.198747 | Value mean/std -0.068358/0.188539 | Adv std 1.083173e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.063700 | AvgAdv -0.000000 | AdvAfterStd 1.092459e-01
[Update 1692] Samples 2048 | Reward mean/std -0.061978/0.201960 | Value mean/std -0.065176/0.178907 | Adv std 9.662120e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.061978 | AvgAdv 0.000000 | AdvAfterStd 7.307371e-02
[Update 1693] Samples 2048 | Reward mean/std -0.061238/0.189657 | Value mean/std -0.058775/0.131920 | Adv std 1.019084e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.061238 | AvgAdv -0.000000 | AdvAfterStd 6.901495e-02
[Update 1694] Samples 2048 | Reward mean/std -0.064649/0.234659 | Value mean/std -0.065859/0.240719 | Adv std 1.552928e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.064649 | AvgAdv -0.000000 | AdvAfterStd 6.923001e-02
[Update 1695] Samples 2048 | Reward mean/std -0.061778/0.187639 | Value mean/std -0.059573/0.133291 | Adv std 1.088251e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.061778 | AvgAdv -0.000000 | AdvAfterStd 8.909144e-02
[Update 1696] Samples 2048 | Reward mean/std -0.055143/0.147593 | Value mean/std -0.055508/0.118310 | Adv std 9.120017e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.055143 | AvgAdv -0.000000 | AdvAfterStd 6.539048e-02
[Update 1697] Samples 2048 | Reward mean/std -0.058988/0.176096 | Value mean/std -0.051107/0.147647 | Adv std 9.215195e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.058988 | AvgAdv 0.000000 | AdvAfterStd 7.243951e-02
[Update 1698] Samples 2048 | Reward mean/std -0.060195/0.186301 | Value mean/std -0.062499/0.152220 | Adv std 1.026159e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.060195 | AvgAdv 0.000000 | AdvAfterStd 7.479894e-02
[Update 1699] Samples 2048 | Reward mean/std -0.059066/0.197557 | Value mean/std -0.064824/0.185673 | Adv std 9.646193e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.059066 | AvgAdv -0.000000 | AdvAfterStd 6.810404e-02
[Update 1700] Samples 2048 | Reward mean/std -0.064003/0.214295 | Value mean/std -0.065017/0.157250 | Adv std 1.107130e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.064003 | AvgAdv 0.000000 | AdvAfterStd 7.380506e-02
Saved checkpoint at update 1700
[Update 1701] Samples 2048 | Reward mean/std -0.062196/0.179531 | Value mean/std -0.062982/0.208696 | Adv std 1.112365e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.062196 | AvgAdv 0.000000 | AdvAfterStd 6.836566e-02
[Update 1702] Samples 2048 | Reward mean/std -0.067015/0.208771 | Value mean/std -0.060028/0.142516 | Adv std 1.231004e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.067015 | AvgAdv 0.000000 | AdvAfterStd 8.120354e-02
[Update 1703] Samples 2048 | Reward mean/std -0.057424/0.166639 | Value mean/std -0.064201/0.175366 | Adv std 8.782066e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.057424 | AvgAdv -0.000000 | AdvAfterStd 7.090312e-02
[Update 1704] Samples 2048 | Reward mean/std -0.054306/0.137267 | Value mean/std -0.055866/0.134856 | Adv std 1.094652e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.054306 | AvgAdv 0.000000 | AdvAfterStd 6.059886e-02
[Update 1705] Samples 2048 | Reward mean/std -0.058845/0.154703 | Value mean/std -0.056400/0.101121 | Adv std 9.679505e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.058845 | AvgAdv -0.000000 | AdvAfterStd 7.731826e-02
[Update 1706] Samples 2048 | Reward mean/std -0.061999/0.195217 | Value mean/std -0.060485/0.154528 | Adv std 1.137574e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.061999 | AvgAdv 0.000000 | AdvAfterStd 6.700624e-02
[Update 1707] Samples 2048 | Reward mean/std -0.059188/0.177290 | Value mean/std -0.060232/0.136321 | Adv std 9.829345e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.059188 | AvgAdv -0.000000 | AdvAfterStd 7.728036e-02
[Update 1708] Samples 2048 | Reward mean/std -0.053937/0.138368 | Value mean/std -0.054932/0.110620 | Adv std 8.993902e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.053937 | AvgAdv -0.000000 | AdvAfterStd 7.273900e-02
[Update 1709] Samples 2048 | Reward mean/std -0.050343/0.142196 | Value mean/std -0.051819/0.125872 | Adv std 7.591368e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.050343 | AvgAdv 0.000000 | AdvAfterStd 6.116792e-02
[Update 1710] Samples 2048 | Reward mean/std -0.064682/0.179261 | Value mean/std -0.058767/0.149342 | Adv std 9.567068e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.064682 | AvgAdv -0.000000 | AdvAfterStd 8.073895e-02
[Update 1711] Samples 2048 | Reward mean/std -0.054704/0.128096 | Value mean/std -0.056075/0.078582 | Adv std 9.416345e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.054704 | AvgAdv 0.000000 | AdvAfterStd 7.075118e-02
[Update 1712] Samples 2048 | Reward mean/std -0.055495/0.180113 | Value mean/std -0.055204/0.159246 | Adv std 9.660027e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.055495 | AvgAdv 0.000000 | AdvAfterStd 7.046116e-02
[Update 1713] Samples 2048 | Reward mean/std -0.058746/0.173065 | Value mean/std -0.060511/0.162858 | Adv std 9.648253e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.058746 | AvgAdv 0.000000 | AdvAfterStd 6.986685e-02
[Update 1714] Samples 2048 | Reward mean/std -0.057709/0.175411 | Value mean/std -0.058092/0.123884 | Adv std 9.670323e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.057709 | AvgAdv -0.000000 | AdvAfterStd 8.148472e-02
[Update 1715] Samples 2048 | Reward mean/std -0.074288/0.254836 | Value mean/std -0.063111/0.194099 | Adv std 1.359590e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.074288 | AvgAdv 0.000000 | AdvAfterStd 8.953983e-02
[Update 1716] Samples 2048 | Reward mean/std -0.057354/0.165702 | Value mean/std -0.062478/0.176504 | Adv std 1.324545e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.057354 | AvgAdv -0.000000 | AdvAfterStd 6.539905e-02
[Update 1717] Samples 2048 | Reward mean/std -0.057756/0.208636 | Value mean/std -0.058993/0.167700 | Adv std 9.190021e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.057756 | AvgAdv -0.000000 | AdvAfterStd 6.985326e-02
[Update 1718] Samples 2048 | Reward mean/std -0.062207/0.171677 | Value mean/std -0.056521/0.154852 | Adv std 1.124486e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.062207 | AvgAdv -0.000000 | AdvAfterStd 7.956433e-02
[Update 1719] Samples 2048 | Reward mean/std -0.056200/0.141602 | Value mean/std -0.059860/0.134130 | Adv std 1.062908e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.056200 | AvgAdv 0.000000 | AdvAfterStd 7.639544e-02
[Update 1720] Samples 2048 | Reward mean/std -0.058537/0.169923 | Value mean/std -0.063783/0.156383 | Adv std 9.028681e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.058537 | AvgAdv 0.000000 | AdvAfterStd 6.607850e-02
[Update 1721] Samples 2048 | Reward mean/std -0.062308/0.231232 | Value mean/std -0.057050/0.176076 | Adv std 1.235392e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.062308 | AvgAdv 0.000000 | AdvAfterStd 8.429183e-02
[Update 1722] Samples 2048 | Reward mean/std -0.061797/0.253994 | Value mean/std -0.069868/0.206474 | Adv std 1.069224e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.061797 | AvgAdv -0.000000 | AdvAfterStd 6.154791e-02
[Update 1723] Samples 2048 | Reward mean/std -0.063838/0.210096 | Value mean/std -0.058657/0.154588 | Adv std 1.388426e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.063838 | AvgAdv -0.000000 | AdvAfterStd 6.508201e-02
[Update 1724] Samples 2048 | Reward mean/std -0.068567/0.256102 | Value mean/std -0.062014/0.191806 | Adv std 1.412292e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.068567 | AvgAdv 0.000000 | AdvAfterStd 7.399754e-02
[Update 1725] Samples 2048 | Reward mean/std -0.062596/0.215865 | Value mean/std -0.060931/0.200446 | Adv std 1.043869e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.062596 | AvgAdv 0.000000 | AdvAfterStd 8.170230e-02
[Update 1726] Samples 2048 | Reward mean/std -0.057251/0.145289 | Value mean/std -0.052873/0.097040 | Adv std 1.009684e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.057251 | AvgAdv 0.000000 | AdvAfterStd 7.857408e-02
[Update 1727] Samples 2048 | Reward mean/std -0.064165/0.287898 | Value mean/std -0.064284/0.186233 | Adv std 1.477770e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.064165 | AvgAdv -0.000000 | AdvAfterStd 7.731743e-02
[Update 1728] Samples 2048 | Reward mean/std -0.059528/0.172418 | Value mean/std -0.053387/0.133050 | Adv std 1.102295e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.059528 | AvgAdv -0.000000 | AdvAfterStd 6.978656e-02
[Update 1729] Samples 2048 | Reward mean/std -0.049743/0.126903 | Value mean/std -0.054423/0.111472 | Adv std 8.458268e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.049743 | AvgAdv -0.000000 | AdvAfterStd 6.656552e-02
[Update 1730] Samples 2048 | Reward mean/std -0.058904/0.170933 | Value mean/std -0.055410/0.173804 | Adv std 1.680410e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.058904 | AvgAdv -0.000000 | AdvAfterStd 8.408704e-02
[Update 1731] Samples 2048 | Reward mean/std -0.053778/0.158180 | Value mean/std -0.059507/0.122276 | Adv std 1.024657e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.053778 | AvgAdv -0.000000 | AdvAfterStd 8.029267e-02
[Update 1732] Samples 2048 | Reward mean/std -0.054745/0.136266 | Value mean/std -0.057524/0.131375 | Adv std 8.841700e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.054745 | AvgAdv 0.000000 | AdvAfterStd 6.806883e-02
[Update 1733] Samples 2048 | Reward mean/std -0.064535/0.227654 | Value mean/std -0.058912/0.206148 | Adv std 9.991170e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.064535 | AvgAdv 0.000000 | AdvAfterStd 8.756647e-02
[Update 1734] Samples 2048 | Reward mean/std -0.062397/0.198439 | Value mean/std -0.059282/0.161569 | Adv std 1.120090e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.062397 | AvgAdv -0.000000 | AdvAfterStd 8.244135e-02
[Update 1735] Samples 2048 | Reward mean/std -0.066023/0.206809 | Value mean/std -0.065329/0.226586 | Adv std 1.471124e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.066023 | AvgAdv -0.000000 | AdvAfterStd 7.559433e-02
[Update 1736] Samples 2048 | Reward mean/std -0.056826/0.168061 | Value mean/std -0.058920/0.121962 | Adv std 1.262366e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.056826 | AvgAdv -0.000000 | AdvAfterStd 8.798410e-02
[Update 1737] Samples 2048 | Reward mean/std -0.059062/0.179331 | Value mean/std -0.058801/0.176072 | Adv std 1.084532e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.059062 | AvgAdv -0.000000 | AdvAfterStd 8.367525e-02
[Update 1738] Samples 2048 | Reward mean/std -0.061040/0.172125 | Value mean/std -0.057623/0.117504 | Adv std 1.200426e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.061040 | AvgAdv 0.000000 | AdvAfterStd 8.800444e-02
[Update 1739] Samples 2048 | Reward mean/std -0.058018/0.180755 | Value mean/std -0.057381/0.149028 | Adv std 1.222412e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.058018 | AvgAdv 0.000000 | AdvAfterStd 7.390482e-02
[Update 1740] Samples 2048 | Reward mean/std -0.064501/0.215773 | Value mean/std -0.058095/0.170617 | Adv std 1.049093e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.064501 | AvgAdv -0.000000 | AdvAfterStd 8.207524e-02
[Update 1741] Samples 2048 | Reward mean/std -0.064966/0.200795 | Value mean/std -0.065147/0.196905 | Adv std 1.270792e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.064966 | AvgAdv -0.000000 | AdvAfterStd 8.454105e-02
[Update 1742] Samples 2048 | Reward mean/std -0.054119/0.146650 | Value mean/std -0.054920/0.109733 | Adv std 1.301093e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.054119 | AvgAdv 0.000000 | AdvAfterStd 9.036458e-02
[Update 1743] Samples 2048 | Reward mean/std -0.066904/0.262817 | Value mean/std -0.057551/0.177515 | Adv std 1.457593e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.066904 | AvgAdv 0.000000 | AdvAfterStd 7.761418e-02
[Update 1744] Samples 2048 | Reward mean/std -0.058107/0.170095 | Value mean/std -0.057212/0.131200 | Adv std 1.013775e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.058107 | AvgAdv 0.000000 | AdvAfterStd 7.064544e-02
[Update 1745] Samples 2048 | Reward mean/std -0.059514/0.181037 | Value mean/std -0.060018/0.153296 | Adv std 9.946538e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.059514 | AvgAdv 0.000000 | AdvAfterStd 8.098793e-02
[Update 1746] Samples 2048 | Reward mean/std -0.057540/0.178977 | Value mean/std -0.060808/0.162743 | Adv std 1.034551e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.057540 | AvgAdv -0.000000 | AdvAfterStd 7.210992e-02
[Update 1747] Samples 2048 | Reward mean/std -0.060823/0.191893 | Value mean/std -0.062251/0.208231 | Adv std 1.278553e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.060823 | AvgAdv -0.000000 | AdvAfterStd 7.279968e-02
[Update 1748] Samples 2048 | Reward mean/std -0.059241/0.193190 | Value mean/std -0.053548/0.132881 | Adv std 1.375684e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.059241 | AvgAdv -0.000000 | AdvAfterStd 7.688769e-02
[Update 1749] Samples 2048 | Reward mean/std -0.071348/0.242806 | Value mean/std -0.066163/0.220847 | Adv std 1.145071e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.071348 | AvgAdv 0.000000 | AdvAfterStd 8.513530e-02
[Update 1750] Samples 2048 | Reward mean/std -0.057240/0.161700 | Value mean/std -0.063869/0.165361 | Adv std 1.103560e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.057240 | AvgAdv 0.000000 | AdvAfterStd 7.539950e-02
Saved checkpoint at update 1750
[Update 1751] Samples 2048 | Reward mean/std -0.057874/0.182117 | Value mean/std -0.054348/0.183957 | Adv std 1.055644e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.057874 | AvgAdv 0.000000 | AdvAfterStd 7.139189e-02
[Update 1752] Samples 2048 | Reward mean/std -0.054080/0.144643 | Value mean/std -0.053245/0.098404 | Adv std 9.757666e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.054080 | AvgAdv 0.000000 | AdvAfterStd 7.303683e-02
[Update 1753] Samples 2048 | Reward mean/std -0.060204/0.175611 | Value mean/std -0.065365/0.146776 | Adv std 1.019465e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.060204 | AvgAdv 0.000000 | AdvAfterStd 7.694493e-02
[Update 1754] Samples 2048 | Reward mean/std -0.052167/0.132716 | Value mean/std -0.055656/0.158122 | Adv std 1.010705e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.052167 | AvgAdv -0.000000 | AdvAfterStd 6.033685e-02
[Update 1755] Samples 2048 | Reward mean/std -0.059045/0.179379 | Value mean/std -0.054876/0.122200 | Adv std 1.144245e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.059045 | AvgAdv 0.000000 | AdvAfterStd 7.771220e-02
[Update 1756] Samples 2048 | Reward mean/std -0.060485/0.174624 | Value mean/std -0.058803/0.150750 | Adv std 1.000028e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.060485 | AvgAdv 0.000000 | AdvAfterStd 7.592054e-02
[Update 1757] Samples 2048 | Reward mean/std -0.061627/0.194569 | Value mean/std -0.063021/0.156523 | Adv std 1.039629e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.061627 | AvgAdv 0.000000 | AdvAfterStd 8.828665e-02
[Update 1758] Samples 2048 | Reward mean/std -0.056869/0.151896 | Value mean/std -0.052790/0.125193 | Adv std 9.046032e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.056869 | AvgAdv -0.000000 | AdvAfterStd 7.562038e-02
[Update 1759] Samples 2048 | Reward mean/std -0.061659/0.211578 | Value mean/std -0.062379/0.156345 | Adv std 1.027825e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.061659 | AvgAdv 0.000000 | AdvAfterStd 6.425616e-02
[Update 1760] Samples 2048 | Reward mean/std -0.058016/0.153109 | Value mean/std -0.059524/0.140143 | Adv std 1.008499e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.058016 | AvgAdv -0.000000 | AdvAfterStd 7.011066e-02
[Update 1761] Samples 2048 | Reward mean/std -0.066175/0.275533 | Value mean/std -0.064890/0.229569 | Adv std 1.010674e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.066175 | AvgAdv 0.000000 | AdvAfterStd 7.864990e-02
[Update 1762] Samples 2048 | Reward mean/std -0.057986/0.150264 | Value mean/std -0.056709/0.101290 | Adv std 1.117285e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.057986 | AvgAdv 0.000000 | AdvAfterStd 7.890951e-02
[Update 1763] Samples 2048 | Reward mean/std -0.062749/0.173204 | Value mean/std -0.061716/0.153775 | Adv std 1.080353e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.062749 | AvgAdv 0.000000 | AdvAfterStd 8.162551e-02
[Update 1764] Samples 2048 | Reward mean/std -0.061136/0.176096 | Value mean/std -0.061085/0.137560 | Adv std 1.084193e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.061136 | AvgAdv 0.000000 | AdvAfterStd 8.912215e-02
[Update 1765] Samples 2048 | Reward mean/std -0.064124/0.213918 | Value mean/std -0.071759/0.185434 | Adv std 9.850527e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.064124 | AvgAdv 0.000000 | AdvAfterStd 7.366632e-02
[Update 1766] Samples 2048 | Reward mean/std -0.060541/0.193628 | Value mean/std -0.059173/0.167331 | Adv std 8.992786e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.060541 | AvgAdv -0.000000 | AdvAfterStd 7.040634e-02
[Update 1767] Samples 2048 | Reward mean/std -0.054510/0.146340 | Value mean/std -0.056238/0.134107 | Adv std 7.621964e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.054510 | AvgAdv 0.000000 | AdvAfterStd 6.390471e-02
[Update 1768] Samples 2048 | Reward mean/std -0.055059/0.144260 | Value mean/std -0.050481/0.104402 | Adv std 9.116884e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.055059 | AvgAdv -0.000000 | AdvAfterStd 7.377253e-02
[Update 1769] Samples 2048 | Reward mean/std -0.061571/0.198282 | Value mean/std -0.065364/0.175301 | Adv std 1.189308e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.061571 | AvgAdv 0.000000 | AdvAfterStd 7.943363e-02
[Update 1770] Samples 2048 | Reward mean/std -0.057801/0.144944 | Value mean/std -0.055552/0.126906 | Adv std 1.001666e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.057801 | AvgAdv 0.000000 | AdvAfterStd 7.065997e-02
[Update 1771] Samples 2048 | Reward mean/std -0.059075/0.169352 | Value mean/std -0.059149/0.132430 | Adv std 1.016237e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.059075 | AvgAdv -0.000000 | AdvAfterStd 8.145615e-02
[Update 1772] Samples 2048 | Reward mean/std -0.064949/0.182891 | Value mean/std -0.063802/0.133162 | Adv std 1.330809e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.064949 | AvgAdv 0.000000 | AdvAfterStd 1.018771e-01
[Update 1773] Samples 2048 | Reward mean/std -0.057904/0.164025 | Value mean/std -0.056508/0.122985 | Adv std 1.087023e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.057904 | AvgAdv -0.000000 | AdvAfterStd 8.075294e-02
[Update 1774] Samples 2048 | Reward mean/std -0.052559/0.145020 | Value mean/std -0.048613/0.121556 | Adv std 9.455811e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.052559 | AvgAdv -0.000000 | AdvAfterStd 6.489456e-02
[Update 1775] Samples 2048 | Reward mean/std -0.054365/0.152601 | Value mean/std -0.056568/0.133734 | Adv std 8.721496e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.054365 | AvgAdv 0.000000 | AdvAfterStd 6.500813e-02
[Update 1776] Samples 2048 | Reward mean/std -0.053451/0.194488 | Value mean/std -0.053655/0.146253 | Adv std 9.397846e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.053451 | AvgAdv 0.000000 | AdvAfterStd 7.094902e-02
[Update 1777] Samples 2048 | Reward mean/std -0.059064/0.177039 | Value mean/std -0.054365/0.148053 | Adv std 1.007950e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.059064 | AvgAdv -0.000000 | AdvAfterStd 7.598001e-02
[Update 1778] Samples 2048 | Reward mean/std -0.058324/0.155865 | Value mean/std -0.057078/0.139532 | Adv std 9.693202e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.058324 | AvgAdv 0.000000 | AdvAfterStd 7.355376e-02
[Update 1779] Samples 2048 | Reward mean/std -0.055804/0.150574 | Value mean/std -0.057652/0.120438 | Adv std 9.479559e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.055804 | AvgAdv -0.000000 | AdvAfterStd 6.543884e-02
[Update 1780] Samples 2048 | Reward mean/std -0.060782/0.230587 | Value mean/std -0.059385/0.188362 | Adv std 1.221982e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.060782 | AvgAdv 0.000000 | AdvAfterStd 7.145790e-02
[Update 1781] Samples 2048 | Reward mean/std -0.054567/0.164265 | Value mean/std -0.055980/0.146895 | Adv std 1.045373e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.054567 | AvgAdv -0.000000 | AdvAfterStd 8.381638e-02
[Update 1782] Samples 2048 | Reward mean/std -0.052939/0.131348 | Value mean/std -0.050147/0.118243 | Adv std 1.025634e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.052939 | AvgAdv 0.000000 | AdvAfterStd 6.539924e-02
[Update 1783] Samples 2048 | Reward mean/std -0.059915/0.216428 | Value mean/std -0.060511/0.136307 | Adv std 1.403534e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.059915 | AvgAdv -0.000000 | AdvAfterStd 6.771135e-02
[Update 1784] Samples 2048 | Reward mean/std -0.056683/0.213057 | Value mean/std -0.058020/0.142631 | Adv std 1.424537e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.056683 | AvgAdv -0.000000 | AdvAfterStd 6.217391e-02
[Update 1785] Samples 2048 | Reward mean/std -0.058287/0.192898 | Value mean/std -0.053413/0.172686 | Adv std 1.082198e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.058287 | AvgAdv -0.000000 | AdvAfterStd 7.756472e-02
[Update 1786] Samples 2048 | Reward mean/std -0.052045/0.134838 | Value mean/std -0.056169/0.125331 | Adv std 9.736143e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.052045 | AvgAdv -0.000000 | AdvAfterStd 7.272810e-02
[Update 1787] Samples 2048 | Reward mean/std -0.070903/0.283101 | Value mean/std -0.067118/0.245893 | Adv std 1.148428e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.070903 | AvgAdv 0.000000 | AdvAfterStd 7.677381e-02
[Update 1788] Samples 2048 | Reward mean/std -0.056052/0.173194 | Value mean/std -0.063708/0.169742 | Adv std 9.010504e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.056052 | AvgAdv 0.000000 | AdvAfterStd 6.781498e-02
[Update 1789] Samples 2048 | Reward mean/std -0.058540/0.183759 | Value mean/std -0.052362/0.154679 | Adv std 8.939071e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.058540 | AvgAdv -0.000000 | AdvAfterStd 7.869858e-02
[Update 1790] Samples 2048 | Reward mean/std -0.060453/0.226291 | Value mean/std -0.058632/0.182109 | Adv std 1.233162e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.060453 | AvgAdv -0.000000 | AdvAfterStd 7.380430e-02
[Update 1791] Samples 2048 | Reward mean/std -0.061495/0.198496 | Value mean/std -0.056167/0.188817 | Adv std 1.418848e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.061495 | AvgAdv 0.000000 | AdvAfterStd 7.428150e-02
[Update 1792] Samples 2048 | Reward mean/std -0.062790/0.171241 | Value mean/std -0.063136/0.173709 | Adv std 1.100715e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.062790 | AvgAdv -0.000000 | AdvAfterStd 7.743132e-02
[Update 1793] Samples 2048 | Reward mean/std -0.057175/0.176534 | Value mean/std -0.062070/0.144390 | Adv std 1.117305e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.057175 | AvgAdv -0.000000 | AdvAfterStd 7.833849e-02
[Update 1794] Samples 2048 | Reward mean/std -0.060560/0.166001 | Value mean/std -0.056642/0.136196 | Adv std 1.070862e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.060560 | AvgAdv 0.000000 | AdvAfterStd 8.110574e-02
[Update 1795] Samples 2048 | Reward mean/std -0.058376/0.204808 | Value mean/std -0.063412/0.196866 | Adv std 9.335592e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.058376 | AvgAdv -0.000000 | AdvAfterStd 7.183344e-02
[Update 1796] Samples 2048 | Reward mean/std -0.049890/0.123965 | Value mean/std -0.049024/0.115509 | Adv std 9.177515e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.049890 | AvgAdv 0.000000 | AdvAfterStd 7.196808e-02
[Update 1797] Samples 2048 | Reward mean/std -0.058055/0.197952 | Value mean/std -0.053713/0.133651 | Adv std 1.095631e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.058055 | AvgAdv 0.000000 | AdvAfterStd 6.680898e-02
[Update 1798] Samples 2048 | Reward mean/std -0.066137/0.276422 | Value mean/std -0.059801/0.189676 | Adv std 1.549701e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.066137 | AvgAdv 0.000000 | AdvAfterStd 7.769103e-02
[Update 1799] Samples 2048 | Reward mean/std -0.061355/0.185881 | Value mean/std -0.061064/0.169178 | Adv std 1.314686e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.061355 | AvgAdv 0.000000 | AdvAfterStd 9.514917e-02
[Update 1800] Samples 2048 | Reward mean/std -0.059318/0.210500 | Value mean/std -0.061155/0.161354 | Adv std 1.123816e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.059318 | AvgAdv 0.000000 | AdvAfterStd 8.404853e-02
Saved checkpoint at update 1800
[Update 1801] Samples 2048 | Reward mean/std -0.054169/0.155759 | Value mean/std -0.056500/0.123033 | Adv std 1.056958e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.054169 | AvgAdv 0.000000 | AdvAfterStd 7.061589e-02
[Update 1802] Samples 2048 | Reward mean/std -0.065781/0.229045 | Value mean/std -0.054956/0.160039 | Adv std 1.152798e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.065781 | AvgAdv 0.000000 | AdvAfterStd 7.861470e-02
[Update 1803] Samples 2048 | Reward mean/std -0.057472/0.186901 | Value mean/std -0.060947/0.188018 | Adv std 1.187413e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.057472 | AvgAdv 0.000000 | AdvAfterStd 8.139360e-02
[Update 1804] Samples 2048 | Reward mean/std -0.057588/0.163476 | Value mean/std -0.057435/0.159713 | Adv std 1.087689e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.057588 | AvgAdv 0.000000 | AdvAfterStd 7.885207e-02
[Update 1805] Samples 2048 | Reward mean/std -0.062289/0.197096 | Value mean/std -0.061982/0.174391 | Adv std 1.017134e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.062289 | AvgAdv 0.000000 | AdvAfterStd 8.243496e-02
[Update 1806] Samples 2048 | Reward mean/std -0.059755/0.185990 | Value mean/std -0.062963/0.169349 | Adv std 1.190321e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.059755 | AvgAdv 0.000000 | AdvAfterStd 7.316258e-02
[Update 1807] Samples 2048 | Reward mean/std -0.056183/0.158780 | Value mean/std -0.057425/0.126167 | Adv std 1.077873e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.056183 | AvgAdv 0.000000 | AdvAfterStd 9.362852e-02
[Update 1808] Samples 2048 | Reward mean/std -0.062651/0.164079 | Value mean/std -0.066462/0.168539 | Adv std 1.008091e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.062651 | AvgAdv 0.000000 | AdvAfterStd 6.718759e-02
[Update 1809] Samples 2048 | Reward mean/std -0.056991/0.185206 | Value mean/std -0.053903/0.159078 | Adv std 9.582204e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.056991 | AvgAdv -0.000000 | AdvAfterStd 7.448947e-02
[Update 1810] Samples 2048 | Reward mean/std -0.058554/0.164628 | Value mean/std -0.055660/0.130540 | Adv std 1.116047e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.058554 | AvgAdv -0.000000 | AdvAfterStd 8.604342e-02
[Update 1811] Samples 2048 | Reward mean/std -0.058020/0.198182 | Value mean/std -0.064069/0.196263 | Adv std 8.773883e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.058020 | AvgAdv 0.000000 | AdvAfterStd 7.765497e-02
[Update 1812] Samples 2048 | Reward mean/std -0.057994/0.182453 | Value mean/std -0.054460/0.129662 | Adv std 1.046063e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.057994 | AvgAdv -0.000000 | AdvAfterStd 7.396781e-02
[Update 1813] Samples 2048 | Reward mean/std -0.059820/0.189454 | Value mean/std -0.058422/0.147732 | Adv std 9.761069e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.059820 | AvgAdv 0.000000 | AdvAfterStd 8.157556e-02
[Update 1814] Samples 2048 | Reward mean/std -0.060367/0.185012 | Value mean/std -0.057223/0.180428 | Adv std 1.124666e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.060367 | AvgAdv 0.000000 | AdvAfterStd 8.535363e-02
[Update 1815] Samples 2048 | Reward mean/std -0.055078/0.179956 | Value mean/std -0.065173/0.153683 | Adv std 1.025124e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.055078 | AvgAdv -0.000000 | AdvAfterStd 8.030879e-02
[Update 1816] Samples 2048 | Reward mean/std -0.062426/0.192399 | Value mean/std -0.062470/0.182448 | Adv std 9.179088e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.062426 | AvgAdv 0.000000 | AdvAfterStd 7.626940e-02
[Update 1817] Samples 2048 | Reward mean/std -0.063055/0.191606 | Value mean/std -0.063743/0.196926 | Adv std 1.037999e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.063055 | AvgAdv 0.000000 | AdvAfterStd 7.657106e-02
[Update 1818] Samples 2048 | Reward mean/std -0.056291/0.157146 | Value mean/std -0.058463/0.133467 | Adv std 8.982985e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.056291 | AvgAdv 0.000000 | AdvAfterStd 6.216301e-02
[Update 1819] Samples 2048 | Reward mean/std -0.060749/0.162267 | Value mean/std -0.059885/0.146269 | Adv std 9.417890e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.060749 | AvgAdv 0.000000 | AdvAfterStd 7.252719e-02
[Update 1820] Samples 2048 | Reward mean/std -0.064131/0.186583 | Value mean/std -0.062757/0.170720 | Adv std 1.140260e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.064131 | AvgAdv -0.000000 | AdvAfterStd 8.464714e-02
[Update 1821] Samples 2048 | Reward mean/std -0.061730/0.203532 | Value mean/std -0.057531/0.143869 | Adv std 1.070616e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.061730 | AvgAdv 0.000000 | AdvAfterStd 6.963754e-02
[Update 1822] Samples 2048 | Reward mean/std -0.059245/0.176515 | Value mean/std -0.062443/0.131875 | Adv std 9.523982e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.059245 | AvgAdv 0.000000 | AdvAfterStd 8.148643e-02
[Update 1823] Samples 2048 | Reward mean/std -0.061400/0.163478 | Value mean/std -0.059054/0.140103 | Adv std 1.055779e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.061400 | AvgAdv 0.000000 | AdvAfterStd 7.162070e-02
[Update 1824] Samples 2048 | Reward mean/std -0.056079/0.157728 | Value mean/std -0.058869/0.122629 | Adv std 1.207414e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.056079 | AvgAdv 0.000000 | AdvAfterStd 6.966458e-02
[Update 1825] Samples 2048 | Reward mean/std -0.060744/0.172203 | Value mean/std -0.061201/0.130952 | Adv std 1.172674e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.060744 | AvgAdv -0.000000 | AdvAfterStd 8.938859e-02
[Update 1826] Samples 2048 | Reward mean/std -0.059818/0.195560 | Value mean/std -0.062246/0.163802 | Adv std 1.065754e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.059818 | AvgAdv -0.000000 | AdvAfterStd 7.824282e-02
[Update 1827] Samples 2048 | Reward mean/std -0.058136/0.158985 | Value mean/std -0.061385/0.131001 | Adv std 9.367812e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.058136 | AvgAdv 0.000000 | AdvAfterStd 8.660228e-02
[Update 1828] Samples 2048 | Reward mean/std -0.065363/0.252842 | Value mean/std -0.065992/0.192528 | Adv std 1.348624e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.065363 | AvgAdv -0.000000 | AdvAfterStd 9.697277e-02
[Update 1829] Samples 2048 | Reward mean/std -0.063842/0.243441 | Value mean/std -0.060220/0.224961 | Adv std 1.063092e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.063842 | AvgAdv -0.000000 | AdvAfterStd 7.706795e-02
[Update 1830] Samples 2048 | Reward mean/std -0.058301/0.168447 | Value mean/std -0.054815/0.121450 | Adv std 9.983075e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.058301 | AvgAdv -0.000000 | AdvAfterStd 7.397394e-02
[Update 1831] Samples 2048 | Reward mean/std -0.071450/0.229103 | Value mean/std -0.066074/0.167544 | Adv std 1.230147e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.071450 | AvgAdv 0.000000 | AdvAfterStd 8.137728e-02
[Update 1832] Samples 2048 | Reward mean/std -0.059027/0.212917 | Value mean/std -0.067343/0.194335 | Adv std 9.311993e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.059027 | AvgAdv -0.000000 | AdvAfterStd 7.392108e-02
[Update 1833] Samples 2048 | Reward mean/std -0.060303/0.184627 | Value mean/std -0.056949/0.143670 | Adv std 1.201268e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.060303 | AvgAdv 0.000000 | AdvAfterStd 7.742468e-02
[Update 1834] Samples 2048 | Reward mean/std -0.051076/0.118444 | Value mean/std -0.048875/0.099832 | Adv std 8.942135e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.051076 | AvgAdv -0.000000 | AdvAfterStd 6.479543e-02
[Update 1835] Samples 2048 | Reward mean/std -0.061559/0.190934 | Value mean/std -0.058733/0.147797 | Adv std 1.017913e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.061559 | AvgAdv 0.000000 | AdvAfterStd 8.610835e-02
[Update 1836] Samples 2048 | Reward mean/std -0.063425/0.178706 | Value mean/std -0.062867/0.171397 | Adv std 9.802673e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.063425 | AvgAdv -0.000000 | AdvAfterStd 7.776527e-02
[Update 1837] Samples 2048 | Reward mean/std -0.056884/0.146927 | Value mean/std -0.055329/0.116204 | Adv std 8.729497e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.056884 | AvgAdv 0.000000 | AdvAfterStd 7.432564e-02
[Update 1838] Samples 2048 | Reward mean/std -0.056724/0.145432 | Value mean/std -0.059430/0.102628 | Adv std 1.033464e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.056724 | AvgAdv -0.000000 | AdvAfterStd 7.837631e-02
[Update 1839] Samples 2048 | Reward mean/std -0.066026/0.253493 | Value mean/std -0.055794/0.134318 | Adv std 1.575970e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.066026 | AvgAdv 0.000000 | AdvAfterStd 8.581907e-02
[Update 1840] Samples 2048 | Reward mean/std -0.059986/0.192837 | Value mean/std -0.064731/0.165821 | Adv std 1.094259e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.059986 | AvgAdv -0.000000 | AdvAfterStd 7.392614e-02
[Update 1841] Samples 2048 | Reward mean/std -0.067106/0.215594 | Value mean/std -0.070686/0.189727 | Adv std 1.221101e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.067106 | AvgAdv -0.000000 | AdvAfterStd 7.697131e-02
[Update 1842] Samples 2048 | Reward mean/std -0.063859/0.236602 | Value mean/std -0.067882/0.249088 | Adv std 9.949290e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.063859 | AvgAdv -0.000000 | AdvAfterStd 8.168911e-02
[Update 1843] Samples 2048 | Reward mean/std -0.054471/0.163238 | Value mean/std -0.060088/0.152160 | Adv std 9.388107e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.054471 | AvgAdv -0.000000 | AdvAfterStd 6.434963e-02
[Update 1844] Samples 2048 | Reward mean/std -0.051182/0.132177 | Value mean/std -0.051266/0.097276 | Adv std 7.847270e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.051182 | AvgAdv 0.000000 | AdvAfterStd 6.434914e-02
[Update 1845] Samples 2048 | Reward mean/std -0.059467/0.186579 | Value mean/std -0.055944/0.153015 | Adv std 1.011821e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.059467 | AvgAdv 0.000000 | AdvAfterStd 7.846102e-02
[Update 1846] Samples 2048 | Reward mean/std -0.066438/0.240693 | Value mean/std -0.061567/0.217712 | Adv std 1.260675e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.066438 | AvgAdv 0.000000 | AdvAfterStd 9.825011e-02
[Update 1847] Samples 2048 | Reward mean/std -0.057048/0.168615 | Value mean/std -0.057233/0.134894 | Adv std 9.815658e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.057048 | AvgAdv 0.000000 | AdvAfterStd 6.743723e-02
[Update 1848] Samples 2048 | Reward mean/std -0.061771/0.183673 | Value mean/std -0.054906/0.158896 | Adv std 9.339793e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.061771 | AvgAdv -0.000000 | AdvAfterStd 7.481388e-02
[Update 1849] Samples 2048 | Reward mean/std -0.059196/0.175988 | Value mean/std -0.060236/0.145719 | Adv std 9.075566e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.059196 | AvgAdv -0.000000 | AdvAfterStd 7.628444e-02
[Update 1850] Samples 2048 | Reward mean/std -0.052100/0.157177 | Value mean/std -0.054452/0.171319 | Adv std 8.951622e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.052100 | AvgAdv 0.000000 | AdvAfterStd 6.523125e-02
Saved checkpoint at update 1850
[Update 1851] Samples 2048 | Reward mean/std -0.056116/0.170075 | Value mean/std -0.050765/0.133023 | Adv std 9.546399e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.056116 | AvgAdv 0.000000 | AdvAfterStd 7.076703e-02
[Update 1852] Samples 2048 | Reward mean/std -0.050938/0.143554 | Value mean/std -0.055211/0.109015 | Adv std 8.911052e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.050938 | AvgAdv 0.000000 | AdvAfterStd 5.963489e-02
[Update 1853] Samples 2048 | Reward mean/std -0.053954/0.143992 | Value mean/std -0.052494/0.123196 | Adv std 8.006910e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.053954 | AvgAdv 0.000000 | AdvAfterStd 6.673271e-02
[Update 1854] Samples 2048 | Reward mean/std -0.061085/0.205735 | Value mean/std -0.057619/0.149866 | Adv std 1.015574e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.061085 | AvgAdv 0.000000 | AdvAfterStd 7.459033e-02
[Update 1855] Samples 2048 | Reward mean/std -0.052160/0.150215 | Value mean/std -0.059437/0.141329 | Adv std 7.489498e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.052160 | AvgAdv 0.000000 | AdvAfterStd 6.436520e-02
[Update 1856] Samples 2048 | Reward mean/std -0.063641/0.237278 | Value mean/std -0.060256/0.169898 | Adv std 1.146752e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.063641 | AvgAdv -0.000000 | AdvAfterStd 7.614976e-02
[Update 1857] Samples 2048 | Reward mean/std -0.067631/0.247332 | Value mean/std -0.073772/0.227710 | Adv std 1.160641e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.067631 | AvgAdv 0.000000 | AdvAfterStd 7.955139e-02
[Update 1858] Samples 2048 | Reward mean/std -0.051817/0.142330 | Value mean/std -0.051703/0.133564 | Adv std 8.555960e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.051817 | AvgAdv -0.000000 | AdvAfterStd 6.649661e-02
[Update 1859] Samples 2048 | Reward mean/std -0.060685/0.216905 | Value mean/std -0.060860/0.199871 | Adv std 1.098108e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.060685 | AvgAdv -0.000000 | AdvAfterStd 6.915014e-02
[Update 1860] Samples 2048 | Reward mean/std -0.062856/0.182376 | Value mean/std -0.058959/0.139874 | Adv std 9.796701e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.062856 | AvgAdv -0.000000 | AdvAfterStd 7.668905e-02
[Update 1861] Samples 2048 | Reward mean/std -0.066355/0.227851 | Value mean/std -0.061790/0.193472 | Adv std 1.218224e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.066355 | AvgAdv -0.000000 | AdvAfterStd 6.745708e-02
[Update 1862] Samples 2048 | Reward mean/std -0.055969/0.149947 | Value mean/std -0.053674/0.112185 | Adv std 9.613907e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.055969 | AvgAdv -0.000000 | AdvAfterStd 8.323418e-02
[Update 1863] Samples 2048 | Reward mean/std -0.057400/0.182006 | Value mean/std -0.059290/0.153608 | Adv std 1.190365e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.057400 | AvgAdv 0.000000 | AdvAfterStd 7.147901e-02
[Update 1864] Samples 2048 | Reward mean/std -0.058214/0.154360 | Value mean/std -0.059221/0.127336 | Adv std 1.020747e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.058214 | AvgAdv -0.000000 | AdvAfterStd 7.474687e-02
[Update 1865] Samples 2048 | Reward mean/std -0.052469/0.151549 | Value mean/std -0.049087/0.102654 | Adv std 8.994045e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.052469 | AvgAdv 0.000000 | AdvAfterStd 6.645792e-02
[Update 1866] Samples 2048 | Reward mean/std -0.050768/0.150344 | Value mean/std -0.051899/0.119900 | Adv std 7.704328e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.050768 | AvgAdv 0.000000 | AdvAfterStd 5.435641e-02
[Update 1867] Samples 2048 | Reward mean/std -0.055099/0.163563 | Value mean/std -0.052340/0.144542 | Adv std 8.712734e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.055099 | AvgAdv 0.000000 | AdvAfterStd 6.806940e-02
[Update 1868] Samples 2048 | Reward mean/std -0.058020/0.164180 | Value mean/std -0.059048/0.142202 | Adv std 1.050424e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.058020 | AvgAdv 0.000000 | AdvAfterStd 7.474209e-02
[Update 1869] Samples 2048 | Reward mean/std -0.060177/0.179603 | Value mean/std -0.053917/0.137028 | Adv std 1.281696e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.060177 | AvgAdv 0.000000 | AdvAfterStd 7.872728e-02
[Update 1870] Samples 2048 | Reward mean/std -0.063108/0.162979 | Value mean/std -0.057409/0.099348 | Adv std 1.157013e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.063108 | AvgAdv 0.000000 | AdvAfterStd 8.154155e-02
[Update 1871] Samples 2048 | Reward mean/std -0.059666/0.178693 | Value mean/std -0.054150/0.125885 | Adv std 1.134107e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.059666 | AvgAdv 0.000000 | AdvAfterStd 7.296327e-02
[Update 1872] Samples 2048 | Reward mean/std -0.062792/0.183749 | Value mean/std -0.059310/0.146334 | Adv std 1.316747e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.062792 | AvgAdv -0.000000 | AdvAfterStd 8.608777e-02
[Update 1873] Samples 2048 | Reward mean/std -0.049978/0.138397 | Value mean/std -0.057781/0.108930 | Adv std 8.952075e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.049978 | AvgAdv 0.000000 | AdvAfterStd 6.287299e-02
[Update 1874] Samples 2048 | Reward mean/std -0.059153/0.176837 | Value mean/std -0.052503/0.124343 | Adv std 1.049471e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.059153 | AvgAdv -0.000000 | AdvAfterStd 8.241688e-02
[Update 1875] Samples 2048 | Reward mean/std -0.058898/0.174057 | Value mean/std -0.059468/0.110894 | Adv std 1.191139e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.058898 | AvgAdv 0.000000 | AdvAfterStd 7.550722e-02
[Update 1876] Samples 2048 | Reward mean/std -0.063264/0.240752 | Value mean/std -0.068836/0.233871 | Adv std 1.274347e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.063264 | AvgAdv -0.000000 | AdvAfterStd 7.838073e-02
[Update 1877] Samples 2048 | Reward mean/std -0.053547/0.146266 | Value mean/std -0.052530/0.138768 | Adv std 9.703103e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.053547 | AvgAdv 0.000000 | AdvAfterStd 7.067703e-02
[Update 1878] Samples 2048 | Reward mean/std -0.064251/0.223618 | Value mean/std -0.054733/0.157639 | Adv std 1.091006e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.064251 | AvgAdv 0.000000 | AdvAfterStd 7.637988e-02
[Update 1879] Samples 2048 | Reward mean/std -0.062168/0.198702 | Value mean/std -0.067507/0.198830 | Adv std 8.647972e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.062168 | AvgAdv -0.000000 | AdvAfterStd 7.092889e-02
[Update 1880] Samples 2048 | Reward mean/std -0.061042/0.261504 | Value mean/std -0.060825/0.185529 | Adv std 1.520956e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.061042 | AvgAdv -0.000000 | AdvAfterStd 9.204344e-02
[Update 1881] Samples 2048 | Reward mean/std -0.064595/0.190512 | Value mean/std -0.059976/0.210771 | Adv std 1.175870e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.064595 | AvgAdv 0.000000 | AdvAfterStd 8.799474e-02
[Update 1882] Samples 2048 | Reward mean/std -0.059661/0.202534 | Value mean/std -0.059279/0.166985 | Adv std 9.217733e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.059661 | AvgAdv 0.000000 | AdvAfterStd 7.290521e-02
[Update 1883] Samples 2048 | Reward mean/std -0.053105/0.137980 | Value mean/std -0.057299/0.128128 | Adv std 8.230714e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.053105 | AvgAdv 0.000000 | AdvAfterStd 7.001269e-02
[Update 1884] Samples 2048 | Reward mean/std -0.053110/0.129524 | Value mean/std -0.047597/0.094503 | Adv std 8.524017e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.053110 | AvgAdv -0.000000 | AdvAfterStd 7.326998e-02
[Update 1885] Samples 2048 | Reward mean/std -0.062485/0.223851 | Value mean/std -0.062639/0.186681 | Adv std 8.629646e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.062485 | AvgAdv 0.000000 | AdvAfterStd 6.749173e-02
[Update 1886] Samples 2048 | Reward mean/std -0.051375/0.162063 | Value mean/std -0.057469/0.144365 | Adv std 9.433509e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.051375 | AvgAdv 0.000000 | AdvAfterStd 6.468938e-02
[Update 1887] Samples 2048 | Reward mean/std -0.062128/0.222071 | Value mean/std -0.053802/0.177360 | Adv std 9.825803e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.062128 | AvgAdv -0.000000 | AdvAfterStd 7.877167e-02
[Update 1888] Samples 2048 | Reward mean/std -0.059195/0.204878 | Value mean/std -0.061551/0.209128 | Adv std 9.064116e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.059195 | AvgAdv 0.000000 | AdvAfterStd 6.874786e-02
[Update 1889] Samples 2048 | Reward mean/std -0.068803/0.278625 | Value mean/std -0.064207/0.209594 | Adv std 1.840183e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.068803 | AvgAdv -0.000000 | AdvAfterStd 6.803200e-02
[Update 1890] Samples 2048 | Reward mean/std -0.056972/0.158151 | Value mean/std -0.064200/0.148759 | Adv std 1.098525e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.056972 | AvgAdv -0.000000 | AdvAfterStd 6.900972e-02
[Update 1891] Samples 2048 | Reward mean/std -0.061046/0.172101 | Value mean/std -0.052208/0.109660 | Adv std 1.103942e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.061046 | AvgAdv -0.000000 | AdvAfterStd 8.026844e-02
[Update 1892] Samples 2048 | Reward mean/std -0.062154/0.212079 | Value mean/std -0.059428/0.195802 | Adv std 8.242533e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.062154 | AvgAdv -0.000000 | AdvAfterStd 6.587840e-02
[Update 1893] Samples 2048 | Reward mean/std -0.059230/0.192190 | Value mean/std -0.063341/0.164294 | Adv std 9.993850e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.059230 | AvgAdv 0.000000 | AdvAfterStd 7.329165e-02
[Update 1894] Samples 2048 | Reward mean/std -0.057949/0.191942 | Value mean/std -0.061875/0.181897 | Adv std 9.617802e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.057949 | AvgAdv 0.000000 | AdvAfterStd 6.567262e-02
[Update 1895] Samples 2048 | Reward mean/std -0.058239/0.162866 | Value mean/std -0.054724/0.143424 | Adv std 9.306730e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.058239 | AvgAdv 0.000000 | AdvAfterStd 6.892649e-02
[Update 1896] Samples 2048 | Reward mean/std -0.059344/0.187793 | Value mean/std -0.062213/0.209146 | Adv std 1.305051e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.059344 | AvgAdv -0.000000 | AdvAfterStd 6.377717e-02
[Update 1897] Samples 2048 | Reward mean/std -0.061943/0.215998 | Value mean/std -0.066906/0.176915 | Adv std 1.190449e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.061943 | AvgAdv 0.000000 | AdvAfterStd 6.665825e-02
[Update 1898] Samples 2048 | Reward mean/std -0.056089/0.174441 | Value mean/std -0.058146/0.142893 | Adv std 9.241333e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.056089 | AvgAdv 0.000000 | AdvAfterStd 7.023938e-02
[Update 1899] Samples 2048 | Reward mean/std -0.055565/0.184234 | Value mean/std -0.054662/0.152078 | Adv std 8.666521e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.055565 | AvgAdv 0.000000 | AdvAfterStd 8.174454e-02
[Update 1900] Samples 2048 | Reward mean/std -0.057412/0.162069 | Value mean/std -0.061564/0.151561 | Adv std 1.038932e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.057412 | AvgAdv -0.000000 | AdvAfterStd 6.905961e-02
Saved checkpoint at update 1900
[Update 1901] Samples 2048 | Reward mean/std -0.055802/0.151878 | Value mean/std -0.057740/0.130033 | Adv std 8.759729e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.055802 | AvgAdv 0.000000 | AdvAfterStd 6.105240e-02
[Update 1902] Samples 2048 | Reward mean/std -0.060667/0.224307 | Value mean/std -0.051907/0.129919 | Adv std 1.290716e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.060667 | AvgAdv 0.000000 | AdvAfterStd 6.648018e-02
[Update 1903] Samples 2048 | Reward mean/std -0.055752/0.173399 | Value mean/std -0.054293/0.150130 | Adv std 8.341083e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.055752 | AvgAdv -0.000000 | AdvAfterStd 6.199297e-02
[Update 1904] Samples 2048 | Reward mean/std -0.054571/0.148617 | Value mean/std -0.058613/0.132696 | Adv std 8.123054e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.054571 | AvgAdv 0.000000 | AdvAfterStd 6.017982e-02
[Update 1905] Samples 2048 | Reward mean/std -0.058573/0.175728 | Value mean/std -0.055286/0.118395 | Adv std 1.215972e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.058573 | AvgAdv 0.000000 | AdvAfterStd 7.031708e-02
[Update 1906] Samples 2048 | Reward mean/std -0.049448/0.121789 | Value mean/std -0.050103/0.103334 | Adv std 8.482279e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.049448 | AvgAdv 0.000000 | AdvAfterStd 7.012281e-02
[Update 1907] Samples 2048 | Reward mean/std -0.058098/0.169659 | Value mean/std -0.055978/0.144307 | Adv std 8.656673e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.058098 | AvgAdv 0.000000 | AdvAfterStd 6.320810e-02
[Update 1908] Samples 2048 | Reward mean/std -0.049315/0.118432 | Value mean/std -0.046903/0.105236 | Adv std 8.444783e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.049315 | AvgAdv 0.000000 | AdvAfterStd 6.341074e-02
[Update 1909] Samples 2048 | Reward mean/std -0.065941/0.249959 | Value mean/std -0.059516/0.190989 | Adv std 1.271374e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.065941 | AvgAdv -0.000000 | AdvAfterStd 8.671854e-02
[Update 1910] Samples 2048 | Reward mean/std -0.059521/0.179852 | Value mean/std -0.059548/0.136567 | Adv std 9.361024e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.059521 | AvgAdv -0.000000 | AdvAfterStd 7.032154e-02
[Update 1911] Samples 2048 | Reward mean/std -0.057590/0.168140 | Value mean/std -0.059433/0.177643 | Adv std 8.891150e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.057590 | AvgAdv 0.000000 | AdvAfterStd 6.798451e-02
[Update 1912] Samples 2048 | Reward mean/std -0.050750/0.120698 | Value mean/std -0.047992/0.090227 | Adv std 8.251834e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.050750 | AvgAdv 0.000000 | AdvAfterStd 6.618113e-02
[Update 1913] Samples 2048 | Reward mean/std -0.059416/0.207646 | Value mean/std -0.056661/0.205995 | Adv std 1.164953e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.059416 | AvgAdv 0.000000 | AdvAfterStd 7.219734e-02
[Update 1914] Samples 2048 | Reward mean/std -0.062586/0.266535 | Value mean/std -0.053469/0.166651 | Adv std 1.552287e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.062586 | AvgAdv 0.000000 | AdvAfterStd 7.367923e-02
[Update 1915] Samples 2048 | Reward mean/std -0.060414/0.175069 | Value mean/std -0.049660/0.134543 | Adv std 1.035061e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.060414 | AvgAdv 0.000000 | AdvAfterStd 7.719159e-02
[Update 1916] Samples 2048 | Reward mean/std -0.063153/0.216219 | Value mean/std -0.060585/0.157687 | Adv std 1.116086e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.063153 | AvgAdv 0.000000 | AdvAfterStd 7.890818e-02
[Update 1917] Samples 2048 | Reward mean/std -0.051693/0.166111 | Value mean/std -0.061037/0.138275 | Adv std 1.056090e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.051693 | AvgAdv -0.000000 | AdvAfterStd 6.448408e-02
[Update 1918] Samples 2048 | Reward mean/std -0.058965/0.233255 | Value mean/std -0.056302/0.160113 | Adv std 1.276870e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.058965 | AvgAdv -0.000000 | AdvAfterStd 8.401562e-02
[Update 1919] Samples 2048 | Reward mean/std -0.059437/0.227358 | Value mean/std -0.055933/0.195829 | Adv std 9.599848e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.059437 | AvgAdv 0.000000 | AdvAfterStd 6.781568e-02
[Update 1920] Samples 2048 | Reward mean/std -0.057166/0.149093 | Value mean/std -0.055144/0.123444 | Adv std 1.104622e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.057166 | AvgAdv 0.000000 | AdvAfterStd 6.678513e-02
[Update 1921] Samples 2048 | Reward mean/std -0.061283/0.183428 | Value mean/std -0.063552/0.143115 | Adv std 1.010782e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.061283 | AvgAdv -0.000000 | AdvAfterStd 7.055132e-02
[Update 1922] Samples 2048 | Reward mean/std -0.055030/0.148419 | Value mean/std -0.056818/0.162584 | Adv std 1.072312e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.055030 | AvgAdv -0.000000 | AdvAfterStd 6.727477e-02
[Update 1923] Samples 2048 | Reward mean/std -0.063606/0.243756 | Value mean/std -0.059295/0.188228 | Adv std 9.881996e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.063606 | AvgAdv -0.000000 | AdvAfterStd 6.194457e-02
[Update 1924] Samples 2048 | Reward mean/std -0.057940/0.164095 | Value mean/std -0.057688/0.122935 | Adv std 1.136963e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.057940 | AvgAdv 0.000000 | AdvAfterStd 8.248813e-02
[Update 1925] Samples 2048 | Reward mean/std -0.058814/0.183341 | Value mean/std -0.062433/0.148947 | Adv std 1.117932e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.058814 | AvgAdv -0.000000 | AdvAfterStd 7.106490e-02
[Update 1926] Samples 2048 | Reward mean/std -0.060135/0.200828 | Value mean/std -0.066938/0.181864 | Adv std 8.380762e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.060135 | AvgAdv 0.000000 | AdvAfterStd 6.349778e-02
[Update 1927] Samples 2048 | Reward mean/std -0.061763/0.205479 | Value mean/std -0.060050/0.166232 | Adv std 9.896216e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.061763 | AvgAdv -0.000000 | AdvAfterStd 7.535625e-02
[Update 1928] Samples 2048 | Reward mean/std -0.055476/0.158667 | Value mean/std -0.065144/0.163513 | Adv std 9.290127e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.055476 | AvgAdv 0.000000 | AdvAfterStd 5.945564e-02
[Update 1929] Samples 2048 | Reward mean/std -0.055284/0.149029 | Value mean/std -0.047690/0.105239 | Adv std 8.814695e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.055284 | AvgAdv 0.000000 | AdvAfterStd 7.174636e-02
[Update 1930] Samples 2048 | Reward mean/std -0.056318/0.173282 | Value mean/std -0.047366/0.146175 | Adv std 8.737659e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.056318 | AvgAdv -0.000000 | AdvAfterStd 7.563223e-02
[Update 1931] Samples 2048 | Reward mean/std -0.054076/0.130067 | Value mean/std -0.053451/0.123054 | Adv std 9.286501e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.054076 | AvgAdv -0.000000 | AdvAfterStd 6.520520e-02
[Update 1932] Samples 2048 | Reward mean/std -0.052800/0.146066 | Value mean/std -0.052389/0.123945 | Adv std 8.666669e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.052800 | AvgAdv 0.000000 | AdvAfterStd 6.645512e-02
[Update 1933] Samples 2048 | Reward mean/std -0.053186/0.134522 | Value mean/std -0.054664/0.127587 | Adv std 8.484305e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.053186 | AvgAdv 0.000000 | AdvAfterStd 6.282677e-02
[Update 1934] Samples 2048 | Reward mean/std -0.060050/0.196653 | Value mean/std -0.056216/0.143003 | Adv std 1.144380e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.060050 | AvgAdv -0.000000 | AdvAfterStd 8.345030e-02
[Update 1935] Samples 2048 | Reward mean/std -0.056812/0.155994 | Value mean/std -0.054576/0.118588 | Adv std 9.159922e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.056812 | AvgAdv 0.000000 | AdvAfterStd 7.247892e-02
[Update 1936] Samples 2048 | Reward mean/std -0.054670/0.142547 | Value mean/std -0.056787/0.124148 | Adv std 8.891616e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.054670 | AvgAdv -0.000000 | AdvAfterStd 6.521631e-02
[Update 1937] Samples 2048 | Reward mean/std -0.059894/0.232846 | Value mean/std -0.051357/0.174895 | Adv std 1.011767e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.059894 | AvgAdv -0.000000 | AdvAfterStd 6.944092e-02
[Update 1938] Samples 2048 | Reward mean/std -0.065710/0.220670 | Value mean/std -0.066394/0.200199 | Adv std 9.808576e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.065710 | AvgAdv 0.000000 | AdvAfterStd 7.401915e-02
[Update 1939] Samples 2048 | Reward mean/std -0.061207/0.210040 | Value mean/std -0.062902/0.192525 | Adv std 9.175237e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.061207 | AvgAdv -0.000000 | AdvAfterStd 7.534388e-02
[Update 1940] Samples 2048 | Reward mean/std -0.053525/0.128444 | Value mean/std -0.051126/0.110703 | Adv std 1.055937e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.053525 | AvgAdv -0.000000 | AdvAfterStd 7.825191e-02
[Update 1941] Samples 2048 | Reward mean/std -0.062585/0.233068 | Value mean/std -0.058309/0.178104 | Adv std 9.779796e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.062585 | AvgAdv 0.000000 | AdvAfterStd 7.653265e-02
[Update 1942] Samples 2048 | Reward mean/std -0.056572/0.191597 | Value mean/std -0.055134/0.204745 | Adv std 1.456324e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.056572 | AvgAdv -0.000000 | AdvAfterStd 6.153311e-02
[Update 1943] Samples 2048 | Reward mean/std -0.055297/0.149558 | Value mean/std -0.057592/0.160967 | Adv std 9.303711e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.055297 | AvgAdv -0.000000 | AdvAfterStd 6.968687e-02
[Update 1944] Samples 2048 | Reward mean/std -0.062617/0.193315 | Value mean/std -0.060171/0.156982 | Adv std 8.873388e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.062617 | AvgAdv 0.000000 | AdvAfterStd 7.083064e-02
[Update 1945] Samples 2048 | Reward mean/std -0.052969/0.147447 | Value mean/std -0.057934/0.128867 | Adv std 8.580973e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.052969 | AvgAdv 0.000000 | AdvAfterStd 6.629699e-02
[Update 1946] Samples 2048 | Reward mean/std -0.055677/0.160223 | Value mean/std -0.058345/0.142740 | Adv std 8.576687e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.055677 | AvgAdv 0.000000 | AdvAfterStd 6.660971e-02
[Update 1947] Samples 2048 | Reward mean/std -0.051460/0.144316 | Value mean/std -0.053887/0.127034 | Adv std 1.212400e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.051460 | AvgAdv -0.000000 | AdvAfterStd 8.626367e-02
[Update 1948] Samples 2048 | Reward mean/std -0.051260/0.145494 | Value mean/std -0.052754/0.122981 | Adv std 7.791522e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.051260 | AvgAdv 0.000000 | AdvAfterStd 6.313528e-02
[Update 1949] Samples 2048 | Reward mean/std -0.058823/0.187757 | Value mean/std -0.052712/0.132972 | Adv std 1.100428e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.058823 | AvgAdv 0.000000 | AdvAfterStd 7.065037e-02
[Update 1950] Samples 2048 | Reward mean/std -0.058558/0.223283 | Value mean/std -0.056086/0.164055 | Adv std 9.667369e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.058558 | AvgAdv 0.000000 | AdvAfterStd 6.754619e-02
Saved checkpoint at update 1950
[Update 1951] Samples 2048 | Reward mean/std -0.065033/0.254915 | Value mean/std -0.062465/0.215805 | Adv std 9.679259e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.065033 | AvgAdv -0.000000 | AdvAfterStd 7.458013e-02
[Update 1952] Samples 2048 | Reward mean/std -0.059384/0.229406 | Value mean/std -0.055148/0.188804 | Adv std 9.519892e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.059384 | AvgAdv -0.000000 | AdvAfterStd 6.248791e-02
[Update 1953] Samples 2048 | Reward mean/std -0.057302/0.177970 | Value mean/std -0.055717/0.172395 | Adv std 1.072602e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.057302 | AvgAdv -0.000000 | AdvAfterStd 7.380134e-02
[Update 1954] Samples 2048 | Reward mean/std -0.058808/0.207907 | Value mean/std -0.058468/0.141000 | Adv std 1.338490e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.058808 | AvgAdv 0.000000 | AdvAfterStd 7.872311e-02
[Update 1955] Samples 2048 | Reward mean/std -0.054966/0.182745 | Value mean/std -0.055920/0.167718 | Adv std 8.892538e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.054966 | AvgAdv 0.000000 | AdvAfterStd 6.019260e-02
[Update 1956] Samples 2048 | Reward mean/std -0.061046/0.219859 | Value mean/std -0.058041/0.200793 | Adv std 9.794200e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.061046 | AvgAdv -0.000000 | AdvAfterStd 6.892413e-02
[Update 1957] Samples 2048 | Reward mean/std -0.053769/0.144710 | Value mean/std -0.053991/0.150431 | Adv std 1.060456e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.053769 | AvgAdv -0.000000 | AdvAfterStd 6.342255e-02
[Update 1958] Samples 2048 | Reward mean/std -0.064755/0.222203 | Value mean/std -0.065914/0.202214 | Adv std 9.347829e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.064755 | AvgAdv -0.000000 | AdvAfterStd 7.620550e-02
[Update 1959] Samples 2048 | Reward mean/std -0.057934/0.199472 | Value mean/std -0.055068/0.168524 | Adv std 9.551904e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.057934 | AvgAdv -0.000000 | AdvAfterStd 8.013545e-02
[Update 1960] Samples 2048 | Reward mean/std -0.055190/0.185058 | Value mean/std -0.056611/0.130474 | Adv std 1.118237e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.055190 | AvgAdv -0.000000 | AdvAfterStd 6.762531e-02
[Update 1961] Samples 2048 | Reward mean/std -0.060146/0.265349 | Value mean/std -0.054673/0.239145 | Adv std 1.386079e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.060146 | AvgAdv -0.000000 | AdvAfterStd 8.219856e-02
[Update 1962] Samples 2048 | Reward mean/std -0.067065/0.276028 | Value mean/std -0.070430/0.302476 | Adv std 1.269048e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.067065 | AvgAdv 0.000000 | AdvAfterStd 9.000931e-02
[Update 1963] Samples 2048 | Reward mean/std -0.058166/0.204435 | Value mean/std -0.052320/0.171952 | Adv std 9.872488e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.058166 | AvgAdv -0.000000 | AdvAfterStd 7.091552e-02
[Update 1964] Samples 2048 | Reward mean/std -0.064041/0.250869 | Value mean/std -0.057860/0.131633 | Adv std 1.837664e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.064041 | AvgAdv -0.000000 | AdvAfterStd 1.091001e-01
[Update 1965] Samples 2048 | Reward mean/std -0.063888/0.230275 | Value mean/std -0.065404/0.253180 | Adv std 1.175696e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.063888 | AvgAdv -0.000000 | AdvAfterStd 8.071829e-02
[Update 1966] Samples 2048 | Reward mean/std -0.070668/0.255890 | Value mean/std -0.070451/0.248843 | Adv std 1.322999e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.070668 | AvgAdv 0.000000 | AdvAfterStd 6.950712e-02
[Update 1967] Samples 2048 | Reward mean/std -0.066581/0.243198 | Value mean/std -0.072722/0.251153 | Adv std 1.131615e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.066581 | AvgAdv -0.000000 | AdvAfterStd 8.272932e-02
[Update 1968] Samples 2048 | Reward mean/std -0.061225/0.186084 | Value mean/std -0.060046/0.129647 | Adv std 1.114440e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.061225 | AvgAdv -0.000000 | AdvAfterStd 8.369140e-02
[Update 1969] Samples 2048 | Reward mean/std -0.065249/0.210599 | Value mean/std -0.061664/0.166816 | Adv std 1.238323e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.065249 | AvgAdv -0.000000 | AdvAfterStd 8.231062e-02
[Update 1970] Samples 2048 | Reward mean/std -0.057425/0.177198 | Value mean/std -0.062253/0.164007 | Adv std 1.058497e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.057425 | AvgAdv -0.000000 | AdvAfterStd 6.573468e-02
[Update 1971] Samples 2048 | Reward mean/std -0.056864/0.199609 | Value mean/std -0.060626/0.187286 | Adv std 8.346153e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.056864 | AvgAdv 0.000000 | AdvAfterStd 7.343078e-02
[Update 1972] Samples 2048 | Reward mean/std -0.059196/0.170931 | Value mean/std -0.051987/0.141597 | Adv std 9.388272e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.059196 | AvgAdv 0.000000 | AdvAfterStd 7.540100e-02
[Update 1973] Samples 2048 | Reward mean/std -0.051121/0.153985 | Value mean/std -0.050178/0.129301 | Adv std 8.289187e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.051121 | AvgAdv 0.000000 | AdvAfterStd 5.872832e-02
[Update 1974] Samples 2048 | Reward mean/std -0.065883/0.239268 | Value mean/std -0.059010/0.216259 | Adv std 9.062593e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.065883 | AvgAdv -0.000000 | AdvAfterStd 8.722740e-02
[Update 1975] Samples 2048 | Reward mean/std -0.062195/0.221771 | Value mean/std -0.062032/0.205502 | Adv std 1.061007e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.062195 | AvgAdv -0.000000 | AdvAfterStd 7.769506e-02
[Update 1976] Samples 2048 | Reward mean/std -0.062464/0.189616 | Value mean/std -0.056717/0.176823 | Adv std 9.758075e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.062464 | AvgAdv -0.000000 | AdvAfterStd 7.821637e-02
[Update 1977] Samples 2048 | Reward mean/std -0.064209/0.225631 | Value mean/std -0.057188/0.150875 | Adv std 1.372639e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.064209 | AvgAdv -0.000000 | AdvAfterStd 8.727326e-02
[Update 1978] Samples 2048 | Reward mean/std -0.059472/0.171750 | Value mean/std -0.062277/0.156005 | Adv std 9.218248e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.059472 | AvgAdv 0.000000 | AdvAfterStd 8.321630e-02
[Update 1979] Samples 2048 | Reward mean/std -0.057752/0.151903 | Value mean/std -0.053086/0.116792 | Adv std 1.025095e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.057752 | AvgAdv 0.000000 | AdvAfterStd 8.782570e-02
[Update 1980] Samples 2048 | Reward mean/std -0.058258/0.192017 | Value mean/std -0.059092/0.153744 | Adv std 1.026847e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.058258 | AvgAdv 0.000000 | AdvAfterStd 8.912037e-02
[Update 1981] Samples 2048 | Reward mean/std -0.058703/0.179045 | Value mean/std -0.063444/0.158749 | Adv std 9.162065e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.058703 | AvgAdv 0.000000 | AdvAfterStd 7.627011e-02
[Update 1982] Samples 2048 | Reward mean/std -0.056328/0.160979 | Value mean/std -0.057762/0.138487 | Adv std 9.910277e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.056328 | AvgAdv 0.000000 | AdvAfterStd 7.829221e-02
[Update 1983] Samples 2048 | Reward mean/std -0.057497/0.181587 | Value mean/std -0.055158/0.154051 | Adv std 1.030745e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.057497 | AvgAdv -0.000000 | AdvAfterStd 7.921590e-02
[Update 1984] Samples 2048 | Reward mean/std -0.059337/0.187425 | Value mean/std -0.062173/0.152028 | Adv std 1.112665e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.059337 | AvgAdv 0.000000 | AdvAfterStd 9.221037e-02
[Update 1985] Samples 2048 | Reward mean/std -0.055143/0.156595 | Value mean/std -0.055977/0.126919 | Adv std 9.654631e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.055143 | AvgAdv -0.000000 | AdvAfterStd 7.246662e-02
[Update 1986] Samples 2048 | Reward mean/std -0.059704/0.200405 | Value mean/std -0.056564/0.178664 | Adv std 1.116607e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.059704 | AvgAdv 0.000000 | AdvAfterStd 7.787042e-02
[Update 1987] Samples 2048 | Reward mean/std -0.053938/0.145450 | Value mean/std -0.053483/0.125771 | Adv std 9.102640e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.053938 | AvgAdv -0.000000 | AdvAfterStd 7.929183e-02
[Update 1988] Samples 2048 | Reward mean/std -0.058931/0.193252 | Value mean/std -0.057152/0.155709 | Adv std 9.048483e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.058931 | AvgAdv 0.000000 | AdvAfterStd 8.245467e-02
[Update 1989] Samples 2048 | Reward mean/std -0.059120/0.178436 | Value mean/std -0.059166/0.150607 | Adv std 1.127070e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.059120 | AvgAdv 0.000000 | AdvAfterStd 6.716949e-02
[Update 1990] Samples 2048 | Reward mean/std -0.055890/0.210647 | Value mean/std -0.061678/0.178685 | Adv std 1.071219e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.055890 | AvgAdv -0.000000 | AdvAfterStd 7.552338e-02
[Update 1991] Samples 2048 | Reward mean/std -0.058747/0.189496 | Value mean/std -0.054084/0.179598 | Adv std 9.026386e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.058747 | AvgAdv -0.000000 | AdvAfterStd 6.829119e-02
[Update 1992] Samples 2048 | Reward mean/std -0.054881/0.153859 | Value mean/std -0.044918/0.111314 | Adv std 9.482964e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.054881 | AvgAdv 0.000000 | AdvAfterStd 6.889254e-02
[Update 1993] Samples 2048 | Reward mean/std -0.056827/0.169854 | Value mean/std -0.052578/0.161599 | Adv std 1.047035e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.056827 | AvgAdv 0.000000 | AdvAfterStd 7.570375e-02
[Update 1994] Samples 2048 | Reward mean/std -0.061632/0.216241 | Value mean/std -0.055480/0.168645 | Adv std 1.067413e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.061632 | AvgAdv 0.000000 | AdvAfterStd 8.999862e-02
[Update 1995] Samples 2048 | Reward mean/std -0.057084/0.171520 | Value mean/std -0.054446/0.138312 | Adv std 8.759643e-02 | ZeroFrac 0.000
Post-update: AvgReward -0.057084 | AvgAdv -0.000000 | AdvAfterStd 6.756283e-02
[Update 1996] Samples 2048 | Reward mean/std -0.065723/0.249754 | Value mean/std -0.064351/0.216913 | Adv std 1.298037e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.065723 | AvgAdv 0.000000 | AdvAfterStd 6.571078e-02
[Update 1997] Samples 2048 | Reward mean/std -0.062823/0.237225 | Value mean/std -0.056274/0.162354 | Adv std 1.229412e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.062823 | AvgAdv -0.000000 | AdvAfterStd 8.530993e-02
[Update 1998] Samples 2048 | Reward mean/std -0.065677/0.249373 | Value mean/std -0.068463/0.201431 | Adv std 1.306227e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.065677 | AvgAdv 0.000000 | AdvAfterStd 9.908690e-02
[Update 1999] Samples 2048 | Reward mean/std -0.057017/0.227925 | Value mean/std -0.057296/0.167321 | Adv std 1.281859e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.057017 | AvgAdv 0.000000 | AdvAfterStd 7.198932e-02
[Update 2000] Samples 2048 | Reward mean/std -0.053796/0.162000 | Value mean/std -0.058920/0.133794 | Adv std 1.200119e-01 | ZeroFrac 0.000
Post-update: AvgReward -0.053796 | AvgAdv 0.000000 | AdvAfterStd 1.040519e-01
Saved checkpoint at update 2000"""

# --- Extract AvgReward using regex ---
avg_rewards = [float(m.group(1)) for m in re.finditer(r"AvgReward ([\-0-9.]+)", log_text)]
updates = list(range(1, len(avg_rewards)+1))

# --- Plot reward curve ---
plt.figure(figsize=(10,5))
plt.plot(updates, avg_rewards, label="AvgReward per update")
plt.xlabel("Update")
plt.ylabel("Average Reward")
plt.title("PPO Training Reward Curve")
plt.grid(True)
plt.legend()
plt.show()


✅ 1. AvgReward

In [ ]:
with torch.no_grad():
    mu, _ = actor(states)
    reward = -((mu - truths) ** 2).mean()
    
reward.item()


2. Per-Action RMSE

In [ ]:
rmse = torch.sqrt(((mu - truths) ** 2).mean(dim=0))
rmse


3. Explained Variance (Critic Quality)

In [ ]:
values = critic(states, backbone)
returns = rewards  # single-step
explained_var = 1 - torch.var(returns - values) / (torch.var(returns) + 1e-8)
